# Path/Folder

# Read Column

In [ ]:
from pathlib import Path
import pandas as pd

# =========================
# FILE PATHS
# =========================
degree_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

company_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"


# =========================
# DISPLAY SETTINGS
# =========================
pd.set_option("display.max_rows", None)        # show all rows
pd.set_option("display.max_columns", None)     # show all columns
pd.set_option("display.max_colwidth", None)    # show full column text
pd.set_option("display.width", 1000)


# =========================
# MEMORY OPTIMIZED COLUMN READER
# =========================
def show_all_columns(file_path, dataset_name):
    print("=" * 80)
    print(dataset_name)
    print("=" * 80)
    
    if not file_path.exists():
        print("FILE NOT FOUND:")
        print(file_path)
        return None
    
    # Memory optimization:
    # nrows=0 reads only the header / column names, not the full data
    header_only = pd.read_csv(file_path, nrows=0)
    
    column_table = pd.DataFrame({
        "column_number": range(1, len(header_only.columns) + 1),
        "column_name": header_only.columns
    })
    
    print("File found:")
    print(file_path)
    print()
    print("Total columns:", len(header_only.columns))
    
    display(column_table)
    
    return column_table


# =========================
# SHOW DEGREE COLUMNS
# =========================
degree_columns = show_all_columns(
    degree_file,
    "DEGREE DATA - IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
)


# =========================
# SHOW COMPANY COLUMNS
# =========================
company_columns = show_all_columns(
    company_file,
    "COMPANY DATA - business_startup_ALL_IN_ONE.csv"
)

# Read row

In [ ]:
degree_preview = pd.read_csv(degree_file, nrows=20)
company_preview = pd.read_csv(company_file, nrows=20)

print("DEGREE DATA - first 20 rows")
display(degree_preview)

print("COMPANY DATA - first 20 rows")
display(company_preview)

# Show 20 row
# DEGREE DATA - ALL COLUMNS
# COMPANY DATA - ALL COLUMNS

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# =========================
# FILE PATHS
# =========================
degree_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

company_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"


# =========================
# DISPLAY SETTINGS
# =========================
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)      # show ALL columns
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)


# =========================
# RANDOM 20 ROWS, ALL COLUMNS
# MEMORY OPTIMIZED
# =========================
def show_random_20_all_columns(file_path, dataset_name, chunk_size=100_000):
    print("=" * 100)
    print(dataset_name)
    print("=" * 100)

    if not file_path.exists():
        print("FILE NOT FOUND:")
        print(file_path)
        return None

    random_samples = []

    for chunk_number, chunk in enumerate(
        pd.read_csv(file_path, chunksize=chunk_size, low_memory=False),
        start=1
    ):
        # Take random rows from each chunk
        sample_size = min(20, len(chunk))
        sample = chunk.sample(n=sample_size, random_state=chunk_number)

        random_samples.append(sample)

    # Combine small samples only
    final_table = pd.concat(random_samples, ignore_index=True)

    # Final random 20 rows
    final_table = final_table.sample(
        n=min(20, len(final_table)),
        random_state=42
    ).reset_index(drop=True)

    print("Random 20 rows, all columns:")
    print("Rows shown:", len(final_table))
    print("Columns shown:", len(final_table.columns))

    display(final_table)

    return final_table


# =========================
# DEGREE DATA - ALL COLUMNS
# =========================
degree_random_all_columns = show_random_20_all_columns(
    degree_file,
    "DEGREE DATA"
)


# =========================
# COMPANY DATA - ALL COLUMNS
# =========================
company_random_all_columns = show_random_20_all_columns(
    company_file,
    "COMPANY DATA"
)

# Show 20 row
# DEGREE DATA - Filter COLUMNS
# COMPANY DATA - Filter COLUMNS

# Html Folder

In [ ]:
# ============================================================
# STACKED BAR DASHBOARD
# Save output directly in Downloads folder
# Memory-optimized version using chunks
# ============================================================

from pathlib import Path
import time
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except:
    display = print


# ------------------------------------------------------------
# 0. FILE PATH SETTINGS
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

DATA_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "FullTest"
    / "All_Data"
)

DEGREE_FILE = DATA_DIR / "IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = DATA_DIR / "business_startup_ALL_IN_ONE.csv"

# Save new output folder directly inside Downloads
OUTPUT_DIR = Path.home() / "Downloads" / "stacked_bar_dashboard_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 250_000

YEAR_MIN = 1988
YEAR_MAX = 2025

# Optional filters
DEGREE_GROUP_FILTER = None
AWARD_LEVEL_FILTER = None
FIELD_GROUP_FILTER = None
INDUSTRY_FILTER = None
METRIC_FILTER = None
STATE_FILTER = None

TOP_N_INDUSTRIES = 12

print("STARTING STACKED BAR DASHBOARD PROJECT")
print()
print("Current Jupyter folder:")
print(PROJECT_DIR)
print()
print("Degree file:")
print(DEGREE_FILE)
print()
print("Company file:")
print(COMPANY_FILE)
print()
print("Output folder saved in Downloads:")
print(OUTPUT_DIR)
print()


# ------------------------------------------------------------
# 1. HELPER FUNCTIONS
# ------------------------------------------------------------

def timer(start):
    return round(time.time() - start, 2)


def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path.name)


def clean_text_series(s):
    return (
        s.astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )


def read_header(path):
    return pd.read_csv(path, nrows=0).columns.tolist()


def require_columns(existing_cols, needed_cols, file_name):
    missing = [c for c in needed_cols if c not in existing_cols]
    if missing:
        raise ValueError(
            f"\nMissing columns in {file_name}:\n{missing}\n\n"
            f"Existing columns are:\n{existing_cols}"
        )


def optional_existing_columns(existing_cols, optional_cols):
    return [c for c in optional_cols if c in existing_cols]


def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def classify_metric(metric_name, metric_category=""):
    text = f"{metric_name} {metric_category}".lower()

    if "job gain" in text or "job gains" in text or "employment gain" in text:
        return "job_gains"

    if "job loss" in text or "job losses" in text or "employment loss" in text:
        return "job_losses"

    if (
        "opening" in text
        or "openings" in text
        or "birth" in text
        or "births" in text
        or "startup" in text
        or "startups" in text
    ):
        return "business_openings"

    if (
        "closing" in text
        or "closings" in text
        or "death" in text
        or "deaths" in text
        or "exit" in text
        or "exits" in text
        or "failure" in text
        or "failures" in text
    ):
        return "business_closings"

    return "other"


# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)
print()


# ------------------------------------------------------------
# 3. READ DEGREE DATA WITH MEMORY OPTIMIZATION
# ------------------------------------------------------------

degree_needed_cols = [
    "year",
    "degree_group",
    "award_level_name",
    "field_group",
    "field_subgroup",
    "major_name",
    "degree_count",
]

degree_existing_cols = read_header(DEGREE_FILE)
require_columns(degree_existing_cols, degree_needed_cols, DEGREE_FILE.name)

print("DEGREE DATA COLUMNS USED:")
for c in degree_needed_cols:
    print("-", c)
print()

start = time.time()
degree_parts = []

print("Reading degree data by chunks...")

for i, chunk in enumerate(
    pd.read_csv(
        DEGREE_FILE,
        usecols=degree_needed_cols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    start=1
):
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["degree_count"] = safe_numeric(chunk["degree_count"])

    chunk = chunk[
        (chunk["year"] >= YEAR_MIN) &
        (chunk["year"] <= YEAR_MAX)
    ].copy()

    if DEGREE_GROUP_FILTER is not None:
        chunk = chunk[
            clean_text_series(chunk["degree_group"]) ==
            str(DEGREE_GROUP_FILTER).strip().upper()
        ]

    if AWARD_LEVEL_FILTER is not None:
        chunk = chunk[
            clean_text_series(chunk["award_level_name"]) ==
            str(AWARD_LEVEL_FILTER).strip().upper()
        ]

    if FIELD_GROUP_FILTER is not None:
        chunk = chunk[
            clean_text_series(chunk["field_group"]) ==
            str(FIELD_GROUP_FILTER).strip().upper()
        ]

    if len(chunk) == 0:
        continue

    text_cols = [
        "degree_group",
        "award_level_name",
        "field_group",
        "field_subgroup",
        "major_name",
    ]

    for col in text_cols:
        chunk[col] = chunk[col].astype("string").fillna("Unknown")

    chunk_agg = (
        chunk
        .groupby(
            [
                "year",
                "degree_group",
                "award_level_name",
                "field_group",
                "field_subgroup",
                "major_name",
            ],
            dropna=False,
            as_index=False
        )["degree_count"]
        .sum()
    )

    degree_parts.append(chunk_agg)

    if i % 10 == 0:
        print(f"  Degree chunks processed: {i} | time: {timer(start)} sec")

if not degree_parts:
    raise ValueError("No degree rows found after filters.")

degree_detail = pd.concat(degree_parts, ignore_index=True)

degree_detail = (
    degree_detail
    .groupby(
        [
            "year",
            "degree_group",
            "award_level_name",
            "field_group",
            "field_subgroup",
            "major_name",
        ],
        dropna=False,
        as_index=False
    )["degree_count"]
    .sum()
)

degree_detail["year"] = degree_detail["year"].astype(int)

degree_detail_path = OUTPUT_DIR / "degree_detail_aggregated.csv"
degree_detail.to_csv(degree_detail_path, index=False)

print()
print("DONE degree data")
print("Rows after aggregation:", len(degree_detail))
print("Saved:", degree_detail_path)
print("Time:", timer(start), "sec")
print()


# ------------------------------------------------------------
# 4. CREATE FIELD-LEVEL DEGREE DATA
# ------------------------------------------------------------

degree_field = (
    degree_detail
    .groupby(
        [
            "year",
            "degree_group",
            "award_level_name",
            "field_group",
        ],
        dropna=False,
        as_index=False
    )["degree_count"]
    .sum()
)

degree_field["field_subgroup"] = "ALL"
degree_field["major_name"] = "ALL"

degree_field_path = OUTPUT_DIR / "degree_field_aggregated.csv"
degree_field.to_csv(degree_field_path, index=False)

print("DONE field-level degree data")
print("Rows:", len(degree_field))
print("Saved:", degree_field_path)
print()


# ------------------------------------------------------------
# 5. READ COMPANY DATA WITH MEMORY OPTIMIZATION
# ------------------------------------------------------------

company_needed_cols = [
    "year",
    "industry_name",
    "industry_code",
    "naics_code",
    "metric_name",
    "metric_category",
    "value",
    "unit",
]

company_optional_cols = [
    "quarter",
    "month",
    "date",
    "state_name",
    "geo_name",
    "firm_age",
    "establishment_age",
    "firm_size",
    "establishment_size",
]

company_existing_cols = read_header(COMPANY_FILE)
require_columns(company_existing_cols, company_needed_cols, COMPANY_FILE.name)

company_usecols = company_needed_cols + optional_existing_columns(
    company_existing_cols,
    company_optional_cols
)

print("COMPANY DATA COLUMNS USED:")
for c in company_usecols:
    print("-", c)
print()

start = time.time()
company_parts = []

print("Reading company data by chunks...")

for i, chunk in enumerate(
    pd.read_csv(
        COMPANY_FILE,
        usecols=company_usecols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    start=1
):
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = safe_numeric(chunk["value"])

    chunk = chunk[
        (chunk["year"] >= YEAR_MIN) &
        (chunk["year"] <= YEAR_MAX)
    ].copy()

    if "state_name" in chunk.columns and STATE_FILTER is not None:
        chunk = chunk[
            clean_text_series(chunk["state_name"]) ==
            str(STATE_FILTER).strip().upper()
        ]

    if INDUSTRY_FILTER is not None:
        chunk = chunk[
            clean_text_series(chunk["industry_name"]) ==
            str(INDUSTRY_FILTER).strip().upper()
        ]

    if METRIC_FILTER is not None:
        chunk = chunk[
            clean_text_series(chunk["metric_name"]) ==
            str(METRIC_FILTER).strip().upper()
        ]

    if len(chunk) == 0:
        continue

    for col in [
        "industry_name",
        "industry_code",
        "naics_code",
        "metric_name",
        "metric_category",
        "unit",
    ]:
        chunk[col] = chunk[col].astype("string").fillna("Unknown")

    chunk_agg = (
        chunk
        .groupby(
            [
                "year",
                "industry_name",
                "industry_code",
                "naics_code",
                "metric_name",
                "metric_category",
                "unit",
            ],
            dropna=False,
            as_index=False
        )["value"]
        .sum()
    )

    company_parts.append(chunk_agg)

    if i % 10 == 0:
        print(f"  Company chunks processed: {i} | time: {timer(start)} sec")

if not company_parts:
    raise ValueError("No company rows found after filters.")

company = pd.concat(company_parts, ignore_index=True)

company = (
    company
    .groupby(
        [
            "year",
            "industry_name",
            "industry_code",
            "naics_code",
            "metric_name",
            "metric_category",
            "unit",
        ],
        dropna=False,
        as_index=False
    )["value"]
    .sum()
)

company["year"] = company["year"].astype(int)

company_path = OUTPUT_DIR / "company_aggregated.csv"
company.to_csv(company_path, index=False)

print()
print("DONE company data")
print("Rows after aggregation:", len(company))
print("Saved:", company_path)
print("Time:", timer(start), "sec")
print()


# ------------------------------------------------------------
# 6. CREATE OR READ MAPPING TABLE
# ------------------------------------------------------------

mapping_path = OUTPUT_DIR / "field_group_to_industry_mapping.csv"

default_mapping = [
    {
        "field_group": "COMPUTER AND INFORMATION SCIENCES",
        "industry_name": "Information",
    },
    {
        "field_group": "COMPUTER AND INFORMATION SCIENCES",
        "industry_name": "Professional, Scientific, and Technical Services",
    },
    {
        "field_group": "COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES",
        "industry_name": "Information",
    },
    {
        "field_group": "COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES",
        "industry_name": "Professional, Scientific, and Technical Services",
    },
    {
        "field_group": "ENGINEERING",
        "industry_name": "Manufacturing",
    },
    {
        "field_group": "ENGINEERING",
        "industry_name": "Construction",
    },
    {
        "field_group": "ENGINEERING",
        "industry_name": "Professional, Scientific, and Technical Services",
    },
    {
        "field_group": "BUSINESS",
        "industry_name": "Finance and Insurance",
    },
    {
        "field_group": "BUSINESS",
        "industry_name": "Management of Companies and Enterprises",
    },
    {
        "field_group": "BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES",
        "industry_name": "Finance and Insurance",
    },
    {
        "field_group": "BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES",
        "industry_name": "Management of Companies and Enterprises",
    },
    {
        "field_group": "HEALTH PROFESSIONS",
        "industry_name": "Health Care and Social Assistance",
    },
    {
        "field_group": "HEALTH PROFESSIONS AND RELATED PROGRAMS",
        "industry_name": "Health Care and Social Assistance",
    },
    {
        "field_group": "EDUCATION",
        "industry_name": "Educational Services",
    },
]

if mapping_path.exists():
    mapping = pd.read_csv(mapping_path)
    print("Using existing mapping file:")
    print(mapping_path)
else:
    mapping = pd.DataFrame(default_mapping)
    mapping.to_csv(mapping_path, index=False)
    print("Created mapping file:")
    print(mapping_path)
    print("You can edit this CSV later to improve the field_group to industry_name connection.")

print()

mapping["field_group_clean"] = clean_text_series(mapping["field_group"])
mapping["industry_name_clean"] = clean_text_series(mapping["industry_name"])

degree_field["field_group_clean"] = clean_text_series(degree_field["field_group"])
company["industry_name_clean"] = clean_text_series(company["industry_name"])

print("Mapping rows:", len(mapping))
print()


# ------------------------------------------------------------
# 7. CALCULATE SUCCESS RATES
# ------------------------------------------------------------

company_for_rate = company.copy()

company_for_rate["metric_role"] = company_for_rate.apply(
    lambda row: classify_metric(
        row["metric_name"],
        row["metric_category"]
    ),
    axis=1
)

rate_base = (
    company_for_rate
    .groupby(
        [
            "year",
            "industry_name",
            "industry_name_clean",
            "metric_role",
        ],
        as_index=False
    )["value"]
    .sum()
)

rate_wide = (
    rate_base
    .pivot_table(
        index=[
            "year",
            "industry_name",
            "industry_name_clean",
        ],
        columns="metric_role",
        values="value",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for col in [
    "job_gains",
    "job_losses",
    "business_openings",
    "business_closings",
]:
    if col not in rate_wide.columns:
        rate_wide[col] = 0

rate_wide["job_success_rate"] = np.where(
    (rate_wide["job_gains"] + rate_wide["job_losses"]) > 0,
    rate_wide["job_gains"] /
    (rate_wide["job_gains"] + rate_wide["job_losses"]) * 100,
    np.nan
)

rate_wide["business_success_rate"] = np.where(
    (rate_wide["business_openings"] + rate_wide["business_closings"]) > 0,
    rate_wide["business_openings"] /
    (rate_wide["business_openings"] + rate_wide["business_closings"]) * 100,
    np.nan
)

success_rate_path = OUTPUT_DIR / "success_rates_by_industry.csv"
rate_wide.to_csv(success_rate_path, index=False)

print("DONE success rates")
print("Saved:", success_rate_path)
print()


# ------------------------------------------------------------
# 8. CREATE FINAL DASHBOARD DATASET
# ------------------------------------------------------------

degree_bridge = degree_field.merge(
    mapping[
        [
            "field_group_clean",
            "industry_name",
            "industry_name_clean",
        ]
    ],
    on="field_group_clean",
    how="inner"
)

final_dashboard = degree_bridge.merge(
    company,
    on=[
        "year",
        "industry_name_clean",
    ],
    how="inner",
    suffixes=("_degree", "_company")
)

final_dashboard["industry_name"] = final_dashboard["industry_name_company"]

final_dashboard = final_dashboard.merge(
    rate_wide[
        [
            "year",
            "industry_name_clean",
            "job_success_rate",
            "business_success_rate",
        ]
    ],
    on=[
        "year",
        "industry_name_clean",
    ],
    how="left"
)

final_dashboard = final_dashboard[
    [
        "year",
        "degree_group",
        "award_level_name",
        "field_group",
        "field_subgroup",
        "major_name",
        "degree_count",
        "industry_name",
        "industry_code",
        "naics_code",
        "metric_name",
        "metric_category",
        "value",
        "unit",
        "job_success_rate",
        "business_success_rate",
    ]
]

final_dashboard_path = OUTPUT_DIR / "final_dashboard_dataset.csv"
final_dashboard.to_csv(final_dashboard_path, index=False)

print("DONE final dashboard dataset")
print("Rows:", len(final_dashboard))
print("Saved:", final_dashboard_path)
print()


# ------------------------------------------------------------
# 9. DISPLAY BASIC CHECKS
# ------------------------------------------------------------

print("=" * 70)
print("BASIC CHECKS")
print("=" * 70)

print()
print("Degree years:")
print(degree_field["year"].min(), "to", degree_field["year"].max())

print()
print("Company years:")
print(company["year"].min(), "to", company["year"].max())

print()
print("Top degree field groups:")
print(
    degree_field
    .groupby("field_group")["degree_count"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print()
print("Top industries:")
print(
    company
    .groupby("industry_name")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print()
print("Metric names found:")
print(
    company
    .groupby("metric_name")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(50)
)

print()
print("Final dashboard sample:")
display(final_dashboard.head(20))


# ------------------------------------------------------------
# 10. CREATE DASHBOARD CHARTS WITH PLOTLY
# ------------------------------------------------------------

try:
    import plotly.express as px
    import plotly.io as pio
except ImportError:
    raise ImportError(
        "Plotly is not installed. Run this in a new cell first:\n"
        "!pip install plotly"
    )


# ----------------------------
# CHART A: DEGREE SUPPLY STACKED BAR
# ----------------------------

chart_a_data = (
    degree_field
    .groupby(
        [
            "year",
            "field_group",
        ],
        as_index=False
    )["degree_count"]
    .sum()
)

fig_a = px.bar(
    chart_a_data,
    x="year",
    y="degree_count",
    color="field_group",
    title="Chart A: Degree Supply by Field Group Over Time",
    labels={
        "year": "Year",
        "degree_count": "Total Degree Count",
        "field_group": "Field Group",
    },
)

fig_a.update_layout(
    barmode="stack",
    xaxis_title="Year",
    yaxis_title="Total Degree Count",
    legend_title="Field Group",
)


# ----------------------------
# CHART B: BUSINESS OUTCOME STACKED BAR
# ----------------------------

if INDUSTRY_FILTER is not None:
    chart_b_source = company[
        clean_text_series(company["industry_name"]) ==
        str(INDUSTRY_FILTER).strip().upper()
    ].copy()
else:
    chart_b_source = company.copy()

chart_b_data = (
    chart_b_source
    .groupby(
        [
            "year",
            "metric_name",
        ],
        as_index=False
    )["value"]
    .sum()
)

fig_b = px.bar(
    chart_b_data,
    x="year",
    y="value",
    color="metric_name",
    title="Chart B: Business and Job Metrics by Metric Name Over Time",
    labels={
        "year": "Year",
        "value": "Metric Value",
        "metric_name": "Metric Name",
    },
)

fig_b.update_layout(
    barmode="stack",
    xaxis_title="Year",
    yaxis_title="Total Value",
    legend_title="Metric Name",
)


# ----------------------------
# CHART C: INDUSTRY SECTOR STACKED BAR
# ----------------------------

if METRIC_FILTER is not None:
    chart_c_source = company[
        clean_text_series(company["metric_name"]) ==
        str(METRIC_FILTER).strip().upper()
    ].copy()
else:
    auto_metric = (
        company
        .groupby("metric_name")["value"]
        .sum()
        .sort_values(ascending=False)
        .index[0]
    )
    chart_c_source = company[company["metric_name"] == auto_metric].copy()
    print()
    print("No METRIC_FILTER selected.")
    print("Chart C auto-selected metric:", auto_metric)

top_industries = (
    chart_c_source
    .groupby("industry_name")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_INDUSTRIES)
    .index
)

chart_c_source = chart_c_source[
    chart_c_source["industry_name"].isin(top_industries)
]

chart_c_data = (
    chart_c_source
    .groupby(
        [
            "year",
            "industry_name",
        ],
        as_index=False
    )["value"]
    .sum()
)

fig_c = px.bar(
    chart_c_data,
    x="year",
    y="value",
    color="industry_name",
    title="Chart C: Industry Activity by Industry Name Over Time",
    labels={
        "year": "Year",
        "value": "Total Value",
        "industry_name": "Industry Name",
    },
)

fig_c.update_layout(
    barmode="stack",
    xaxis_title="Year",
    yaxis_title="Total Value",
    legend_title="Industry Name",
)


# ----------------------------
# CHART D: JOB SUCCESS RATE
# ----------------------------

job_rate_data = rate_wide.dropna(subset=["job_success_rate"]).copy()

if len(job_rate_data) > 0:
    top_job_industries = (
        job_rate_data
        .groupby("industry_name")["job_gains"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N_INDUSTRIES)
        .index
    )

    job_rate_data = job_rate_data[
        job_rate_data["industry_name"].isin(top_job_industries)
    ]

fig_d = px.line(
    job_rate_data,
    x="year",
    y="job_success_rate",
    color="industry_name",
    markers=True,
    title="Chart D: Job Success Rate by Industry Over Time",
    labels={
        "year": "Year",
        "job_success_rate": "Job Success Rate (%)",
        "industry_name": "Industry Name",
    },
)

fig_d.update_layout(
    xaxis_title="Year",
    yaxis_title="Job Success Rate (%)",
    legend_title="Industry Name",
)


# ----------------------------
# CHART E: BUSINESS SUCCESS RATE
# ----------------------------

business_rate_data = rate_wide.dropna(subset=["business_success_rate"]).copy()

if len(business_rate_data) > 0:
    top_business_industries = (
        business_rate_data
        .groupby("industry_name")["business_openings"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N_INDUSTRIES)
        .index
    )

    business_rate_data = business_rate_data[
        business_rate_data["industry_name"].isin(top_business_industries)
    ]

fig_e = px.line(
    business_rate_data,
    x="year",
    y="business_success_rate",
    color="industry_name",
    markers=True,
    title="Chart E: Business Success Rate by Industry Over Time",
    labels={
        "year": "Year",
        "business_success_rate": "Business Success Rate (%)",
        "industry_name": "Industry Name",
    },
)

fig_e.update_layout(
    xaxis_title="Year",
    yaxis_title="Business Success Rate (%)",
    legend_title="Industry Name",
)


# ------------------------------------------------------------
# 11. SHOW CHARTS IN NOTEBOOK
# ------------------------------------------------------------

fig_a.show()
fig_b.show()
fig_c.show()
fig_d.show()
fig_e.show()


# ------------------------------------------------------------
# 12. SAVE ALL CHARTS INTO ONE HTML DASHBOARD
# ------------------------------------------------------------

dashboard_html_path = OUTPUT_DIR / "degree_supply_vs_industry_dashboard.html"

html_parts = []

html_parts.append("""
<html>
<head>
<title>Degree Supply vs Industry Business and Job Outcomes Over Time</title>
</head>
<body>
<h1>Degree Supply vs Industry Business and Job Outcomes Over Time</h1>

<p>
This dashboard connects degree completion trends to industry sectors and compares them with
business openings, business closings, job gains, job losses, and success rates over time.
</p>

<h2>Output Location</h2>
<p>This dashboard and CSV output files are saved in the Downloads folder.</p>

<h2>Filters Used</h2>
<ul>
<li>Year range: {year_min} to {year_max}</li>
<li>Degree group filter: {degree_group_filter}</li>
<li>Award level filter: {award_level_filter}</li>
<li>Field group filter: {field_group_filter}</li>
<li>Industry filter: {industry_filter}</li>
<li>Metric filter: {metric_filter}</li>
<li>State filter: {state_filter}</li>
</ul>
""".format(
    year_min=YEAR_MIN,
    year_max=YEAR_MAX,
    degree_group_filter=DEGREE_GROUP_FILTER,
    award_level_filter=AWARD_LEVEL_FILTER,
    field_group_filter=FIELD_GROUP_FILTER,
    industry_filter=INDUSTRY_FILTER,
    metric_filter=METRIC_FILTER,
    state_filter=STATE_FILTER,
))

html_parts.append(pio.to_html(fig_a, full_html=False, include_plotlyjs="cdn"))
html_parts.append(pio.to_html(fig_b, full_html=False, include_plotlyjs=False))
html_parts.append(pio.to_html(fig_c, full_html=False, include_plotlyjs=False))
html_parts.append(pio.to_html(fig_d, full_html=False, include_plotlyjs=False))
html_parts.append(pio.to_html(fig_e, full_html=False, include_plotlyjs=False))

html_parts.append("""
</body>
</html>
""")

dashboard_html_path.write_text("\n".join(html_parts), encoding="utf-8")


# ------------------------------------------------------------
# 13. FINAL MESSAGE
# ------------------------------------------------------------

print()
print("=" * 70)
print("DONE")
print("=" * 70)
print()
print("Your new output folder is saved here:")
print(OUTPUT_DIR)
print()
print("HTML dashboard:")
print(dashboard_html_path)
print()
print("CSV files saved:")
print("-", degree_detail_path)
print("-", degree_field_path)
print("-", company_path)
print("-", mapping_path)
print("-", success_rate_path)
print("-", final_dashboard_path)
print()
print("Important:")
print("This code reads your original files but does NOT edit them.")
print("It only creates new CSV and HTML files inside the Downloads output folder.")

# Missing year Column row

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# READ ONLY - ALL COLUMN YEAR RANGE + UNIQUE COUNT TABLE
# Does not save files
# Does not edit files
# Memory optimized with chunks
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

YEAR_START = 1970
YEAR_END = 2025
CHUNKSIZE = 100_000


def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path.name)


def get_columns(path):
    return list(pd.read_csv(path, nrows=0).columns)


def check_column_table(path, dataset_name, year_col="year"):
    print("=" * 90)
    print(f"CHECKING: {dataset_name}")
    print("=" * 90)

    columns = get_columns(path)

    if year_col not in columns:
        raise ValueError(f"Missing year column: {year_col}")

    year_sets = {col: set() for col in columns}
    unique_sets = {col: set() for col in columns}

    reader = pd.read_csv(
        path,
        chunksize=CHUNKSIZE,
        dtype=str,
        low_memory=False,
        na_values=["", " ", "NA", "N/A", "nan", "NaN", "null", "NULL", "."]
    )

    total_rows = 0
    start_time = time.time()

    for chunk_num, chunk in enumerate(reader, start=1):
        total_rows += len(chunk)

        year_num = pd.to_numeric(chunk[year_col], errors="coerce")

        valid_year_mask = year_num.between(YEAR_START, YEAR_END, inclusive="both")
        valid_years = year_num[valid_year_mask].astype("Int64")

        for col in columns:
            col_data = chunk[col]

            has_data = col_data.notna()

            # Unique count
            unique_sets[col].update(col_data[has_data].unique())

            # Year range where this column has data
            years_for_col = valid_years[valid_year_mask & has_data].dropna().unique()
            year_sets[col].update(int(y) for y in years_for_col)

        if chunk_num % 10 == 0:
            seconds = round(time.time() - start_time, 1)
            print(f"chunks: {chunk_num:,} | rows read: {total_rows:,} | seconds: {seconds}")

    records = []

    for i, col in enumerate(columns, start=1):
        years = sorted(year_sets[col])

        if len(years) == 0:
            year_range = "no data"
        else:
            year_range = f"{years[0]}-{years[-1]}"

        records.append({
            "dataset": dataset_name,
            "column_number": i,
            "column_name": col,
            "year_range_with_data": year_range,
            "unique_word_count": len(unique_sets[col])
        })

    result = pd.DataFrame(records)

    return result


# ============================================================
# RUN
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_table = check_column_table(DEGREE_FILE, "degree")
company_table = check_column_table(COMPANY_FILE, "company")

final_table = pd.concat([degree_table, company_table], ignore_index=True)

# Show all rows, no limit
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(final_table)

print("=" * 90)
print("DONE - READ ONLY")
print("=" * 90)
print("Total columns checked:", len(final_table))
print("Degree columns:", len(degree_table))
print("Company columns:", len(company_table))
print("No files changed.")
print("No files saved.")

| dataset |   column_number | column_name          | year_range_with_data               |        unique_word_count |
| ------- | --------------: | -------------------- | ---------------------------------- | -----------------------: |
| degree  | all 242 columns | every column checked | oldest year–newest year per column | unique values per column |
| company |  all 34 columns | every column checked | oldest year–newest year per column | unique values per column |


# Stacked chart 1

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time

# ============================================================
# READ ONLY - MEMORY OPTIMIZED STACKED CHARTS
# Does NOT edit original CSV files
# Reads only needed columns
# Uses chunks for large files
# Saves chart images to Downloads
# ============================================================

start_time = time.time()

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads/Stacked_Chart_ReadOnly_Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000


# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)
print()


# ------------------------------------------------------------
# 3. DEGREE STACKED CHART
#    year + degree_group + degree_count
# ------------------------------------------------------------
def build_degree_summary():
    print("=" * 80)
    print("READING DEGREE DATA - MEMORY OPTIMIZED")
    print("=" * 80)

    usecols = ["year", "degree_group", "degree_count"]
    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(DEGREE_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce")

        chunk = chunk.dropna(subset=["year", "degree_group", "degree_count"])
        chunk["year"] = chunk["year"].astype(int)

        grouped = (
            chunk
            .groupby(["year", "degree_group"], as_index=False)["degree_count"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed degree chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "degree_group"], as_index=False)["degree_count"]
        .sum()
    )

    return final


degree_summary = build_degree_summary()

degree_pivot = (
    degree_summary
    .pivot(index="year", columns="degree_group", values="degree_count")
    .fillna(0)
    .sort_index()
)

print()
print("DEGREE SUMMARY PREVIEW")
display(degree_pivot.head(20))

plt.figure(figsize=(16, 8))
degree_pivot.plot(kind="bar", stacked=True, figsize=(16, 8))

plt.title("Degree Counts by Year Stacked by Degree Group")
plt.xlabel("Year")
plt.ylabel("Total Degree Count")
plt.xticks(rotation=90)
plt.tight_layout()

degree_chart_path = OUTPUT_DIR / "degree_stacked_by_degree_group.png"
plt.savefig(degree_chart_path, dpi=200)
plt.show()

print("Saved degree chart:")
print(degree_chart_path)
print()


# ------------------------------------------------------------
# 4. COMPANY STACKED CHART
#    year + metric_category + value
# ------------------------------------------------------------
def build_company_summary():
    print("=" * 80)
    print("READING COMPANY DATA - MEMORY OPTIMIZED")
    print("=" * 80)

    usecols = ["year", "metric_category", "value"]
    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(COMPANY_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

        chunk = chunk.dropna(subset=["year", "metric_category", "value"])
        chunk["year"] = chunk["year"].astype(int)

        grouped = (
            chunk
            .groupby(["year", "metric_category"], as_index=False)["value"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed company chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "metric_category"], as_index=False)["value"]
        .sum()
    )

    return final


company_summary = build_company_summary()

company_pivot = (
    company_summary
    .pivot(index="year", columns="metric_category", values="value")
    .fillna(0)
    .sort_index()
)

print()
print("COMPANY SUMMARY PREVIEW")
display(company_pivot.head(20))

plt.figure(figsize=(16, 8))
company_pivot.plot(kind="bar", stacked=True, figsize=(16, 8))

plt.title("Company Data by Year Stacked by Metric Category")
plt.xlabel("Year")
plt.ylabel("Total Value")
plt.xticks(rotation=90)
plt.tight_layout()

company_chart_path = OUTPUT_DIR / "company_stacked_by_metric_category.png"
plt.savefig(company_chart_path, dpi=200)
plt.show()

print("Saved company chart:")
print(company_chart_path)
print()


# ------------------------------------------------------------
# 5. SAVE SMALL SUMMARY TABLES ONLY
#    Original files are not changed
# ------------------------------------------------------------
degree_summary_path = OUTPUT_DIR / "degree_stacked_summary.csv"
company_summary_path = OUTPUT_DIR / "company_stacked_summary.csv"

degree_summary.to_csv(degree_summary_path, index=False)
company_summary.to_csv(company_summary_path, index=False)

print("=" * 80)
print("DONE")
print("=" * 80)
print("Original files were NOT edited.")
print("Output folder:")
print(OUTPUT_DIR)
print()
print("Saved files:")
print(degree_chart_path.name)
print(company_chart_path.name)
print(degree_summary_path.name)
print(company_summary_path.name)
print()
print(f"Total time: {time.time() - start_time:.2f} seconds")


# Stacked chart 2

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time

# ============================================================
# READ ONLY - MEMORY OPTIMIZED STACKED CHARTS
# Degree: year stacked by FIELD OF STUDY
# Company: year stacked by INDUSTRIAL SECTOR
# Fixes older year missing by using fallback count columns
# Does NOT edit original files
# ============================================================

start_time = time.time()

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads/Stacked_Chart_Field_Industry_ReadOnly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000

# Keep chart readable
TOP_FIELD_GROUPS = 15
TOP_INDUSTRIES = 15

# For company, this gives the oldest years, 1978+
# Change to None if you want all metric categories mixed together
COMPANY_METRIC_CATEGORY_FILTER = "startup_survival_failure_proxy"


# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)
print()


# ------------------------------------------------------------
# 3. SAFE COLUMN CHECK
# ------------------------------------------------------------
def get_existing_columns(path, wanted_cols):
    all_cols = pd.read_csv(path, nrows=0).columns.tolist()
    return [c for c in wanted_cols if c in all_cols]


# ------------------------------------------------------------
# 4. DEGREE COUNT FIX
# ------------------------------------------------------------
def add_fixed_degree_count(chunk):
    """
    Fix older years:
    degree_count is sometimes 0 for old IPEDS years.
    Use total columns if degree_count is missing or zero.
    """

    # Start with degree_count
    if "degree_count" in chunk.columns:
        count = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)
    else:
        count = pd.Series(0, index=chunk.index)

    # Fallback 1: CTOTALT, newer total column
    if "CTOTALT" in chunk.columns:
        ctotalt = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)
        count = count.mask(count <= 0, ctotalt)

    # Fallback 2: uppercase CRACE15 + CRACE16
    if "CRACE15" in chunk.columns and "CRACE16" in chunk.columns:
        crace_upper = (
            pd.to_numeric(chunk["CRACE15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["CRACE16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, crace_upper)

    # Fallback 3: lowercase crace15 + crace16, old IPEDS
    if "crace15" in chunk.columns and "crace16" in chunk.columns:
        crace_lower = (
            pd.to_numeric(chunk["crace15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["crace16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, crace_lower)

    chunk["degree_count_fixed"] = count
    return chunk


# ------------------------------------------------------------
# 5. BUILD DEGREE FIELD OF STUDY SUMMARY
# ------------------------------------------------------------
def build_degree_field_summary():
    print("=" * 80)
    print("READING DEGREE DATA - FIELD OF STUDY - MEMORY OPTIMIZED")
    print("=" * 80)

    wanted_cols = [
        "year",
        "field_group",
        "degree_count",
        "CTOTALT",
        "CRACE15",
        "CRACE16",
        "crace15",
        "crace16",
    ]

    usecols = get_existing_columns(DEGREE_FILE, wanted_cols)

    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(DEGREE_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk.dropna(subset=["year", "field_group"])
        chunk["year"] = chunk["year"].astype(int)

        chunk["field_group"] = chunk["field_group"].fillna("Unknown Field")
        chunk["field_group"] = chunk["field_group"].astype(str).str.strip()
        chunk.loc[chunk["field_group"].eq(""), "field_group"] = "Unknown Field"

        chunk = add_fixed_degree_count(chunk)

        chunk = chunk[chunk["degree_count_fixed"] > 0]

        grouped = (
            chunk
            .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed degree chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
        .sum()
    )

    return final


degree_field_summary = build_degree_field_summary()

# Keep top fields, group the rest as Other
top_fields = (
    degree_field_summary
    .groupby("field_group")["degree_count_fixed"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_FIELD_GROUPS)
    .index
)

degree_field_summary["field_group_chart"] = degree_field_summary["field_group"].where(
    degree_field_summary["field_group"].isin(top_fields),
    "Other Field Groups"
)

degree_field_chart_data = (
    degree_field_summary
    .groupby(["year", "field_group_chart"], as_index=False)["degree_count_fixed"]
    .sum()
)

degree_field_pivot = (
    degree_field_chart_data
    .pivot(index="year", columns="field_group_chart", values="degree_count_fixed")
    .fillna(0)
    .sort_index()
)

print()
print("DEGREE FIELD OF STUDY SUMMARY PREVIEW")
display(degree_field_pivot.head(25))

ax = degree_field_pivot.plot(kind="bar", stacked=True, figsize=(18, 9))

ax.set_title("Degree Counts by Year Stacked by Field of Study")
ax.set_xlabel("Year")
ax.set_ylabel("Total Degree Count")
plt.xticks(rotation=90)
plt.tight_layout()

degree_chart_path = OUTPUT_DIR / "degree_stacked_by_field_of_study.png"
plt.savefig(degree_chart_path, dpi=200)
plt.show()

print("Saved degree field chart:")
print(degree_chart_path)
print()


# ------------------------------------------------------------
# 6. BUILD COMPANY INDUSTRIAL SECTOR SUMMARY
# ------------------------------------------------------------
def build_company_industry_summary():
    print("=" * 80)
    print("READING COMPANY DATA - INDUSTRIAL SECTOR - MEMORY OPTIMIZED")
    print("=" * 80)

    wanted_cols = [
        "year",
        "industry_name",
        "industry_code",
        "metric_category",
        "value",
    ]

    usecols = get_existing_columns(COMPANY_FILE, wanted_cols)

    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(COMPANY_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

        chunk = chunk.dropna(subset=["year", "value"])
        chunk["year"] = chunk["year"].astype(int)

        # Optional filter to keep the company chart meaningful
        if COMPANY_METRIC_CATEGORY_FILTER is not None and "metric_category" in chunk.columns:
            chunk = chunk[chunk["metric_category"] == COMPANY_METRIC_CATEGORY_FILTER]

        if chunk.empty:
            continue

        # Older years may not have industry_name, so use industry_code fallback
        industry_name = chunk["industry_name"].astype(str).str.strip()
        industry_code = chunk["industry_code"].astype(str).str.strip()

        bad_names = industry_name.isna() | industry_name.eq("") | industry_name.eq("nan") | industry_name.eq("None")

        chunk["industry_sector"] = industry_name
        chunk.loc[bad_names, "industry_sector"] = "Industry Code " + industry_code

        chunk["industry_sector"] = chunk["industry_sector"].fillna("Unknown Industry")
        chunk.loc[chunk["industry_sector"].eq("Industry Code nan"), "industry_sector"] = "Unknown Industry"
        chunk.loc[chunk["industry_sector"].eq("Industry Code None"), "industry_sector"] = "Unknown Industry"

        chunk = chunk[chunk["value"] > 0]

        grouped = (
            chunk
            .groupby(["year", "industry_sector"], as_index=False)["value"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed company chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "industry_sector"], as_index=False)["value"]
        .sum()
    )

    return final


company_industry_summary = build_company_industry_summary()

# Keep top industries, group the rest as Other
top_industries = (
    company_industry_summary
    .groupby("industry_sector")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_INDUSTRIES)
    .index
)

company_industry_summary["industry_sector_chart"] = company_industry_summary["industry_sector"].where(
    company_industry_summary["industry_sector"].isin(top_industries),
    "Other Industrial Sectors"
)

company_industry_chart_data = (
    company_industry_summary
    .groupby(["year", "industry_sector_chart"], as_index=False)["value"]
    .sum()
)

company_industry_pivot = (
    company_industry_chart_data
    .pivot(index="year", columns="industry_sector_chart", values="value")
    .fillna(0)
    .sort_index()
)

print()
print("COMPANY INDUSTRIAL SECTOR SUMMARY PREVIEW")
display(company_industry_pivot.head(25))

ax = company_industry_pivot.plot(kind="bar", stacked=True, figsize=(18, 9))

ax.set_title("Company Data by Year Stacked by Industrial Sector")
ax.set_xlabel("Year")
ax.set_ylabel("Total Value")
plt.xticks(rotation=90)
plt.tight_layout()

company_chart_path = OUTPUT_DIR / "company_stacked_by_industrial_sector.png"
plt.savefig(company_chart_path, dpi=200)
plt.show()

print("Saved company industry chart:")
print(company_chart_path)
print()


# ------------------------------------------------------------
# 7. SAVE SUMMARY TABLES ONLY
# ------------------------------------------------------------
degree_summary_path = OUTPUT_DIR / "degree_field_of_study_summary.csv"
company_summary_path = OUTPUT_DIR / "company_industrial_sector_summary.csv"

degree_field_summary.to_csv(degree_summary_path, index=False)
company_industry_summary.to_csv(company_summary_path, index=False)

print("=" * 80)
print("DONE")
print("=" * 80)
print("Original files were NOT edited.")
print("Output folder:")
print(OUTPUT_DIR)
print()
print("Saved files:")
print(degree_chart_path.name)
print(company_chart_path.name)
print(degree_summary_path.name)
print(company_summary_path.name)
print()
print(f"Total time: {time.time() - start_time:.2f} seconds")

# Stacked 3 - field of study vs industry

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time

# ============================================================
# READ ONLY - MEMORY OPTIMIZED STACKED CHARTS
# Degree: Field of Study by Year
# Company: Industrial Sector by Year
#
# Fixes:
# 1. Older degree years with 0 degree_count
# 2. Older company years with industry_code but no industry_name
#
# Original CSV files are NOT edited
# ============================================================

start_time = time.time()

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads/Stacked_Chart_Field_Industry_ReadOnly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000

TOP_FIELD_GROUPS = 15
TOP_INDUSTRIES = 15

# Use this to keep company chart meaningful.
# This gives older company years from 1978 forward.
COMPANY_METRIC_CATEGORY_FILTER = "startup_survival_failure_proxy"


# ------------------------------------------------------------
# 2. NAICS INDUSTRY CODE NAME MAP
# ------------------------------------------------------------
NAICS_SECTOR_MAP = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31": "Manufacturing",
    "32": "Manufacturing",
    "33": "Manufacturing",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44": "Retail Trade",
    "45": "Retail Trade",
    "44-45": "Retail Trade",
    "48": "Transportation and Warehousing",
    "49": "Transportation and Warehousing",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services except Public Administration",
    "92": "Public Administration",
    "0": "All Industries",
    "00": "All Industries",
}


# ------------------------------------------------------------
# 3. CHECK FILES
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)
print()


# ------------------------------------------------------------
# 4. COLUMN HELPER
# ------------------------------------------------------------
def get_existing_columns(path, wanted_cols):
    all_cols = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in all_cols]


def clean_code(value):
    """
    Cleans industry codes like:
    23.0 -> 23
    31-33 -> 31-33
    """
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    return value


# ------------------------------------------------------------
# 5. DEGREE COUNT FIX FOR OLDER YEARS
# ------------------------------------------------------------
def add_fixed_degree_count(chunk):
    """
    Older IPEDS years may have degree_count as 0.
    This creates degree_count_fixed using fallback total columns.
    """

    if "degree_count" in chunk.columns:
        count = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)
    else:
        count = pd.Series(0, index=chunk.index)

    # Newer total column
    if "CTOTALT" in chunk.columns:
        ctotalt = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)
        count = count.mask(count <= 0, ctotalt)

    # Uppercase old total male + female
    if "CRACE15" in chunk.columns and "CRACE16" in chunk.columns:
        upper_total = (
            pd.to_numeric(chunk["CRACE15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["CRACE16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, upper_total)

    # Lowercase old total male + female
    if "crace15" in chunk.columns and "crace16" in chunk.columns:
        lower_total = (
            pd.to_numeric(chunk["crace15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["crace16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, lower_total)

    chunk["degree_count_fixed"] = count

    return chunk


# ------------------------------------------------------------
# 6. DEGREE STACKED CHART BY FIELD OF STUDY
# ------------------------------------------------------------
def build_degree_field_summary():
    print("=" * 80)
    print("READING DEGREE DATA - FIELD OF STUDY - MEMORY OPTIMIZED")
    print("=" * 80)

    wanted_cols = [
        "year",
        "field_group",
        "degree_count",
        "CTOTALT",
        "CRACE15",
        "CRACE16",
        "crace15",
        "crace16",
    ]

    usecols = get_existing_columns(DEGREE_FILE, wanted_cols)

    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(
            DEGREE_FILE,
            usecols=usecols,
            chunksize=CHUNKSIZE,
            low_memory=False
        ),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk.dropna(subset=["year", "field_group"])
        chunk["year"] = chunk["year"].astype(int)

        chunk["field_group"] = chunk["field_group"].astype(str).str.strip()
        chunk.loc[chunk["field_group"].isin(["", "nan", "None"]), "field_group"] = "Unknown Field"

        chunk = add_fixed_degree_count(chunk)
        chunk = chunk[chunk["degree_count_fixed"] > 0]

        grouped = (
            chunk
            .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed degree chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
        .sum()
    )

    return final


degree_field_summary = build_degree_field_summary()

top_fields = (
    degree_field_summary
    .groupby("field_group")["degree_count_fixed"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_FIELD_GROUPS)
    .index
)

degree_field_summary["field_group_chart"] = degree_field_summary["field_group"].where(
    degree_field_summary["field_group"].isin(top_fields),
    "Other Field Groups"
)

degree_chart_data = (
    degree_field_summary
    .groupby(["year", "field_group_chart"], as_index=False)["degree_count_fixed"]
    .sum()
)

degree_pivot = (
    degree_chart_data
    .pivot(index="year", columns="field_group_chart", values="degree_count_fixed")
    .fillna(0)
    .sort_index()
)

print()
print("DEGREE FIELD OF STUDY SUMMARY PREVIEW")
display(degree_pivot.head(25))

ax = degree_pivot.plot(kind="bar", stacked=True, figsize=(18, 9))

ax.set_title("Degree Counts by Year Stacked by Field of Study")
ax.set_xlabel("Year")
ax.set_ylabel("Total Degree Count")
plt.xticks(rotation=90)
plt.tight_layout()

degree_chart_path = OUTPUT_DIR / "degree_stacked_by_field_of_study.png"
plt.savefig(degree_chart_path, dpi=200)
plt.show()

print("Saved degree chart:")
print(degree_chart_path)
print()


# ------------------------------------------------------------
# 7. COMPANY STACKED CHART BY INDUSTRIAL SECTOR
# ------------------------------------------------------------
def build_company_industry_summary():
    print("=" * 80)
    print("READING COMPANY DATA - INDUSTRIAL SECTOR - MEMORY OPTIMIZED")
    print("=" * 80)

    wanted_cols = [
        "year",
        "industry_name",
        "industry_code",
        "metric_category",
        "value",
    ]

    usecols = get_existing_columns(COMPANY_FILE, wanted_cols)

    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(
            COMPANY_FILE,
            usecols=usecols,
            chunksize=CHUNKSIZE,
            low_memory=False
        ),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

        chunk = chunk.dropna(subset=["year", "value"])
        chunk["year"] = chunk["year"].astype(int)

        # Keep one metric category so numbers are not mixed
        if COMPANY_METRIC_CATEGORY_FILTER is not None and "metric_category" in chunk.columns:
            chunk = chunk[chunk["metric_category"] == COMPANY_METRIC_CATEGORY_FILTER]

        if chunk.empty:
            continue

        # Clean industry code
        chunk["industry_code_clean"] = chunk["industry_code"].apply(clean_code)

        # Clean industry name
        if "industry_name" in chunk.columns:
            chunk["industry_name_clean"] = chunk["industry_name"].astype(str).str.strip()
        else:
            chunk["industry_name_clean"] = ""

        bad_name = chunk["industry_name_clean"].isin(["", "nan", "None"])

        # Start with industry name
        chunk["industry_sector"] = chunk["industry_name_clean"]

        # For older years, fill name from industry code map
        chunk.loc[bad_name, "industry_sector"] = (
            chunk.loc[bad_name, "industry_code_clean"]
            .map(NAICS_SECTOR_MAP)
        )

        # Final fallback
        missing_after_map = chunk["industry_sector"].isna() | chunk["industry_sector"].isin(["", "nan", "None"])
        chunk.loc[missing_after_map, "industry_sector"] = "Unknown Industry"

        chunk = chunk[chunk["value"] > 0]

        grouped = (
            chunk
            .groupby(["year", "industry_sector"], as_index=False)["value"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed company chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "industry_sector"], as_index=False)["value"]
        .sum()
    )

    return final


company_industry_summary = build_company_industry_summary()

top_industries = (
    company_industry_summary
    .groupby("industry_sector")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_INDUSTRIES)
    .index
)

company_industry_summary["industry_sector_chart"] = company_industry_summary["industry_sector"].where(
    company_industry_summary["industry_sector"].isin(top_industries),
    "Other Industrial Sectors"
)

company_chart_data = (
    company_industry_summary
    .groupby(["year", "industry_sector_chart"], as_index=False)["value"]
    .sum()
)

company_pivot = (
    company_chart_data
    .pivot(index="year", columns="industry_sector_chart", values="value")
    .fillna(0)
    .sort_index()
)

print()
print("COMPANY INDUSTRIAL SECTOR SUMMARY PREVIEW")
display(company_pivot.head(25))

ax = company_pivot.plot(kind="bar", stacked=True, figsize=(18, 9))

ax.set_title("Company Data by Year Stacked by Industrial Sector")
ax.set_xlabel("Year")
ax.set_ylabel("Total Value")
plt.xticks(rotation=90)
plt.tight_layout()

company_chart_path = OUTPUT_DIR / "company_stacked_by_industrial_sector.png"
plt.savefig(company_chart_path, dpi=200)
plt.show()

print("Saved company chart:")
print(company_chart_path)
print()


# ------------------------------------------------------------
# 8. SAVE SUMMARY TABLES ONLY
# ------------------------------------------------------------
degree_summary_path = OUTPUT_DIR / "degree_field_of_study_summary.csv"
company_summary_path = OUTPUT_DIR / "company_industrial_sector_summary.csv"

degree_field_summary.to_csv(degree_summary_path, index=False)
company_industry_summary.to_csv(company_summary_path, index=False)

print("=" * 80)
print("DONE")
print("=" * 80)
print("Original files were NOT edited.")
print("Output folder:")
print(OUTPUT_DIR)
print()
print("Saved files:")
print(degree_chart_path.name)
print(company_chart_path.name)
print(degree_summary_path.name)
print(company_summary_path.name)
print()
print(f"Total time: {time.time() - start_time:.2f} seconds")

# Connect 1 - field of study vs industry

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time

# ============================================================
# READ ONLY - MEMORY OPTIMIZED CONNECTED CHART PROJECT
#
# Creates:
# 1. Aligned stacked bar chart
# 2. Correlation heatmap
# 3. Sankey flow chart, if plotly is installed
#
# Original CSV files are NOT edited.
# ============================================================

start_time = time.time()

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

OUTPUT_DIR = Path.home() / "Downloads/Connected_Degree_Industry_Charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 100_000

TOP_FIELD_GROUPS = 12
TOP_INDUSTRIES = 12

COMPANY_METRIC_CATEGORY_FILTER = "startup_survival_failure_proxy"


# ------------------------------------------------------------
# 2. NAICS INDUSTRY CODE NAME MAP
# ------------------------------------------------------------
NAICS_SECTOR_MAP = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31": "Manufacturing",
    "32": "Manufacturing",
    "33": "Manufacturing",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44": "Retail Trade",
    "45": "Retail Trade",
    "44-45": "Retail Trade",
    "48": "Transportation and Warehousing",
    "49": "Transportation and Warehousing",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services except Public Administration",
    "92": "Public Administration",
    "0": "All Industries",
    "00": "All Industries",
}


# ------------------------------------------------------------
# 3. MANUAL FIELD → INDUSTRY CONNECTION MAP
#    This is for Sankey chart only.
# ------------------------------------------------------------
FIELD_TO_INDUSTRY_MAP = {
    "COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES": [
        "Information",
        "Professional, Scientific, and Technical Services",
    ],
    "ENGINEERING": [
        "Manufacturing",
        "Construction",
        "Professional, Scientific, and Technical Services",
    ],
    "BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES": [
        "Finance and Insurance",
        "Management of Companies and Enterprises",
        "Retail Trade",
    ],
    "HEALTH PROFESSIONS AND RELATED PROGRAMS": [
        "Health Care and Social Assistance",
    ],
    "EDUCATION": [
        "Educational Services",
    ],
    "CONSTRUCTION TRADES": [
        "Construction",
    ],
    "MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS": [
        "Manufacturing",
        "Transportation and Warehousing",
    ],
    "TRANSPORTATION AND MATERIALS MOVING": [
        "Transportation and Warehousing",
    ],
    "COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS": [
        "Information",
    ],
    "VISUAL AND PERFORMING ARTS": [
        "Arts, Entertainment, and Recreation",
    ],
    "CULINARY, ENTERTAINMENT, AND PERSONAL SERVICES": [
        "Accommodation and Food Services",
        "Other Services except Public Administration",
    ],
    "LEGAL PROFESSIONS AND STUDIES": [
        "Professional, Scientific, and Technical Services",
    ],
    "AGRICULTURE, AGRICULTURE OPERATIONS, AND RELATED SCIENCES": [
        "Agriculture, Forestry, Fishing and Hunting",
    ],
}


# ------------------------------------------------------------
# 4. BASIC HELPERS
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def get_existing_columns(path, wanted_cols):
    all_cols = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in all_cols]


def clean_code(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)
print()


# ------------------------------------------------------------
# 5. DEGREE COUNT FIX
# ------------------------------------------------------------
def add_fixed_degree_count(chunk):
    if "degree_count" in chunk.columns:
        count = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)
    else:
        count = pd.Series(0, index=chunk.index)

    if "CTOTALT" in chunk.columns:
        ctotalt = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)
        count = count.mask(count <= 0, ctotalt)

    if "CRACE15" in chunk.columns and "CRACE16" in chunk.columns:
        upper_total = (
            pd.to_numeric(chunk["CRACE15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["CRACE16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, upper_total)

    if "crace15" in chunk.columns and "crace16" in chunk.columns:
        lower_total = (
            pd.to_numeric(chunk["crace15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["crace16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, lower_total)

    chunk["degree_count_fixed"] = count
    return chunk


# ------------------------------------------------------------
# 6. READ DEGREE DATA BY FIELD OF STUDY
# ------------------------------------------------------------
def build_degree_field_summary():
    print("=" * 80)
    print("READING DEGREE DATA - FIELD OF STUDY")
    print("=" * 80)

    wanted_cols = [
        "year",
        "field_group",
        "degree_count",
        "CTOTALT",
        "CRACE15",
        "CRACE16",
        "crace15",
        "crace16",
    ]

    usecols = get_existing_columns(DEGREE_FILE, wanted_cols)
    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(DEGREE_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk.dropna(subset=["year", "field_group"])
        chunk["year"] = chunk["year"].astype(int)

        chunk["field_group"] = chunk["field_group"].astype(str).str.strip()
        chunk.loc[chunk["field_group"].isin(["", "nan", "None"]), "field_group"] = "Unknown Field"

        chunk = add_fixed_degree_count(chunk)
        chunk = chunk[chunk["degree_count_fixed"] > 0]

        grouped = (
            chunk
            .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed degree chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
        .sum()
    )

    return final


degree_summary = build_degree_field_summary()


# ------------------------------------------------------------
# 7. READ COMPANY DATA BY INDUSTRIAL SECTOR
# ------------------------------------------------------------
def build_company_industry_summary():
    print("=" * 80)
    print("READING COMPANY DATA - INDUSTRIAL SECTOR")
    print("=" * 80)

    wanted_cols = [
        "year",
        "industry_name",
        "industry_code",
        "metric_category",
        "value",
    ]

    usecols = get_existing_columns(COMPANY_FILE, wanted_cols)
    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(COMPANY_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

        chunk = chunk.dropna(subset=["year", "value"])
        chunk["year"] = chunk["year"].astype(int)

        if COMPANY_METRIC_CATEGORY_FILTER is not None and "metric_category" in chunk.columns:
            chunk = chunk[chunk["metric_category"] == COMPANY_METRIC_CATEGORY_FILTER]

        if chunk.empty:
            continue

        chunk["industry_code_clean"] = chunk["industry_code"].apply(clean_code)

        if "industry_name" in chunk.columns:
            chunk["industry_name_clean"] = chunk["industry_name"].astype(str).str.strip()
        else:
            chunk["industry_name_clean"] = ""

        bad_name = chunk["industry_name_clean"].isin(["", "nan", "None"])

        chunk["industry_sector"] = chunk["industry_name_clean"]

        chunk.loc[bad_name, "industry_sector"] = (
            chunk.loc[bad_name, "industry_code_clean"]
            .map(NAICS_SECTOR_MAP)
        )

        missing_after_map = (
            chunk["industry_sector"].isna()
            | chunk["industry_sector"].isin(["", "nan", "None"])
        )

        chunk.loc[missing_after_map, "industry_sector"] = "Unknown Industry"

        chunk = chunk[chunk["value"] > 0]

        grouped = (
            chunk
            .groupby(["year", "industry_sector"], as_index=False)["value"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed company chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "industry_sector"], as_index=False)["value"]
        .sum()
    )

    return final


company_summary = build_company_industry_summary()


# ------------------------------------------------------------
# 8. MAKE TOP GROUPS
# ------------------------------------------------------------
top_fields = (
    degree_summary
    .groupby("field_group")["degree_count_fixed"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_FIELD_GROUPS)
    .index
)

degree_summary["field_group_chart"] = degree_summary["field_group"].where(
    degree_summary["field_group"].isin(top_fields),
    "Other Field Groups"
)

degree_chart_data = (
    degree_summary
    .groupby(["year", "field_group_chart"], as_index=False)["degree_count_fixed"]
    .sum()
)

degree_pivot = (
    degree_chart_data
    .pivot(index="year", columns="field_group_chart", values="degree_count_fixed")
    .fillna(0)
    .sort_index()
)


top_industries = (
    company_summary
    .groupby("industry_sector")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_INDUSTRIES)
    .index
)

company_summary["industry_sector_chart"] = company_summary["industry_sector"].where(
    company_summary["industry_sector"].isin(top_industries),
    "Other Industrial Sectors"
)

company_chart_data = (
    company_summary
    .groupby(["year", "industry_sector_chart"], as_index=False)["value"]
    .sum()
)

company_pivot = (
    company_chart_data
    .pivot(index="year", columns="industry_sector_chart", values="value")
    .fillna(0)
    .sort_index()
)


# ------------------------------------------------------------
# 9. CHART 1 - ALIGNED STACKED BAR
# ------------------------------------------------------------
print("=" * 80)
print("CREATING CHART 1: ALIGNED STACKED BAR")
print("=" * 80)

common_years = sorted(set(degree_pivot.index).intersection(set(company_pivot.index)))

degree_aligned = degree_pivot.loc[common_years]
company_aligned = company_pivot.loc[common_years]

fig, axes = plt.subplots(2, 1, figsize=(20, 14), sharex=True)

degree_aligned.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set_title("Field of Study Degrees by Year")
axes[0].set_ylabel("Degree Count")
axes[0].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)

company_aligned.plot(kind="bar", stacked=True, ax=axes[1], width=0.85)
axes[1].set_title("Industrial Sector Company Activity by Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Company Value")
axes[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)

plt.xticks(rotation=90)
plt.tight_layout()

aligned_chart_path = OUTPUT_DIR / "01_aligned_stacked_bar_degree_field_and_industry.png"
plt.savefig(aligned_chart_path, dpi=200)
plt.show()

print("Saved:", aligned_chart_path)
print()


# ------------------------------------------------------------
# 10. CHART 2 - CORRELATION HEATMAP
# ------------------------------------------------------------
print("=" * 80)
print("CREATING CHART 2: CORRELATION HEATMAP")
print("=" * 80)

# Use percent share so big sectors do not dominate only by size
degree_share = degree_aligned.div(degree_aligned.sum(axis=1), axis=0).replace([np.inf, -np.inf], 0).fillna(0)
company_share = company_aligned.div(company_aligned.sum(axis=1), axis=0).replace([np.inf, -np.inf], 0).fillna(0)

corr_table = pd.DataFrame(
    index=degree_share.columns,
    columns=company_share.columns,
    dtype=float
)

for field in degree_share.columns:
    for industry in company_share.columns:
        corr_table.loc[field, industry] = degree_share[field].corr(company_share[industry])

corr_table = corr_table.fillna(0)

plt.figure(figsize=(18, 10))
plt.imshow(corr_table.values, aspect="auto")
plt.colorbar(label="Correlation score")

plt.xticks(range(len(corr_table.columns)), corr_table.columns, rotation=90)
plt.yticks(range(len(corr_table.index)), corr_table.index)

plt.title("Correlation Heatmap: Field of Study vs Industrial Sector")
plt.xlabel("Industrial Sector")
plt.ylabel("Field of Study")
plt.tight_layout()

heatmap_path = OUTPUT_DIR / "02_correlation_heatmap_field_vs_industry.png"
plt.savefig(heatmap_path, dpi=200)
plt.show()

print("Saved:", heatmap_path)
print()


# ------------------------------------------------------------
# 11. CHART 3 - SANKEY FLOW CHART
# ------------------------------------------------------------
print("=" * 80)
print("CREATING CHART 3: SANKEY FLOW CHART")
print("=" * 80)

latest_degree_year = int(degree_summary["year"].max())

latest_degree = (
    degree_summary[degree_summary["year"] == latest_degree_year]
    .groupby("field_group", as_index=False)["degree_count_fixed"]
    .sum()
)

sankey_rows = []

for _, row in latest_degree.iterrows():
    field = row["field_group"]
    degree_value = row["degree_count_fixed"]

    field_upper = str(field).upper().strip()

    matched_industries = None

    for map_field, industries in FIELD_TO_INDUSTRY_MAP.items():
        if map_field in field_upper or field_upper in map_field:
            matched_industries = industries
            break

    if matched_industries is None:
        continue

    flow_value = degree_value / len(matched_industries)

    for industry in matched_industries:
        sankey_rows.append({
            "field_group": field,
            "industry_sector": industry,
            "flow_value": flow_value,
            "note": "Manual mapping, not direct person-level job match"
        })

sankey_df = pd.DataFrame(sankey_rows)

sankey_csv_path = OUTPUT_DIR / "03_sankey_manual_mapping_data.csv"
sankey_df.to_csv(sankey_csv_path, index=False)

print("Saved Sankey data:", sankey_csv_path)

try:
    import plotly.graph_objects as go

    all_nodes = list(pd.concat([
        sankey_df["field_group"],
        sankey_df["industry_sector"]
    ]).drop_duplicates())

    node_index = {name: i for i, name in enumerate(all_nodes)}

    sources = sankey_df["field_group"].map(node_index)
    targets = sankey_df["industry_sector"].map(node_index)
    values = sankey_df["flow_value"]

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=20,
            thickness=20,
            label=all_nodes
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )])

    fig.update_layout(
        title_text=f"Sankey Flow: Field of Study to Industrial Sector Mapping, {latest_degree_year}",
        font_size=10,
        width=1200,
        height=800
    )

    sankey_html_path = OUTPUT_DIR / "03_sankey_field_to_industry_manual_mapping.html"
    fig.write_html(sankey_html_path)

    print("Saved Sankey chart:", sankey_html_path)

except ImportError:
    print("Plotly is not installed, so Sankey HTML chart was skipped.")
    print("The Sankey CSV was still saved.")


# ------------------------------------------------------------
# 12. SAVE SUMMARY TABLES
# ------------------------------------------------------------
degree_summary_path = OUTPUT_DIR / "degree_field_summary_all_years.csv"
company_summary_path = OUTPUT_DIR / "company_industry_summary_all_years.csv"
corr_path = OUTPUT_DIR / "correlation_field_vs_industry.csv"

degree_summary.to_csv(degree_summary_path, index=False)
company_summary.to_csv(company_summary_path, index=False)
corr_table.to_csv(corr_path)

print()
print("=" * 80)
print("DONE")
print("=" * 80)
print("Original files were NOT edited.")
print("Output folder:")
print(OUTPUT_DIR)
print()
print("Saved files:")
print(aligned_chart_path.name)
print(heatmap_path.name)
print(sankey_csv_path.name)
print(degree_summary_path.name)
print(company_summary_path.name)
print(corr_path.name)
print()
print(f"Total time: {time.time() - start_time:.2f} seconds")

# Degree-to-Industry Career Path Comparison: Industry Job vs Startup Success and Failure Risk - 1

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import time

# ============================================================
# READ ONLY - MEMORY OPTIMIZED
# NO SAVE VERSION
#
# Goal:
# Compare whether each field of study looks better for:
#   1. Industry job path
#   2. Startup path
#
# Shows charts and tables only.
# Does NOT save files.
# Does NOT edit original CSV files.
# ============================================================

start_time = time.time()

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000

COMPARE_START_YEAR = 2010
COMPARE_END_YEAR = None

TOP_FIELDS_FOR_CHART = 15
DROP_ALL_INDUSTRIES = True


# ------------------------------------------------------------
# 2. INDUSTRY CODE TO NAME MAP
# ------------------------------------------------------------
NAICS_SECTOR_MAP = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31": "Manufacturing",
    "32": "Manufacturing",
    "33": "Manufacturing",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44": "Retail Trade",
    "45": "Retail Trade",
    "44-45": "Retail Trade",
    "48": "Transportation and Warehousing",
    "49": "Transportation and Warehousing",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services except Public Administration",
    "92": "Public Administration",
    "0": "All Industries",
    "00": "All Industries",
}


# ------------------------------------------------------------
# 3. FIELD OF STUDY TO INDUSTRY MAP
# ------------------------------------------------------------
FIELD_TO_INDUSTRY_MAP = {
    "COMPUTER AND INFORMATION SCIENCES": [
        "Information",
        "Professional, Scientific, and Technical Services",
    ],
    "ENGINEERING": [
        "Manufacturing",
        "Construction",
        "Professional, Scientific, and Technical Services",
    ],
    "ENGINEERING TECHNOLOGIES": [
        "Manufacturing",
        "Construction",
    ],
    "BUSINESS, MANAGEMENT, MARKETING": [
        "Finance and Insurance",
        "Management of Companies and Enterprises",
        "Retail Trade",
        "Wholesale Trade",
    ],
    "HEALTH PROFESSIONS": [
        "Health Care and Social Assistance",
    ],
    "EDUCATION": [
        "Educational Services",
    ],
    "CONSTRUCTION TRADES": [
        "Construction",
    ],
    "MECHANIC AND REPAIR": [
        "Manufacturing",
        "Transportation and Warehousing",
    ],
    "TRANSPORTATION AND MATERIALS MOVING": [
        "Transportation and Warehousing",
    ],
    "COMMUNICATION, JOURNALISM": [
        "Information",
    ],
    "VISUAL AND PERFORMING ARTS": [
        "Arts, Entertainment, and Recreation",
        "Information",
    ],
    "PERSONAL AND CULINARY SERVICES": [
        "Accommodation and Food Services",
        "Other Services except Public Administration",
    ],
    "CULINARY": [
        "Accommodation and Food Services",
    ],
    "LEGAL PROFESSIONS": [
        "Professional, Scientific, and Technical Services",
    ],
    "AGRICULTURE": [
        "Agriculture, Forestry, Fishing and Hunting",
    ],
    "ARCHITECTURE": [
        "Construction",
        "Professional, Scientific, and Technical Services",
    ],
    "BIOLOGICAL AND BIOMEDICAL SCIENCES": [
        "Health Care and Social Assistance",
        "Professional, Scientific, and Technical Services",
    ],
    "MATHEMATICS AND STATISTICS": [
        "Professional, Scientific, and Technical Services",
        "Finance and Insurance",
    ],
    "PHYSICAL SCIENCES": [
        "Professional, Scientific, and Technical Services",
        "Manufacturing",
    ],
    "SECURITY AND PROTECTIVE SERVICES": [
        "Public Administration",
        "Administrative and Support and Waste Management",
    ],
    "HOMELAND SECURITY": [
        "Public Administration",
        "Administrative and Support and Waste Management",
    ],
}


# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def get_existing_columns(path, wanted_cols):
    all_cols = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in all_cols]


def clean_code(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    return value


def standardize_industry_name(name):
    if pd.isna(name):
        return "Unknown Industry"

    s = str(name).strip()

    if s in ["", "nan", "None"]:
        return "Unknown Industry"

    low = s.lower()

    if "agriculture" in low:
        return "Agriculture, Forestry, Fishing and Hunting"
    if "mining" in low:
        return "Mining, Quarrying, and Oil and Gas Extraction"
    if "utilities" in low:
        return "Utilities"
    if "construction" in low:
        return "Construction"
    if "manufacturing" in low:
        return "Manufacturing"
    if "wholesale" in low:
        return "Wholesale Trade"
    if "retail" in low:
        return "Retail Trade"
    if "transportation" in low or "warehousing" in low:
        return "Transportation and Warehousing"
    if "information" in low:
        return "Information"
    if "finance" in low or "insurance" in low:
        return "Finance and Insurance"
    if "real estate" in low:
        return "Real Estate and Rental and Leasing"
    if "professional" in low or "scientific" in low or "technical" in low:
        return "Professional, Scientific, and Technical Services"
    if "management of companies" in low:
        return "Management of Companies and Enterprises"
    if "administrative" in low or "waste management" in low:
        return "Administrative and Support and Waste Management"
    if "educational" in low:
        return "Educational Services"
    if "health care" in low or "social assistance" in low:
        return "Health Care and Social Assistance"
    if "arts" in low or "entertainment" in low or "recreation" in low:
        return "Arts, Entertainment, and Recreation"
    if "accommodation" in low or "food services" in low:
        return "Accommodation and Food Services"
    if "other services" in low:
        return "Other Services except Public Administration"
    if "public administration" in low:
        return "Public Administration"
    if "all industries" in low or low == "total":
        return "All Industries"

    return s


def short_label(text, max_len=45):
    text = str(text)
    if len(text) <= max_len:
        return text
    return text[:max_len - 3] + "..."


def safe_number(x):
    if pd.isna(x):
        return 0
    return x


def map_field_to_industries(field_name):
    field_upper = str(field_name).upper().strip()
    matches = []

    for key, industries in FIELD_TO_INDUSTRY_MAP.items():
        key_upper = key.upper().strip()

        if key_upper in field_upper or field_upper in key_upper:
            matches.extend(industries)

    clean_matches = []
    for item in matches:
        if item not in clean_matches:
            clean_matches.append(item)

    return clean_matches


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)
print()


# ------------------------------------------------------------
# 5. DEGREE COUNT FIX
# ------------------------------------------------------------
def add_fixed_degree_count(chunk):
    if "degree_count" in chunk.columns:
        count = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)
    else:
        count = pd.Series(0, index=chunk.index)

    if "CTOTALT" in chunk.columns:
        ctotalt = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)
        count = count.mask(count <= 0, ctotalt)

    if "CRACE15" in chunk.columns and "CRACE16" in chunk.columns:
        upper_total = (
            pd.to_numeric(chunk["CRACE15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["CRACE16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, upper_total)

    if "crace15" in chunk.columns and "crace16" in chunk.columns:
        lower_total = (
            pd.to_numeric(chunk["crace15"], errors="coerce").fillna(0)
            + pd.to_numeric(chunk["crace16"], errors="coerce").fillna(0)
        )
        count = count.mask(count <= 0, lower_total)

    chunk["degree_count_fixed"] = count
    return chunk


# ------------------------------------------------------------
# 6. READ DEGREE DATA
# ------------------------------------------------------------
def build_degree_summary():
    print("=" * 90)
    print("READING DEGREE DATA - FIELD OF STUDY")
    print("=" * 90)

    wanted_cols = [
        "year",
        "field_group",
        "degree_count",
        "CTOTALT",
        "CRACE15",
        "CRACE16",
        "crace15",
        "crace16",
    ]

    usecols = get_existing_columns(DEGREE_FILE, wanted_cols)
    summaries = []

    for i, chunk in enumerate(
        pd.read_csv(
            DEGREE_FILE,
            usecols=usecols,
            chunksize=CHUNKSIZE,
            low_memory=False
        ),
        start=1
    ):
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk.dropna(subset=["year", "field_group"])
        chunk["year"] = chunk["year"].astype(int)

        chunk["field_group"] = chunk["field_group"].astype(str).str.strip()
        chunk.loc[chunk["field_group"].isin(["", "nan", "None"]), "field_group"] = "Unknown Field"

        chunk = add_fixed_degree_count(chunk)
        chunk = chunk[chunk["degree_count_fixed"] > 0]

        grouped = (
            chunk
            .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed degree chunks: {i}")

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
        .sum()
    )

    return final


degree_summary = build_degree_summary()
print()


# ------------------------------------------------------------
# 7. CLASSIFY COMPANY METRICS
# ------------------------------------------------------------
def add_company_signal_type(chunk):
    for col in ["metric_category", "metric_name", "metric_code"]:
        if col not in chunk.columns:
            chunk[col] = ""

    cat = chunk["metric_category"].fillna("").astype(str).str.lower()
    name = chunk["metric_name"].fillna("").astype(str).str.lower()
    code = chunk["metric_code"].fillna("").astype(str).str.lower()

    name_code_text = (name + " " + code).str.replace("_", " ", regex=False)

    signal = pd.Series("other", index=chunk.index)

    job_success_re = (
        r"job creation|job gain|job gains|gross job gain|openings|opening|hires|"
        r"employment gain|expansion"
    )

    job_risk_re = (
        r"job destruction|job loss|job losses|gross job loss|closings|closing|"
        r"separations|employment loss|contraction"
    )

    startup_success_re = (
        r"business application|new business|formation|startup|start up|birth|births|"
        r"entry|entries|survival|survive|firmbirth|estab entry|establishment entry"
    )

    startup_failure_re = (
        r"bankruptcy|failure|fail|death|deaths|exit|exits|closure|closures|"
        r"firmdeath|estab exit|establishment exit"
    )

    business_app_cat = cat.str.contains(
        r"business_applications|business applications|new_business|new business|formation",
        regex=True,
        na=False
    )

    bankruptcy_cat = cat.str.contains(
        r"bankruptcy",
        regex=True,
        na=False
    )

    job_cat = cat.str.contains(
        r"job_flows|job flows|openings|closings|gains|losses|bdm|bed",
        regex=True,
        na=False
    )

    startup_cat = cat.str.contains(
        r"startup|survival|failure|bds|birth|death",
        regex=True,
        na=False
    )

    signal.loc[business_app_cat] = "startup_success"
    signal.loc[bankruptcy_cat] = "startup_failure_risk"

    signal.loc[job_cat & name_code_text.str.contains(job_success_re, regex=True, na=False)] = "job_success"
    signal.loc[job_cat & name_code_text.str.contains(job_risk_re, regex=True, na=False)] = "job_loss_risk"

    signal.loc[startup_cat & name_code_text.str.contains(startup_success_re, regex=True, na=False)] = "startup_success"
    signal.loc[startup_cat & name_code_text.str.contains(startup_failure_re, regex=True, na=False)] = "startup_failure_risk"

    signal.loc[(signal == "other") & name_code_text.str.contains(job_success_re, regex=True, na=False)] = "job_success"
    signal.loc[(signal == "other") & name_code_text.str.contains(job_risk_re, regex=True, na=False)] = "job_loss_risk"
    signal.loc[(signal == "other") & name_code_text.str.contains(startup_success_re, regex=True, na=False)] = "startup_success"
    signal.loc[(signal == "other") & name_code_text.str.contains(startup_failure_re, regex=True, na=False)] = "startup_failure_risk"

    chunk["signal_type"] = signal
    return chunk


# ------------------------------------------------------------
# 8. READ COMPANY DATA
# ------------------------------------------------------------
def build_company_signal_summary():
    print("=" * 90)
    print("READING COMPANY DATA - JOB VS STARTUP SIGNALS")
    print("=" * 90)

    wanted_cols = [
        "year",
        "industry_name",
        "industry_code",
        "metric_category",
        "metric_code",
        "metric_name",
        "value",
        "unit",
    ]

    usecols = get_existing_columns(COMPANY_FILE, wanted_cols)

    summaries = []
    metric_examples = []

    for i, chunk in enumerate(
        pd.read_csv(
            COMPANY_FILE,
            usecols=usecols,
            chunksize=CHUNKSIZE,
            low_memory=False
        ),
        start=1
    ):
        for col in ["industry_name", "industry_code", "metric_category", "metric_code", "metric_name", "unit"]:
            if col not in chunk.columns:
                chunk[col] = ""

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

        chunk = chunk.dropna(subset=["year", "value"])
        chunk["year"] = chunk["year"].astype(int)

        metric_examples.append(
            chunk[["metric_category", "metric_code", "metric_name", "unit"]]
            .drop_duplicates()
        )

        chunk["industry_code_clean"] = chunk["industry_code"].apply(clean_code)

        chunk["industry_name_clean"] = chunk["industry_name"].astype(str).str.strip()
        bad_name = chunk["industry_name_clean"].isin(["", "nan", "None"])

        chunk["industry_sector"] = chunk["industry_name_clean"]

        chunk.loc[bad_name, "industry_sector"] = (
            chunk.loc[bad_name, "industry_code_clean"]
            .map(NAICS_SECTOR_MAP)
        )

        chunk["industry_sector"] = chunk["industry_sector"].apply(standardize_industry_name)

        if DROP_ALL_INDUSTRIES:
            chunk = chunk[chunk["industry_sector"] != "All Industries"]

        chunk = add_company_signal_type(chunk)

        chunk = chunk[chunk["signal_type"] != "other"]
        chunk = chunk[chunk["value"] > 0]

        if chunk.empty:
            continue

        grouped = (
            chunk
            .groupby(["year", "industry_sector", "signal_type"], as_index=False)["value"]
            .sum()
        )

        summaries.append(grouped)

        if i % 10 == 0:
            print(f"Processed company chunks: {i}")

    metric_check = pd.concat(metric_examples, ignore_index=True).drop_duplicates()
    metric_check = add_company_signal_type(metric_check)

    print()
    print("METRIC CLASSIFICATION COUNTS")
    display(metric_check["signal_type"].value_counts().reset_index().rename(
        columns={"index": "signal_type", "signal_type": "count"}
    ))

    print()
    print("METRIC CLASSIFICATION CHECK")
    display(metric_check.sort_values(["signal_type", "metric_category", "metric_name"]).head(50))

    if len(summaries) == 0:
        raise ValueError(
            "No company metrics were classified into job/startup signals. "
            "Check the METRIC CLASSIFICATION CHECK table above."
        )

    final = pd.concat(summaries, ignore_index=True)

    final = (
        final
        .groupby(["year", "industry_sector", "signal_type"], as_index=False)["value"]
        .sum()
    )

    return final


company_signal_summary = build_company_signal_summary()
print()


# ------------------------------------------------------------
# 9. CHANGE CALCULATION
# ------------------------------------------------------------
def compute_group_change(df, group_cols, value_col, year_col="year", start_year=None, end_year=None):
    df = df.copy()
    df = df.dropna(subset=[year_col, value_col])
    df = df[df[value_col] > 0]

    if start_year is not None:
        df = df[df[year_col] >= start_year]

    if end_year is not None:
        df = df[df[year_col] <= end_year]

    if df.empty:
        return pd.DataFrame()

    agg = (
        df
        .groupby(group_cols + [year_col], as_index=False)[value_col]
        .sum()
        .sort_values(group_cols + [year_col])
    )

    first_rows = (
        agg
        .groupby(group_cols, as_index=False)
        .first()
        .rename(columns={year_col: "first_year", value_col: "first_value"})
    )

    last_rows = (
        agg
        .groupby(group_cols, as_index=False)
        .last()
        .rename(columns={year_col: "last_year", value_col: "last_value"})
    )

    out = pd.merge(first_rows, last_rows, on=group_cols, how="inner")

    out["change_pct"] = np.where(
        out["first_value"] == 0,
        np.nan,
        ((out["last_value"] - out["first_value"]) / out["first_value"].abs()) * 100
    )

    out["change_pct"] = out["change_pct"].replace([np.inf, -np.inf], np.nan)

    return out


degree_change = compute_group_change(
    degree_summary,
    group_cols=["field_group"],
    value_col="degree_count_fixed",
    start_year=COMPARE_START_YEAR,
    end_year=COMPARE_END_YEAR
)

company_change = compute_group_change(
    company_signal_summary,
    group_cols=["industry_sector", "signal_type"],
    value_col="value",
    start_year=COMPARE_START_YEAR,
    end_year=COMPARE_END_YEAR
)

company_change_wide = (
    company_change
    .pivot(index="industry_sector", columns="signal_type", values="change_pct")
)

for col in ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]:
    if col not in company_change_wide.columns:
        company_change_wide[col] = np.nan

company_change_wide = company_change_wide[
    ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]
]


# ------------------------------------------------------------
# 10. BUILD FINAL COMPARISON TABLE
# ------------------------------------------------------------
def mean_signal_for_industries(industry_list, signal_col):
    existing = [x for x in industry_list if x in company_change_wide.index]

    if len(existing) == 0:
        return np.nan

    return company_change_wide.loc[existing, signal_col].mean(skipna=True)


degree_lookup = degree_change.set_index("field_group")

comparison_rows = []

for field in sorted(degree_summary["field_group"].dropna().unique()):
    industries = map_field_to_industries(field)

    if len(industries) == 0:
        continue

    if field not in degree_lookup.index:
        continue

    degree_row = degree_lookup.loc[field]

    degree_change_pct = degree_row["change_pct"]
    latest_degree_count = degree_row["last_value"]
    latest_degree_year = degree_row["last_year"]

    job_success_change = mean_signal_for_industries(industries, "job_success")
    job_loss_risk_change = mean_signal_for_industries(industries, "job_loss_risk")
    startup_success_change = mean_signal_for_industries(industries, "startup_success")
    startup_failure_risk_change = mean_signal_for_industries(industries, "startup_failure_risk")

    job_path_score = safe_number(job_success_change) - safe_number(job_loss_risk_change)
    startup_path_score = safe_number(startup_success_change) - safe_number(startup_failure_risk_change)

    path_gap = job_path_score - startup_path_score

    available_signals = sum(
        pd.notna(x)
        for x in [
            job_success_change,
            job_loss_risk_change,
            startup_success_change,
            startup_failure_risk_change,
        ]
    )

    if available_signals == 0:
        better_path = "Not enough company data"
    elif path_gap > 10:
        better_path = "Industry job"
    elif path_gap < -10:
        better_path = "Startup"
    else:
        better_path = "Mixed / close"

    comparison_rows.append({
        "field_of_study": field,
        "mapped_industrial_sectors": "; ".join(industries),
        "degree_change_pct": degree_change_pct,
        "latest_degree_year": latest_degree_year,
        "latest_degree_count": latest_degree_count,
        "job_success_change_pct": job_success_change,
        "job_loss_risk_change_pct": job_loss_risk_change,
        "startup_success_change_pct": startup_success_change,
        "startup_failure_risk_change_pct": startup_failure_risk_change,
        "job_path_score": job_path_score,
        "startup_path_score": startup_path_score,
        "job_minus_startup_score": path_gap,
        "better_path": better_path,
        "available_company_signals": available_signals,
    })

comparison_table = pd.DataFrame(comparison_rows)

comparison_table = comparison_table.sort_values(
    ["latest_degree_count", "field_of_study"],
    ascending=[False, True]
)

print("=" * 90)
print("FINAL CAREER PATH COMPARISON TABLE")
print("=" * 90)
display(comparison_table.head(30))


# ------------------------------------------------------------
# 11. FLOWCHART
# ------------------------------------------------------------
def draw_flowchart():
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 8)
    ax.axis("off")

    def box(x, y, w, h, text):
        patch = FancyBboxPatch(
            (x, y),
            w,
            h,
            boxstyle="round,pad=0.04",
            linewidth=1.5,
            facecolor="white",
            edgecolor="black"
        )
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11, wrap=True)

    def arrow(x1, y1, x2, y2):
        ax.annotate(
            "",
            xy=(x2, y2),
            xytext=(x1, y1),
            arrowprops=dict(arrowstyle="->", linewidth=1.5)
        )

    box(0.5, 5.8, 2.2, 1.0, "Field of study\nIPEDS degree data")
    box(3.4, 5.8, 2.2, 1.0, "Map to\nindustrial sector")
    box(6.3, 6.6, 2.3, 0.9, "Path 1:\nIndustry job")
    box(6.3, 5.0, 2.3, 0.9, "Path 2:\nOpen startup")
    box(9.2, 6.6, 2.3, 0.9, "Job success change\nminus job loss risk")
    box(9.2, 5.0, 2.3, 0.9, "Startup success change\nminus failure risk")
    box(4.4, 2.7, 3.3, 1.0, "Compare scores")
    box(4.4, 1.1, 3.3, 1.0, "Better path:\nIndustry job, startup,\nor mixed")

    arrow(2.7, 6.3, 3.4, 6.3)
    arrow(5.6, 6.3, 6.3, 7.05)
    arrow(5.6, 6.3, 6.3, 5.45)
    arrow(8.6, 7.05, 9.2, 7.05)
    arrow(8.6, 5.45, 9.2, 5.45)
    arrow(10.35, 6.6, 6.1, 3.7)
    arrow(10.35, 5.0, 6.1, 3.7)
    arrow(6.05, 2.7, 6.05, 2.1)

    ax.set_title(
        "Degree-to-Industry Career Path Logic: Job Success vs Startup Success and Failure Risk",
        fontsize=15,
        pad=20
    )

    plt.tight_layout()
    plt.show()


draw_flowchart()


# ------------------------------------------------------------
# 12. CHART 1 - JOB PATH VS STARTUP PATH SCORE
# ------------------------------------------------------------
chart_df = comparison_table.copy()
chart_df = chart_df[chart_df["available_company_signals"] > 0]
chart_df = chart_df.head(TOP_FIELDS_FOR_CHART).copy()

chart_df["field_label"] = chart_df["field_of_study"].apply(short_label)
chart_df = chart_df.sort_values("latest_degree_count", ascending=True)

y = np.arange(len(chart_df))
height = 0.38

plt.figure(figsize=(16, 10))
plt.barh(y - height / 2, chart_df["job_path_score"], height, label="Industry job path score")
plt.barh(y + height / 2, chart_df["startup_path_score"], height, label="Startup path score")

plt.axvline(0, linewidth=1)
plt.yticks(y, chart_df["field_label"])
plt.xlabel("Score: success change minus risk change")
plt.title("Job Path vs Startup Path Score by Field of Study")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 13. CHART 2 - BETTER PATH GAP
# ------------------------------------------------------------
gap_df = chart_df.copy()
gap_df = gap_df.sort_values("job_minus_startup_score", ascending=True)

plt.figure(figsize=(16, 10))
plt.barh(gap_df["field_label"], gap_df["job_minus_startup_score"])
plt.axvline(0, linewidth=1)

plt.xlabel("Job score minus startup score")
plt.title("Better Path Gap: Positive = Industry Job, Negative = Startup")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 14. CHART 3 - STARTUP SUCCESS VS FAILURE RISK
# ------------------------------------------------------------
startup_df = chart_df.copy()

plt.figure(figsize=(14, 10))
plt.scatter(
    startup_df["startup_success_change_pct"].fillna(0),
    startup_df["startup_failure_risk_change_pct"].fillna(0)
)

for _, row in startup_df.iterrows():
    plt.annotate(
        row["field_label"],
        (
            safe_number(row["startup_success_change_pct"]),
            safe_number(row["startup_failure_risk_change_pct"])
        ),
        fontsize=8
    )

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("Startup success change %")
plt.ylabel("Startup failure risk change %")
plt.title("Startup Opportunity vs Startup Failure Risk by Field of Study")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 15. SHOW FINAL TABLE AGAIN AFTER CHARTS
# ------------------------------------------------------------
print("=" * 90)
print("FINAL TABLE AFTER CHARTS")
print("=" * 90)

display(comparison_table.head(30))

print()
print("=" * 90)
print("DONE - NO FILES SAVED")
print("=" * 90)
print("Original files were NOT edited.")
print(f"Total time: {time.time() - start_time:.2f} seconds")

# Degree-to-Industry Career Path Comparison: Industry Job vs Startup Success and Failure Risk - 2

In [ ]:
# ------------------------------------------------------------
# 14. READABLE YEAR CHARTS - NO SAVE
# Shows year on x-axis
# Replaces the old crowded scatter plot
# ------------------------------------------------------------

TOP_FIELDS_FOR_YEAR_CHART = 8
YEAR_START_FOR_CHART = COMPARE_START_YEAR
YEAR_END_FOR_CHART = COMPARE_END_YEAR


def make_company_yearly_change_table(company_signal_summary, start_year=None, end_year=None):
    comp = company_signal_summary.copy()

    comp["year"] = pd.to_numeric(comp["year"], errors="coerce")
    comp["value"] = pd.to_numeric(comp["value"], errors="coerce")
    comp = comp.dropna(subset=["year", "value"])
    comp["year"] = comp["year"].astype(int)
    comp = comp[comp["value"] > 0]

    if start_year is not None:
        comp = comp[comp["year"] >= start_year]

    if end_year is not None:
        comp = comp[comp["year"] <= end_year]

    grouped = (
        comp
        .groupby(["industry_sector", "signal_type", "year"], as_index=False)["value"]
        .sum()
        .sort_values(["industry_sector", "signal_type", "year"])
    )

    base = (
        grouped
        .groupby(["industry_sector", "signal_type"], as_index=False)
        .first()[["industry_sector", "signal_type", "year", "value"]]
        .rename(columns={"year": "base_year", "value": "base_value"})
    )

    out = grouped.merge(base, on=["industry_sector", "signal_type"], how="left")

    out["change_from_start_pct"] = np.where(
        out["base_value"] == 0,
        np.nan,
        ((out["value"] - out["base_value"]) / out["base_value"].abs()) * 100
    )

    out["change_from_start_pct"] = out["change_from_start_pct"].replace(
        [np.inf, -np.inf],
        np.nan
    )

    return out


company_year_change = make_company_yearly_change_table(
    company_signal_summary,
    start_year=YEAR_START_FOR_CHART,
    end_year=YEAR_END_FOR_CHART
)

company_year_wide = (
    company_year_change
    .pivot_table(
        index=["year", "industry_sector"],
        columns="signal_type",
        values="change_from_start_pct",
        aggfunc="mean"
    )
    .reset_index()
)

for col in ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]:
    if col not in company_year_wide.columns:
        company_year_wide[col] = np.nan


# Pick top fields so chart is readable
top_fields = (
    comparison_table[comparison_table["available_company_signals"] > 0]
    .head(TOP_FIELDS_FOR_YEAR_CHART)["field_of_study"]
    .tolist()
)

degree_year = degree_summary.copy()
degree_year = degree_year[degree_year["field_group"].isin(top_fields)]

if YEAR_START_FOR_CHART is not None:
    degree_year = degree_year[degree_year["year"] >= YEAR_START_FOR_CHART]

if YEAR_END_FOR_CHART is not None:
    degree_year = degree_year[degree_year["year"] <= YEAR_END_FOR_CHART]

degree_year = (
    degree_year
    .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
    .sum()
)


# Build year-by-year path table
rows = []

all_years = sorted(company_year_wide["year"].dropna().unique())

for field in top_fields:
    industries = map_field_to_industries(field)

    degree_lookup_year = (
        degree_year[degree_year["field_group"] == field]
        .set_index("year")["degree_count_fixed"]
        .to_dict()
    )

    for year in all_years:
        sector_rows = company_year_wide[
            (company_year_wide["year"] == year) &
            (company_year_wide["industry_sector"].isin(industries))
        ]

        if sector_rows.empty:
            continue

        job_success_change = sector_rows["job_success"].mean(skipna=True)
        job_loss_risk_change = sector_rows["job_loss_risk"].mean(skipna=True)
        startup_success_change = sector_rows["startup_success"].mean(skipna=True)
        startup_failure_risk_change = sector_rows["startup_failure_risk"].mean(skipna=True)

        job_path_score = safe_number(job_success_change) - safe_number(job_loss_risk_change)
        startup_path_score = safe_number(startup_success_change) - safe_number(startup_failure_risk_change)

        gap = job_path_score - startup_path_score

        if gap > 10:
            better_path = "Industry job"
        elif gap < -10:
            better_path = "Startup"
        else:
            better_path = "Mixed / close"

        rows.append({
            "year": year,
            "field_of_study": field,
            "mapped_industrial_sectors": "; ".join(industries),
            "degree_count": degree_lookup_year.get(year, np.nan),
            "job_success_change_pct": job_success_change,
            "job_loss_risk_change_pct": job_loss_risk_change,
            "startup_success_change_pct": startup_success_change,
            "startup_failure_risk_change_pct": startup_failure_risk_change,
            "job_path_score": job_path_score,
            "startup_path_score": startup_path_score,
            "job_minus_startup_score": gap,
            "better_path": better_path
        })

yearly_path_table = pd.DataFrame(rows)


print("=" * 90)
print("YEAR-BY-YEAR PATH TABLE")
print("=" * 90)

display(
    yearly_path_table[
        [
            "year",
            "field_of_study",
            "degree_count",
            "job_path_score",
            "startup_path_score",
            "job_minus_startup_score",
            "better_path"
        ]
    ]
    .sort_values(["field_of_study", "year"])
    .tail(40)
)


# ------------------------------------------------------------
# CHART A - DEGREE COUNT BY YEAR
# ------------------------------------------------------------
plt.figure(figsize=(15, 7))

for field in top_fields:
    one = degree_year[degree_year["field_group"] == field].sort_values("year")

    if one.empty:
        continue

    plt.plot(
        one["year"],
        one["degree_count_fixed"],
        marker="o",
        linewidth=1.5,
        label=short_label(field, 45)
    )

plt.xlabel("Year")
plt.ylabel("Degree count")
plt.title("Degree Count by Year and Field of Study")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# CHART B - ONE READABLE JOB VS STARTUP CHART PER FIELD
# ------------------------------------------------------------
for field in top_fields:
    one = yearly_path_table[
        yearly_path_table["field_of_study"] == field
    ].sort_values("year")

    if one.empty:
        continue

    plt.figure(figsize=(13, 5))

    plt.plot(
        one["year"],
        one["job_path_score"],
        marker="o",
        linewidth=1.8,
        label="Industry job path score"
    )

    plt.plot(
        one["year"],
        one["startup_path_score"],
        marker="o",
        linewidth=1.8,
        label="Startup path score"
    )

    plt.axhline(0, linewidth=1)

    plt.xlabel("Year")
    plt.ylabel("Score: success change % minus risk change %")
    plt.title("Yearly Job Path vs Startup Path\n" + short_label(field, 80))
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# CHART C - BETTER PATH GAP BY YEAR
# Positive = industry job looks better
# Negative = startup looks better
# ------------------------------------------------------------
plt.figure(figsize=(15, 7))

for field in top_fields:
    one = yearly_path_table[
        yearly_path_table["field_of_study"] == field
    ].sort_values("year")

    if one.empty:
        continue

    plt.plot(
        one["year"],
        one["job_minus_startup_score"],
        marker="o",
        linewidth=1.5,
        label=short_label(field, 45)
    )

plt.axhline(0, linewidth=1)

plt.xlabel("Year")
plt.ylabel("Job score minus startup score")
plt.title("Better Path Gap by Year: Positive = Industry Job, Negative = Startup")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# LATEST YEAR SUMMARY TABLE
# ------------------------------------------------------------
latest_year_summary = (
    yearly_path_table
    .sort_values("year")
    .groupby("field_of_study", as_index=False)
    .tail(1)
    .sort_values("degree_count", ascending=False)
)

print("=" * 90)
print("LATEST YEAR SUMMARY")
print("=" * 90)

display(
    latest_year_summary[
        [
            "year",
            "field_of_study",
            "degree_count",
            "job_path_score",
            "startup_path_score",
            "job_minus_startup_score",
            "better_path"
        ]
    ]
)

print("=" * 90)
print("DONE - YEAR CHARTS SHOWN ONLY, NO FILES SAVED")
print("=" * 90)

# Degree-to-Industry Career Path Comparison: Industry Job vs Startup Success and Failure Risk - 3 Diverge

In [ ]:
# ------------------------------------------------------------
# YEAR BAR CHARTS
# Side-by-side bars by year
# Success = positive
# Failure risk = negative
# No save
# ------------------------------------------------------------

TOP_FIELDS_FOR_YEAR_BAR = 8

top_fields = (
    comparison_table[comparison_table["available_company_signals"] > 0]
    .head(TOP_FIELDS_FOR_YEAR_BAR)["field_of_study"]
    .tolist()
)

print("=" * 90)
print("YEAR-BY-YEAR BAR CHART DATA")
print("=" * 90)

display(
    yearly_path_table[
        [
            "year",
            "field_of_study",
            "job_success_change_pct",
            "job_loss_risk_change_pct",
            "startup_success_change_pct",
            "startup_failure_risk_change_pct",
            "better_path"
        ]
    ]
    .sort_values(["field_of_study", "year"])
)


# ------------------------------------------------------------
# ONE BAR CHART PER FIELD
# ------------------------------------------------------------
for field in top_fields:
    one = yearly_path_table[
        yearly_path_table["field_of_study"] == field
    ].sort_values("year").copy()

    if one.empty:
        continue

    x = np.arange(len(one))
    width = 0.36

    years = one["year"].astype(int).astype(str)

    # positive side
    job_success = one["job_success_change_pct"].fillna(0).to_numpy()
    startup_success = one["startup_success_change_pct"].fillna(0).to_numpy()

    # negative side
    job_failure = -one["job_loss_risk_change_pct"].fillna(0).to_numpy()
    startup_failure = -one["startup_failure_risk_change_pct"].fillna(0).to_numpy()

    plt.figure(figsize=(16, 7))

    # JOB BAR = LEFT OF YEAR
    plt.bar(
        x - width / 2,
        job_success,
        width=width,
        label="Job success change %"
    )
    plt.bar(
        x - width / 2,
        job_failure,
        width=width,
        label="Job loss risk change %"
    )

    # STARTUP BAR = RIGHT OF YEAR
    plt.bar(
        x + width / 2,
        startup_success,
        width=width,
        label="Startup success change %"
    )
    plt.bar(
        x + width / 2,
        startup_failure,
        width=width,
        label="Startup failure risk change %"
    )

    plt.axhline(0, linewidth=1.2)

    plt.xticks(x, years, rotation=45)
    plt.xlabel("Year")
    plt.ylabel("Change %")
    plt.title(
        "Success vs Failure by Year\n"
        + short_label(field, 90)
        + "\nLeft bar = Job path | Right bar = Startup path"
    )
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("-" * 90)
    print("TABLE:", field)
    display(
        one[
            [
                "year",
                "job_success_change_pct",
                "job_loss_risk_change_pct",
                "startup_success_change_pct",
                "startup_failure_risk_change_pct",
                "job_path_score",
                "startup_path_score",
                "job_minus_startup_score",
                "better_path"
            ]
        ]
    )


# ------------------------------------------------------------
# SUMMARY BAR CHART FOR LATEST YEAR ONLY
# One chart across fields
# ------------------------------------------------------------
latest_year_summary = (
    yearly_path_table
    .sort_values("year")
    .groupby("field_of_study", as_index=False)
    .tail(1)
    .copy()
)

latest_year_summary = latest_year_summary.sort_values("degree_count", ascending=False)
latest_year_summary["field_label"] = latest_year_summary["field_of_study"].apply(lambda x: short_label(x, 45))

x = np.arange(len(latest_year_summary))
width = 0.36

job_success = latest_year_summary["job_success_change_pct"].fillna(0).to_numpy()
startup_success = latest_year_summary["startup_success_change_pct"].fillna(0).to_numpy()

job_failure = -latest_year_summary["job_loss_risk_change_pct"].fillna(0).to_numpy()
startup_failure = -latest_year_summary["startup_failure_risk_change_pct"].fillna(0).to_numpy()

plt.figure(figsize=(18, 8))

# job = left
plt.bar(x - width / 2, job_success, width=width, label="Job success change %")
plt.bar(x - width / 2, job_failure, width=width, label="Job loss risk change %")

# startup = right
plt.bar(x + width / 2, startup_success, width=width, label="Startup success change %")
plt.bar(x + width / 2, startup_failure, width=width, label="Startup failure risk change %")

plt.axhline(0, linewidth=1.2)
plt.xticks(x, latest_year_summary["field_label"], rotation=45, ha="right")
plt.xlabel("Field of study")
plt.ylabel("Change %")
plt.title("Latest Year: Job vs Startup Success and Failure by Field")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 90)
print("LATEST YEAR SUMMARY TABLE")
print("=" * 90)

display(
    latest_year_summary[
        [
            "year",
            "field_of_study",
            "degree_count",
            "job_success_change_pct",
            "job_loss_risk_change_pct",
            "startup_success_change_pct",
            "startup_failure_risk_change_pct",
            "job_path_score",
            "startup_path_score",
            "job_minus_startup_score",
            "better_path"
        ]
    ]
)

print("=" * 90)
print("DONE - NO FILES SAVED")
print("=" * 90)

# Degree-to-Industry Career Path Comparison: Industry Job vs Startup Success and Failure Risk - 4 Table only and connect flow chart

In [ ]:
# ============================================================
# NO SAVE VERSION
# 1. Field of study -> industry sector connection chart FIRST
# 2. Connection table
# 3. Year success/failure TABLE instead of bar chart
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

TOP_FIELDS_CONNECTION = 12
TOP_FIELDS_TABLE = 12

# ------------------------------------------------------------
# A. BUILD FIELD -> INDUSTRY CONNECTION TABLE
# ------------------------------------------------------------
latest_degree_rows = (
    degree_summary
    .sort_values("year")
    .groupby("field_group", as_index=False)
    .tail(1)
    .rename(columns={
        "field_group": "field_of_study",
        "year": "latest_degree_year",
        "degree_count_fixed": "latest_degree_count"
    })
)

degree_year_range = (
    degree_summary
    .groupby("field_group", as_index=False)
    .agg(
        first_degree_year=("year", "min"),
        last_degree_year=("year", "max")
    )
    .rename(columns={"field_group": "field_of_study"})
)

field_degree_info = latest_degree_rows.merge(
    degree_year_range,
    on="field_of_study",
    how="left"
)

field_degree_info = field_degree_info.sort_values(
    "latest_degree_count",
    ascending=False
)

company_sector_info = (
    company_signal_summary
    .groupby("industry_sector", as_index=False)
    .agg(
        company_first_year=("year", "min"),
        company_last_year=("year", "max"),
        available_signal_types=("signal_type", lambda x: ", ".join(sorted(x.dropna().unique()))),
        company_rows=("value", "count")
    )
)

sector_lookup = company_sector_info.set_index("industry_sector")

connection_rows = []

for _, row in field_degree_info.iterrows():
    field = row["field_of_study"]
    industries = map_field_to_industries(field)

    if len(industries) == 0:
        connection_rows.append({
            "field_of_study": field,
            "connected_industry_sector": "No mapped sector",
            "latest_degree_year": row["latest_degree_year"],
            "latest_degree_count": row["latest_degree_count"],
            "first_degree_year": row["first_degree_year"],
            "last_degree_year": row["last_degree_year"],
            "company_first_year": np.nan,
            "company_last_year": np.nan,
            "available_signal_types": "No company signal",
            "connection_status": "No field-to-industry map"
        })
        continue

    for industry in industries:
        if industry in sector_lookup.index:
            s = sector_lookup.loc[industry]

            connection_rows.append({
                "field_of_study": field,
                "connected_industry_sector": industry,
                "latest_degree_year": row["latest_degree_year"],
                "latest_degree_count": row["latest_degree_count"],
                "first_degree_year": row["first_degree_year"],
                "last_degree_year": row["last_degree_year"],
                "company_first_year": s["company_first_year"],
                "company_last_year": s["company_last_year"],
                "available_signal_types": s["available_signal_types"],
                "connection_status": "Connected with company data"
            })
        else:
            connection_rows.append({
                "field_of_study": field,
                "connected_industry_sector": industry,
                "latest_degree_year": row["latest_degree_year"],
                "latest_degree_count": row["latest_degree_count"],
                "first_degree_year": row["first_degree_year"],
                "last_degree_year": row["last_degree_year"],
                "company_first_year": np.nan,
                "company_last_year": np.nan,
                "available_signal_types": "No company signal",
                "connection_status": "Mapped, but no company data"
            })

connection_table = pd.DataFrame(connection_rows)

connection_table = connection_table.sort_values(
    ["latest_degree_count", "field_of_study", "connected_industry_sector"],
    ascending=[False, True, True]
)

# ------------------------------------------------------------
# B. CONNECTION CHART FIRST
# Field of study on left
# Industry sector on right
# ------------------------------------------------------------
chart_connections = (
    connection_table[
        connection_table["connection_status"] == "Connected with company data"
    ]
    .head(TOP_FIELDS_CONNECTION * 3)
    .copy()
)

top_chart_fields = (
    chart_connections
    .drop_duplicates("field_of_study")
    .head(TOP_FIELDS_CONNECTION)["field_of_study"]
    .tolist()
)

chart_connections = chart_connections[
    chart_connections["field_of_study"].isin(top_chart_fields)
].copy()

fields = chart_connections["field_of_study"].drop_duplicates().tolist()
industries = chart_connections["connected_industry_sector"].drop_duplicates().tolist()

field_y = {field: i for i, field in enumerate(fields)}
industry_y = {industry: i for i, industry in enumerate(industries)}

max_y = max(len(fields), len(industries))

plt.figure(figsize=(18, max(8, max_y * 0.6)))
ax = plt.gca()

# left field labels
for field in fields:
    y = field_y[field]
    ax.scatter(0, y, s=120)
    ax.text(
        -0.03,
        y,
        short_label(field, 50),
        ha="right",
        va="center",
        fontsize=10
    )

# right industry labels
for industry in industries:
    y = industry_y[industry]
    ax.scatter(1, y, s=120)
    ax.text(
        1.03,
        y,
        short_label(industry, 55),
        ha="left",
        va="center",
        fontsize=10
    )

# connection lines
for _, row in chart_connections.iterrows():
    field = row["field_of_study"]
    industry = row["connected_industry_sector"]

    y1 = field_y[field]
    y2 = industry_y[industry]

    degree_count = row["latest_degree_count"]

    if pd.isna(degree_count) or degree_count <= 0:
        line_width = 1
    else:
        line_width = 1 + min(4, np.log10(degree_count + 1) / 2)

    ax.plot(
        [0, 1],
        [y1, y2],
        linewidth=line_width,
        alpha=0.45
    )

ax.set_xlim(-0.55, 1.55)
ax.set_ylim(-1, max_y)
ax.axis("off")

plt.title(
    "Field of Study to Industry Sector Connection Chart\n"
    "Example: Computer and Information Sciences connects to Information",
    fontsize=15
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# C. CONNECTION TABLE
# ------------------------------------------------------------
print("=" * 100)
print("FIELD OF STUDY -> INDUSTRY SECTOR CONNECTION TABLE")
print("=" * 100)

connection_display = connection_table.copy()

for col in ["latest_degree_count"]:
    connection_display[col] = connection_display[col].apply(
        lambda x: "No data" if pd.isna(x) else f"{x:,.0f}"
    )

for col in ["company_first_year", "company_last_year"]:
    connection_display[col] = connection_display[col].apply(
        lambda x: "No data" if pd.isna(x) else int(x)
    )

display(
    connection_display[
        [
            "field_of_study",
            "connected_industry_sector",
            "latest_degree_year",
            "latest_degree_count",
            "first_degree_year",
            "last_degree_year",
            "company_first_year",
            "company_last_year",
            "available_signal_types",
            "connection_status"
        ]
    ]
    .head(50)
)


# ------------------------------------------------------------
# D. WHY NaN HAPPENS TABLE
# This checks which industry sectors have which signals
# ------------------------------------------------------------
signal_matrix = (
    company_signal_summary
    .groupby(["industry_sector", "signal_type"], as_index=False)
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        row_count=("value", "count")
    )
)

signal_matrix["year_range"] = (
    signal_matrix["first_year"].astype(str)
    + "-"
    + signal_matrix["last_year"].astype(str)
)

signal_pivot = (
    signal_matrix
    .pivot(
        index="industry_sector",
        columns="signal_type",
        values="year_range"
    )
    .reset_index()
)

for col in ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]:
    if col not in signal_pivot.columns:
        signal_pivot[col] = "No data"

signal_pivot = signal_pivot[
    [
        "industry_sector",
        "job_success",
        "job_loss_risk",
        "startup_success",
        "startup_failure_risk"
    ]
].fillna("No data")

print("=" * 100)
print("WHY NaN HAPPENS: INDUSTRY SECTOR SIGNAL AVAILABILITY")
print("=" * 100)

display(signal_pivot)


# ------------------------------------------------------------
# E. BUILD YEARLY TABLE WITH NaN SHOWN AS 'No data'
# ------------------------------------------------------------
def make_company_yearly_change_table(company_signal_summary, start_year=None, end_year=None):
    comp = company_signal_summary.copy()

    comp["year"] = pd.to_numeric(comp["year"], errors="coerce")
    comp["value"] = pd.to_numeric(comp["value"], errors="coerce")
    comp = comp.dropna(subset=["year", "value"])
    comp["year"] = comp["year"].astype(int)
    comp = comp[comp["value"] > 0]

    if start_year is not None:
        comp = comp[comp["year"] >= start_year]

    if end_year is not None:
        comp = comp[comp["year"] <= end_year]

    grouped = (
        comp
        .groupby(["industry_sector", "signal_type", "year"], as_index=False)["value"]
        .sum()
        .sort_values(["industry_sector", "signal_type", "year"])
    )

    base = (
        grouped
        .groupby(["industry_sector", "signal_type"], as_index=False)
        .first()[["industry_sector", "signal_type", "year", "value"]]
        .rename(columns={"year": "base_year", "value": "base_value"})
    )

    out = grouped.merge(
        base,
        on=["industry_sector", "signal_type"],
        how="left"
    )

    out["change_from_start_pct"] = np.where(
        out["base_value"] == 0,
        np.nan,
        ((out["value"] - out["base_value"]) / out["base_value"].abs()) * 100
    )

    out["change_from_start_pct"] = out["change_from_start_pct"].replace(
        [np.inf, -np.inf],
        np.nan
    )

    return out


company_year_change = make_company_yearly_change_table(
    company_signal_summary,
    start_year=COMPARE_START_YEAR,
    end_year=COMPARE_END_YEAR
)

company_year_wide = (
    company_year_change
    .pivot_table(
        index=["year", "industry_sector"],
        columns="signal_type",
        values="change_from_start_pct",
        aggfunc="mean"
    )
    .reset_index()
)

for col in ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]:
    if col not in company_year_wide.columns:
        company_year_wide[col] = np.nan


top_fields_for_table = (
    comparison_table[comparison_table["available_company_signals"] > 0]
    .head(TOP_FIELDS_TABLE)["field_of_study"]
    .tolist()
)

degree_year = degree_summary.copy()

degree_year = degree_year[
    degree_year["field_group"].isin(top_fields_for_table)
]

if COMPARE_START_YEAR is not None:
    degree_year = degree_year[degree_year["year"] >= COMPARE_START_YEAR]

if COMPARE_END_YEAR is not None:
    degree_year = degree_year[degree_year["year"] <= COMPARE_END_YEAR]

degree_year = (
    degree_year
    .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
    .sum()
)

rows = []

all_years = sorted(company_year_wide["year"].dropna().unique())

for field in top_fields_for_table:
    industries = map_field_to_industries(field)

    degree_lookup_year = (
        degree_year[degree_year["field_group"] == field]
        .set_index("year")["degree_count_fixed"]
        .to_dict()
    )

    for year in all_years:
        sector_rows = company_year_wide[
            (company_year_wide["year"] == year) &
            (company_year_wide["industry_sector"].isin(industries))
        ]

        if sector_rows.empty:
            rows.append({
                "year": year,
                "field_of_study": field,
                "connected_industry_sectors": "; ".join(industries),
                "degree_count": degree_lookup_year.get(year, np.nan),
                "job_success_change_pct": np.nan,
                "job_loss_risk_change_pct": np.nan,
                "startup_success_change_pct": np.nan,
                "startup_failure_risk_change_pct": np.nan,
                "job_path_score": np.nan,
                "startup_path_score": np.nan,
                "job_minus_startup_score": np.nan,
                "better_path": "No company data for mapped sector",
                "missing_reason": "No matching company sector data for this year"
            })
            continue

        job_success_change = sector_rows["job_success"].mean(skipna=True)
        job_loss_risk_change = sector_rows["job_loss_risk"].mean(skipna=True)
        startup_success_change = sector_rows["startup_success"].mean(skipna=True)
        startup_failure_risk_change = sector_rows["startup_failure_risk"].mean(skipna=True)

        raw_metrics = {
            "job_success_change_pct": job_success_change,
            "job_loss_risk_change_pct": job_loss_risk_change,
            "startup_success_change_pct": startup_success_change,
            "startup_failure_risk_change_pct": startup_failure_risk_change
        }

        missing_cols = [
            name.replace("_change_pct", "").replace("_", " ")
            for name, value in raw_metrics.items()
            if pd.isna(value)
        ]

        if len(missing_cols) == 0:
            missing_reason = "Complete"
        else:
            missing_reason = "Missing: " + ", ".join(missing_cols)

        job_path_score = safe_number(job_success_change) - safe_number(job_loss_risk_change)
        startup_path_score = safe_number(startup_success_change) - safe_number(startup_failure_risk_change)
        gap = job_path_score - startup_path_score

        if len(missing_cols) == 4:
            better_path = "No data"
        elif gap > 10:
            better_path = "Industry job"
        elif gap < -10:
            better_path = "Startup"
        else:
            better_path = "Mixed / close"

        rows.append({
            "year": year,
            "field_of_study": field,
            "connected_industry_sectors": "; ".join(industries),
            "degree_count": degree_lookup_year.get(year, np.nan),
            "job_success_change_pct": job_success_change,
            "job_loss_risk_change_pct": job_loss_risk_change,
            "startup_success_change_pct": startup_success_change,
            "startup_failure_risk_change_pct": startup_failure_risk_change,
            "job_path_score": job_path_score,
            "startup_path_score": startup_path_score,
            "job_minus_startup_score": gap,
            "better_path": better_path,
            "missing_reason": missing_reason
        })

yearly_path_table_clean = pd.DataFrame(rows)

# ------------------------------------------------------------
# F. DISPLAY TABLE ONLY
# No bar chart here
# ------------------------------------------------------------
table_display = yearly_path_table_clean.copy()

percent_cols = [
    "job_success_change_pct",
    "job_loss_risk_change_pct",
    "startup_success_change_pct",
    "startup_failure_risk_change_pct",
    "job_path_score",
    "startup_path_score",
    "job_minus_startup_score"
]

for col in percent_cols:
    table_display[col] = table_display[col].apply(
        lambda x: "No data" if pd.isna(x) else f"{x:,.1f}%"
    )

table_display["degree_count"] = table_display["degree_count"].apply(
    lambda x: "No data" if pd.isna(x) else f"{x:,.0f}"
)

print("=" * 100)
print("YEARLY SUCCESS / FAILURE TABLE")
print("=" * 100)

display(
    table_display[
        [
            "year",
            "field_of_study",
            "connected_industry_sectors",
            "degree_count",
            "job_success_change_pct",
            "job_loss_risk_change_pct",
            "startup_success_change_pct",
            "startup_failure_risk_change_pct",
            "job_path_score",
            "startup_path_score",
            "job_minus_startup_score",
            "better_path",
            "missing_reason"
        ]
    ]
    .sort_values(["field_of_study", "year"])
)

print("=" * 100)
print("DONE - CONNECTION CHART + TABLES ONLY, NO FILES SAVED")
print("=" * 100)

# Degree-to-Industry Career Path Comparison: Industry Job vs Startup Success and Failure Risk - 5 Better chart

In [ ]:
# ============================================================
# FULL NO-SAVE VERSION
# DEGREE + COMPANY DATA
# ALL YEARS FROM WHATEVER THE DATA SHOWS
# ALL FIELDS
# FIXES MISSING DATA AS MUCH AS POSSIBLE
#
# What this does:
# 1. Reads degree CSV with chunks
# 2. Reads company/startup/job CSV with chunks
# 3. Builds field of study -> industry sector connection
# 4. Uses exact industry data first
# 5. If exact signal is missing, uses broader industry fallback
# 6. If still missing, shows "No data" instead of NaN
# 7. Shows charts only, does NOT save files
# ============================================================

from pathlib import Path
import re
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 140)

# ============================================================
# FILE PATHS
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000

# None = show all years from the data
START_YEAR = None
END_YEAR = None

# None = show all fields
MAX_FIELDS_TO_SHOW = None

# Prevent giant side-scroll bars in Jupyter.
# This only caps the chart display length.
# The table still keeps real values.
DISPLAY_CAP = 200

SHOW_CONNECTION_TABLE = True
SHOW_SIGNAL_AVAILABILITY_TABLE = True
SHOW_MISSING_DATA_TABLE = True
SHOW_OVERVIEW_CHART = True
SHOW_TABLE_EACH_FIELD = True

# This removes broken edge-year rows if a source drops to -99.9 because the
# latest year is incomplete. Set False if you want to see even those rows.
DROP_EXTREME_EDGE_ROWS = False


# ============================================================
# BASIC HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def elapsed(start_time):
    return f"{time.time() - start_time:,.1f}s"


def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .replace({
            "No data": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "": np.nan,
            " ": np.nan
        }),
        errors="coerce"
    )


def fmt_num(x):
    if pd.isna(x):
        return "No data"
    return f"{x:,.0f}"


def fmt_pct(x):
    if pd.isna(x):
        return "No data"
    return f"{x:,.1f}%"


def short_label(text, max_len=75):
    text = str(text)
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."


def cap_values(values, cap=DISPLAY_CAP):
    return np.clip(values, -cap, cap)


def safe_mean(values):
    values = [v for v in values if pd.notna(v)]
    if len(values) == 0:
        return np.nan
    return float(np.mean(values))


def safe_score(success_value, risk_value):
    has_success = pd.notna(success_value)
    has_risk = pd.notna(risk_value)

    if not has_success and not has_risk:
        return np.nan

    success_part = 0 if pd.isna(success_value) else success_value
    risk_part = 0 if pd.isna(risk_value) else risk_value

    return success_part - risk_part


def choose_better_path(job_score, startup_score):
    if pd.isna(job_score) and pd.isna(startup_score):
        return "No data"

    if pd.isna(job_score):
        return "Startup only"

    if pd.isna(startup_score):
        return "Industry job only"

    gap = job_score - startup_score

    if gap > 10:
        return "Industry job"
    elif gap < -10:
        return "Startup"
    else:
        return "Mixed / close"


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)


# ============================================================
# INDUSTRY NORMALIZATION
# ============================================================

NAICS_SECTOR_MAP = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31": "Manufacturing",
    "32": "Manufacturing",
    "33": "Manufacturing",
    "42": "Wholesale Trade",
    "44": "Retail Trade",
    "45": "Retail Trade",
    "48": "Transportation and Warehousing",
    "49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services except Public Administration",
    "92": "Public Administration"
}

BROAD_SECTOR_FALLBACKS = {
    "Agriculture, Forestry, Fishing and Hunting": ["Goods-producing", "Total private"],
    "Mining, Quarrying, and Oil and Gas Extraction": ["Goods-producing", "Total private"],
    "Utilities": ["Service-providing", "Total private"],
    "Construction": ["Goods-producing", "Total private"],
    "Manufacturing": ["Goods-producing", "Total private"],
    "Wholesale Trade": ["Service-providing", "Total private"],
    "Retail Trade": ["Service-providing", "Total private"],
    "Transportation and Warehousing": ["Service-providing", "Total private"],
    "Information": ["Service-providing", "Total private"],
    "Finance and Insurance": ["Financial activities", "Service-providing", "Total private"],
    "Real Estate and Rental and Leasing": ["Financial activities", "Service-providing", "Total private"],
    "Professional, Scientific, and Technical Services": ["Service-providing", "Total private"],
    "Management of Companies and Enterprises": ["Service-providing", "Total private"],
    "Administrative and Support and Waste Management": ["Service-providing", "Total private"],
    "Educational Services": ["Education and health services", "Service-providing", "Total private"],
    "Health Care and Social Assistance": ["Education and health services", "Service-providing", "Total private"],
    "Arts, Entertainment, and Recreation": ["Leisure and hospitality", "Service-providing", "Total private"],
    "Accommodation and Food Services": ["Leisure and hospitality", "Service-providing", "Total private"],
    "Other Services except Public Administration": ["Service-providing", "Total private"],
    "Public Administration": ["Service-providing", "Total private"]
}


def normalize_industry_from_code(code):
    if pd.isna(code):
        return None

    text = str(code).upper().strip()

    # Examples: NAICS51, NAICS 51, 51, 511, 541
    text = text.replace("NAICS", "").replace("-", "").replace(" ", "")

    if len(text) < 2:
        return None

    two = text[:2]

    if two in NAICS_SECTOR_MAP:
        return NAICS_SECTOR_MAP[two]

    return None


def normalize_industry_name(name):
    if pd.isna(name):
        return None

    raw = str(name).strip()
    low = raw.lower()

    # If the name itself is like NAICS51
    code_sector = normalize_industry_from_code(raw)
    if code_sector is not None:
        return code_sector

    known_names = [
        "Agriculture, Forestry, Fishing and Hunting",
        "Mining, Quarrying, and Oil and Gas Extraction",
        "Utilities",
        "Construction",
        "Manufacturing",
        "Wholesale Trade",
        "Retail Trade",
        "Transportation and Warehousing",
        "Information",
        "Finance and Insurance",
        "Real Estate and Rental and Leasing",
        "Professional, Scientific, and Technical Services",
        "Management of Companies and Enterprises",
        "Administrative and Support and Waste Management",
        "Educational Services",
        "Health Care and Social Assistance",
        "Arts, Entertainment, and Recreation",
        "Accommodation and Food Services",
        "Other Services except Public Administration",
        "Public Administration",
        "Goods-producing",
        "Service-providing",
        "Total private",
        "Financial activities",
        "Education and health services",
        "Leisure and hospitality"
    ]

    for item in known_names:
        if low == item.lower():
            return item

    if "education and health" in low:
        return "Education and health services"
    if "financial activities" in low:
        return "Financial activities"
    if "leisure and hospitality" in low:
        return "Leisure and hospitality"
    if "goods-producing" in low or "goods producing" in low:
        return "Goods-producing"
    if "service-providing" in low or "service providing" in low:
        return "Service-providing"
    if "total private" in low:
        return "Total private"

    for code, sector in NAICS_SECTOR_MAP.items():
        if sector.lower() in low:
            return sector

    return raw


def get_company_industry_sector(row):
    # Prefer industry_name, then industry_code, then naics_code
    for col in ["industry_name", "industry_code", "naics_code"]:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() != "":
                if col in ["industry_code", "naics_code"]:
                    sector = normalize_industry_from_code(value)
                else:
                    sector = normalize_industry_name(value)

                if sector is not None:
                    return sector

    return "Unknown Industry"


# ============================================================
# METRIC -> SIGNAL CLASSIFICATION
# ============================================================

def classify_signal_types(metric_name, metric_category="", dataset="", source_table=""):
    text = " ".join([
        str(metric_name) if pd.notna(metric_name) else "",
        str(metric_category) if pd.notna(metric_category) else "",
        str(dataset) if pd.notna(dataset) else "",
        str(source_table) if pd.notna(source_table) else ""
    ]).lower()

    signals = set()

    # Bankruptcy / death / closing / failure
    if any(k in text for k in [
        "bankrupt", "death", "deaths", "exit", "exits", "failure", "failures",
        "closing", "closings", "closure", "closures"
    ]):
        signals.add("startup_failure_risk")

    # Business birth / startup / opening / formation
    if any(k in text for k in [
        "birth", "births", "startup", "startups", "business application",
        "applications", "formation", "formations", "opening", "openings",
        "entry", "entries", "survival", "survivor", "survive"
    ]):
        signals.add("startup_success")

    # Job success / gains
    if any(k in text for k in [
        "job gain", "job gains", "gross job gain", "gross job gains",
        "employment gain", "employment gains", "jobs gained", "job creation",
        "jobs created", "employment increase", "employment growth"
    ]):
        signals.add("job_success")

    # Job loss risk / losses
    if any(k in text for k in [
        "job loss", "job losses", "gross job loss", "gross job losses",
        "employment loss", "employment losses", "jobs lost", "employment decline",
        "employment decrease", "layoff", "layoffs"
    ]):
        signals.add("job_loss_risk")

    # If the metric says opening/closing and source is BLS BED/BDM, openings/closings
    # are useful as business movement signals. Do not force them as job signals unless
    # the metric explicitly mentions job/employment.
    if ("opening" in text or "openings" in text) and ("job" in text or "employment" in text):
        signals.add("job_success")

    if ("closing" in text or "closings" in text) and ("job" in text or "employment" in text):
        signals.add("job_loss_risk")

    return sorted(signals)


# ============================================================
# FIELD OF STUDY -> INDUSTRY MAP
# ============================================================

def map_field_to_industries(field):
    if pd.isna(field):
        return []

    f = str(field).upper()

    if "UNKNOWN" in f or "UNMAPPED" in f or "NONSTANDARD" in f or "OTHER / UNKNOWN" in f:
        return []

    industries = []

    def add(items):
        for item in items:
            if item not in industries:
                industries.append(item)

    if "COMPUTER" in f or "INFORMATION SCIENCES" in f or "INFORMATION TECHNOLOGY" in f:
        add(["Information", "Professional, Scientific, and Technical Services"])

    if "ENGINEERING" in f:
        add(["Manufacturing", "Construction", "Professional, Scientific, and Technical Services"])

    if "BUSINESS" in f or "MANAGEMENT" in f or "MARKETING" in f:
        add(["Finance and Insurance", "Management of Companies and Enterprises", "Retail Trade", "Wholesale Trade"])

    if "HEALTH" in f or "CLINICAL" in f or "NURSING" in f or "MEDICAL" in f:
        add(["Health Care and Social Assistance"])

    if "BIOLOGICAL" in f or "BIOMEDICAL" in f:
        add(["Health Care and Social Assistance", "Professional, Scientific, and Technical Services"])

    if "EDUCATION" in f:
        add(["Educational Services"])

    if "VISUAL" in f or "PERFORMING" in f or "ARTS" in f:
        add(["Arts, Entertainment, and Recreation", "Information"])

    if "COMMUNICATION" in f or "JOURNALISM" in f:
        add(["Information"])

    if "CULINARY" in f or "PERSONAL" in f or "ENTERTAINMENT" in f:
        add(["Accommodation and Food Services", "Other Services except Public Administration"])

    if "MECHANIC" in f or "REPAIR" in f or "TECHNICIAN" in f:
        add(["Manufacturing", "Transportation and Warehousing"])

    if "HOMELAND" in f or "SECURITY" in f or "LAW ENFORCEMENT" in f or "FIREFIGHTING" in f or "PROTECTIVE" in f:
        add(["Public Administration", "Administrative and Support and Waste Management"])

    if "AGRICULTURE" in f or "NATURAL RESOURCES" in f or "CONSERVATION" in f:
        add(["Agriculture, Forestry, Fishing and Hunting"])

    if "ARCHITECTURE" in f:
        add(["Construction", "Professional, Scientific, and Technical Services"])

    if "LEGAL" in f or "LAW" in f:
        add(["Professional, Scientific, and Technical Services", "Public Administration"])

    if "MATHEMATICS" in f or "STATISTICS" in f:
        add(["Professional, Scientific, and Technical Services", "Finance and Insurance", "Information"])

    if "PHYSICAL SCIENCES" in f or "SCIENCE TECHNOLOGIES" in f:
        add(["Professional, Scientific, and Technical Services"])

    if "PSYCHOLOGY" in f:
        add(["Health Care and Social Assistance", "Professional, Scientific, and Technical Services"])

    if "SOCIAL SCIENCES" in f:
        add(["Public Administration", "Professional, Scientific, and Technical Services"])

    if "PUBLIC ADMINISTRATION" in f or "SOCIAL SERVICE" in f:
        add(["Public Administration", "Health Care and Social Assistance"])

    if "TRANSPORTATION" in f:
        add(["Transportation and Warehousing"])

    if "CONSTRUCTION TRADES" in f:
        add(["Construction"])

    if "PRECISION PRODUCTION" in f:
        add(["Manufacturing"])

    if "FAMILY" in f or "CONSUMER" in f:
        add(["Health Care and Social Assistance", "Educational Services", "Other Services except Public Administration"])

    if "PARKS" in f or "RECREATION" in f or "FITNESS" in f:
        add(["Arts, Entertainment, and Recreation", "Health Care and Social Assistance"])

    return industries


# ============================================================
# READ DEGREE DATA WITH CHUNKS
# ============================================================

def read_degree_summary(path, chunksize=100_000):
    t0 = time.time()

    need_cols = ["year", "degree_count", "field_group"]

    header = pd.read_csv(path, nrows=0)
    existing = list(header.columns)

    missing = [c for c in need_cols if c not in existing]
    if missing:
        raise ValueError(f"Degree file missing required columns: {missing}")

    pieces = []
    total_rows = 0

    print("=" * 100)
    print("READING DEGREE DATA")
    print("=" * 100)

    for i, chunk in enumerate(pd.read_csv(path, usecols=need_cols, chunksize=chunksize, low_memory=False), start=1):
        total_rows += len(chunk)

        chunk["year"] = clean_number(chunk["year"])
        chunk["degree_count"] = clean_number(chunk["degree_count"])
        chunk["field_group"] = chunk["field_group"].astype(str).str.strip()

        chunk = chunk.dropna(subset=["year", "field_group"])
        chunk["year"] = chunk["year"].astype(int)
        chunk["degree_count"] = chunk["degree_count"].fillna(0)

        if START_YEAR is not None:
            chunk = chunk[chunk["year"] >= START_YEAR]

        if END_YEAR is not None:
            chunk = chunk[chunk["year"] <= END_YEAR]

        grouped = (
            chunk
            .groupby(["year", "field_group"], as_index=False)["degree_count"]
            .sum()
            .rename(columns={"degree_count": "degree_count_fixed"})
        )

        pieces.append(grouped)

        if i == 1 or i % 10 == 0:
            print(f"[{elapsed(t0)}] degree chunks read: {i:,} | rows checked: {total_rows:,}")

    out = (
        pd.concat(pieces, ignore_index=True)
        .groupby(["year", "field_group"], as_index=False)["degree_count_fixed"]
        .sum()
    )

    print(f"[{elapsed(t0)}] degree rows checked: {total_rows:,}")
    print("Degree year range:", out["year"].min(), "to", out["year"].max())
    print("Degree fields:", out["field_group"].nunique())

    return out


degree_summary = read_degree_summary(DEGREE_FILE, CHUNKSIZE)


# ============================================================
# READ COMPANY DATA WITH CHUNKS
# ============================================================

def read_company_signal_summary(path, chunksize=100_000):
    t0 = time.time()

    header = pd.read_csv(path, nrows=0)
    existing = list(header.columns)

    possible_cols = [
        "source",
        "dataset",
        "source_table",
        "year",
        "industry_name",
        "industry_code",
        "naics_code",
        "metric_name",
        "metric_category",
        "value",
        "unit"
    ]

    usecols = [c for c in possible_cols if c in existing]

    required = ["year", "value", "metric_name"]
    missing = [c for c in required if c not in usecols]

    if missing:
        raise ValueError(f"Company file missing required columns: {missing}")

    pieces = []
    unclassified_examples = []
    total_rows = 0
    total_signal_rows = 0

    print("=" * 100)
    print("READING COMPANY / STARTUP / JOB DATA")
    print("=" * 100)

    for i, chunk in enumerate(pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False), start=1):
        total_rows += len(chunk)

        chunk["year"] = clean_number(chunk["year"])
        chunk["value"] = clean_number(chunk["value"])

        chunk = chunk.dropna(subset=["year", "value"])
        chunk["year"] = chunk["year"].astype(int)

        if START_YEAR is not None:
            chunk = chunk[chunk["year"] >= START_YEAR]

        if END_YEAR is not None:
            chunk = chunk[chunk["year"] <= END_YEAR]

        chunk = chunk[chunk["value"] > 0].copy()

        if chunk.empty:
            continue

        chunk["industry_sector"] = chunk.apply(get_company_industry_sector, axis=1)

        if "metric_category" not in chunk.columns:
            chunk["metric_category"] = ""

        if "dataset" not in chunk.columns:
            chunk["dataset"] = ""

        if "source_table" not in chunk.columns:
            chunk["source_table"] = ""

        chunk["signal_type_list"] = chunk.apply(
            lambda r: classify_signal_types(
                r.get("metric_name", ""),
                r.get("metric_category", ""),
                r.get("dataset", ""),
                r.get("source_table", "")
            ),
            axis=1
        )

        bad = chunk[chunk["signal_type_list"].apply(len) == 0]
        if len(bad) > 0 and len(unclassified_examples) < 20:
            sample = bad[["metric_name", "metric_category", "dataset"]].drop_duplicates().head(20)
            unclassified_examples.append(sample)

        chunk = chunk[chunk["signal_type_list"].apply(len) > 0].copy()

        if chunk.empty:
            continue

        chunk = chunk.explode("signal_type_list")
        chunk = chunk.rename(columns={"signal_type_list": "signal_type"})

        total_signal_rows += len(chunk)

        grouped = (
            chunk
            .groupby(["year", "industry_sector", "signal_type"], as_index=False)["value"]
            .sum()
        )

        pieces.append(grouped)

        if i == 1 or i % 10 == 0:
            print(
                f"[{elapsed(t0)}] company chunks read: {i:,} | "
                f"rows checked: {total_rows:,} | signal rows: {total_signal_rows:,}"
            )

    if len(pieces) == 0:
        print("No company signal rows were classified.")
        if len(unclassified_examples) > 0:
            print("Unclassified metric examples:")
            display(pd.concat(unclassified_examples, ignore_index=True).drop_duplicates().head(50))
        raise ValueError("No usable company signal rows found.")

    out = (
        pd.concat(pieces, ignore_index=True)
        .groupby(["year", "industry_sector", "signal_type"], as_index=False)["value"]
        .sum()
    )

    print(f"[{elapsed(t0)}] company rows checked: {total_rows:,}")
    print(f"[{elapsed(t0)}] company signal rows after classification: {total_signal_rows:,}")
    print("Company year range:", out["year"].min(), "to", out["year"].max())
    print("Company sectors:", out["industry_sector"].nunique())
    print("Signal types:", sorted(out["signal_type"].dropna().unique()))

    if len(unclassified_examples) > 0:
        print("=" * 100)
        print("NOTE: Some metric names were not used because they did not match success/failure signals.")
        print("Showing examples only:")
        print("=" * 100)
        display(pd.concat(unclassified_examples, ignore_index=True).drop_duplicates().head(30))

    return out


company_signal_summary = read_company_signal_summary(COMPANY_FILE, CHUNKSIZE)


# ============================================================
# COMPANY CHANGE FROM FIRST AVAILABLE YEAR
# ============================================================

def make_company_yearly_change_table(company_signal_summary):
    grouped = (
        company_signal_summary
        .groupby(["industry_sector", "signal_type", "year"], as_index=False)["value"]
        .sum()
        .sort_values(["industry_sector", "signal_type", "year"])
    )

    base = (
        grouped
        .groupby(["industry_sector", "signal_type"], as_index=False)
        .first()[["industry_sector", "signal_type", "year", "value"]]
        .rename(columns={"year": "base_year", "value": "base_value"})
    )

    out = grouped.merge(
        base,
        on=["industry_sector", "signal_type"],
        how="left"
    )

    out["change_from_start_pct"] = np.where(
        out["base_value"] == 0,
        np.nan,
        ((out["value"] - out["base_value"]) / out["base_value"].abs()) * 100
    )

    out["change_from_start_pct"] = out["change_from_start_pct"].replace([np.inf, -np.inf], np.nan)

    return out


company_year_change = make_company_yearly_change_table(company_signal_summary)

company_year_wide = (
    company_year_change
    .pivot_table(
        index=["year", "industry_sector"],
        columns="signal_type",
        values="change_from_start_pct",
        aggfunc="mean"
    )
    .reset_index()
)

for col in ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]:
    if col not in company_year_wide.columns:
        company_year_wide[col] = np.nan

company_year_wide = company_year_wide[
    [
        "year",
        "industry_sector",
        "job_success",
        "job_loss_risk",
        "startup_success",
        "startup_failure_risk"
    ]
].copy()


# ============================================================
# SIGNAL AVAILABILITY TABLE
# ============================================================

signal_availability = (
    company_signal_summary
    .groupby(["industry_sector", "signal_type"], as_index=False)
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        rows=("value", "count")
    )
)

signal_availability["year_range"] = (
    signal_availability["first_year"].astype(str)
    + "-"
    + signal_availability["last_year"].astype(str)
)

signal_pivot = (
    signal_availability
    .pivot_table(
        index="industry_sector",
        columns="signal_type",
        values="year_range",
        aggfunc="first"
    )
    .reset_index()
)

for col in ["job_success", "job_loss_risk", "startup_success", "startup_failure_risk"]:
    if col not in signal_pivot.columns:
        signal_pivot[col] = "No data"

signal_pivot = signal_pivot[
    [
        "industry_sector",
        "job_success",
        "job_loss_risk",
        "startup_success",
        "startup_failure_risk"
    ]
].fillna("No data")

if SHOW_SIGNAL_AVAILABILITY_TABLE:
    print("=" * 100)
    print("SIGNAL AVAILABILITY TABLE")
    print("This explains why some values are missing.")
    print("=" * 100)
    display(signal_pivot)


# ============================================================
# FIELD -> INDUSTRY CONNECTION TABLE
# ============================================================

latest_degree = (
    degree_summary
    .sort_values("year")
    .groupby("field_group", as_index=False)
    .tail(1)
    .rename(columns={
        "field_group": "field_of_study",
        "year": "latest_degree_year",
        "degree_count_fixed": "latest_degree_count"
    })
)

degree_year_range = (
    degree_summary
    .groupby("field_group", as_index=False)
    .agg(
        first_degree_year=("year", "min"),
        last_degree_year=("year", "max")
    )
    .rename(columns={"field_group": "field_of_study"})
)

field_degree_info = latest_degree.merge(
    degree_year_range,
    on="field_of_study",
    how="left"
)

field_degree_info = field_degree_info.sort_values("latest_degree_count", ascending=False)

company_sector_info = (
    company_signal_summary
    .groupby("industry_sector", as_index=False)
    .agg(
        company_first_year=("year", "min"),
        company_last_year=("year", "max"),
        available_signal_types=("signal_type", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
)

sector_lookup = company_sector_info.set_index("industry_sector")

connection_rows = []

for _, row in field_degree_info.iterrows():
    field = row["field_of_study"]
    industries = map_field_to_industries(field)

    if len(industries) == 0:
        connection_rows.append({
            "field_of_study": field,
            "connected_industry_sector": "No mapped sector",
            "latest_degree_year": row["latest_degree_year"],
            "latest_degree_count": row["latest_degree_count"],
            "first_degree_year": row["first_degree_year"],
            "last_degree_year": row["last_degree_year"],
            "company_first_year": np.nan,
            "company_last_year": np.nan,
            "available_signal_types": "No company signal",
            "connection_status": "No field-to-industry map"
        })
    else:
        for industry in industries:
            if industry in sector_lookup.index:
                s = sector_lookup.loc[industry]

                connection_rows.append({
                    "field_of_study": field,
                    "connected_industry_sector": industry,
                    "latest_degree_year": row["latest_degree_year"],
                    "latest_degree_count": row["latest_degree_count"],
                    "first_degree_year": row["first_degree_year"],
                    "last_degree_year": row["last_degree_year"],
                    "company_first_year": s["company_first_year"],
                    "company_last_year": s["company_last_year"],
                    "available_signal_types": s["available_signal_types"],
                    "connection_status": "Connected with exact company sector"
                })
            else:
                fallback = BROAD_SECTOR_FALLBACKS.get(industry, [])
                fallback_available = [x for x in fallback if x in sector_lookup.index]

                if len(fallback_available) > 0:
                    firsts = []
                    lasts = []
                    signals = []

                    for fb in fallback_available:
                        s = sector_lookup.loc[fb]
                        firsts.append(s["company_first_year"])
                        lasts.append(s["company_last_year"])
                        signals.append(s["available_signal_types"])

                    connection_rows.append({
                        "field_of_study": field,
                        "connected_industry_sector": industry,
                        "latest_degree_year": row["latest_degree_year"],
                        "latest_degree_count": row["latest_degree_count"],
                        "first_degree_year": row["first_degree_year"],
                        "last_degree_year": row["last_degree_year"],
                        "company_first_year": min(firsts),
                        "company_last_year": max(lasts),
                        "available_signal_types": ", ".join(sorted(set(", ".join(signals).split(", ")))),
                        "connection_status": "Filled by broader sector fallback"
                    })
                else:
                    connection_rows.append({
                        "field_of_study": field,
                        "connected_industry_sector": industry,
                        "latest_degree_year": row["latest_degree_year"],
                        "latest_degree_count": row["latest_degree_count"],
                        "first_degree_year": row["first_degree_year"],
                        "last_degree_year": row["last_degree_year"],
                        "company_first_year": np.nan,
                        "company_last_year": np.nan,
                        "available_signal_types": "No company signal",
                        "connection_status": "Mapped, but no company data"
                    })

connection_table = pd.DataFrame(connection_rows)

connection_table = connection_table.sort_values(
    ["latest_degree_count", "field_of_study", "connected_industry_sector"],
    ascending=[False, True, True]
)

if SHOW_CONNECTION_TABLE:
    connection_display = connection_table.copy()

    connection_display["latest_degree_count"] = connection_display["latest_degree_count"].apply(fmt_num)
    connection_display["company_first_year"] = connection_display["company_first_year"].apply(fmt_num)
    connection_display["company_last_year"] = connection_display["company_last_year"].apply(fmt_num)

    print("=" * 100)
    print("FIELD OF STUDY -> INDUSTRY SECTOR CONNECTION TABLE")
    print("=" * 100)
    display(connection_display)


# ============================================================
# EXACT + FALLBACK SIGNAL LOOKUP
# ============================================================

wide_lookup = company_year_wide.set_index(["year", "industry_sector"])

def lookup_signal_for_industries(year, industries, signal):
    exact_values = []

    for industry in industries:
        key = (year, industry)
        if key in wide_lookup.index:
            value = wide_lookup.loc[key, signal]
            if isinstance(value, pd.Series):
                value = value.mean(skipna=True)
            if pd.notna(value):
                exact_values.append(value)

    if len(exact_values) > 0:
        return safe_mean(exact_values), "Exact sector"

    fallback_sectors = []

    for industry in industries:
        for fb in BROAD_SECTOR_FALLBACKS.get(industry, []):
            if fb not in fallback_sectors:
                fallback_sectors.append(fb)

    fallback_values = []

    for fb in fallback_sectors:
        key = (year, fb)
        if key in wide_lookup.index:
            value = wide_lookup.loc[key, signal]
            if isinstance(value, pd.Series):
                value = value.mean(skipna=True)
            if pd.notna(value):
                fallback_values.append(value)

    if len(fallback_values) > 0:
        return safe_mean(fallback_values), "Broader sector fallback"

    return np.nan, "No data"


# ============================================================
# BUILD YEARLY PATH TABLE WITH ALL YEARS
# ============================================================

degree_fields = sorted(degree_summary["field_group"].dropna().unique())

yearly_rows = []

print("=" * 100)
print("BUILDING YEARLY PATH TABLE")
print("Using all years found for each field from degree data + company data")
print("=" * 100)

for field_i, field in enumerate(degree_fields, start=1):
    industries = map_field_to_industries(field)

    field_degree = degree_summary[degree_summary["field_group"] == field].copy()

    degree_lookup = (
        field_degree
        .set_index("year")["degree_count_fixed"]
        .to_dict()
    )

    degree_years = set(field_degree["year"].dropna().astype(int).unique())

    company_years = set()

    if len(industries) > 0:
        exact_years = company_year_wide[
            company_year_wide["industry_sector"].isin(industries)
        ]["year"].dropna().astype(int).unique()

        company_years.update(exact_years)

        fallback_sectors = []

        for industry in industries:
            for fb in BROAD_SECTOR_FALLBACKS.get(industry, []):
                if fb not in fallback_sectors:
                    fallback_sectors.append(fb)

        fallback_years = company_year_wide[
            company_year_wide["industry_sector"].isin(fallback_sectors)
        ]["year"].dropna().astype(int).unique()

        company_years.update(fallback_years)

    all_years_for_field = sorted(degree_years.union(company_years))

    if START_YEAR is not None:
        all_years_for_field = [y for y in all_years_for_field if y >= START_YEAR]

    if END_YEAR is not None:
        all_years_for_field = [y for y in all_years_for_field if y <= END_YEAR]

    if len(all_years_for_field) == 0:
        continue

    for year in all_years_for_field:
        degree_count = degree_lookup.get(year, np.nan)

        if len(industries) == 0:
            yearly_rows.append({
                "year": year,
                "field_of_study": field,
                "connected_industry_sectors": "No mapped sector",
                "degree_count": degree_count,
                "job_success_change_pct": np.nan,
                "job_loss_risk_change_pct": np.nan,
                "startup_success_change_pct": np.nan,
                "startup_failure_risk_change_pct": np.nan,
                "job_path_score": np.nan,
                "startup_path_score": np.nan,
                "job_minus_startup_score": np.nan,
                "better_path": "No data",
                "missing_reason": "No field-to-industry map",
                "data_quality": "Degree only / no industry map"
            })
            continue

        job_success, job_success_source = lookup_signal_for_industries(year, industries, "job_success")
        job_loss, job_loss_source = lookup_signal_for_industries(year, industries, "job_loss_risk")
        startup_success, startup_success_source = lookup_signal_for_industries(year, industries, "startup_success")
        startup_failure, startup_failure_source = lookup_signal_for_industries(year, industries, "startup_failure_risk")

        signal_values = {
            "job_success": job_success,
            "job_loss_risk": job_loss,
            "startup_success": startup_success,
            "startup_failure_risk": startup_failure
        }

        signal_sources = {
            "job_success": job_success_source,
            "job_loss_risk": job_loss_source,
            "startup_success": startup_success_source,
            "startup_failure_risk": startup_failure_source
        }

        missing_signals = [k for k, v in signal_values.items() if pd.isna(v)]
        fallback_signals = [k for k, v in signal_sources.items() if v == "Broader sector fallback"]

        if len(missing_signals) == 0 and len(fallback_signals) == 0:
            missing_reason = "Complete exact sector data"
            data_quality = "Complete exact"
        elif len(missing_signals) == 0 and len(fallback_signals) > 0:
            missing_reason = "Filled by fallback: " + ", ".join(fallback_signals)
            data_quality = "Complete with fallback"
        elif len(missing_signals) < 4:
            missing_reason = "Missing after fallback: " + ", ".join(missing_signals)
            if len(fallback_signals) > 0:
                missing_reason += " | Filled by fallback: " + ", ".join(fallback_signals)
            data_quality = "Partial"
        else:
            missing_reason = "No company signal data for this field-year"
            data_quality = "Degree only / no company signal"

        job_path_score = safe_score(job_success, job_loss)
        startup_path_score = safe_score(startup_success, startup_failure)

        if pd.notna(job_path_score) and pd.notna(startup_path_score):
            gap = job_path_score - startup_path_score
        else:
            gap = np.nan

        better_path = choose_better_path(job_path_score, startup_path_score)

        yearly_rows.append({
            "year": year,
            "field_of_study": field,
            "connected_industry_sectors": "; ".join(industries),
            "degree_count": degree_count,
            "job_success_change_pct": job_success,
            "job_loss_risk_change_pct": job_loss,
            "startup_success_change_pct": startup_success,
            "startup_failure_risk_change_pct": startup_failure,
            "job_path_score": job_path_score,
            "startup_path_score": startup_path_score,
            "job_minus_startup_score": gap,
            "better_path": better_path,
            "missing_reason": missing_reason,
            "data_quality": data_quality
        })

    if field_i == 1 or field_i % 10 == 0:
        print(f"Processed fields: {field_i:,} / {len(degree_fields):,}")

yearly_path_table_clean = pd.DataFrame(yearly_rows)

if DROP_EXTREME_EDGE_ROWS:
    extreme_mask = (
        yearly_path_table_clean["job_success_change_pct"].le(-90).fillna(False) |
        yearly_path_table_clean["startup_success_change_pct"].le(-90).fillna(False)
    )

    print("Removed extreme edge rows:", int(extreme_mask.sum()))
    yearly_path_table_clean = yearly_path_table_clean.loc[~extreme_mask].copy()

print("=" * 100)
print("DONE BUILDING YEARLY PATH TABLE")
print("First year:", yearly_path_table_clean["year"].min())
print("Last year:", yearly_path_table_clean["year"].max())
print("Rows:", len(yearly_path_table_clean))
print("=" * 100)


# ============================================================
# MISSING DATA CHECK TABLE
# ============================================================

if SHOW_MISSING_DATA_TABLE:
    missing_summary = (
        yearly_path_table_clean
        .groupby("field_of_study", as_index=False)
        .agg(
            first_year=("year", "min"),
            last_year=("year", "max"),
            years_total=("year", "nunique"),
            degree_rows_missing=("degree_count", lambda x: x.isna().sum()),
            job_success_missing=("job_success_change_pct", lambda x: x.isna().sum()),
            job_loss_missing=("job_loss_risk_change_pct", lambda x: x.isna().sum()),
            startup_success_missing=("startup_success_change_pct", lambda x: x.isna().sum()),
            startup_failure_missing=("startup_failure_risk_change_pct", lambda x: x.isna().sum()),
            complete_exact_rows=("data_quality", lambda x: (x == "Complete exact").sum()),
            fallback_rows=("data_quality", lambda x: (x == "Complete with fallback").sum()),
            partial_rows=("data_quality", lambda x: (x == "Partial").sum()),
            degree_only_rows=("data_quality", lambda x: x.astype(str).str.contains("Degree only", na=False).sum())
        )
    )

    print("=" * 100)
    print("MISSING DATA CHECK BY FIELD")
    print("Missing is fixed where possible by broader-sector fallback.")
    print("Remaining missing means the source file does not contain that signal.")
    print("=" * 100)
    display(missing_summary)


# ============================================================
# ADD CHART HELPER COLUMNS
# ============================================================

def add_chart_columns(df):
    out = df.copy()

    js = out["job_success_change_pct"]
    jl = out["job_loss_risk_change_pct"]
    ss = out["startup_success_change_pct"]
    sf = out["startup_failure_risk_change_pct"]

    out["job_good_strength"] = js.fillna(0).clip(lower=0) + (-jl.fillna(0)).clip(lower=0)
    out["startup_good_strength"] = ss.fillna(0).clip(lower=0) + (-sf.fillna(0)).clip(lower=0)

    out["job_bad_pressure"] = jl.fillna(0).clip(lower=0) + (-js.fillna(0)).clip(lower=0)
    out["startup_bad_pressure"] = sf.fillna(0).clip(lower=0) + (-ss.fillna(0)).clip(lower=0)

    return out


chart_df = add_chart_columns(yearly_path_table_clean)

usable_fields = sorted(chart_df["field_of_study"].dropna().unique())

if MAX_FIELDS_TO_SHOW is not None:
    usable_fields = usable_fields[:MAX_FIELDS_TO_SHOW]

print("=" * 100)
print("TOTAL FIELDS TO DISPLAY:", len(usable_fields))
print("NO FILES SAVED")
print("=" * 100)


# ============================================================
# OVERVIEW CHART + TABLE
# ============================================================

overview_rows = []

for field in usable_fields:
    temp = chart_df[chart_df["field_of_study"] == field].copy()

    avg_job = temp["job_path_score"].mean(skipna=True)
    avg_startup = temp["startup_path_score"].mean(skipna=True)

    if pd.notna(avg_job) and pd.notna(avg_startup):
        avg_gap = avg_job - avg_startup
    else:
        avg_gap = np.nan

    overall = choose_better_path(avg_job, avg_startup)

    overview_rows.append({
        "field_of_study": field,
        "first_year": temp["year"].min(),
        "last_year": temp["year"].max(),
        "years_used": temp["year"].nunique(),
        "avg_job_path_score": avg_job,
        "avg_startup_path_score": avg_startup,
        "avg_job_minus_startup_score": avg_gap,
        "overall_better_path": overall
    })

overview_df = pd.DataFrame(overview_rows)

overview_df = overview_df.sort_values(
    "avg_job_minus_startup_score",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

if SHOW_OVERVIEW_CHART:
    plt.figure(figsize=(13, max(7, len(overview_df) * 0.38)))
    ax = plt.gca()

    y = np.arange(len(overview_df))

    startup_left = -overview_df["avg_startup_path_score"].fillna(0).abs().to_numpy()
    job_right = overview_df["avg_job_path_score"].fillna(0).abs().to_numpy()

    ax.barh(y, cap_values(startup_left), label="Startup path score")
    ax.barh(y, cap_values(job_right), label="Industry job path score")

    ax.axvline(0, linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels([short_label(x, 65) for x in overview_df["field_of_study"]])
    ax.invert_yaxis()

    ax.set_xlim(-DISPLAY_CAP, DISPLAY_CAP)
    ax.set_xlabel("Average path score")
    ax.set_ylabel("Field of study")
    ax.set_title(
        "All Fields Overview\n"
        "Left = startup path score | Right = industry job path score"
    )

    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()
    plt.close()

overview_display = overview_df.copy()

for col in ["avg_job_path_score", "avg_startup_path_score", "avg_job_minus_startup_score"]:
    overview_display[col] = overview_display[col].apply(fmt_pct)

print("=" * 100)
print("OVERVIEW TABLE")
print("=" * 100)
display(overview_display)


# ============================================================
# ALL FIELD CHARTS
# ============================================================

for field in overview_df["field_of_study"].tolist():

    field_df = chart_df[chart_df["field_of_study"] == field].copy()
    field_df = field_df.sort_values("year").reset_index(drop=True)

    if field_df.empty:
        continue

    print("\n" + "=" * 100)
    print("FIELD:", field)
    print("YEARS:", field_df["year"].min(), "to", field_df["year"].max())
    print("=" * 100)

    y = np.arange(len(field_df))

    # --------------------------------------------------------
    # CHART 1: SUCCESS VS RISK
    # --------------------------------------------------------
    plt.figure(figsize=(13, max(7, len(field_df) * 0.28)))
    ax = plt.gca()

    left_base = np.zeros(len(field_df))
    right_base = np.zeros(len(field_df))

    left_stack = [
        ("job_bad_pressure", "Job bad pressure"),
        ("startup_bad_pressure", "Startup bad pressure")
    ]

    right_stack = [
        ("job_good_strength", "Job good strength"),
        ("startup_good_strength", "Startup good strength")
    ]

    for col, label in left_stack:
        values = -field_df[col].fillna(0).to_numpy()
        values = cap_values(values)
        ax.barh(y, values, left=left_base, label=label)
        left_base += values

    for col, label in right_stack:
        values = field_df[col].fillna(0).to_numpy()
        values = cap_values(values)
        ax.barh(y, values, left=right_base, label=label)
        right_base += values

    ax.axvline(0, linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(field_df["year"].astype(str))
    ax.invert_yaxis()

    ax.set_xlim(-DISPLAY_CAP, DISPLAY_CAP)
    ax.set_xlabel("Percent change from first available signal year")
    ax.set_ylabel("Year")
    ax.set_title(
        short_label(field, 85) +
        "\nLeft = bad pressure / risk | Right = good success strength"
    )

    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()
    plt.close()

    # --------------------------------------------------------
    # CHART 2: STARTUP VS INDUSTRY JOB PATH SCORE
    # --------------------------------------------------------
    plt.figure(figsize=(13, max(7, len(field_df) * 0.28)))
    ax = plt.gca()

    startup_actual = field_df["startup_path_score"].fillna(0).to_numpy()
    job_actual = field_df["job_path_score"].fillna(0).to_numpy()

    startup_left = -np.abs(startup_actual)
    job_right = np.abs(job_actual)

    startup_left_plot = cap_values(startup_left)
    job_right_plot = cap_values(job_right)

    ax.barh(y, startup_left_plot, label="Startup path score")
    ax.barh(y, job_right_plot, label="Industry job path score")

    ax.axvline(0, linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(field_df["year"].astype(str))
    ax.invert_yaxis()

    ax.set_xlim(-DISPLAY_CAP, DISPLAY_CAP)
    ax.set_xlabel("Path score")
    ax.set_ylabel("Year")
    ax.set_title(
        short_label(field, 85) +
        "\nLeft = startup path score | Right = industry job path score"
    )

    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()
    plt.close()

    # --------------------------------------------------------
    # FIELD TABLE
    # --------------------------------------------------------
    if SHOW_TABLE_EACH_FIELD:
        table_view = field_df[
            [
                "year",
                "connected_industry_sectors",
                "degree_count",
                "job_success_change_pct",
                "job_loss_risk_change_pct",
                "startup_success_change_pct",
                "startup_failure_risk_change_pct",
                "job_path_score",
                "startup_path_score",
                "job_minus_startup_score",
                "better_path",
                "data_quality",
                "missing_reason"
            ]
        ].copy()

        table_view["degree_count"] = table_view["degree_count"].apply(fmt_num)

        percent_cols = [
            "job_success_change_pct",
            "job_loss_risk_change_pct",
            "startup_success_change_pct",
            "startup_failure_risk_change_pct",
            "job_path_score",
            "startup_path_score",
            "job_minus_startup_score"
        ]

        for col in percent_cols:
            table_view[col] = table_view[col].apply(fmt_pct)

        display(table_view)

print("\n" + "=" * 100)
print("DONE - ALL YEARS FROM DATA, ALL FIELDS, NO FILES SAVED")
print("=" * 100)

# table check for 0 - 1

In [ ]:
# ============================================================
# SHORT TABLE BY YEAR - ALL FIELDS
# NO FIELD FILTER
# NO SAVE
# Run after yearly_path_table_clean is created
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

# ------------------------------------------------------------
# OPTIONAL FILTERS
# Leave as None to show all data
# ------------------------------------------------------------
FILTER_START_YEAR = None
FILTER_END_YEAR = None

FILTER_BETTER_PATH = None
# example:
# FILTER_BETTER_PATH = "Industry job"
# FILTER_BETTER_PATH = "Startup"
# FILTER_BETTER_PATH = "Mixed / close"

FILTER_DATA_QUALITY = None
# example:
# FILTER_DATA_QUALITY = "Complete exact"
# FILTER_DATA_QUALITY = "Complete with fallback"
# FILTER_DATA_QUALITY = "Partial"

SHOW_ONLY_ROWS_WITH_DEGREE_COUNT = False
SHOW_ONLY_ROWS_WITH_COMPANY_SCORE = False


# ------------------------------------------------------------
# CHECK DATA
# ------------------------------------------------------------
if "yearly_path_table_clean" not in globals():
    raise NameError("Run the code that creates yearly_path_table_clean first.")

df = yearly_path_table_clean.copy()


# ------------------------------------------------------------
# CLEAN NUMBERS
# ------------------------------------------------------------
def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .replace({
            "No data": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "": np.nan
        }),
        errors="coerce"
    )


def fmt_num(x):
    if pd.isna(x):
        return "No data"
    return f"{x:,.0f}"


def fmt_pct(x):
    if pd.isna(x):
        return "No data"
    return f"{x:,.1f}%"


number_cols = [
    "year",
    "degree_count",
    "job_path_score",
    "startup_path_score",
    "job_minus_startup_score"
]

for col in number_cols:
    if col in df.columns:
        df[col] = clean_number(df[col])

df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)


# ------------------------------------------------------------
# APPLY FILTERS
# NO FIELD FILTER HERE
# THIS KEEPS ALL FIELDS
# ------------------------------------------------------------
if FILTER_START_YEAR is not None:
    df = df[df["year"] >= FILTER_START_YEAR]

if FILTER_END_YEAR is not None:
    df = df[df["year"] <= FILTER_END_YEAR]

if FILTER_BETTER_PATH is not None:
    df = df[
        df["better_path"]
        .astype(str)
        .str.contains(FILTER_BETTER_PATH, case=False, na=False)
    ]

if FILTER_DATA_QUALITY is not None and "data_quality" in df.columns:
    df = df[
        df["data_quality"]
        .astype(str)
        .str.contains(FILTER_DATA_QUALITY, case=False, na=False)
    ]

if SHOW_ONLY_ROWS_WITH_DEGREE_COUNT:
    df = df[df["degree_count"].notna()]

if SHOW_ONLY_ROWS_WITH_COMPANY_SCORE:
    df = df[
        df["job_path_score"].notna() |
        df["startup_path_score"].notna()
    ]


# ------------------------------------------------------------
# SHORT TABLE BY YEAR - ALL FIELDS COMBINED
# ------------------------------------------------------------
short_by_year = (
    df
    .groupby("year", as_index=False)
    .agg(
        total_fields=("field_of_study", "nunique"),
        total_degree_count=("degree_count", "sum"),

        avg_job_path_score=("job_path_score", "mean"),
        avg_startup_path_score=("startup_path_score", "mean"),
        avg_job_minus_startup_score=("job_minus_startup_score", "mean"),

        industry_job_rows=("better_path", lambda x: (x == "Industry job").sum()),
        startup_rows=("better_path", lambda x: (x == "Startup").sum()),
        mixed_rows=("better_path", lambda x: (x == "Mixed / close").sum()),
        no_data_rows=("better_path", lambda x: x.astype(str).str.contains("No data", case=False, na=False).sum()),

        degree_missing_rows=("degree_count", lambda x: x.isna().sum()),
        job_score_missing_rows=("job_path_score", lambda x: x.isna().sum()),
        startup_score_missing_rows=("startup_path_score", lambda x: x.isna().sum())
    )
)

short_by_year["better_overall_by_year"] = np.where(
    short_by_year["avg_job_minus_startup_score"] > 10,
    "Industry job",
    np.where(
        short_by_year["avg_job_minus_startup_score"] < -10,
        "Startup",
        "Mixed / close"
    )
)

short_by_year = short_by_year[
    [
        "year",
        "total_fields",
        "total_degree_count",
        "avg_job_path_score",
        "avg_startup_path_score",
        "avg_job_minus_startup_score",
        "better_overall_by_year",
        "industry_job_rows",
        "startup_rows",
        "mixed_rows",
        "no_data_rows",
        "degree_missing_rows",
        "job_score_missing_rows",
        "startup_score_missing_rows"
    ]
]


# ------------------------------------------------------------
# FORMAT DISPLAY
# ------------------------------------------------------------
display_table = short_by_year.copy()

display_table["total_degree_count"] = display_table["total_degree_count"].apply(fmt_num)

for col in [
    "avg_job_path_score",
    "avg_startup_path_score",
    "avg_job_minus_startup_score"
]:
    display_table[col] = display_table[col].apply(fmt_pct)


print("=" * 100)
print("SHORT TABLE BY YEAR - ALL FIELDS")
print("=" * 100)
print("Field filter: NONE")
print("Year filter:", FILTER_START_YEAR, "to", FILTER_END_YEAR)
print("Better path filter:", FILTER_BETTER_PATH)
print("Data quality filter:", FILTER_DATA_QUALITY)
print("=" * 100)

display(display_table)

print("=" * 100)
print("DONE - ALL FIELDS SHORT TABLE BY YEAR, NOTHING SAVED")
print("=" * 100)

# table check for 0 - 2 Double check

In [ ]:
# ============================================================
# FIXED SHORT TABLE BY YEAR - ALL FIELDS
# NO SAVE
#
# Fixes:
# 1. Uses ALL fields, no field filter
# 2. Re-reads degree file to fix old-year degree counts
# 3. Uses fallback count columns:
#    degree_count -> CTOTALT -> CTOTALM + CTOTALW -> crace15 + crace16
# 4. Does NOT turn missing degree years into fake 0
# 5. Shows short readable table by year
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

# ------------------------------------------------------------
# FILE PATH
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

CHUNKSIZE = 100_000

# Optional year filter. Leave None for all years.
FILTER_START_YEAR = None
FILTER_END_YEAR = None

# Optional: only show years with degree data
SHOW_ONLY_YEARS_WITH_DEGREE_DATA = False

# Optional: only show years with company/path score
SHOW_ONLY_YEARS_WITH_COMPANY_SCORE = False


# ------------------------------------------------------------
# CHECK DATA
# ------------------------------------------------------------
if "yearly_path_table_clean" not in globals():
    raise NameError("Run the code that creates yearly_path_table_clean first.")

if not DEGREE_FILE.exists():
    raise FileNotFoundError(f"Missing degree file: {DEGREE_FILE}")

print("FOUND DEGREE FILE:", DEGREE_FILE)


# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------
def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .replace({
            "No data": np.nan,
            "No degree data": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "": np.nan,
            " ": np.nan
        }),
        errors="coerce"
    )


def fmt_num(x):
    if pd.isna(x):
        return "No degree data"
    return f"{x:,.0f}"


def fmt_pct(x):
    if pd.isna(x):
        return "No data"
    return f"{x:,.1f}%"


def year_label_from_gap(x):
    if pd.isna(x):
        return "No data"
    elif x > 10:
        return "Industry job"
    elif x < -10:
        return "Startup"
    else:
        return "Mixed / close"


def first_positive_value(df, columns):
    """
    Return one count column using first positive value from candidate columns.
    This avoids old years showing 0 when another total column has data.
    """
    result = pd.Series(np.nan, index=df.index, dtype="float64")

    for col in columns:
        if col in df.columns:
            val = clean_number(df[col])
            mask = (result.isna() | result.le(0)) & val.notna() & val.gt(0)
            result.loc[mask] = val.loc[mask]

    return result


def pair_sum_if_positive(df, col_a, col_b):
    if col_a in df.columns and col_b in df.columns:
        a = clean_number(df[col_a]).fillna(0)
        b = clean_number(df[col_b]).fillna(0)
        total = a + b
        total = total.where(total > 0, np.nan)
        return total

    return pd.Series(np.nan, index=df.index, dtype="float64")


# ------------------------------------------------------------
# 1. REBUILD DEGREE TOTAL BY YEAR FROM RAW DEGREE FILE
# ------------------------------------------------------------
header_cols = list(pd.read_csv(DEGREE_FILE, nrows=0).columns)

needed_possible_cols = [
    "year",
    "field_group",

    # final cleaned count
    "degree_count",

    # IPEDS total count columns
    "CTOTALT", "ctotalt",
    "CTOTALM", "CTOTALW",
    "ctotalm", "ctotalw",

    # older race total style columns
    "crace15", "crace16",
    "CRACE15", "CRACE16"
]

usecols = [c for c in needed_possible_cols if c in header_cols]

if "year" not in usecols:
    raise ValueError("Degree file missing year column.")

if "field_group" not in usecols:
    raise ValueError("Degree file missing field_group column.")

print("=" * 100)
print("READING DEGREE DATA AGAIN TO FIX OLD YEAR TOTALS")
print("Using count columns found:")
print([c for c in usecols if c not in ["year", "field_group"]])
print("=" * 100)

degree_pieces = []
total_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(DEGREE_FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    total_rows += len(chunk)

    chunk["year"] = clean_number(chunk["year"])
    chunk = chunk.dropna(subset=["year"])
    chunk["year"] = chunk["year"].astype(int)

    if FILTER_START_YEAR is not None:
        chunk = chunk[chunk["year"] >= FILTER_START_YEAR]

    if FILTER_END_YEAR is not None:
        chunk = chunk[chunk["year"] <= FILTER_END_YEAR]

    if chunk.empty:
        continue

    chunk["field_group"] = chunk["field_group"].astype(str).str.strip()

    # First choice: degree_count or CTOTALT
    direct_total = first_positive_value(
        chunk,
        ["degree_count", "CTOTALT", "ctotalt"]
    )

    # Second choice: CTOTALM + CTOTALW
    gender_total = pair_sum_if_positive(chunk, "CTOTALM", "CTOTALW")
    gender_total_lower = pair_sum_if_positive(chunk, "ctotalm", "ctotalw")

    # Third choice: older crace15 + crace16
    old_total = pair_sum_if_positive(chunk, "crace15", "crace16")
    old_total_upper = pair_sum_if_positive(chunk, "CRACE15", "CRACE16")

    chunk["degree_count_fixed"] = direct_total

    for fallback_total in [gender_total, gender_total_lower, old_total, old_total_upper]:
        mask = (
            chunk["degree_count_fixed"].isna() |
            chunk["degree_count_fixed"].le(0)
        ) & fallback_total.notna() & fallback_total.gt(0)

        chunk.loc[mask, "degree_count_fixed"] = fallback_total.loc[mask]

    # Keep only real positive degree counts
    chunk.loc[chunk["degree_count_fixed"].le(0), "degree_count_fixed"] = np.nan

    grouped = (
        chunk
        .dropna(subset=["degree_count_fixed"])
        .groupby(["year", "field_group"], as_index=False)
        .agg(
            degree_count_fixed=("degree_count_fixed", lambda x: x.sum(min_count=1))
        )
    )

    if not grouped.empty:
        degree_pieces.append(grouped)

    if chunk_number == 1 or chunk_number % 10 == 0:
        print(f"Chunk {chunk_number:,} | rows checked {total_rows:,}")

if len(degree_pieces) == 0:
    raise ValueError("No positive degree counts found from degree file.")

degree_by_year_field = (
    pd.concat(degree_pieces, ignore_index=True)
    .groupby(["year", "field_group"], as_index=False)
    .agg(
        degree_count_fixed=("degree_count_fixed", lambda x: x.sum(min_count=1))
    )
)

degree_by_year = (
    degree_by_year_field
    .groupby("year", as_index=False)
    .agg(
        total_degree_count_fixed=("degree_count_fixed", lambda x: x.sum(min_count=1)),
        fields_with_degree_data=("field_group", "nunique")
    )
)

print("=" * 100)
print("DONE FIXING DEGREE YEAR TOTALS")
print("Degree year range:", degree_by_year["year"].min(), "to", degree_by_year["year"].max())
print("=" * 100)


# ------------------------------------------------------------
# 2. CLEAN YEARLY PATH TABLE
# ------------------------------------------------------------
df = yearly_path_table_clean.copy()

number_cols = [
    "year",
    "degree_count",
    "job_path_score",
    "startup_path_score",
    "job_minus_startup_score"
]

for col in number_cols:
    if col in df.columns:
        df[col] = clean_number(df[col])

df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

if FILTER_START_YEAR is not None:
    df = df[df["year"] >= FILTER_START_YEAR]

if FILTER_END_YEAR is not None:
    df = df[df["year"] <= FILTER_END_YEAR]


# ------------------------------------------------------------
# 3. SHORT COMPANY / PATH SUMMARY BY YEAR
# ------------------------------------------------------------
company_year_summary = (
    df
    .groupby("year", as_index=False)
    .agg(
        total_fields_in_path_table=("field_of_study", "nunique"),

        avg_job_path_score=("job_path_score", "mean"),
        avg_startup_path_score=("startup_path_score", "mean"),
        avg_job_minus_startup_score=("job_minus_startup_score", "mean"),

        industry_job_rows=("better_path", lambda x: (x == "Industry job").sum()),
        startup_rows=("better_path", lambda x: (x == "Startup").sum()),
        mixed_rows=("better_path", lambda x: (x == "Mixed / close").sum()),
        no_data_rows=("better_path", lambda x: x.astype(str).str.contains("No data", case=False, na=False).sum()),

        job_score_missing_rows=("job_path_score", lambda x: x.isna().sum()),
        startup_score_missing_rows=("startup_path_score", lambda x: x.isna().sum())
    )
)

company_year_summary["better_overall_by_year"] = company_year_summary[
    "avg_job_minus_startup_score"
].apply(year_label_from_gap)


# ------------------------------------------------------------
# 4. MERGE FIXED DEGREE TOTAL INTO YEAR SUMMARY
# ------------------------------------------------------------
all_years = sorted(
    set(company_year_summary["year"].dropna().astype(int).unique())
    .union(set(degree_by_year["year"].dropna().astype(int).unique()))
)

year_frame = pd.DataFrame({"year": all_years})

short_by_year = (
    year_frame
    .merge(degree_by_year, on="year", how="left")
    .merge(company_year_summary, on="year", how="left")
)

# Do not let missing values become fake 0
count_cols = [
    "fields_with_degree_data",
    "total_fields_in_path_table",
    "industry_job_rows",
    "startup_rows",
    "mixed_rows",
    "no_data_rows",
    "job_score_missing_rows",
    "startup_score_missing_rows"
]

for col in count_cols:
    if col in short_by_year.columns:
        short_by_year[col] = short_by_year[col].fillna(0).astype(int)

# Better label should be No data when score gap is missing
short_by_year["better_overall_by_year"] = short_by_year[
    "avg_job_minus_startup_score"
].apply(year_label_from_gap)


# ------------------------------------------------------------
# 5. OPTIONAL FILTERS
# ------------------------------------------------------------
if SHOW_ONLY_YEARS_WITH_DEGREE_DATA:
    short_by_year = short_by_year[
        short_by_year["total_degree_count_fixed"].notna()
    ]

if SHOW_ONLY_YEARS_WITH_COMPANY_SCORE:
    short_by_year = short_by_year[
        short_by_year["avg_job_path_score"].notna() |
        short_by_year["avg_startup_path_score"].notna()
    ]


# ------------------------------------------------------------
# 6. FINAL SHORT TABLE
# ------------------------------------------------------------
short_by_year = short_by_year[
    [
        "year",
        "fields_with_degree_data",
        "total_fields_in_path_table",
        "total_degree_count_fixed",

        "avg_job_path_score",
        "avg_startup_path_score",
        "avg_job_minus_startup_score",
        "better_overall_by_year",

        "industry_job_rows",
        "startup_rows",
        "mixed_rows",
        "no_data_rows",

        "job_score_missing_rows",
        "startup_score_missing_rows"
    ]
].sort_values("year").reset_index(drop=True)

display_table = short_by_year.copy()

display_table = display_table.rename(columns={
    "total_degree_count_fixed": "total_degree_count"
})

display_table["total_degree_count"] = display_table["total_degree_count"].apply(fmt_num)

for col in [
    "avg_job_path_score",
    "avg_startup_path_score",
    "avg_job_minus_startup_score"
]:
    display_table[col] = display_table[col].apply(fmt_pct)

print("=" * 100)
print("FIXED SHORT TABLE BY YEAR - ALL FIELDS")
print("=" * 100)
print("Field filter: NONE, all fields included")
print("Year filter:", FILTER_START_YEAR, "to", FILTER_END_YEAR)
print("Fake 0 degree totals fixed: yes")
print("=" * 100)

display(display_table)

print("=" * 100)
print("DONE - SHORT TABLE ONLY, NOTHING SAVED")
print("=" * 100)

# table check for 0 - 2 all column by year

In [ ]:
# ============================================================
# READ ONLY - SHORT TABLE BY YEAR FROM BOTH FILES
# DEGREE + COMPANY
# ALL YEARS FROM WHATEVER DATA EXISTS
# NO SAVE
# MEMORY OPTIMIZED WITH CHUNKS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

# ------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000

# None = use all years from data
START_YEAR = None
END_YEAR = None


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .replace({
            "No data": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "": np.nan,
            " ": np.nan
        }),
        errors="coerce"
    )


def fmt_num(x):
    if pd.isna(x):
        return "No data"
    return f"{x:,.0f}"


def classify_company_signal(metric_name, metric_category="", dataset="", source_table=""):
    text = " ".join([
        str(metric_name),
        str(metric_category),
        str(dataset),
        str(source_table)
    ]).lower()

    signals = []

    if any(k in text for k in [
        "birth", "births", "startup", "startups", "formation",
        "business application", "opening", "openings", "entry", "entries",
        "survival", "survivor", "survive"
    ]):
        signals.append("startup_success")

    if any(k in text for k in [
        "death", "deaths", "exit", "exits", "failure", "failures",
        "closing", "closings", "closure", "closures", "bankrupt"
    ]):
        signals.append("startup_failure_risk")

    if any(k in text for k in [
        "job gain", "job gains", "gross job gain", "gross job gains",
        "employment gain", "employment gains", "jobs gained",
        "job creation", "jobs created", "employment growth"
    ]):
        signals.append("job_success")

    if any(k in text for k in [
        "job loss", "job losses", "gross job loss", "gross job losses",
        "employment loss", "employment losses", "jobs lost",
        "layoff", "layoffs", "employment decline"
    ]):
        signals.append("job_loss_risk")

    if len(signals) == 0:
        return "other_company_metric"

    return "; ".join(sorted(set(signals)))


def make_degree_count_fixed(chunk):
    """
    Fix degree count for old and new IPEDS years.
    Tries:
    1. degree_count
    2. CTOTALT / ctotalt
    3. CTOTALM + CTOTALW
    4. ctotalm + ctotalw
    5. crace15 + crace16
    6. CRACE15 + CRACE16
    """

    result = pd.Series(np.nan, index=chunk.index, dtype="float64")

    # direct total columns
    for col in ["degree_count", "CTOTALT", "ctotalt"]:
        if col in chunk.columns:
            value = clean_number(chunk[col])
            mask = result.isna() & value.notna() & value.gt(0)
            result.loc[mask] = value.loc[mask]

    # pair total columns
    pair_cols = [
        ("CTOTALM", "CTOTALW"),
        ("ctotalm", "ctotalw"),
        ("crace15", "crace16"),
        ("CRACE15", "CRACE16")
    ]

    for col_a, col_b in pair_cols:
        if col_a in chunk.columns and col_b in chunk.columns:
            a = clean_number(chunk[col_a]).fillna(0)
            b = clean_number(chunk[col_b]).fillna(0)
            total = a + b
            total = total.where(total > 0, np.nan)

            mask = result.isna() & total.notna() & total.gt(0)
            result.loc[mask] = total.loc[mask]

    return result


check_file(DEGREE_FILE)
check_file(COMPANY_FILE)


# ============================================================
# 1. DEGREE TABLE BY YEAR
# ============================================================
degree_header = list(pd.read_csv(DEGREE_FILE, nrows=0).columns)

degree_possible_cols = [
    "year",
    "unitid",
    "field_group",
    "field_subgroup",
    "major_name",
    "degree_group",
    "award_level_name",
    "degree_count",
    "CTOTALT", "ctotalt",
    "CTOTALM", "CTOTALW",
    "ctotalm", "ctotalw",
    "crace15", "crace16",
    "CRACE15", "CRACE16"
]

degree_usecols = [c for c in degree_possible_cols if c in degree_header]

if "year" not in degree_usecols:
    raise ValueError("Degree file missing year column.")

print("=" * 100)
print("READING DEGREE FILE BY YEAR")
print("=" * 100)
print("Degree columns used:")
print(degree_usecols)

degree_parts = []
degree_total_rows = 0
t0 = time.time()

for i, chunk in enumerate(
    pd.read_csv(DEGREE_FILE, usecols=degree_usecols, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    degree_total_rows += len(chunk)

    chunk["year"] = clean_number(chunk["year"])
    chunk = chunk.dropna(subset=["year"])
    chunk["year"] = chunk["year"].astype(int)

    if START_YEAR is not None:
        chunk = chunk[chunk["year"] >= START_YEAR]

    if END_YEAR is not None:
        chunk = chunk[chunk["year"] <= END_YEAR]

    if chunk.empty:
        continue

    chunk["degree_count_fixed"] = make_degree_count_fixed(chunk)

    # do not turn missing count into fake 0
    chunk.loc[chunk["degree_count_fixed"].le(0), "degree_count_fixed"] = np.nan

    group_dict = {
        "degree_rows": ("year", "size"),
        "degree_rows_with_count": ("degree_count_fixed", lambda x: x.notna().sum()),
        "total_degree_count": ("degree_count_fixed", lambda x: x.sum(min_count=1))
    }

    if "unitid" in chunk.columns:
        group_dict["degree_schools"] = ("unitid", "nunique")

    if "field_group" in chunk.columns:
        group_dict["degree_fields"] = ("field_group", "nunique")

    if "field_subgroup" in chunk.columns:
        group_dict["degree_subfields"] = ("field_subgroup", "nunique")

    if "major_name" in chunk.columns:
        group_dict["degree_majors"] = ("major_name", "nunique")

    if "degree_group" in chunk.columns:
        group_dict["degree_groups"] = ("degree_group", "nunique")

    if "award_level_name" in chunk.columns:
        group_dict["award_levels"] = ("award_level_name", "nunique")

    grouped = chunk.groupby("year", as_index=False).agg(**group_dict)
    degree_parts.append(grouped)

    if i == 1 or i % 10 == 0:
        print(f"Degree chunk {i:,} | rows checked {degree_total_rows:,} | elapsed {time.time() - t0:,.1f}s")

degree_by_year_raw = pd.concat(degree_parts, ignore_index=True)

degree_agg = {
    "degree_rows": "sum",
    "degree_rows_with_count": "sum",
    "total_degree_count": lambda x: x.sum(min_count=1)
}

for col in [
    "degree_schools",
    "degree_fields",
    "degree_subfields",
    "degree_majors",
    "degree_groups",
    "award_levels"
]:
    if col in degree_by_year_raw.columns:
        degree_agg[col] = "max"

degree_by_year = (
    degree_by_year_raw
    .groupby("year", as_index=False)
    .agg(degree_agg)
)

print("=" * 100)
print("DEGREE YEAR RANGE:", degree_by_year["year"].min(), "to", degree_by_year["year"].max())
print("=" * 100)


# ============================================================
# 2. COMPANY TABLE BY YEAR
# ============================================================
company_header = list(pd.read_csv(COMPANY_FILE, nrows=0).columns)

company_possible_cols = [
    "year",
    "source",
    "dataset",
    "source_table",
    "geo_level",
    "state_name",
    "county_name",
    "msa_name",
    "industry_code",
    "industry_name",
    "naics_code",
    "metric_name",
    "metric_category",
    "value",
    "unit",
    "seasonal"
]

company_usecols = [c for c in company_possible_cols if c in company_header]

if "year" not in company_usecols:
    raise ValueError("Company file missing year column.")

if "value" not in company_usecols:
    raise ValueError("Company file missing value column.")

print("=" * 100)
print("READING COMPANY FILE BY YEAR")
print("=" * 100)
print("Company columns used:")
print(company_usecols)

company_parts = []
company_total_rows = 0
t0 = time.time()

for i, chunk in enumerate(
    pd.read_csv(COMPANY_FILE, usecols=company_usecols, chunksize=CHUNKSIZE, low_memory=False),
    start=1
):
    company_total_rows += len(chunk)

    chunk["year"] = clean_number(chunk["year"])
    chunk["value"] = clean_number(chunk["value"])

    chunk = chunk.dropna(subset=["year"])
    chunk["year"] = chunk["year"].astype(int)

    if START_YEAR is not None:
        chunk = chunk[chunk["year"] >= START_YEAR]

    if END_YEAR is not None:
        chunk = chunk[chunk["year"] <= END_YEAR]

    if chunk.empty:
        continue

    if "metric_category" not in chunk.columns:
        chunk["metric_category"] = ""

    if "dataset" not in chunk.columns:
        chunk["dataset"] = ""

    if "source_table" not in chunk.columns:
        chunk["source_table"] = ""

    chunk["signal_type"] = chunk.apply(
        lambda r: classify_company_signal(
            r.get("metric_name", ""),
            r.get("metric_category", ""),
            r.get("dataset", ""),
            r.get("source_table", "")
        ),
        axis=1
    )

    group_dict = {
        "company_rows": ("year", "size"),
        "company_rows_with_value": ("value", lambda x: x.notna().sum()),
        "total_company_value": ("value", lambda x: x.sum(min_count=1)),
        "company_metrics": ("metric_name", "nunique")
    }

    for col, out_name in [
        ("source", "company_sources"),
        ("dataset", "company_datasets"),
        ("source_table", "company_source_tables"),
        ("geo_level", "company_geo_levels"),
        ("state_name", "company_states"),
        ("county_name", "company_counties"),
        ("msa_name", "company_msas"),
        ("industry_name", "company_industries"),
        ("industry_code", "company_industry_codes"),
        ("naics_code", "company_naics_codes"),
        ("unit", "company_units")
    ]:
        if col in chunk.columns:
            group_dict[out_name] = (col, "nunique")

    grouped = chunk.groupby("year", as_index=False).agg(**group_dict)

    signal_counts = (
        chunk
        .groupby(["year", "signal_type"], as_index=False)
        .size()
        .pivot(index="year", columns="signal_type", values="size")
        .reset_index()
    )

    for col in [
        "startup_success",
        "startup_failure_risk",
        "job_success",
        "job_loss_risk",
        "other_company_metric"
    ]:
        if col not in signal_counts.columns:
            signal_counts[col] = 0

    signal_counts = signal_counts[
        [
            "year",
            "startup_success",
            "startup_failure_risk",
            "job_success",
            "job_loss_risk",
            "other_company_metric"
        ]
    ]

    grouped = grouped.merge(signal_counts, on="year", how="left")

    company_parts.append(grouped)

    if i == 1 or i % 10 == 0:
        print(f"Company chunk {i:,} | rows checked {company_total_rows:,} | elapsed {time.time() - t0:,.1f}s")

company_by_year_raw = pd.concat(company_parts, ignore_index=True)

company_agg = {
    "company_rows": "sum",
    "company_rows_with_value": "sum",
    "total_company_value": lambda x: x.sum(min_count=1),
    "company_metrics": "max",
    "startup_success": "sum",
    "startup_failure_risk": "sum",
    "job_success": "sum",
    "job_loss_risk": "sum",
    "other_company_metric": "sum"
}

for col in [
    "company_sources",
    "company_datasets",
    "company_source_tables",
    "company_geo_levels",
    "company_states",
    "company_counties",
    "company_msas",
    "company_industries",
    "company_industry_codes",
    "company_naics_codes",
    "company_units"
]:
    if col in company_by_year_raw.columns:
        company_agg[col] = "max"

company_by_year = (
    company_by_year_raw
    .groupby("year", as_index=False)
    .agg(company_agg)
)

print("=" * 100)
print("COMPANY YEAR RANGE:", company_by_year["year"].min(), "to", company_by_year["year"].max())
print("=" * 100)


# ============================================================
# 3. MERGE DEGREE + COMPANY BY YEAR
# ============================================================
all_years = sorted(
    set(degree_by_year["year"].dropna().astype(int).unique())
    .union(set(company_by_year["year"].dropna().astype(int).unique()))
)

year_table = pd.DataFrame({"year": all_years})

year_table = (
    year_table
    .merge(degree_by_year, on="year", how="left")
    .merge(company_by_year, on="year", how="left")
)

year_table["has_degree_data"] = year_table["degree_rows"].notna()
year_table["has_company_data"] = year_table["company_rows"].notna()

year_table["year_data_status"] = np.select(
    [
        year_table["has_degree_data"] & year_table["has_company_data"],
        year_table["has_degree_data"] & ~year_table["has_company_data"],
        ~year_table["has_degree_data"] & year_table["has_company_data"]
    ],
    [
        "Degree + company data",
        "Degree only",
        "Company only"
    ],
    default="No data"
)

# Fill row/count columns only. Do not fill total_degree_count with 0.
count_cols = [
    "degree_rows",
    "degree_rows_with_count",
    "degree_schools",
    "degree_fields",
    "degree_subfields",
    "degree_majors",
    "degree_groups",
    "award_levels",
    "company_rows",
    "company_rows_with_value",
    "company_sources",
    "company_datasets",
    "company_source_tables",
    "company_geo_levels",
    "company_states",
    "company_counties",
    "company_msas",
    "company_industries",
    "company_industry_codes",
    "company_naics_codes",
    "company_metrics",
    "company_units",
    "startup_success",
    "startup_failure_risk",
    "job_success",
    "job_loss_risk",
    "other_company_metric"
]

for col in count_cols:
    if col in year_table.columns:
        year_table[col] = year_table[col].fillna(0).astype(int)

# ------------------------------------------------------------
# FINAL SHORT TABLE
# ------------------------------------------------------------
wanted_cols = [
    "year",
    "year_data_status",

    "total_degree_count",
    "degree_rows",
    "degree_rows_with_count",
    "degree_schools",
    "degree_fields",
    "degree_subfields",
    "degree_majors",
    "degree_groups",
    "award_levels",

    "total_company_value",
    "company_rows",
    "company_rows_with_value",
    "company_sources",
    "company_datasets",
    "company_source_tables",
    "company_geo_levels",
    "company_states",
    "company_counties",
    "company_msas",
    "company_industries",
    "company_industry_codes",
    "company_naics_codes",
    "company_metrics",

    "startup_success",
    "startup_failure_risk",
    "job_success",
    "job_loss_risk",
    "other_company_metric"
]

wanted_cols = [c for c in wanted_cols if c in year_table.columns]

final_table = year_table[wanted_cols].copy()

display_table = final_table.copy()

for col in ["total_degree_count", "total_company_value"]:
    if col in display_table.columns:
        display_table[col] = display_table[col].apply(fmt_num)

print("=" * 100)
print("SHORT YEAR TABLE - DEGREE + COMPANY")
print("=" * 100)
print("Degree file years:", degree_by_year["year"].min(), "to", degree_by_year["year"].max())
print("Company file years:", company_by_year["year"].min(), "to", company_by_year["year"].max())
print("Final table years:", final_table["year"].min(), "to", final_table["year"].max())
print("No files saved.")
print("=" * 100)

display(display_table)

print("=" * 100)
print("DONE")
print("=" * 100)

# Degree + company data from 1984 to 2024

| Chart                          | Use                                            |
| ------------------------------ | ---------------------------------------------- |
| **Side-by-side bar chart**     | Compare job openings vs job closings by sector |
| **Opposite stacked bar chart** | Success side vs failure side                   |
| **Heatmap**                    | Degree field connected to industry sector      |
| **Flow/Sankey chart**          | Degree field → sector → outcome                |


# Code 1 D + C - some success

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DEGREE + COMPANY DATA CHARTS
# 1984 to 2024
# READ ONLY / NO SAVE
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 1984
END_YEAR = 2024
CHUNKSIZE = 100_000

USE_NATIONAL_ONLY = True   # True = use U.S. total if available
TOP_N_SECTORS = 12
TOP_N_FIELDS = 20


# ============================================================
# 1. BASIC HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def get_columns(path):
    return [str(c).strip().lower() for c in pd.read_csv(path, nrows=0).columns]


def pick_col(columns, exact_names, contains_words=None):
    for name in exact_names:
        if name in columns:
            return name

    if contains_words:
        for word in contains_words:
            matches = [c for c in columns if word in c]
            if matches:
                return matches[0]

    return None


# ============================================================
# 2. DEGREE FIELD → INDUSTRY SECTOR MAPPING
# ============================================================

def map_degree_field_to_sector(field):
    text = str(field).lower()

    if any(x in text for x in ["computer", "information science", "data", "mathematics", "statistics"]):
        return "Information"

    if any(x in text for x in ["communication", "journalism", "library"]):
        return "Information"

    if any(x in text for x in ["business", "management", "marketing", "accounting", "finance"]):
        if any(x in text for x in ["finance", "accounting"]):
            return "Financial activities"
        return "Professional and business services"

    if any(x in text for x in ["engineering", "mechanic", "precision production", "manufacturing"]):
        return "Manufacturing"

    if any(x in text for x in ["construction", "architecture"]):
        return "Construction"

    if any(x in text for x in ["health", "nursing", "medicine", "clinical", "dental"]):
        return "Education and health services"

    if any(x in text for x in ["education", "teacher"]):
        return "Education and health services"

    if any(x in text for x in ["agriculture", "natural resources", "mining"]):
        return "Natural resources and mining"

    if any(x in text for x in ["transportation", "logistics"]):
        return "Transportation and warehousing"

    if any(x in text for x in ["visual", "performing", "arts", "culinary", "recreation", "parks"]):
        return "Leisure and hospitality"

    if any(x in text for x in ["legal", "law", "public administration", "social sciences", "psychology"]):
        return "Professional and business services"

    return "Professional and business services"


# ============================================================
# 3. COMPANY INDUSTRY → CLEAN SECTOR
# ============================================================

def map_company_to_sector(industry_name, industry_code=None):
    text = str(industry_name).lower()
    code = str(industry_code).strip().lower()

    if text in ["nan", "", "none"]:
        text = code

    # Remove total rows
    if any(x in text for x in ["total private", "goods-producing", "service-providing", "total"]):
        return np.nan

    if any(x in text for x in ["natural resources", "mining", "agriculture"]) or code in ["11", "21", "100010"]:
        return "Natural resources and mining"

    if "construction" in text or code in ["23", "100020"]:
        return "Construction"

    if "manufacturing" in text or code in ["31-33", "31", "32", "33", "100030"]:
        return "Manufacturing"

    if "wholesale" in text or code in ["42", "200010"]:
        return "Wholesale trade"

    if "retail" in text or code in ["44-45", "44", "45", "200020"]:
        return "Retail trade"

    if any(x in text for x in ["transportation", "warehousing"]) or code in ["48-49", "48", "49", "200030"]:
        return "Transportation and warehousing"

    if "utilities" in text or code in ["22", "200040"]:
        return "Utilities"

    if "information" in text or code in ["51", "200050"]:
        return "Information"

    if any(x in text for x in ["financial", "finance", "insurance", "real estate"]) or code in ["52", "53", "200060"]:
        return "Financial activities"

    if any(x in text for x in ["professional", "business services", "management", "administrative"]) or code in ["54", "55", "56", "200070"]:
        return "Professional and business services"

    if any(x in text for x in ["education", "health"]) or code in ["61", "62", "200080"]:
        return "Education and health services"

    if any(x in text for x in ["leisure", "hospitality", "arts", "entertainment", "accommodation", "food"]) or code in ["71", "72", "200090"]:
        return "Leisure and hospitality"

    if "other services" in text or code in ["81", "200100"]:
        return "Other services"

    return np.nan


# ============================================================
# 4. COMPANY METRIC NAME CLEANING
# ============================================================

def normalize_metric(metric):
    text = str(metric).lower().strip()

    if "opening" in text:
        return "Job openings"

    if "closing" in text:
        return "Job closings"

    if "gain" in text:
        return "Job gains"

    if "loss" in text:
        return "Job losses"

    if "birth" in text:
        return "Establishment births"

    if "death" in text:
        return "Establishment deaths"

    return None


KEEP_OUTCOMES = [
    "Job openings",
    "Job closings",
    "Job gains",
    "Job losses",
    "Establishment births",
    "Establishment deaths"
]

SUCCESS_OUTCOMES = [
    "Job openings",
    "Job gains",
    "Establishment births"
]

FAILURE_OUTCOMES = [
    "Job closings",
    "Job losses",
    "Establishment deaths"
]


# ============================================================
# 5. READ DEGREE DATA
# ============================================================

def read_degree_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])
    field_col = pick_col(
        columns,
        ["field_group", "degree_field", "degree_fields", "field_of_study", "cip_field"],
        ["field"]
    )
    count_col = pick_col(
        columns,
        ["degree_count", "total_degree_count", "count", "degrees"],
        ["degree_count"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in degree file.")
    if field_col is None:
        raise ValueError("Could not find degree field column in degree file.")

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year_clean"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year_clean"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["degree_field"] = chunk[field_col].astype(str).str.strip()

        if count_col is not None:
            chunk["degree_count"] = pd.to_numeric(chunk[count_col], errors="coerce").fillna(0)
        else:
            chunk["degree_count"] = 1

        chunk = chunk[chunk["degree_count"] > 0]
        chunk["sector"] = chunk["degree_field"].apply(map_degree_field_to_sector)

        grouped = (
            chunk.groupby(["year_clean", "degree_field", "sector"], as_index=False)["degree_count"]
            .sum()
            .rename(columns={"year_clean": "year"})
        )

        results.append(grouped)

    if not results:
        raise ValueError("No degree rows found for the selected years.")

    degree_data = pd.concat(results, ignore_index=True)

    degree_data = (
        degree_data.groupby(["year", "degree_field", "sector"], as_index=False)["degree_count"]
        .sum()
    )

    return degree_data


# ============================================================
# 6. READ COMPANY DATA
# ============================================================

def detect_national_filter(path):
    sample = pd.read_csv(path, nrows=200_000, low_memory=False)
    sample = clean_columns(sample)

    possible_geo_cols = [
        "geo_level", "state_name", "area_name", "geography", "region", "state"
    ]

    for col in possible_geo_cols:
        if col in sample.columns:
            text = sample[col].astype(str).str.lower()
            if text.str.contains("u.s. totals|us totals|united states|national", regex=True, na=False).any():
                return col

    return None


def read_company_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])
    industry_col = pick_col(
        columns,
        ["industry_name_clean", "industry_name", "industry", "sector"],
        ["industry"]
    )
    industry_code_col = pick_col(
        columns,
        ["industry_code", "industrycode", "naics", "naics_code"],
        ["code"]
    )

    metric_col = pick_col(
        columns,
        ["metric_name", "metric", "measure", "variable", "data_type"],
        ["metric"]
    )
    value_col = pick_col(
        columns,
        ["value", "metric_value", "amount", "count"],
        ["value"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in company file.")

    national_col = detect_national_filter(path) if USE_NATIONAL_ONLY else None
    if USE_NATIONAL_ONLY and national_col:
        print(f"Using national rows based on column: {national_col}")
    elif USE_NATIONAL_ONLY:
        print("No national-only column detected. Using all company rows.")

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year_clean"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year_clean"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        if national_col and national_col in chunk.columns:
            text = chunk[national_col].astype(str).str.lower()
            chunk = chunk[text.str.contains("u.s. totals|us totals|united states|national", regex=True, na=False)]

        if chunk.empty:
            continue

        # Long format: metric_name + value
        if metric_col in chunk.columns and value_col in chunk.columns:
            chunk["outcome"] = chunk[metric_col].apply(normalize_metric)
            chunk["value"] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

            if industry_col in chunk.columns:
                industry_name = chunk[industry_col]
            else:
                industry_name = ""

            if industry_code_col in chunk.columns:
                industry_code = chunk[industry_code_col]
            else:
                industry_code = ""

            chunk["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = chunk[
                chunk["outcome"].isin(KEEP_OUTCOMES)
                & chunk["sector"].notna()
                & (chunk["value"] != 0)
            ]

            grouped = (
                keep.groupby(["year_clean", "sector", "outcome"], as_index=False)["value"]
                .sum()
                .rename(columns={"year_clean": "year"})
            )

            results.append(grouped)

        # Wide format: columns are openings/closings/gains/losses/births/deaths
        else:
            metric_columns = {}
            for col in chunk.columns:
                clean_metric = normalize_metric(col)
                if clean_metric is not None:
                    metric_columns[col] = clean_metric

            if not metric_columns:
                continue

            id_cols = [year_col]
            if industry_col:
                id_cols.append(industry_col)
            if industry_code_col:
                id_cols.append(industry_code_col)

            temp = chunk[id_cols + list(metric_columns.keys())].copy()

            temp = temp.melt(
                id_vars=id_cols,
                value_vars=list(metric_columns.keys()),
                var_name="raw_metric",
                value_name="value"
            )

            temp["outcome"] = temp["raw_metric"].map(metric_columns)
            temp["value"] = pd.to_numeric(temp["value"], errors="coerce").fillna(0)

            if industry_col and industry_col in temp.columns:
                industry_name = temp[industry_col]
            else:
                industry_name = ""

            if industry_code_col and industry_code_col in temp.columns:
                industry_code = temp[industry_code_col]
            else:
                industry_code = ""

            temp["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = temp[
                temp["outcome"].isin(KEEP_OUTCOMES)
                & temp["sector"].notna()
                & (temp["value"] != 0)
            ]

            grouped = (
                keep.groupby(["year_clean", "sector", "outcome"], as_index=False)["value"]
                .sum()
                .rename(columns={"year_clean": "year"})
            )

            results.append(grouped)

    if not results:
        raise ValueError("No company rows found for the selected years and metrics.")

    company_data = pd.concat(results, ignore_index=True)

    company_data = (
        company_data.groupby(["year", "sector", "outcome"], as_index=False)["value"]
        .sum()
    )

    return company_data


# ============================================================
# 7. LOAD DATA
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_data = read_degree_data(DEGREE_FILE)
company_data = read_company_data(COMPANY_FILE)

print()
print("Degree data year range:", int(degree_data["year"].min()), "to", int(degree_data["year"].max()))
print("Company data year range:", int(company_data["year"].min()), "to", int(company_data["year"].max()))
print("Degree rows after cleaning:", len(degree_data))
print("Company rows after cleaning:", len(company_data))


# ============================================================
# 8. MAKE CONNECTED TABLE
# ============================================================

degree_sector = (
    degree_data.groupby(["degree_field", "sector"], as_index=False)["degree_count"]
    .sum()
)

company_sector = (
    company_data.groupby(["sector", "outcome"], as_index=False)["value"]
    .sum()
)

company_pivot = (
    company_sector.pivot_table(
        index="sector",
        columns="outcome",
        values="value",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for col in KEEP_OUTCOMES:
    if col not in company_pivot.columns:
        company_pivot[col] = 0

connected_table = degree_sector.merge(company_pivot, on="sector", how="left").fillna(0)

connected_table["success_total"] = connected_table[SUCCESS_OUTCOMES].sum(axis=1)
connected_table["failure_total"] = connected_table[FAILURE_OUTCOMES].sum(axis=1)
connected_table["company_outcome_total"] = connected_table["success_total"] + connected_table["failure_total"]

connected_table = connected_table[
    (connected_table["degree_count"] > 0)
    & (connected_table["company_outcome_total"] > 0)
]

connected_table = connected_table.sort_values(
    ["company_outcome_total", "degree_count"],
    ascending=False
)

print()
print("CONNECTED TABLE: Degree field → industry sector → company outcomes")
display(connected_table.head(50))


# ============================================================
# 9. CHART 1
# SIDE-BY-SIDE BAR CHART
# JOB OPENINGS VS JOB CLOSINGS BY SECTOR
# ============================================================

sector_open_close = company_pivot.copy()

sector_open_close["open_close_total"] = (
    sector_open_close["Job openings"] + sector_open_close["Job closings"]
)

sector_open_close = sector_open_close[
    sector_open_close["open_close_total"] > 0
].sort_values("open_close_total", ascending=False).head(TOP_N_SECTORS)

plot_df = sector_open_close.set_index("sector")[["Job openings", "Job closings"]]

ax = plot_df.plot(
    kind="bar",
    figsize=(14, 6)
)

plt.title("Degree + Company Data 1984–2024: Job Openings vs Job Closings by Industry Sector")
plt.xlabel("Industry sector")
plt.ylabel("Total nonzero company value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


# ============================================================
# 10. CHART 2
# OPPOSITE STACKED BAR CHART
# SUCCESS SIDE VS FAILURE SIDE
# ============================================================

sector_success_failure = company_pivot.copy()

sector_success_failure["success_total"] = sector_success_failure[SUCCESS_OUTCOMES].sum(axis=1)
sector_success_failure["failure_total"] = sector_success_failure[FAILURE_OUTCOMES].sum(axis=1)
sector_success_failure["total"] = (
    sector_success_failure["success_total"] + sector_success_failure["failure_total"]
)

sector_success_failure = sector_success_failure[
    sector_success_failure["total"] > 0
].sort_values("total", ascending=False).head(TOP_N_SECTORS)

sector_success_failure = sector_success_failure.set_index("sector")

success_df = sector_success_failure[SUCCESS_OUTCOMES]
failure_df = -sector_success_failure[FAILURE_OUTCOMES]

fig, ax = plt.subplots(figsize=(15, 7))

success_df.plot(
    kind="bar",
    stacked=True,
    ax=ax
)

failure_df.plot(
    kind="bar",
    stacked=True,
    ax=ax
)

ax.axhline(0, linewidth=1)
plt.title("Degree + Company Data 1984–2024: Success Outcomes vs Failure Outcomes by Sector")
plt.xlabel("Industry sector")
plt.ylabel("Success above 0 / Failure below 0")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


# ============================================================
# 11. CHART 3
# HEATMAP
# DEGREE FIELD CONNECTED TO INDUSTRY SECTOR
# ============================================================

top_fields = (
    degree_sector.groupby("degree_field")["degree_count"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_FIELDS)
    .index
)

heat_data = degree_sector[degree_sector["degree_field"].isin(top_fields)]

heat_pivot = heat_data.pivot_table(
    index="degree_field",
    columns="sector",
    values="degree_count",
    aggfunc="sum",
    fill_value=0
)

heat_pivot = heat_pivot.loc[
    heat_pivot.sum(axis=1).sort_values(ascending=False).index
]

plt.figure(figsize=(15, 9))
plt.imshow(np.log1p(heat_pivot.values), aspect="auto")
plt.colorbar(label="log(1 + degree count)")

plt.title("Degree Field → Industry Sector Connection Heatmap, 1984–2024")
plt.xlabel("Industry sector")
plt.ylabel("Degree field")

plt.xticks(
    ticks=np.arange(len(heat_pivot.columns)),
    labels=heat_pivot.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(len(heat_pivot.index)),
    labels=heat_pivot.index
)

plt.tight_layout()
plt.show()


# ============================================================
# 12. CHART 4
# FLOW / SANKEY CHART
# DEGREE FIELD → SECTOR → OUTCOME
# ============================================================

try:
    import plotly.graph_objects as go

    sankey_fields = (
        degree_sector.groupby("degree_field")["degree_count"]
        .sum()
        .sort_values(ascending=False)
        .head(12)
        .index
    )

    sankey_degree_sector = degree_sector[
        degree_sector["degree_field"].isin(sankey_fields)
    ].copy()

    sankey_sectors = sankey_degree_sector["sector"].unique().tolist()

    sankey_company = company_pivot[
        company_pivot["sector"].isin(sankey_sectors)
    ].copy()

    # Allocate sector company outcomes to degree fields by degree share.
    # This makes the Sankey readable, but it is NOT a true hiring probability.
    sector_degree_total = (
        sankey_degree_sector.groupby("sector")["degree_count"]
        .sum()
        .reset_index()
        .rename(columns={"degree_count": "sector_degree_total"})
    )

    sankey_degree_sector = sankey_degree_sector.merge(
        sector_degree_total,
        on="sector",
        how="left"
    )

    sankey_company["sector_outcome_total"] = sankey_company[KEEP_OUTCOMES].sum(axis=1)

    sankey_degree_sector = sankey_degree_sector.merge(
        sankey_company[["sector", "sector_outcome_total"]],
        on="sector",
        how="left"
    ).fillna(0)

    sankey_degree_sector["allocated_value"] = (
        sankey_degree_sector["degree_count"]
        / sankey_degree_sector["sector_degree_total"]
        * sankey_degree_sector["sector_outcome_total"]
    )

    sankey_degree_sector = sankey_degree_sector[
        sankey_degree_sector["allocated_value"] > 0
    ]

    degree_nodes = sankey_degree_sector["degree_field"].unique().tolist()
    sector_nodes = sankey_degree_sector["sector"].unique().tolist()
    outcome_nodes = KEEP_OUTCOMES

    labels = degree_nodes + sector_nodes + outcome_nodes
    label_to_id = {label: i for i, label in enumerate(labels)}

    sources = []
    targets = []
    values = []

    # Degree field → sector
    for _, row in sankey_degree_sector.iterrows():
        sources.append(label_to_id[row["degree_field"]])
        targets.append(label_to_id[row["sector"]])
        values.append(row["allocated_value"])

    # Sector → outcome
    for _, row in sankey_company.iterrows():
        for outcome in KEEP_OUTCOMES:
            value = row[outcome]
            if value > 0:
                sources.append(label_to_id[row["sector"]])
                targets.append(label_to_id[outcome])
                values.append(value)

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=15,
                    thickness=18,
                    line=dict(width=0.5),
                    label=labels
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values
                )
            )
        ]
    )

    fig.update_layout(
        title_text="Degree Field → Industry Sector → Company Outcomes, 1984–2024",
        font_size=11,
        height=750
    )

    fig.show()

except ImportError:
    print("Plotly is not installed. To show the Sankey chart, run this once:")
    print("pip install plotly")


# ============================================================
# 13. SIMPLE SUMMARY TABLE BY SECTOR
# ============================================================

sector_summary = company_pivot.copy()
sector_summary["success_total"] = sector_summary[SUCCESS_OUTCOMES].sum(axis=1)
sector_summary["failure_total"] = sector_summary[FAILURE_OUTCOMES].sum(axis=1)

sector_summary["net_success_minus_failure"] = (
    sector_summary["success_total"] - sector_summary["failure_total"]
)

sector_summary = sector_summary.sort_values(
    "net_success_minus_failure",
    ascending=False
)

print()
print("SECTOR SUMMARY TABLE")
display(sector_summary)

# Code 2 D + C

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# ONE FULL CONNECTED CHART
# Degree Field → Industry Sector → Company Outcome
# 1984–2024
# READ ONLY / NO SAVE
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 1984
END_YEAR = 2024
CHUNKSIZE = 100_000

# Use U.S. totals if the company file has them.
# If the file does not have U.S. totals, the code will use all rows.
USE_US_TOTAL_ONLY = True

# For readability.
# Set MAX_DEGREE_FIELDS = None if you really want every degree field in the Sankey chart.
MAX_DEGREE_FIELDS = 40

KEEP_OUTCOMES = [
    "Job openings",
    "Job closings",
    "Job gains",
    "Job losses",
    "Establishment births",
    "Establishment deaths"
]

SUCCESS_OUTCOMES = [
    "Job openings",
    "Job gains",
    "Establishment births"
]

FAILURE_OUTCOMES = [
    "Job closings",
    "Job losses",
    "Establishment deaths"
]


# ============================================================
# 1. BASIC HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def get_columns(path):
    return [str(c).strip().lower() for c in pd.read_csv(path, nrows=0).columns]


def pick_col(columns, exact_names, contains_words=None):
    """
    Try to find a column by exact name first.
    If not found, try partial matching.
    """
    for name in exact_names:
        if name in columns:
            return name

    if contains_words:
        for word in contains_words:
            matches = [c for c in columns if word in c]
            if matches:
                return matches[0]

    return None


# ============================================================
# 2. DEGREE FIELD → INDUSTRY SECTOR MAPPING
# ============================================================

def map_degree_field_to_sector(field):
    """
    This connects degree field names to broad industry sectors.

    Important:
    This is not a true person-level hiring connection.
    It is a project mapping:
    degree field → likely related industry sector.
    """
    text = str(field).lower()

    if any(x in text for x in [
        "computer", "information science", "information sciences",
        "data", "mathematics", "statistics", "cyber", "software"
    ]):
        return "Information"

    if any(x in text for x in [
        "communication", "journalism", "library"
    ]):
        return "Information"

    if any(x in text for x in [
        "business", "management", "marketing", "entrepreneurship",
        "human resources", "administration"
    ]):
        return "Professional and business services"

    if any(x in text for x in [
        "accounting", "finance", "financial", "insurance", "real estate"
    ]):
        return "Financial activities"

    if any(x in text for x in [
        "engineering", "mechanic", "mechanical", "electrical",
        "industrial", "precision", "manufacturing", "technology/technician"
    ]):
        return "Manufacturing"

    if any(x in text for x in [
        "construction", "architecture", "architectural"
    ]):
        return "Construction"

    if any(x in text for x in [
        "health", "nursing", "medical", "medicine", "clinical",
        "dental", "pharmacy", "therapy", "public health"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "education", "teacher", "teaching"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "agriculture", "natural resources", "mining", "forestry",
        "fishing", "wildlife"
    ]):
        return "Natural resources and mining"

    if any(x in text for x in [
        "transportation", "logistics", "aviation", "pilot"
    ]):
        return "Transportation and warehousing"

    if any(x in text for x in [
        "culinary", "hospitality", "hotel", "restaurant",
        "parks", "recreation", "fitness", "leisure"
    ]):
        return "Leisure and hospitality"

    if any(x in text for x in [
        "visual", "performing", "arts", "music", "drama", "film"
    ]):
        return "Leisure and hospitality"

    if any(x in text for x in [
        "legal", "law", "paralegal", "public administration",
        "social sciences", "psychology", "sociology", "political"
    ]):
        return "Professional and business services"

    if any(x in text for x in [
        "family", "consumer", "human services", "social work"
    ]):
        return "Education and health services"

    return "Professional and business services"


# ============================================================
# 3. COMPANY INDUSTRY → CLEAN SECTOR MAPPING
# ============================================================

def map_company_to_sector(industry_name=None, industry_code=None):
    """
    This cleans company industry names/codes into the same broad sector names
    used by the degree mapping.
    """
    text = str(industry_name).lower().strip()
    code = str(industry_code).lower().strip()

    if text in ["nan", "", "none", "null"]:
        text = code

    # Remove high-level total rows
    if any(x in text for x in [
        "total private", "goods-producing", "service-providing",
        "all industries", "total all", "total"
    ]):
        return np.nan

    if any(x in text for x in ["natural resources", "mining", "agriculture"]) or code in ["11", "21", "100010"]:
        return "Natural resources and mining"

    if "construction" in text or code in ["23", "100020"]:
        return "Construction"

    if "manufacturing" in text or code in ["31", "32", "33", "31-33", "100030"]:
        return "Manufacturing"

    if "wholesale" in text or code in ["42", "200010"]:
        return "Wholesale trade"

    if "retail" in text or code in ["44", "45", "44-45", "200020"]:
        return "Retail trade"

    if any(x in text for x in ["transportation", "warehousing"]) or code in ["48", "49", "48-49", "200030"]:
        return "Transportation and warehousing"

    if "utilities" in text or code in ["22", "200040"]:
        return "Utilities"

    if "information" in text or code in ["51", "200050"]:
        return "Information"

    if any(x in text for x in ["financial", "finance", "insurance", "real estate"]) or code in ["52", "53", "200060"]:
        return "Financial activities"

    if any(x in text for x in [
        "professional", "business services", "management",
        "administrative", "support services"
    ]) or code in ["54", "55", "56", "200070"]:
        return "Professional and business services"

    if any(x in text for x in ["education", "educational", "health", "health care", "social assistance"]) or code in ["61", "62", "200080"]:
        return "Education and health services"

    if any(x in text for x in [
        "leisure", "hospitality", "arts", "entertainment",
        "recreation", "accommodation", "food services"
    ]) or code in ["71", "72", "200090"]:
        return "Leisure and hospitality"

    if "other services" in text or code in ["81", "200100"]:
        return "Other services"

    return np.nan


# ============================================================
# 4. COMPANY METRIC CLEANING
# ============================================================

def normalize_metric(metric):
    """
    Converts different company column names into clean outcome names.
    Works for both:
    - long format: metric_name + value
    - wide format: openings, closings, job_gains, job_losses, births, deaths
    """
    text = str(metric).lower().strip()

    text = text.replace("_", " ")
    text = text.replace("-", " ")

    if "opening" in text or text == "openings":
        return "Job openings"

    if "closing" in text or text == "closings":
        return "Job closings"

    if "gain" in text:
        return "Job gains"

    if "loss" in text:
        return "Job losses"

    if "birth" in text:
        return "Establishment births"

    if "death" in text:
        return "Establishment deaths"

    return None


# ============================================================
# 5. DETECT U.S. TOTAL ROWS IN COMPANY FILE
# ============================================================

def detect_us_total_column(path):
    sample = pd.read_csv(path, nrows=200_000, low_memory=False)
    sample = clean_columns(sample)

    possible_cols = [
        "state_name",
        "area_name",
        "geo_name",
        "geography",
        "geo_level",
        "region",
        "state"
    ]

    for col in possible_cols:
        if col in sample.columns:
            text = sample[col].astype(str).str.lower()
            if text.str.contains(
                "u.s. totals|us totals|united states|national",
                regex=True,
                na=False
            ).any():
                return col

    return None


# ============================================================
# 6. READ DEGREE DATA
# ============================================================

def read_degree_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    field_col = pick_col(
        columns,
        [
            "field_group",
            "degree_field",
            "degree_fields",
            "field_of_study",
            "cip_field",
            "field"
        ],
        ["field"]
    )

    count_col = pick_col(
        columns,
        [
            "degree_count",
            "total_degree_count",
            "count",
            "degrees"
        ],
        ["degree_count"]
    )

    if year_col is None:
        raise ValueError("Could not find a year column in degree file.")

    if field_col is None:
        raise ValueError("Could not find a degree field column in degree file.")

    print()
    print("DEGREE COLUMNS USED")
    print("year column:", year_col)
    print("field column:", field_col)
    print("count column:", count_col)

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year_clean"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year_clean"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["degree_field"] = chunk[field_col].astype(str).str.strip()

        if count_col is not None and count_col in chunk.columns:
            chunk["degree_count"] = pd.to_numeric(chunk[count_col], errors="coerce").fillna(0)
        else:
            chunk["degree_count"] = 1

        chunk = chunk[
            (chunk["degree_field"].notna())
            & (chunk["degree_field"].astype(str).str.lower() != "nan")
            & (chunk["degree_count"] > 0)
        ]

        if chunk.empty:
            continue

        chunk["sector"] = chunk["degree_field"].apply(map_degree_field_to_sector)

        grouped = (
            chunk.groupby(["degree_field", "sector"], as_index=False)["degree_count"]
            .sum()
        )

        results.append(grouped)

    if not results:
        raise ValueError("No degree data found after filtering years and nonzero degree counts.")

    degree_data = pd.concat(results, ignore_index=True)

    degree_data = (
        degree_data.groupby(["degree_field", "sector"], as_index=False)["degree_count"]
        .sum()
    )

    degree_data = degree_data[degree_data["degree_count"] > 0]

    return degree_data


# ============================================================
# 7. READ COMPANY DATA
# ============================================================

def read_company_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    industry_col = pick_col(
        columns,
        [
            "industry_name_clean",
            "industry_name",
            "industry",
            "sector"
        ],
        ["industry"]
    )

    industry_code_col = pick_col(
        columns,
        [
            "industry_code",
            "industrycode",
            "naics",
            "naics_code"
        ],
        ["code", "naics"]
    )

    metric_col = pick_col(
        columns,
        [
            "metric_name",
            "metric",
            "measure",
            "variable",
            "data_type"
        ],
        ["metric", "measure"]
    )

    value_col = pick_col(
        columns,
        [
            "value",
            "metric_value",
            "amount"
        ],
        ["value"]
    )

    if year_col is None:
        raise ValueError("Could not find a year column in company file.")

    if industry_col is None and industry_code_col is None:
        raise ValueError("Could not find industry name or industry code column in company file.")

    us_total_col = detect_us_total_column(path) if USE_US_TOTAL_ONLY else None

    print()
    print("COMPANY COLUMNS USED")
    print("year column:", year_col)
    print("industry name column:", industry_col)
    print("industry code column:", industry_code_col)
    print("metric column:", metric_col)
    print("value column:", value_col)

    if USE_US_TOTAL_ONLY and us_total_col:
        print("U.S. total filter column:", us_total_col)
    elif USE_US_TOTAL_ONLY:
        print("U.S. total column not detected, using all company rows.")

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year_clean"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year_clean"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        # Optional U.S. total filter
        if USE_US_TOTAL_ONLY and us_total_col and us_total_col in chunk.columns:
            geo_text = chunk[us_total_col].astype(str).str.lower()
            chunk = chunk[
                geo_text.str.contains(
                    "u.s. totals|us totals|united states|national",
                    regex=True,
                    na=False
                )
            ]

        if chunk.empty:
            continue

        # ------------------------------------------------------------
        # Case A: Long format
        # Example columns:
        # year, industry_name, metric_name, value
        # ------------------------------------------------------------

        if metric_col is not None and value_col is not None and metric_col in chunk.columns and value_col in chunk.columns:
            chunk["outcome"] = chunk[metric_col].apply(normalize_metric)
            chunk["value"] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

            if industry_col is not None and industry_col in chunk.columns:
                industry_name = chunk[industry_col]
            else:
                industry_name = pd.Series([""] * len(chunk), index=chunk.index)

            if industry_code_col is not None and industry_code_col in chunk.columns:
                industry_code = chunk[industry_code_col]
            else:
                industry_code = pd.Series([""] * len(chunk), index=chunk.index)

            chunk["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = chunk[
                chunk["outcome"].isin(KEEP_OUTCOMES)
                & chunk["sector"].notna()
                & (chunk["value"] > 0)
            ]

            if keep.empty:
                continue

            grouped = (
                keep.groupby(["sector", "outcome"], as_index=False)["value"]
                .sum()
            )

            results.append(grouped)

        # ------------------------------------------------------------
        # Case B: Wide format
        # Example columns:
        # openings, closings, job_gains, job_losses, business_births, business_deaths
        # ------------------------------------------------------------

        else:
            metric_columns = {}

            for col in chunk.columns:
                clean_metric = normalize_metric(col)
                if clean_metric is not None:
                    metric_columns[col] = clean_metric

            if not metric_columns:
                continue

            id_cols = []

            if industry_col is not None and industry_col in chunk.columns:
                id_cols.append(industry_col)

            if industry_code_col is not None and industry_code_col in chunk.columns:
                id_cols.append(industry_code_col)

            temp = chunk[id_cols + list(metric_columns.keys())].copy()

            temp = temp.melt(
                id_vars=id_cols,
                value_vars=list(metric_columns.keys()),
                var_name="raw_metric",
                value_name="value"
            )

            temp["outcome"] = temp["raw_metric"].map(metric_columns)
            temp["value"] = pd.to_numeric(temp["value"], errors="coerce").fillna(0)

            if industry_col is not None and industry_col in temp.columns:
                industry_name = temp[industry_col]
            else:
                industry_name = pd.Series([""] * len(temp), index=temp.index)

            if industry_code_col is not None and industry_code_col in temp.columns:
                industry_code = temp[industry_code_col]
            else:
                industry_code = pd.Series([""] * len(temp), index=temp.index)

            temp["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = temp[
                temp["outcome"].isin(KEEP_OUTCOMES)
                & temp["sector"].notna()
                & (temp["value"] > 0)
            ]

            if keep.empty:
                continue

            grouped = (
                keep.groupby(["sector", "outcome"], as_index=False)["value"]
                .sum()
            )

            results.append(grouped)

    if not results:
        raise ValueError("No company data found after filtering years, sectors, outcomes, and nonzero values.")

    company_data = pd.concat(results, ignore_index=True)

    company_data = (
        company_data.groupby(["sector", "outcome"], as_index=False)["value"]
        .sum()
    )

    company_data = company_data[company_data["value"] > 0]

    return company_data


# ============================================================
# 8. LOAD BOTH DATASETS
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_data = read_degree_data(DEGREE_FILE)
company_data = read_company_data(COMPANY_FILE)

print()
print("=" * 80)
print("DATA LOADED")
print("=" * 80)
print("Degree rows after grouping:", len(degree_data))
print("Company rows after grouping:", len(company_data))

print()
print("Degree sectors:")
print(sorted(degree_data["sector"].dropna().unique()))

print()
print("Company sectors:")
print(sorted(company_data["sector"].dropna().unique()))


# ============================================================
# 9. MAKE THE TRUE CONNECTED DATA
# Degree Field → Sector → Outcome
# ============================================================

degree_sector = (
    degree_data.groupby(["degree_field", "sector"], as_index=False)["degree_count"]
    .sum()
)

sector_outcome = (
    company_data.groupby(["sector", "outcome"], as_index=False)["value"]
    .sum()
)

degree_sector = degree_sector[degree_sector["degree_count"] > 0]
sector_outcome = sector_outcome[sector_outcome["value"] > 0]

# Keep only sectors that exist in BOTH datasets.
# This prevents errors like KeyError: 'Construction'
common_sectors = sorted(
    set(degree_sector["sector"].dropna()) &
    set(sector_outcome["sector"].dropna())
)

degree_sector = degree_sector[degree_sector["sector"].isin(common_sectors)]
sector_outcome = sector_outcome[sector_outcome["sector"].isin(common_sectors)]

if degree_sector.empty:
    raise ValueError("No connected degree-sector rows found.")

if sector_outcome.empty:
    raise ValueError("No connected sector-outcome rows found.")

print()
print("=" * 80)
print("CONNECTED SECTORS USED")
print("=" * 80)
print(common_sectors)


# ============================================================
# 10. OPTIONAL: LIMIT DEGREE FIELDS TO KEEP CHART READABLE
# ============================================================

if MAX_DEGREE_FIELDS is not None:
    top_degree_fields = (
        degree_sector.groupby("degree_field")["degree_count"]
        .sum()
        .sort_values(ascending=False)
        .head(MAX_DEGREE_FIELDS)
        .index
    )

    degree_sector = degree_sector[degree_sector["degree_field"].isin(top_degree_fields)]

    # Recalculate common sectors after limiting degree fields
    common_sectors = sorted(
        set(degree_sector["sector"].dropna()) &
        set(sector_outcome["sector"].dropna())
    )

    degree_sector = degree_sector[degree_sector["sector"].isin(common_sectors)]
    sector_outcome = sector_outcome[sector_outcome["sector"].isin(common_sectors)]


# ============================================================
# 11. CONNECTED TABLE
# ============================================================

connected_table = degree_sector.merge(
    sector_outcome,
    on="sector",
    how="inner"
)

connected_table["connection"] = (
    connected_table["degree_field"]
    + " → "
    + connected_table["sector"]
    + " → "
    + connected_table["outcome"]
)

connected_table = connected_table[
    (connected_table["degree_count"] > 0)
    & (connected_table["value"] > 0)
]

connected_table = connected_table.sort_values(
    ["sector", "degree_count", "value"],
    ascending=[True, False, False]
)

print()
print("=" * 80)
print("CONNECTED TABLE")
print("Degree Field → Industry Sector → Outcome")
print("=" * 80)

display(connected_table.head(200))


# ============================================================
# 12. SUMMARY TABLE BY SECTOR
# ============================================================

sector_summary = sector_outcome.pivot_table(
    index="sector",
    columns="outcome",
    values="value",
    aggfunc="sum",
    fill_value=0
).reset_index()

for outcome in KEEP_OUTCOMES:
    if outcome not in sector_summary.columns:
        sector_summary[outcome] = 0

sector_summary["success_total"] = sector_summary[SUCCESS_OUTCOMES].sum(axis=1)
sector_summary["failure_total"] = sector_summary[FAILURE_OUTCOMES].sum(axis=1)
sector_summary["net_success_minus_failure"] = (
    sector_summary["success_total"] - sector_summary["failure_total"]
)

sector_summary = sector_summary.sort_values(
    "net_success_minus_failure",
    ascending=False
)

print()
print("=" * 80)
print("SECTOR SUMMARY")
print("=" * 80)

display(sector_summary)


# ============================================================
# 13. ONE FULL CONNECTED SANKEY CHART
# Degree Field → Industry Sector → Company Outcome
# ============================================================

try:
    import plotly.graph_objects as go

    # ------------------------------------------------------------
    # Create node labels
    # ------------------------------------------------------------

    degree_nodes = sorted(degree_sector["degree_field"].dropna().unique().tolist())
    sector_nodes = sorted(degree_sector["sector"].dropna().unique().tolist())

    outcome_nodes = [
        outcome for outcome in KEEP_OUTCOMES
        if outcome in sector_outcome["outcome"].dropna().unique()
    ]

    labels = degree_nodes + sector_nodes + outcome_nodes
    label_to_id = {label: i for i, label in enumerate(labels)}

    sources = []
    targets = []
    values = []

    # ------------------------------------------------------------
    # Degree Field → Sector links
    # ------------------------------------------------------------

    for _, row in degree_sector.iterrows():
        degree_field = row["degree_field"]
        sector = row["sector"]
        value = row["degree_count"]

        if degree_field in label_to_id and sector in label_to_id and value > 0:
            sources.append(label_to_id[degree_field])
            targets.append(label_to_id[sector])
            values.append(value)

    # ------------------------------------------------------------
    # Sector → Outcome links
    # ------------------------------------------------------------

    for _, row in sector_outcome.iterrows():
        sector = row["sector"]
        outcome = row["outcome"]
        value = row["value"]

        if sector in label_to_id and outcome in label_to_id and value > 0:
            sources.append(label_to_id[sector])
            targets.append(label_to_id[outcome])
            values.append(value)

    if len(sources) == 0:
        raise ValueError("No Sankey links were created.")

    # ------------------------------------------------------------
    # Build chart
    # ------------------------------------------------------------

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=18,
                    thickness=20,
                    line=dict(width=0.5),
                    label=labels
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values
                )
            )
        ]
    )

    fig.update_layout(
        title_text=(
            "Degree Field → Industry Sector → Job Openings, Closings, Gains, Losses, "
            "Establishment Births, and Establishment Deaths, 1984–2024"
        ),
        font_size=11,
        height=900
    )

    fig.show()

except ImportError:
    print()
    print("Plotly is not installed.")
    print("Run this one time in a new Jupyter cell:")
    print("pip install plotly")

# Code 3 D + C - sackle chart Success

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# STACKED CHART
# Degree + Company Data, 1984–2024
# READ ONLY / NO SAVE
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 1984
END_YEAR = 2024
CHUNKSIZE = 100_000

USE_US_TOTAL_ONLY = True
TOP_N_SECTORS = 12

SUCCESS_OUTCOMES = [
    "Job openings",
    "Job gains",
    "Establishment births"
]

FAILURE_OUTCOMES = [
    "Job closings",
    "Job losses",
    "Establishment deaths"
]

KEEP_OUTCOMES = SUCCESS_OUTCOMES + FAILURE_OUTCOMES


# ============================================================
# BASIC HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def get_columns(path):
    return [str(c).strip().lower() for c in pd.read_csv(path, nrows=0).columns]


def pick_col(columns, exact_names, contains_words=None):
    for name in exact_names:
        if name in columns:
            return name

    if contains_words:
        for word in contains_words:
            matches = [c for c in columns if word in c]
            if matches:
                return matches[0]

    return None


def compact_number(x, pos=None):
    x = abs(x)
    if x >= 1_000_000_000:
        return f"{x/1_000_000_000:.1f}B"
    if x >= 1_000_000:
        return f"{x/1_000_000:.1f}M"
    if x >= 1_000:
        return f"{x/1_000:.1f}K"
    return f"{x:.0f}"


# ============================================================
# DEGREE FIELD → INDUSTRY SECTOR
# ============================================================

def map_degree_field_to_sector(field):
    text = str(field).lower()

    if any(x in text for x in ["computer", "information science", "data", "software", "cyber", "mathematics", "statistics"]):
        return "Information"

    if any(x in text for x in ["communication", "journalism", "library"]):
        return "Information"

    if any(x in text for x in ["business", "management", "marketing", "administration", "entrepreneurship"]):
        return "Professional and business services"

    if any(x in text for x in ["accounting", "finance", "financial", "insurance", "real estate"]):
        return "Financial activities"

    if any(x in text for x in ["engineering", "mechanic", "mechanical", "electrical", "industrial", "manufacturing", "precision"]):
        return "Manufacturing"

    if any(x in text for x in ["construction", "architecture", "architectural"]):
        return "Construction"

    if any(x in text for x in ["health", "nursing", "medical", "clinical", "dental", "pharmacy", "therapy", "public health"]):
        return "Education and health services"

    if any(x in text for x in ["education", "teacher", "teaching"]):
        return "Education and health services"

    if any(x in text for x in ["agriculture", "natural resources", "mining", "forestry", "wildlife"]):
        return "Natural resources and mining"

    if any(x in text for x in ["transportation", "logistics", "aviation"]):
        return "Transportation and warehousing"

    if any(x in text for x in ["culinary", "hospitality", "hotel", "restaurant", "recreation", "leisure", "arts", "music", "film"]):
        return "Leisure and hospitality"

    return "Professional and business services"


# ============================================================
# COMPANY INDUSTRY → INDUSTRY SECTOR
# ============================================================

def map_company_to_sector(industry_name=None, industry_code=None):
    text = str(industry_name).lower().strip()
    code = str(industry_code).lower().strip()

    if text in ["nan", "", "none", "null"]:
        text = code

    if any(x in text for x in ["total private", "goods-producing", "service-providing", "all industries", "total"]):
        return np.nan

    if any(x in text for x in ["natural resources", "mining", "agriculture"]) or code in ["11", "21", "100010"]:
        return "Natural resources and mining"

    if "construction" in text or code in ["23", "100020"]:
        return "Construction"

    if "manufacturing" in text or code in ["31", "32", "33", "31-33", "100030"]:
        return "Manufacturing"

    if "wholesale" in text or code in ["42", "200010"]:
        return "Wholesale trade"

    if "retail" in text or code in ["44", "45", "44-45", "200020"]:
        return "Retail trade"

    if any(x in text for x in ["transportation", "warehousing"]) or code in ["48", "49", "48-49", "200030"]:
        return "Transportation and warehousing"

    if "utilities" in text or code in ["22", "200040"]:
        return "Utilities"

    if "information" in text or code in ["51", "200050"]:
        return "Information"

    if any(x in text for x in ["financial", "finance", "insurance", "real estate"]) or code in ["52", "53", "200060"]:
        return "Financial activities"

    if any(x in text for x in ["professional", "business services", "management", "administrative", "support services"]) or code in ["54", "55", "56", "200070"]:
        return "Professional and business services"

    if any(x in text for x in ["education", "educational", "health", "health care", "social assistance"]) or code in ["61", "62", "200080"]:
        return "Education and health services"

    if any(x in text for x in ["leisure", "hospitality", "arts", "entertainment", "recreation", "accommodation", "food services"]) or code in ["71", "72", "200090"]:
        return "Leisure and hospitality"

    if "other services" in text or code in ["81", "200100"]:
        return "Other services"

    return np.nan


# ============================================================
# COMPANY METRIC CLEANING
# ============================================================

def normalize_metric(metric):
    text = str(metric).lower().strip()
    text = text.replace("_", " ").replace("-", " ")

    if "opening" in text or text == "openings":
        return "Job openings"

    if "closing" in text or text == "closings":
        return "Job closings"

    if "gain" in text:
        return "Job gains"

    if "loss" in text:
        return "Job losses"

    if "birth" in text:
        return "Establishment births"

    if "death" in text:
        return "Establishment deaths"

    return None


# ============================================================
# DETECT U.S. TOTAL ROWS
# ============================================================

def detect_us_total_column(path):
    sample = pd.read_csv(path, nrows=200_000, low_memory=False)
    sample = clean_columns(sample)

    possible_cols = [
        "state_name",
        "area_name",
        "geo_name",
        "geography",
        "geo_level",
        "region",
        "state"
    ]

    for col in possible_cols:
        if col in sample.columns:
            text = sample[col].astype(str).str.lower()
            if text.str.contains("u.s. totals|us totals|united states|national", regex=True, na=False).any():
                return col

    return None


# ============================================================
# READ DEGREE DATA
# ============================================================

def read_degree_sector_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    field_col = pick_col(
        columns,
        ["field_group", "degree_field", "degree_fields", "field_of_study", "cip_field", "field"],
        ["field"]
    )

    count_col = pick_col(
        columns,
        ["degree_count", "total_degree_count", "count", "degrees"],
        ["degree_count"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in degree file.")

    if field_col is None:
        raise ValueError("Could not find degree field column in degree file.")

    print()
    print("DEGREE COLUMNS USED")
    print("year:", year_col)
    print("degree field:", field_col)
    print("degree count:", count_col)

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["degree_field"] = chunk[field_col].astype(str).str.strip()

        if count_col is not None and count_col in chunk.columns:
            chunk["degree_count"] = pd.to_numeric(chunk[count_col], errors="coerce").fillna(0)
        else:
            chunk["degree_count"] = 1

        chunk = chunk[
            (chunk["degree_count"] > 0)
            & (chunk["degree_field"].str.lower() != "nan")
        ]

        if chunk.empty:
            continue

        chunk["sector"] = chunk["degree_field"].apply(map_degree_field_to_sector)

        grouped = (
            chunk.groupby(["sector", "degree_field"], as_index=False)["degree_count"]
            .sum()
        )

        results.append(grouped)

    if not results:
        raise ValueError("No degree data found.")

    degree_sector = pd.concat(results, ignore_index=True)

    degree_sector = (
        degree_sector.groupby(["sector", "degree_field"], as_index=False)["degree_count"]
        .sum()
    )

    return degree_sector


# ============================================================
# READ COMPANY DATA
# ============================================================

def read_company_outcome_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    industry_col = pick_col(
        columns,
        ["industry_name_clean", "industry_name", "industry", "sector"],
        ["industry"]
    )

    industry_code_col = pick_col(
        columns,
        ["industry_code", "industrycode", "naics", "naics_code"],
        ["code", "naics"]
    )

    metric_col = pick_col(
        columns,
        ["metric_name", "metric", "measure", "variable", "data_type"],
        ["metric", "measure"]
    )

    value_col = pick_col(
        columns,
        ["value", "metric_value", "amount"],
        ["value"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in company file.")

    us_total_col = detect_us_total_column(path) if USE_US_TOTAL_ONLY else None

    print()
    print("COMPANY COLUMNS USED")
    print("year:", year_col)
    print("industry name:", industry_col)
    print("industry code:", industry_code_col)
    print("metric:", metric_col)
    print("value:", value_col)

    if USE_US_TOTAL_ONLY and us_total_col:
        print("U.S. total filter:", us_total_col)
    elif USE_US_TOTAL_ONLY:
        print("No U.S. total column found, using all company rows.")

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        if USE_US_TOTAL_ONLY and us_total_col and us_total_col in chunk.columns:
            geo_text = chunk[us_total_col].astype(str).str.lower()
            chunk = chunk[
                geo_text.str.contains("u.s. totals|us totals|united states|national", regex=True, na=False)
            ]

        if chunk.empty:
            continue

        # Long format
        if metric_col is not None and value_col is not None and metric_col in chunk.columns and value_col in chunk.columns:
            chunk["outcome"] = chunk[metric_col].apply(normalize_metric)
            chunk["value"] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

            industry_name = chunk[industry_col] if industry_col in chunk.columns else pd.Series([""] * len(chunk), index=chunk.index)
            industry_code = chunk[industry_code_col] if industry_code_col in chunk.columns else pd.Series([""] * len(chunk), index=chunk.index)

            chunk["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = chunk[
                chunk["sector"].notna()
                & chunk["outcome"].isin(KEEP_OUTCOMES)
                & (chunk["value"] > 0)
            ]

            if keep.empty:
                continue

            grouped = (
                keep.groupby(["sector", "outcome"], as_index=False)["value"]
                .sum()
            )

            results.append(grouped)

        # Wide format
        else:
            metric_columns = {}

            for col in chunk.columns:
                clean_metric = normalize_metric(col)
                if clean_metric is not None:
                    metric_columns[col] = clean_metric

            if not metric_columns:
                continue

            id_cols = []

            if industry_col is not None and industry_col in chunk.columns:
                id_cols.append(industry_col)

            if industry_code_col is not None and industry_code_col in chunk.columns:
                id_cols.append(industry_code_col)

            temp = chunk[id_cols + list(metric_columns.keys())].copy()

            temp = temp.melt(
                id_vars=id_cols,
                value_vars=list(metric_columns.keys()),
                var_name="raw_metric",
                value_name="value"
            )

            temp["outcome"] = temp["raw_metric"].map(metric_columns)
            temp["value"] = pd.to_numeric(temp["value"], errors="coerce").fillna(0)

            industry_name = temp[industry_col] if industry_col in temp.columns else pd.Series([""] * len(temp), index=temp.index)
            industry_code = temp[industry_code_col] if industry_code_col in temp.columns else pd.Series([""] * len(temp), index=temp.index)

            temp["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = temp[
                temp["sector"].notna()
                & temp["outcome"].isin(KEEP_OUTCOMES)
                & (temp["value"] > 0)
            ]

            if keep.empty:
                continue

            grouped = (
                keep.groupby(["sector", "outcome"], as_index=False)["value"]
                .sum()
            )

            results.append(grouped)

    if not results:
        raise ValueError("No company data found.")

    company_outcome = pd.concat(results, ignore_index=True)

    company_outcome = (
        company_outcome.groupby(["sector", "outcome"], as_index=False)["value"]
        .sum()
    )

    return company_outcome


# ============================================================
# LOAD DATA
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_sector = read_degree_sector_data(DEGREE_FILE)
company_outcome = read_company_outcome_data(COMPANY_FILE)

print()
print("Degree-sector rows:", len(degree_sector))
print("Company-outcome rows:", len(company_outcome))


# ============================================================
# CONNECT DEGREE + COMPANY BY INDUSTRY SECTOR
# ============================================================

degree_sector_summary = (
    degree_sector.groupby("sector", as_index=False)
    .agg(
        total_degree_count=("degree_count", "sum"),
        degree_field_count=("degree_field", "nunique")
    )
)

common_sectors = sorted(
    set(degree_sector_summary["sector"].dropna())
    & set(company_outcome["sector"].dropna())
)

degree_sector_summary = degree_sector_summary[
    degree_sector_summary["sector"].isin(common_sectors)
]

company_outcome = company_outcome[
    company_outcome["sector"].isin(common_sectors)
]

company_pivot = company_outcome.pivot_table(
    index="sector",
    columns="outcome",
    values="value",
    aggfunc="sum",
    fill_value=0
).reset_index()

for col in KEEP_OUTCOMES:
    if col not in company_pivot.columns:
        company_pivot[col] = 0

connected_summary = degree_sector_summary.merge(
    company_pivot,
    on="sector",
    how="inner"
)

connected_summary["success_total"] = connected_summary[SUCCESS_OUTCOMES].sum(axis=1)
connected_summary["failure_total"] = connected_summary[FAILURE_OUTCOMES].sum(axis=1)
connected_summary["outcome_total"] = connected_summary["success_total"] + connected_summary["failure_total"]

connected_summary = connected_summary[
    connected_summary["outcome_total"] > 0
].sort_values("outcome_total", ascending=False)

connected_summary = connected_summary.head(TOP_N_SECTORS)

print()
print("CONNECTED SUMMARY TABLE")
display(connected_summary)


# ============================================================
# STACKED BAR CHART
# Success above zero, failure below zero
# ============================================================

plot_df = connected_summary.set_index("sector")

x = np.arange(len(plot_df.index))

fig, ax = plt.subplots(figsize=(15, 7))

# Success stacked above 0
bottom_success = np.zeros(len(plot_df))

for col in SUCCESS_OUTCOMES:
    values = plot_df[col].values
    ax.bar(x, values, bottom=bottom_success, label=col)
    bottom_success += values

# Failure stacked below 0
bottom_failure = np.zeros(len(plot_df))

for col in FAILURE_OUTCOMES:
    values = -plot_df[col].values
    ax.bar(x, values, bottom=bottom_failure, label=col)
    bottom_failure += values

ax.axhline(0, linewidth=1)

ax.set_title(
    "Stacked Success vs Failure Outcomes by Industry Sector, 1984–2024",
    fontsize=14
)

ax.set_xlabel("Industry sector")
ax.set_ylabel("Company outcome value")

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=45, ha="right")

ax.yaxis.set_major_formatter(FuncFormatter(compact_number))

ax.legend(
    title="Outcome",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Code 4 D + C - sackle chart some year

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

# ============================================================
# FIELD OF STUDY → INDUSTRY SECTOR → COMPANY OUTCOME BY YEAR
# Degree + Company Data, 1984–2024
# READ ONLY / NO SAVE
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 1984
END_YEAR = 2024

# These are the years you want to show clearly.
# The code still reads 1984–2024 by year.
SHOW_YEARS = [1984, 2000, 2010, 2024]

CHUNKSIZE = 100_000
USE_US_TOTAL_ONLY = True
TOP_N_SECTORS_PER_YEAR = 10
TOP_N_FIELDS_PER_SECTOR = 3

SUCCESS_OUTCOMES = [
    "Job openings",
    "Job gains",
    "Establishment births"
]

FAILURE_OUTCOMES = [
    "Job closings",
    "Job losses",
    "Establishment deaths"
]

KEEP_OUTCOMES = SUCCESS_OUTCOMES + FAILURE_OUTCOMES


# ============================================================
# DISPLAY SETTINGS
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)


# ============================================================
# BASIC HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def get_columns(path):
    return [str(c).strip().lower() for c in pd.read_csv(path, nrows=0).columns]


def pick_col(columns, exact_names, contains_words=None):
    for name in exact_names:
        if name in columns:
            return name

    if contains_words:
        for word in contains_words:
            matches = [c for c in columns if word in c]
            if matches:
                return matches[0]

    return None


def compact_number(x, pos=None):
    x = abs(float(x))

    if x >= 1_000_000_000:
        return f"{x / 1_000_000_000:.1f}B"
    if x >= 1_000_000:
        return f"{x / 1_000_000:.1f}M"
    if x >= 1_000:
        return f"{x / 1_000:.1f}K"

    return f"{x:.0f}"


def clean_industry_code(code):
    code = str(code).lower().strip()

    if code.endswith(".0"):
        code = code[:-2]

    return code


# ============================================================
# DEGREE FIELD → INDUSTRY SECTOR
# ============================================================

def map_degree_field_to_sector(field):
    text = str(field).lower()

    if any(x in text for x in [
        "computer", "information science", "data", "software",
        "cyber", "mathematics", "statistics"
    ]):
        return "Information"

    if any(x in text for x in [
        "communication", "journalism", "library"
    ]):
        return "Information"

    if any(x in text for x in [
        "business", "management", "marketing",
        "administration", "entrepreneurship"
    ]):
        return "Professional and business services"

    if any(x in text for x in [
        "accounting", "finance", "financial", "insurance", "real estate"
    ]):
        return "Financial activities"

    if any(x in text for x in [
        "engineering", "mechanic", "mechanical", "electrical",
        "industrial", "manufacturing", "precision"
    ]):
        return "Manufacturing"

    if any(x in text for x in [
        "construction", "architecture", "architectural"
    ]):
        return "Construction"

    if any(x in text for x in [
        "health", "nursing", "medical", "clinical", "dental",
        "pharmacy", "therapy", "public health"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "education", "teacher", "teaching"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "agriculture", "natural resources", "mining", "forestry", "wildlife"
    ]):
        return "Natural resources and mining"

    if any(x in text for x in [
        "transportation", "logistics", "aviation"
    ]):
        return "Transportation and warehousing"

    if any(x in text for x in [
        "culinary", "hospitality", "hotel", "restaurant",
        "recreation", "leisure", "arts", "music", "film"
    ]):
        return "Leisure and hospitality"

    return "Professional and business services"


# ============================================================
# COMPANY INDUSTRY → INDUSTRY SECTOR
# ============================================================

def map_company_to_sector(industry_name=None, industry_code=None):
    text = str(industry_name).lower().strip()
    code = clean_industry_code(industry_code)

    if text in ["nan", "", "none", "null"]:
        text = code

    if any(x in text for x in [
        "total private", "goods-producing", "service-providing",
        "all industries", "total"
    ]):
        return np.nan

    if any(x in text for x in ["natural resources", "mining", "agriculture"]) or code in ["11", "21", "100010"]:
        return "Natural resources and mining"

    if "construction" in text or code in ["23", "100020"]:
        return "Construction"

    if "manufacturing" in text or code in ["31", "32", "33", "31-33", "100030"]:
        return "Manufacturing"

    if "wholesale" in text or code in ["42", "200010"]:
        return "Wholesale trade"

    if "retail" in text or code in ["44", "45", "44-45", "200020"]:
        return "Retail trade"

    if any(x in text for x in ["transportation", "warehousing"]) or code in ["48", "49", "48-49", "200030"]:
        return "Transportation and warehousing"

    if "utilities" in text or code in ["22", "200040"]:
        return "Utilities"

    if "information" in text or code in ["51", "200050"]:
        return "Information"

    if any(x in text for x in ["financial", "finance", "insurance", "real estate"]) or code in ["52", "53", "200060"]:
        return "Financial activities"

    if any(x in text for x in [
        "professional", "business services", "management",
        "administrative", "support services"
    ]) or code in ["54", "55", "56", "200070"]:
        return "Professional and business services"

    if any(x in text for x in [
        "education", "educational", "health", "health care", "social assistance"
    ]) or code in ["61", "62", "200080"]:
        return "Education and health services"

    if any(x in text for x in [
        "leisure", "hospitality", "arts", "entertainment", "recreation",
        "accommodation", "food services"
    ]) or code in ["71", "72", "200090"]:
        return "Leisure and hospitality"

    if "other services" in text or code in ["81", "200100"]:
        return "Other services"

    return np.nan


# ============================================================
# COMPANY METRIC CLEANING
# ============================================================

def normalize_metric(metric):
    text = str(metric).lower().strip()
    text = text.replace("_", " ").replace("-", " ")

    if "opening" in text or text == "openings":
        return "Job openings"

    if "closing" in text or text == "closings":
        return "Job closings"

    if "gain" in text:
        return "Job gains"

    if "loss" in text:
        return "Job losses"

    if "birth" in text:
        return "Establishment births"

    if "death" in text:
        return "Establishment deaths"

    return None


# ============================================================
# DETECT U.S. TOTAL ROWS
# ============================================================

def detect_us_total_column(path):
    sample = pd.read_csv(path, nrows=200_000, low_memory=False)
    sample = clean_columns(sample)

    possible_cols = [
        "state_name",
        "area_name",
        "geo_name",
        "geography",
        "geo_level",
        "region",
        "state"
    ]

    for col in possible_cols:
        if col in sample.columns:
            text = sample[col].astype(str).str.lower()
            found_total = text.str.contains(
                "u.s. totals|us totals|united states|national",
                regex=True,
                na=False
            ).any()

            if found_total:
                return col

    return None


# ============================================================
# READ DEGREE DATA BY YEAR
# ============================================================

def read_degree_sector_year_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    field_col = pick_col(
        columns,
        [
            "field_group",
            "degree_field",
            "degree_fields",
            "field_of_study",
            "cip_field",
            "field"
        ],
        ["field"]
    )

    count_col = pick_col(
        columns,
        ["degree_count", "total_degree_count", "count", "degrees"],
        ["degree_count"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in degree file.")

    if field_col is None:
        raise ValueError("Could not find degree field column in degree file.")

    print()
    print("DEGREE COLUMNS USED")
    print("year:", year_col)
    print("field of study:", field_col)
    print("degree count:", count_col)

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)
        chunk["degree_field"] = chunk[field_col].astype(str).str.strip()

        if count_col is not None and count_col in chunk.columns:
            chunk["degree_count"] = pd.to_numeric(chunk[count_col], errors="coerce").fillna(0)
        else:
            chunk["degree_count"] = 1

        chunk = chunk[
            (chunk["degree_count"] > 0)
            & (chunk["degree_field"].str.lower() != "nan")
            & (chunk["degree_field"].str.strip() != "")
        ]

        if chunk.empty:
            continue

        chunk["sector"] = chunk["degree_field"].apply(map_degree_field_to_sector)

        grouped = (
            chunk.groupby(["year", "sector", "degree_field"], as_index=False)["degree_count"]
            .sum()
        )

        results.append(grouped)

    if not results:
        raise ValueError("No degree data found.")

    degree_sector_year = pd.concat(results, ignore_index=True)

    degree_sector_year = (
        degree_sector_year
        .groupby(["year", "sector", "degree_field"], as_index=False)["degree_count"]
        .sum()
    )

    return degree_sector_year


# ============================================================
# READ COMPANY DATA BY YEAR
# ============================================================

def read_company_outcome_year_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    industry_col = pick_col(
        columns,
        ["industry_name_clean", "industry_name", "industry", "sector"],
        ["industry"]
    )

    industry_code_col = pick_col(
        columns,
        ["industry_code", "industrycode", "naics", "naics_code"],
        ["code", "naics"]
    )

    metric_col = pick_col(
        columns,
        ["metric_name", "metric", "measure", "variable", "data_type"],
        ["metric", "measure"]
    )

    value_col = pick_col(
        columns,
        ["value", "metric_value", "amount"],
        ["value"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in company file.")

    us_total_col = detect_us_total_column(path) if USE_US_TOTAL_ONLY else None

    print()
    print("COMPANY COLUMNS USED")
    print("year:", year_col)
    print("industry name:", industry_col)
    print("industry code:", industry_code_col)
    print("metric:", metric_col)
    print("value:", value_col)

    if USE_US_TOTAL_ONLY and us_total_col:
        print("U.S. total filter:", us_total_col)
    elif USE_US_TOTAL_ONLY:
        print("No U.S. total column found, using all company rows.")

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)

        if USE_US_TOTAL_ONLY and us_total_col and us_total_col in chunk.columns:
            geo_text = chunk[us_total_col].astype(str).str.lower()
            chunk = chunk[
                geo_text.str.contains(
                    "u.s. totals|us totals|united states|national",
                    regex=True,
                    na=False
                )
            ]

        if chunk.empty:
            continue

        # -------------------------------
        # Long format company data
        # -------------------------------
        if (
            metric_col is not None
            and value_col is not None
            and metric_col in chunk.columns
            and value_col in chunk.columns
        ):
            chunk["outcome"] = chunk[metric_col].apply(normalize_metric)
            chunk["value"] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

            if industry_col is not None and industry_col in chunk.columns:
                industry_name = chunk[industry_col]
            else:
                industry_name = pd.Series([""] * len(chunk), index=chunk.index)

            if industry_code_col is not None and industry_code_col in chunk.columns:
                industry_code = chunk[industry_code_col]
            else:
                industry_code = pd.Series([""] * len(chunk), index=chunk.index)

            chunk["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = chunk[
                chunk["sector"].notna()
                & chunk["outcome"].isin(KEEP_OUTCOMES)
                & (chunk["value"] > 0)
            ]

            if keep.empty:
                continue

            grouped = (
                keep.groupby(["year", "sector", "outcome"], as_index=False)["value"]
                .sum()
            )

            results.append(grouped)

        # -------------------------------
        # Wide format company data
        # -------------------------------
        else:
            metric_columns = {}

            for col in chunk.columns:
                clean_metric = normalize_metric(col)
                if clean_metric is not None:
                    metric_columns[col] = clean_metric

            if not metric_columns:
                continue

            id_cols = ["year"]

            if industry_col is not None and industry_col in chunk.columns:
                id_cols.append(industry_col)

            if industry_code_col is not None and industry_code_col in chunk.columns:
                id_cols.append(industry_code_col)

            temp = chunk[id_cols + list(metric_columns.keys())].copy()

            temp = temp.melt(
                id_vars=id_cols,
                value_vars=list(metric_columns.keys()),
                var_name="raw_metric",
                value_name="value"
            )

            temp["outcome"] = temp["raw_metric"].map(metric_columns)
            temp["value"] = pd.to_numeric(temp["value"], errors="coerce").fillna(0)

            if industry_col is not None and industry_col in temp.columns:
                industry_name = temp[industry_col]
            else:
                industry_name = pd.Series([""] * len(temp), index=temp.index)

            if industry_code_col is not None and industry_code_col in temp.columns:
                industry_code = temp[industry_code_col]
            else:
                industry_code = pd.Series([""] * len(temp), index=temp.index)

            temp["sector"] = [
                map_company_to_sector(n, c)
                for n, c in zip(industry_name, industry_code)
            ]

            keep = temp[
                temp["sector"].notna()
                & temp["outcome"].isin(KEEP_OUTCOMES)
                & (temp["value"] > 0)
            ]

            if keep.empty:
                continue

            grouped = (
                keep.groupby(["year", "sector", "outcome"], as_index=False)["value"]
                .sum()
            )

            results.append(grouped)

    if not results:
        raise ValueError("No company data found.")

    company_outcome_year = pd.concat(results, ignore_index=True)

    company_outcome_year = (
        company_outcome_year
        .groupby(["year", "sector", "outcome"], as_index=False)["value"]
        .sum()
    )

    return company_outcome_year


# ============================================================
# MAKE TOP FIELD TEXT FOR EACH YEAR + SECTOR
# ============================================================

def make_top_field_table(degree_sector_year):
    rows = []

    for (year, sector), group in degree_sector_year.groupby(["year", "sector"]):
        top_fields = (
            group.sort_values("degree_count", ascending=False)
            .head(TOP_N_FIELDS_PER_SECTOR)
        )

        field_text = " | ".join(
            [
                f"{row['degree_field']} ({compact_number(row['degree_count'])})"
                for _, row in top_fields.iterrows()
            ]
        )

        rows.append(
            {
                "year": year,
                "sector": sector,
                "top_connected_fields_of_study": field_text
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# BUILD CONNECTED TABLE
# ============================================================

def build_connected_summary(degree_sector_year, company_outcome_year):
    degree_summary = (
        degree_sector_year
        .groupby(["year", "sector"], as_index=False)
        .agg(
            total_degree_count=("degree_count", "sum"),
            degree_field_count=("degree_field", "nunique")
        )
    )

    top_field_table = make_top_field_table(degree_sector_year)

    degree_summary = degree_summary.merge(
        top_field_table,
        on=["year", "sector"],
        how="left"
    )

    company_pivot = company_outcome_year.pivot_table(
        index=["year", "sector"],
        columns="outcome",
        values="value",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    company_pivot.columns.name = None

    for col in KEEP_OUTCOMES:
        if col not in company_pivot.columns:
            company_pivot[col] = 0

    connected = degree_summary.merge(
        company_pivot,
        on=["year", "sector"],
        how="inner"
    )

    connected["success_total"] = connected[SUCCESS_OUTCOMES].sum(axis=1)
    connected["failure_total"] = connected[FAILURE_OUTCOMES].sum(axis=1)
    connected["outcome_total"] = connected["success_total"] + connected["failure_total"]

    connected = connected[
        (connected["total_degree_count"] > 0)
        & (connected["outcome_total"] > 0)
    ].copy()

    connected["net_success_minus_failure"] = (
        connected["success_total"] - connected["failure_total"]
    )

    connected["success_rate_percent"] = np.where(
        connected["outcome_total"] > 0,
        connected["success_total"] / connected["outcome_total"] * 100,
        np.nan
    )

    connected["rank_in_year"] = (
        connected.groupby("year")["outcome_total"]
        .rank(method="first", ascending=False)
    )

    connected = connected.sort_values(
        ["year", "rank_in_year"],
        ascending=[True, True]
    )

    return connected


# ============================================================
# PLOT STACKED SUCCESS VS FAILURE BY YEAR
# ============================================================

def plot_year_stacked_chart(connected_year_df, year):
    plot_df = connected_year_df.copy()

    plot_df = plot_df.sort_values("outcome_total", ascending=False).head(TOP_N_SECTORS_PER_YEAR)

    if plot_df.empty:
        print(f"No connected data found for {year}.")
        return

    plot_df = plot_df.set_index("sector")

    x = np.arange(len(plot_df.index))

    fig, ax = plt.subplots(figsize=(15, 7))

    # Success stacked above zero
    bottom_success = np.zeros(len(plot_df))

    for col in SUCCESS_OUTCOMES:
        values = plot_df[col].values
        ax.bar(x, values, bottom=bottom_success, label=col)
        bottom_success += values

    # Failure stacked below zero
    bottom_failure = np.zeros(len(plot_df))

    for col in FAILURE_OUTCOMES:
        values = -plot_df[col].values
        ax.bar(x, values, bottom=bottom_failure, label=col)
        bottom_failure += values

    ax.axhline(0, linewidth=1)

    ax.set_title(
        f"Field of Study → Industry Sector → Success vs Failure Outcomes, {year}",
        fontsize=14
    )

    ax.set_xlabel("Connected industry sector")
    ax.set_ylabel("Company outcome value")

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=45, ha="right")

    ax.yaxis.set_major_formatter(FuncFormatter(compact_number))

    ax.legend(
        title="Outcome",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# RUN EVERYTHING
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_sector_year = read_degree_sector_year_data(DEGREE_FILE)
company_outcome_year = read_company_outcome_year_data(COMPANY_FILE)

print()
print("Degree field-year-sector rows:", len(degree_sector_year))
print("Company year-sector-outcome rows:", len(company_outcome_year))

connected_all_years = build_connected_summary(
    degree_sector_year,
    company_outcome_year
)

connected_top_years = connected_all_years[
    connected_all_years["rank_in_year"] <= TOP_N_SECTORS_PER_YEAR
].copy()

print()
print("CONNECTED ALL YEARS TABLE CREATED")
print("This table is connected by: year + field of study + industry sector + company outcome")
print("Rows:", len(connected_all_years))

available_years = sorted(connected_all_years["year"].unique())
print()
print("Available connected years:")
print(available_years)


# ============================================================
# DISPLAY TABLES FIRST, THEN CHARTS
# ============================================================

display_cols = [
    "year",
    "top_connected_fields_of_study",
    "sector",
    "total_degree_count",
    "degree_field_count",
    "Job openings",
    "Job gains",
    "Establishment births",
    "Job closings",
    "Job losses",
    "Establishment deaths",
    "success_total",
    "failure_total",
    "net_success_minus_failure",
    "success_rate_percent"
]

for year in SHOW_YEARS:
    year_df = connected_top_years[
        connected_top_years["year"] == year
    ].copy()

    print()
    print("=" * 120)
    print(f"FIELD OF STUDY → INDUSTRY CONNECTION TABLE, {year}")
    print("=" * 120)

    if year_df.empty:
        print(f"No connected data found for {year}.")
        continue

    table_df = year_df[display_cols].copy()

    table_df["success_rate_percent"] = table_df["success_rate_percent"].round(2)

    display(table_df)

    plot_year_stacked_chart(year_df, year)


# ============================================================
# OPTIONAL: DISPLAY ONE CLEAN COMBINED TABLE FOR THE 4 YEARS
# ============================================================

final_milestone_table = connected_top_years[
    connected_top_years["year"].isin(SHOW_YEARS)
][display_cols].copy()

final_milestone_table["success_rate_percent"] = (
    final_milestone_table["success_rate_percent"].round(2)
)

print()
print("=" * 120)
print("FINAL COMBINED CONNECTION TABLE: 1984, 2000, 2010, 2024")
print("=" * 120)

display(final_milestone_table)

# Connect 1 code

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# CONNECTED TABLE ONLY
# Field of Study → Industry Sector → Company Activity
# Years 1984–2024
# READ ONLY / NO SAVE
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 1984
END_YEAR = 2024
CHUNKSIZE = 100_000

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


# ============================================================
# HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def get_columns(path):
    return [str(c).strip().lower() for c in pd.read_csv(path, nrows=0).columns]


def pick_col(columns, exact_names, contains_words=None):
    for name in exact_names:
        if name in columns:
            return name

    if contains_words:
        for word in contains_words:
            matches = [c for c in columns if word in c]
            if matches:
                return matches[0]

    return None


def compact_number(x):
    if pd.isna(x):
        return ""
    x = float(x)

    if abs(x) >= 1_000_000_000:
        return f"{x / 1_000_000_000:.1f}B"
    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.1f}M"
    if abs(x) >= 1_000:
        return f"{x / 1_000:.1f}K"

    return f"{x:.0f}"


# ============================================================
# FIELD OF STUDY → INDUSTRY SECTOR
# ============================================================

def map_degree_field_to_sector(field):
    text = str(field).lower()

    if any(x in text for x in [
        "computer", "information science", "data", "software",
        "cyber", "mathematics", "statistics"
    ]):
        return "Information"

    if any(x in text for x in [
        "communication", "journalism", "library"
    ]):
        return "Information"

    if any(x in text for x in [
        "business", "management", "marketing",
        "administration", "entrepreneurship"
    ]):
        return "Professional and business services"

    if any(x in text for x in [
        "accounting", "finance", "financial", "insurance", "real estate"
    ]):
        return "Financial activities"

    if any(x in text for x in [
        "engineering", "mechanic", "mechanical", "electrical",
        "industrial", "manufacturing", "precision"
    ]):
        return "Manufacturing"

    if any(x in text for x in [
        "construction", "architecture", "architectural"
    ]):
        return "Construction"

    if any(x in text for x in [
        "health", "nursing", "medical", "clinical", "dental",
        "pharmacy", "therapy", "public health"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "education", "teacher", "teaching"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "agriculture", "natural resources", "mining", "forestry", "wildlife"
    ]):
        return "Natural resources and mining"

    if any(x in text for x in [
        "transportation", "logistics", "aviation"
    ]):
        return "Transportation and warehousing"

    if any(x in text for x in [
        "culinary", "hospitality", "hotel", "restaurant",
        "recreation", "leisure", "arts", "music", "film"
    ]):
        return "Leisure and hospitality"

    return "Professional and business services"


# ============================================================
# COMPANY INDUSTRY → INDUSTRY SECTOR
# ============================================================

def clean_code(code):
    code = str(code).lower().strip()

    if code.endswith(".0"):
        code = code[:-2]

    return code


def map_company_to_sector(industry_name=None, industry_code=None):
    text = str(industry_name).lower().strip()
    code = clean_code(industry_code)

    if text in ["nan", "", "none", "null"]:
        text = code

    if any(x in text for x in [
        "total private", "goods-producing", "service-providing",
        "all industries", "total"
    ]):
        return np.nan

    if any(x in text for x in ["natural resources", "mining", "agriculture"]) or code.startswith(("11", "21")):
        return "Natural resources and mining"

    if "construction" in text or code.startswith("23"):
        return "Construction"

    if "manufacturing" in text or code.startswith(("31", "32", "33")) or code in ["31-33"]:
        return "Manufacturing"

    if "wholesale" in text or code.startswith("42"):
        return "Wholesale trade"

    if "retail" in text or code.startswith(("44", "45")) or code in ["44-45"]:
        return "Retail trade"

    if any(x in text for x in ["transportation", "warehousing"]) or code.startswith(("48", "49")) or code in ["48-49"]:
        return "Transportation and warehousing"

    if "utilities" in text or code.startswith("22"):
        return "Utilities"

    if "information" in text or code.startswith("51"):
        return "Information"

    if any(x in text for x in ["financial", "finance", "insurance", "real estate"]) or code.startswith(("52", "53")):
        return "Financial activities"

    if any(x in text for x in [
        "professional", "business services", "management",
        "administrative", "support services"
    ]) or code.startswith(("54", "55", "56")):
        return "Professional and business services"

    if any(x in text for x in [
        "education", "educational", "health", "health care", "social assistance"
    ]) or code.startswith(("61", "62")):
        return "Education and health services"

    if any(x in text for x in [
        "leisure", "hospitality", "arts", "entertainment",
        "recreation", "accommodation", "food services"
    ]) or code.startswith(("71", "72")):
        return "Leisure and hospitality"

    if "other services" in text or code.startswith("81"):
        return "Other services"

    return np.nan


# ============================================================
# METRIC CLEANING
# This is optional.
# The table still keeps company_total_value even if metric does not match.
# ============================================================

def normalize_metric(metric):
    text = str(metric).lower().strip()
    text = text.replace("_", " ").replace("-", " ")

    if "opening" in text or text == "openings":
        return "job_openings"

    if "closing" in text or text == "closings":
        return "job_closings"

    if "gain" in text:
        return "job_gains"

    if "loss" in text:
        return "job_losses"

    if "birth" in text:
        return "establishment_births"

    if "death" in text:
        return "establishment_deaths"

    return None


# ============================================================
# READ DEGREE DATA BY YEAR + FIELD + SECTOR
# ============================================================

def read_degree_connection_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    field_col = pick_col(
        columns,
        ["field_group", "degree_field", "degree_fields", "field_of_study", "cip_field", "field"],
        ["field"]
    )

    count_col = pick_col(
        columns,
        ["degree_count", "total_degree_count", "count", "degrees"],
        ["degree_count"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in degree file.")

    if field_col is None:
        raise ValueError("Could not find field of study column in degree file.")

    print()
    print("DEGREE COLUMNS USED")
    print("year:", year_col)
    print("field of study:", field_col)
    print("degree count:", count_col)

    results = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)
        chunk["field_of_study"] = chunk[field_col].astype(str).str.strip()

        if count_col is not None and count_col in chunk.columns:
            chunk["degree_count"] = pd.to_numeric(chunk[count_col], errors="coerce").fillna(0)
        else:
            chunk["degree_count"] = 1

        chunk = chunk[
            (chunk["degree_count"] > 0)
            & (chunk["field_of_study"].str.lower() != "nan")
            & (chunk["field_of_study"] != "")
        ]

        if chunk.empty:
            continue

        chunk["connected_industry_sector"] = chunk["field_of_study"].apply(map_degree_field_to_sector)

        grouped = (
            chunk.groupby(
                ["year", "field_of_study", "connected_industry_sector"],
                as_index=False
            )["degree_count"]
            .sum()
        )

        results.append(grouped)

    if not results:
        raise ValueError("No degree data found.")

    degree_table = pd.concat(results, ignore_index=True)

    degree_table = (
        degree_table
        .groupby(["year", "field_of_study", "connected_industry_sector"], as_index=False)["degree_count"]
        .sum()
    )

    return degree_table


# ============================================================
# READ COMPANY DATA BY YEAR + SECTOR
# This does NOT drop 1984 or 2000 just because metrics are different.
# ============================================================

def read_company_connection_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    industry_col = pick_col(
        columns,
        ["industry_name_clean", "industry_name", "industry", "sector"],
        ["industry"]
    )

    industry_code_col = pick_col(
        columns,
        ["industry_code", "industrycode", "naics", "naics_code"],
        ["industry_code", "code", "naics"]
    )

    metric_col = pick_col(
        columns,
        ["metric_name", "metric", "measure", "variable", "data_type"],
        ["metric", "measure"]
    )

    value_col = pick_col(
        columns,
        ["value", "metric_value", "amount"],
        ["value"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in company file.")

    if value_col is None:
        raise ValueError("Could not find value column in company file.")

    print()
    print("COMPANY COLUMNS USED")
    print("year:", year_col)
    print("industry name:", industry_col)
    print("industry code:", industry_code_col)
    print("metric:", metric_col)
    print("value:", value_col)
    print("U.S. total filter: OFF, using all company rows")

    all_summary = []
    metric_summary = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)
        chunk["company_value"] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

        if industry_col is not None and industry_col in chunk.columns:
            industry_name = chunk[industry_col]
        else:
            industry_name = pd.Series([""] * len(chunk), index=chunk.index)

        if industry_code_col is not None and industry_code_col in chunk.columns:
            industry_code = chunk[industry_code_col]
        else:
            industry_code = pd.Series([""] * len(chunk), index=chunk.index)

        chunk["connected_industry_sector"] = [
            map_company_to_sector(n, c)
            for n, c in zip(industry_name, industry_code)
        ]

        chunk = chunk[
            chunk["connected_industry_sector"].notna()
            & (chunk["company_value"] > 0)
        ]

        if chunk.empty:
            continue

        if metric_col is not None and metric_col in chunk.columns:
            chunk["raw_metric"] = chunk[metric_col].astype(str).str.strip()
            chunk["clean_metric"] = chunk["raw_metric"].apply(normalize_metric)
        else:
            chunk["raw_metric"] = "unknown_metric"
            chunk["clean_metric"] = None

        # Total company activity by year and sector
        grouped_all = (
            chunk.groupby(["year", "connected_industry_sector"], as_index=False)
            .agg(
                company_total_value=("company_value", "sum"),
                company_rows=("company_value", "size"),
                company_metric_count=("raw_metric", "nunique")
            )
        )

        all_summary.append(grouped_all)

        # Optional success/failure columns only when metric names match
        known_metrics = chunk[chunk["clean_metric"].notna()].copy()

        if not known_metrics.empty:
            grouped_metric = (
                known_metrics.groupby(
                    ["year", "connected_industry_sector", "clean_metric"],
                    as_index=False
                )["company_value"]
                .sum()
            )

            metric_summary.append(grouped_metric)

    if not all_summary:
        raise ValueError("No mapped company data found.")

    company_table = pd.concat(all_summary, ignore_index=True)

    company_table = (
        company_table
        .groupby(["year", "connected_industry_sector"], as_index=False)
        .agg(
            company_total_value=("company_total_value", "sum"),
            company_rows=("company_rows", "sum"),
            company_metric_count=("company_metric_count", "max")
        )
    )

    if metric_summary:
        metric_table = pd.concat(metric_summary, ignore_index=True)

        metric_table = (
            metric_table
            .groupby(["year", "connected_industry_sector", "clean_metric"], as_index=False)["company_value"]
            .sum()
        )

        metric_pivot = metric_table.pivot_table(
            index=["year", "connected_industry_sector"],
            columns="clean_metric",
            values="company_value",
            aggfunc="sum",
            fill_value=0
        ).reset_index()

        metric_pivot.columns.name = None

        company_table = company_table.merge(
            metric_pivot,
            on=["year", "connected_industry_sector"],
            how="left"
        )

    metric_cols = [
        "job_openings",
        "job_gains",
        "establishment_births",
        "job_closings",
        "job_losses",
        "establishment_deaths"
    ]

    for col in metric_cols:
        if col not in company_table.columns:
            company_table[col] = 0
        else:
            company_table[col] = company_table[col].fillna(0)

    return company_table


# ============================================================
# RUN
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_table = read_degree_connection_data(DEGREE_FILE)
company_table = read_company_connection_data(COMPANY_FILE)

print()
print("Degree rows after grouping:", len(degree_table))
print("Company rows after grouping:", len(company_table))


# ============================================================
# CONNECT TABLE
# Field of study connects to company by:
# year + connected_industry_sector
# ============================================================

connected_table = degree_table.merge(
    company_table,
    on=["year", "connected_industry_sector"],
    how="left"
)

connected_table["connection_status"] = np.where(
    connected_table["company_total_value"].notna(),
    "connected",
    "degree sector only - no mapped company sector"
)

numeric_cols = [
    "company_total_value",
    "company_rows",
    "company_metric_count",
    "job_openings",
    "job_gains",
    "establishment_births",
    "job_closings",
    "job_losses",
    "establishment_deaths"
]

for col in numeric_cols:
    connected_table[col] = connected_table[col].fillna(0)

connected_table["success_total"] = (
    connected_table["job_openings"]
    + connected_table["job_gains"]
    + connected_table["establishment_births"]
)

connected_table["failure_total"] = (
    connected_table["job_closings"]
    + connected_table["job_losses"]
    + connected_table["establishment_deaths"]
)

connected_table["net_success_minus_failure"] = (
    connected_table["success_total"] - connected_table["failure_total"]
)

connected_table = connected_table.sort_values(
    ["year", "connected_industry_sector", "degree_count"],
    ascending=[True, True, False]
)


# ============================================================
# FINAL TABLE 1984–2024
# ============================================================

final_cols = [
    "year",
    "field_of_study",
    "connected_industry_sector",
    "degree_count",
    "company_total_value",
    "company_rows",
    "company_metric_count",
    "job_openings",
    "job_gains",
    "establishment_births",
    "job_closings",
    "job_losses",
    "establishment_deaths",
    "success_total",
    "failure_total",
    "net_success_minus_failure",
    "connection_status"
]

final_connected_table = connected_table[final_cols].copy()

print()
print("=" * 120)
print("FINAL CONNECTED TABLE: FIELD OF STUDY → INDUSTRY SECTOR → COMPANY ACTIVITY, 1984–2024")
print("=" * 120)
print("One row = one year + one field of study + its connected industry sector")
print("Years:", final_connected_table["year"].min(), "to", final_connected_table["year"].max())
print("Rows:", len(final_connected_table))

display(final_connected_table)


# ============================================================
# OPTIONAL SMALLER SUMMARY TABLE
# One row = year + industry sector
# ============================================================

summary_by_year_sector = (
    final_connected_table
    .groupby(["year", "connected_industry_sector"], as_index=False)
    .agg(
        total_degree_count=("degree_count", "sum"),
        field_of_study_count=("field_of_study", "nunique"),
        company_total_value=("company_total_value", "max"),
        company_rows=("company_rows", "max"),
        company_metric_count=("company_metric_count", "max"),
        job_openings=("job_openings", "max"),
        job_gains=("job_gains", "max"),
        establishment_births=("establishment_births", "max"),
        job_closings=("job_closings", "max"),
        job_losses=("job_losses", "max"),
        establishment_deaths=("establishment_deaths", "max"),
        success_total=("success_total", "max"),
        failure_total=("failure_total", "max"),
        net_success_minus_failure=("net_success_minus_failure", "max")
    )
    .sort_values(["year", "connected_industry_sector"])
)

print()
print("=" * 120)
print("SUMMARY TABLE: YEAR + INDUSTRY SECTOR")
print("=" * 120)

display(summary_by_year_sector)

# Check fix before 2008 code

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# FULL FIXED CONNECTED TABLE
# Field of Study → Industry Sector → Company Activity
# 1984–2024
# READ ONLY / NO SAVE
#
# FIX:
# Older years before 2008 may have degree_count = 0.
# This code uses fallback count columns so 1984–2007 do not disappear.
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

START_YEAR = 1984
END_YEAR = 2024
CHUNKSIZE = 100_000

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


# ============================================================
# BASIC HELPERS
# ============================================================

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def make_unique_columns(columns):
    """
    Fix duplicate column names after cleaning.
    Example:
    cipcode, cipcode -> cipcode, cipcode_2
    """
    seen = {}
    new_cols = []

    for col in columns:
        col = str(col).strip().lower()

        if col not in seen:
            seen[col] = 1
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")

    return new_cols


def clean_columns(df):
    df = df.copy()
    df.columns = make_unique_columns(df.columns)
    return df


def get_columns(path):
    columns = pd.read_csv(path, nrows=0).columns
    return make_unique_columns(columns)


def pick_col(columns, exact_names, contains_words=None):
    for name in exact_names:
        if name in columns:
            return name

    if contains_words:
        for word in contains_words:
            matches = [c for c in columns if word in c]
            if matches:
                return matches[0]

    return None


def clean_text_series(s):
    """
    If duplicate column names make pandas return a DataFrame,
    use the first column.
    """
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]

    s = s.astype(str).str.strip()
    s = s.replace(
        ["nan", "NaN", "None", "NONE", "none", "null", "NULL", ""],
        np.nan
    )
    return s


def compact_number(x):
    if pd.isna(x):
        return ""

    x = float(x)

    if abs(x) >= 1_000_000_000:
        return f"{x / 1_000_000_000:.1f}B"
    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.1f}M"
    if abs(x) >= 1_000:
        return f"{x / 1_000:.1f}K"

    return f"{x:.0f}"


# ============================================================
# FIELD OF STUDY FALLBACK
# ============================================================

def make_field_of_study(chunk):
    possible_cols = [
        "field_group",
        "degree_field",
        "degree_fields",
        "field_of_study",
        "cip_field",
        "field_subgroup",
        "degree_subfield",
        "degree_subfields",
        "subfield",
        "major_name",
        "degree_major",
        "degree_majors",
        "major",
        "cipcode",
        "cip_code"
    ]

    field = pd.Series(np.nan, index=chunk.index)
    used_cols = []

    for col in possible_cols:
        if col in chunk.columns:
            temp = clean_text_series(chunk[col])
            field = field.fillna(temp)
            used_cols.append(col)

    if not used_cols:
        raise ValueError("No usable field of study column found.")

    return field, used_cols


# ============================================================
# DEGREE COUNT FALLBACK
# This is the important fix for before 2008.
# ============================================================

def sum_existing_columns(chunk, cols):
    """
    Sum only columns that exist.
    Returns a Series of 0 if none exist.
    """
    existing = [c for c in cols if c in chunk.columns]

    if not existing:
        return pd.Series(0, index=chunk.index, dtype="float64"), []

    total = pd.Series(0, index=chunk.index, dtype="float64")

    for col in existing:
        data = chunk[col]

        if isinstance(data, pd.DataFrame):
            data = data.iloc[:, 0]

        total = total + pd.to_numeric(data, errors="coerce").fillna(0)

    return total, existing


def build_degree_count(chunk, count_col):
    """
    Uses degree_count first.
    If degree_count is 0 or missing, fallback to older IPEDS count columns.

    Fallback order:
    1. degree_count / count column
    2. ctotal / ctotalt / total completer columns
    3. totalm + totalw
    4. bal_m + bal_w
    5. crace15 + crace16
    6. crace01-crace14
    7. row_count_1 backup, only to keep old years visible
    """

    count = pd.Series(0, index=chunk.index, dtype="float64")
    source = pd.Series("missing_or_zero", index=chunk.index)

    # -----------------------------
    # 1. Main degree_count column
    # -----------------------------
    if count_col is not None and count_col in chunk.columns:
        data = chunk[count_col]

        if isinstance(data, pd.DataFrame):
            data = data.iloc[:, 0]

        main_count = pd.to_numeric(data, errors="coerce").fillna(0)
        mask = main_count > 0

        count.loc[mask] = main_count.loc[mask]
        source.loc[mask] = "main_degree_count"

    # -----------------------------
    # 2. Total-like columns
    # -----------------------------
    total_like_cols = [
        "ctotal",
        "ctotalt",
        "total",
        "total_count",
        "total_degrees",
        "total_degree_count",
        "grand_total"
    ]

    total_like, used_total_like = sum_existing_columns(chunk, total_like_cols)
    mask = (count <= 0) & (total_like > 0)

    count.loc[mask] = total_like.loc[mask]
    source.loc[mask] = "total_like_columns"

    # -----------------------------
    # 3. totalm + totalw
    # -----------------------------
    total_mw, used_total_mw = sum_existing_columns(chunk, ["totalm", "totalw"])
    mask = (count <= 0) & (total_mw > 0)

    count.loc[mask] = total_mw.loc[mask]
    source.loc[mask] = "totalm_totalw"

    # -----------------------------
    # 4. bal_m + bal_w
    # -----------------------------
    bal_mw, used_bal = sum_existing_columns(chunk, ["bal_m", "bal_w"])
    mask = (count <= 0) & (bal_mw > 0)

    count.loc[mask] = bal_mw.loc[mask]
    source.loc[mask] = "bal_m_bal_w"

    # -----------------------------
    # 5. crace15 + crace16
    # -----------------------------
    crace_15_16, used_crace_15_16 = sum_existing_columns(chunk, ["crace15", "crace16"])
    mask = (count <= 0) & (crace_15_16 > 0)

    count.loc[mask] = crace_15_16.loc[mask]
    source.loc[mask] = "crace15_crace16"

    # -----------------------------
    # 6. crace01-crace14
    # -----------------------------
    crace_01_14_cols = [f"crace{i:02d}" for i in range(1, 15)]
    crace_01_14, used_crace_01_14 = sum_existing_columns(chunk, crace_01_14_cols)
    mask = (count <= 0) & (crace_01_14 > 0)

    count.loc[mask] = crace_01_14.loc[mask]
    source.loc[mask] = "crace01_to_crace14"

    # -----------------------------
    # 7. Final backup
    # This keeps old year rows visible.
    # It is not a true degree count if no count columns exist.
    # -----------------------------
    if "field_of_study" in chunk.columns:
        mask = (count <= 0) & (chunk["field_of_study"].notna())
        count.loc[mask] = 1
        source.loc[mask] = "row_count_1_backup"

    used_cols = {
        "total_like": used_total_like,
        "totalm_totalw": used_total_mw,
        "bal_m_bal_w": used_bal,
        "crace15_crace16": used_crace_15_16,
        "crace01_to_crace14": used_crace_01_14
    }

    return count, source, used_cols


# ============================================================
# DEGREE FIELD → INDUSTRY SECTOR
# ============================================================

def map_degree_field_to_sector(field):
    text = str(field).lower()

    if any(x in text for x in [
        "computer", "information science", "information technology",
        "data", "software", "cyber", "mathematics", "statistics",
        "computer and information"
    ]):
        return "Information"

    if any(x in text for x in [
        "communication", "journalism", "library"
    ]):
        return "Information"

    if any(x in text for x in [
        "business", "management", "marketing",
        "administration", "entrepreneurship"
    ]):
        return "Professional and business services"

    if any(x in text for x in [
        "accounting", "finance", "financial", "insurance", "real estate"
    ]):
        return "Financial activities"

    if any(x in text for x in [
        "engineering", "mechanic", "mechanical", "electrical",
        "industrial", "manufacturing", "precision"
    ]):
        return "Manufacturing"

    if any(x in text for x in [
        "construction", "architecture", "architectural"
    ]):
        return "Construction"

    if any(x in text for x in [
        "health", "nursing", "medical", "clinical", "dental",
        "pharmacy", "therapy", "public health", "medicine"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "education", "teacher", "teaching"
    ]):
        return "Education and health services"

    if any(x in text for x in [
        "agriculture", "natural resources", "mining", "forestry", "wildlife"
    ]):
        return "Natural resources and mining"

    if any(x in text for x in [
        "transportation", "logistics", "aviation"
    ]):
        return "Transportation and warehousing"

    if any(x in text for x in [
        "culinary", "hospitality", "hotel", "restaurant",
        "recreation", "leisure", "arts", "music", "film",
        "visual and performing arts"
    ]):
        return "Leisure and hospitality"

    return "Professional and business services"


# ============================================================
# COMPANY INDUSTRY → INDUSTRY SECTOR
# Includes older company industry codes before 2008.
# ============================================================

def clean_code(code):
    code = str(code).lower().strip()

    if code.endswith(".0"):
        code = code[:-2]

    return code


def map_company_to_sector(industry_name=None, industry_code=None):
    text = str(industry_name).lower().strip()
    code = clean_code(industry_code)

    if text in ["nan", "", "none", "null"]:
        text = code

    # Remove total rows because they do not represent one sector.
    if any(x in text for x in [
        "total private", "goods-producing", "service-providing",
        "all industries", "total"
    ]):
        return np.nan

    if any(x in text for x in [
        "natural resources", "mining", "agriculture"
    ]) or code in ["11", "21", "100010"]:
        return "Natural resources and mining"

    if "construction" in text or code in ["23", "100020"]:
        return "Construction"

    if "manufacturing" in text or code in ["31", "32", "33", "31-33", "100030"]:
        return "Manufacturing"

    if "wholesale" in text or code in ["42", "200010"]:
        return "Wholesale trade"

    if "retail" in text or code in ["44", "45", "44-45", "200020"]:
        return "Retail trade"

    if any(x in text for x in [
        "transportation", "warehousing"
    ]) or code in ["48", "49", "48-49", "200030"]:
        return "Transportation and warehousing"

    if "utilities" in text or code in ["22", "200040"]:
        return "Utilities"

    if "information" in text or code in ["51", "200050"]:
        return "Information"

    if any(x in text for x in [
        "financial", "finance", "insurance", "real estate"
    ]) or code in ["52", "53", "200060"]:
        return "Financial activities"

    if any(x in text for x in [
        "professional", "business services", "management",
        "administrative", "support services"
    ]) or code in ["54", "55", "56", "200070"]:
        return "Professional and business services"

    if any(x in text for x in [
        "education", "educational", "health", "health care", "social assistance"
    ]) or code in ["61", "62", "200080"]:
        return "Education and health services"

    if any(x in text for x in [
        "leisure", "hospitality", "arts", "entertainment",
        "recreation", "accommodation", "food services"
    ]) or code in ["71", "72", "200090"]:
        return "Leisure and hospitality"

    if "other services" in text or code in ["81", "200100"]:
        return "Other services"

    return np.nan


# ============================================================
# METRIC CLEANING
# ============================================================

def normalize_metric(metric):
    text = str(metric).lower().strip()
    text = text.replace("_", " ").replace("-", " ")

    if "opening" in text or text == "openings":
        return "job_openings"

    if "closing" in text or text == "closings":
        return "job_closings"

    if "gain" in text:
        return "job_gains"

    if "loss" in text:
        return "job_losses"

    if "birth" in text:
        return "establishment_births"

    if "death" in text:
        return "establishment_deaths"

    return None


# ============================================================
# READ DEGREE DATA
# ============================================================

def read_degree_connection_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    count_col = pick_col(
        columns,
        ["degree_count", "total_degree_count", "count", "degrees"],
        ["degree_count"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in degree file.")

    print()
    print("=" * 120)
    print("DEGREE COLUMNS USED")
    print("=" * 120)
    print("year:", year_col)
    print("degree count:", count_col)
    print("field of study: fallback system")

    results = []
    year_debug = []
    field_cols_seen = set()
    count_source_debug = []
    fallback_cols_seen = {}

    chunk_number = 0

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk_number += 1
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)

        # Field fallback first
        chunk["field_of_study"], used_cols = make_field_of_study(chunk)
        field_cols_seen.update(used_cols)

        # Degree count fallback fix
        chunk["degree_count"], chunk["degree_count_source"], used_fallback_cols = build_degree_count(
            chunk,
            count_col
        )

        for key, val in used_fallback_cols.items():
            if key not in fallback_cols_seen:
                fallback_cols_seen[key] = set()
            fallback_cols_seen[key].update(val)

        # Debug before final filter
        year_part = (
            chunk.groupby("year", as_index=False)
            .agg(
                raw_degree_rows=("year", "size"),
                rows_with_field=("field_of_study", lambda x: x.notna().sum()),
                rows_with_positive_degree_count=("degree_count", lambda x: (x > 0).sum())
            )
        )
        year_debug.append(year_part)

        source_part = (
            chunk.groupby(["year", "degree_count_source"], as_index=False)
            .size()
            .rename(columns={"size": "rows"})
        )
        count_source_debug.append(source_part)

        # Keep only useful rows
        chunk = chunk[
            (chunk["degree_count"] > 0)
            & (chunk["field_of_study"].notna())
        ]

        if chunk.empty:
            continue

        chunk["connected_industry_sector"] = chunk["field_of_study"].apply(
            map_degree_field_to_sector
        )

        grouped = (
            chunk.groupby(
                ["year", "field_of_study", "connected_industry_sector"],
                as_index=False
            )["degree_count"]
            .sum()
        )

        results.append(grouped)

    if not results:
        raise ValueError("No degree data found after fallback count fix.")

    degree_table = pd.concat(results, ignore_index=True)

    degree_table = (
        degree_table
        .groupby(
            ["year", "field_of_study", "connected_industry_sector"],
            as_index=False
        )["degree_count"]
        .sum()
    )

    degree_year_debug = pd.concat(year_debug, ignore_index=True)
    degree_year_debug = (
        degree_year_debug
        .groupby("year", as_index=False)
        .agg(
            raw_degree_rows=("raw_degree_rows", "sum"),
            rows_with_field=("rows_with_field", "sum"),
            rows_with_positive_degree_count=("rows_with_positive_degree_count", "sum")
        )
        .sort_values("year")
    )

    degree_count_source_debug = pd.concat(count_source_debug, ignore_index=True)
    degree_count_source_debug = (
        degree_count_source_debug
        .groupby(["year", "degree_count_source"], as_index=False)["rows"]
        .sum()
        .sort_values(["year", "degree_count_source"])
    )

    print()
    print("field columns used as fallback:")
    print(sorted(field_cols_seen))

    print()
    print("degree count fallback columns seen:")
    for key, val in fallback_cols_seen.items():
        print(key, ":", sorted(val))

    print()
    print("degree year range after fallback:")
    print(degree_table["year"].min(), "to", degree_table["year"].max())

    print()
    print("=" * 120)
    print("DEGREE YEAR CHECK")
    print("This should show 1984 to 2024 if old degree rows exist.")
    print("=" * 120)
    display(degree_year_debug)

    print()
    print("=" * 120)
    print("DEGREE COUNT SOURCE CHECK")
    print("This shows where the degree_count came from by year.")
    print("=" * 120)
    display(degree_count_source_debug)

    return degree_table


# ============================================================
# READ COMPANY DATA
# ============================================================

def read_company_connection_data(path):
    columns = get_columns(path)

    year_col = pick_col(columns, ["year"], ["year"])

    industry_col = pick_col(
        columns,
        ["industry_name_clean", "industry_name", "industry", "sector"],
        ["industry"]
    )

    industry_code_col = pick_col(
        columns,
        ["industry_code", "industrycode", "naics", "naics_code"],
        ["industry_code", "code", "naics"]
    )

    metric_col = pick_col(
        columns,
        ["metric_name", "metric", "measure", "variable", "data_type"],
        ["metric", "measure"]
    )

    value_col = pick_col(
        columns,
        ["value", "metric_value", "amount"],
        ["value"]
    )

    if year_col is None:
        raise ValueError("Could not find year column in company file.")

    if value_col is None:
        raise ValueError("Could not find value column in company file.")

    print()
    print("=" * 120)
    print("COMPANY COLUMNS USED")
    print("=" * 120)
    print("year:", year_col)
    print("industry name:", industry_col)
    print("industry code:", industry_code_col)
    print("metric:", metric_col)
    print("value:", value_col)

    all_summary = []
    metric_summary = []
    company_year_debug = []

    for chunk in pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)

        chunk["year"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[chunk["year"].between(START_YEAR, END_YEAR)]

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)

        value_data = chunk[value_col]
        if isinstance(value_data, pd.DataFrame):
            value_data = value_data.iloc[:, 0]

        chunk["company_value"] = pd.to_numeric(
            value_data,
            errors="coerce"
        ).fillna(0)

        if industry_col is not None and industry_col in chunk.columns:
            industry_name = chunk[industry_col]
            if isinstance(industry_name, pd.DataFrame):
                industry_name = industry_name.iloc[:, 0]
        else:
            industry_name = pd.Series([""] * len(chunk), index=chunk.index)

        if industry_code_col is not None and industry_code_col in chunk.columns:
            industry_code = chunk[industry_code_col]
            if isinstance(industry_code, pd.DataFrame):
                industry_code = industry_code.iloc[:, 0]
        else:
            industry_code = pd.Series([""] * len(chunk), index=chunk.index)

        chunk["connected_industry_sector"] = [
            map_company_to_sector(n, c)
            for n, c in zip(industry_name, industry_code)
        ]

        debug_part = (
            chunk.groupby("year", as_index=False)
            .agg(
                raw_company_rows=("year", "size"),
                rows_with_positive_company_value=("company_value", lambda x: (x > 0).sum()),
                rows_mapped_to_sector=("connected_industry_sector", lambda x: x.notna().sum())
            )
        )
        company_year_debug.append(debug_part)

        chunk = chunk[
            chunk["connected_industry_sector"].notna()
            & (chunk["company_value"] > 0)
        ]

        if chunk.empty:
            continue

        if metric_col is not None and metric_col in chunk.columns:
            metric_data = chunk[metric_col]
            if isinstance(metric_data, pd.DataFrame):
                metric_data = metric_data.iloc[:, 0]

            chunk["raw_metric"] = metric_data.astype(str).str.strip()
            chunk["clean_metric"] = chunk["raw_metric"].apply(normalize_metric)
        else:
            chunk["raw_metric"] = "unknown_metric"
            chunk["clean_metric"] = None

        grouped_all = (
            chunk.groupby(["year", "connected_industry_sector"], as_index=False)
            .agg(
                company_total_value=("company_value", "sum"),
                company_rows=("company_value", "size"),
                company_metric_count=("raw_metric", "nunique")
            )
        )

        all_summary.append(grouped_all)

        known = chunk[chunk["clean_metric"].notna()].copy()

        if not known.empty:
            grouped_metric = (
                known.groupby(
                    ["year", "connected_industry_sector", "clean_metric"],
                    as_index=False
                )["company_value"]
                .sum()
            )

            metric_summary.append(grouped_metric)

    if not all_summary:
        raise ValueError("No mapped company data found.")

    company_table = pd.concat(all_summary, ignore_index=True)

    company_table = (
        company_table
        .groupby(["year", "connected_industry_sector"], as_index=False)
        .agg(
            company_total_value=("company_total_value", "sum"),
            company_rows=("company_rows", "sum"),
            company_metric_count=("company_metric_count", "max")
        )
    )

    if metric_summary:
        metric_table = pd.concat(metric_summary, ignore_index=True)

        metric_table = (
            metric_table
            .groupby(
                ["year", "connected_industry_sector", "clean_metric"],
                as_index=False
            )["company_value"]
            .sum()
        )

        metric_pivot = metric_table.pivot_table(
            index=["year", "connected_industry_sector"],
            columns="clean_metric",
            values="company_value",
            aggfunc="sum",
            fill_value=0
        ).reset_index()

        metric_pivot.columns.name = None

        company_table = company_table.merge(
            metric_pivot,
            on=["year", "connected_industry_sector"],
            how="left"
        )

    metric_cols = [
        "job_openings",
        "job_gains",
        "establishment_births",
        "job_closings",
        "job_losses",
        "establishment_deaths"
    ]

    for col in metric_cols:
        if col not in company_table.columns:
            company_table[col] = 0
        else:
            company_table[col] = company_table[col].fillna(0)

    company_year_debug = pd.concat(company_year_debug, ignore_index=True)
    company_year_debug = (
        company_year_debug
        .groupby("year", as_index=False)
        .agg(
            raw_company_rows=("raw_company_rows", "sum"),
            rows_with_positive_company_value=("rows_with_positive_company_value", "sum"),
            rows_mapped_to_sector=("rows_mapped_to_sector", "sum")
        )
        .sort_values("year")
    )

    print()
    print("company year range after mapping:")
    print(company_table["year"].min(), "to", company_table["year"].max())

    print()
    print("=" * 120)
    print("COMPANY YEAR CHECK")
    print("If this starts at 2008, then the company file itself only has usable mapped company data from 2008 forward.")
    print("=" * 120)
    display(company_year_debug)

    return company_table


# ============================================================
# RUN
# ============================================================

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_table = read_degree_connection_data(DEGREE_FILE)
company_table = read_company_connection_data(COMPANY_FILE)

print()
print("=" * 120)
print("GROUPED ROW COUNTS")
print("=" * 120)
print("Degree grouped rows:", len(degree_table))
print("Company grouped rows:", len(company_table))


# ============================================================
# CONNECT DEGREE + COMPANY
# Connection key:
# year + connected_industry_sector
# ============================================================

connected_table = degree_table.merge(
    company_table,
    on=["year", "connected_industry_sector"],
    how="left"
)

connected_table["connection_status"] = np.where(
    connected_table["company_total_value"].notna(),
    "connected",
    "degree sector only - no mapped company sector"
)

numeric_cols = [
    "company_total_value",
    "company_rows",
    "company_metric_count",
    "job_openings",
    "job_gains",
    "establishment_births",
    "job_closings",
    "job_losses",
    "establishment_deaths"
]

for col in numeric_cols:
    connected_table[col] = connected_table[col].fillna(0)

connected_table["success_total"] = (
    connected_table["job_openings"]
    + connected_table["job_gains"]
    + connected_table["establishment_births"]
)

connected_table["failure_total"] = (
    connected_table["job_closings"]
    + connected_table["job_losses"]
    + connected_table["establishment_deaths"]
)

connected_table["net_success_minus_failure"] = (
    connected_table["success_total"] - connected_table["failure_total"]
)

connected_table = connected_table.sort_values(
    ["year", "connected_industry_sector", "degree_count"],
    ascending=[True, True, False]
)


# ============================================================
# FINAL DETAILED TABLE
# One row = year + field of study + connected industry sector
# ============================================================

final_cols = [
    "year",
    "field_of_study",
    "connected_industry_sector",
    "degree_count",
    "company_total_value",
    "company_rows",
    "company_metric_count",
    "job_openings",
    "job_gains",
    "establishment_births",
    "job_closings",
    "job_losses",
    "establishment_deaths",
    "success_total",
    "failure_total",
    "net_success_minus_failure",
    "connection_status"
]

final_connected_table = connected_table[final_cols].copy()

print()
print("=" * 120)
print("FINAL CONNECTED TABLE: FIELD OF STUDY → INDUSTRY SECTOR → COMPANY ACTIVITY, 1984–2024")
print("=" * 120)
print("One row = one year + one field of study + its connected industry sector")
print("Years:", final_connected_table["year"].min(), "to", final_connected_table["year"].max())
print("Rows:", len(final_connected_table))
print()
print("Connection status count:")
print(final_connected_table["connection_status"].value_counts())

display(final_connected_table)


# ============================================================
# CHECK TABLE BY YEAR
# This shows whether 1984–2007 are connecting or degree-only.
# ============================================================

year_check = (
    final_connected_table
    .groupby(["year", "connection_status"], as_index=False)
    .size()
    .rename(columns={"size": "rows"})
    .sort_values(["year", "connection_status"])
)

print()
print("=" * 120)
print("CHECK: CONNECTION ROWS BY YEAR")
print("=" * 120)
display(year_check)


# ============================================================
# CLEAN SUMMARY TABLE
# One row = year + industry sector
# ============================================================

summary_by_year_sector = (
    final_connected_table
    .groupby(["year", "connected_industry_sector"], as_index=False)
    .agg(
        total_degree_count=("degree_count", "sum"),
        field_of_study_count=("field_of_study", "nunique"),
        company_total_value=("company_total_value", "max"),
        company_rows=("company_rows", "max"),
        company_metric_count=("company_metric_count", "max"),
        job_openings=("job_openings", "max"),
        job_gains=("job_gains", "max"),
        establishment_births=("establishment_births", "max"),
        job_closings=("job_closings", "max"),
        job_losses=("job_losses", "max"),
        establishment_deaths=("establishment_deaths", "max"),
        success_total=("success_total", "max"),
        failure_total=("failure_total", "max"),
        net_success_minus_failure=("net_success_minus_failure", "max")
    )
    .sort_values(["year", "connected_industry_sector"])
)

print()
print("=" * 120)
print("SUMMARY TABLE: YEAR + INDUSTRY SECTOR")
print("=" * 120)
display(summary_by_year_sector)


# ============================================================
# IMPORTANT YEARS ONLY
# This makes it easy to confirm old years are now visible.
# ============================================================

important_years = [1984, 1990, 2000, 2007, 2008, 2010, 2024]

important_table = final_connected_table[
    final_connected_table["year"].isin(important_years)
].copy()

print()
print("=" * 120)
print("IMPORTANT YEARS ONLY: 1984, 1990, 2000, 2007, 2008, 2010, 2024")
print("=" * 120)
display(important_table)


# ============================================================
# SIMPLE FINAL YEAR LIST
# ============================================================

available_years = sorted(final_connected_table["year"].dropna().unique())

print()
print("=" * 120)
print("AVAILABLE YEARS IN FINAL TABLE")
print("=" * 120)
print(available_years)

| Chart goal                          | Use these columns together                                                                               | Formula                                                                |
| ----------------------------------- | -------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------- |
| Degree → Industry → success/failure | `field_of_study`, `connected_industry_sector`, `success_total`, `failure_total`                          | `success_total = job_openings + job_gains + establishment_births`      |
| Job market only                     | `field_of_study`, `connected_industry_sector`, `job_openings`, `job_closings`, `job_gains`, `job_losses` | `job_net = job_openings + job_gains - job_closings - job_losses`       |
| Startup only                        | `field_of_study`, `connected_industry_sector`, `establishment_births`, `establishment_deaths`            | `startup_net = establishment_births - establishment_deaths`            |
| Overall good vs bad                 | `field_of_study`, `connected_industry_sector`, `success_total`, `failure_total`                          | `net_success_minus_failure = success_total - failure_total`            |
| Success rate                        | `success_total`, `failure_total`                                                                         | `success_rate = success_total / (success_total + failure_total) * 100` |
| Failure rate                        | `success_total`, `failure_total`                                                                         | `failure_rate = failure_total / (success_total + failure_total) * 100` |


# Formula to use

| Chart goal                          | Use these columns together                                                                               | Formula                                                                |
| ----------------------------------- | -------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------- |
| Degree → Industry → success/failure | `field_of_study`, `connected_industry_sector`, `success_total`, `failure_total`                          | `success_total = job_openings + job_gains + establishment_births`      |
| Job market only                     | `field_of_study`, `connected_industry_sector`, `job_openings`, `job_closings`, `job_gains`, `job_losses` | `job_net = job_openings + job_gains - job_closings - job_losses`       |
| Startup only                        | `field_of_study`, `connected_industry_sector`, `establishment_births`, `establishment_deaths`            | `startup_net = establishment_births - establishment_deaths`            |
| Overall good vs bad                 | `field_of_study`, `connected_industry_sector`, `success_total`, `failure_total`                          | `net_success_minus_failure = success_total - failure_total`            |
| Success rate                        | `success_total`, `failure_total`                                                                         | `success_rate = success_total / (success_total + failure_total) * 100` |
| Failure rate                        | `success_total`, `failure_total`                                                                         | `failure_rate = failure_total / (success_total + failure_total) * 100` |


# Check for 0 Fail?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ============================================================
# YEAR ZERO CHECK TABLE FIRST
# Reads company file directly
# No chart
# No save
# Memory optimized with chunks
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

YEAR_START = 1984
YEAR_END = 2024
CHUNKSIZE = 100_000

if not COMPANY_FILE.exists():
    raise FileNotFoundError(f"Missing file: {COMPANY_FILE}")

print("FOUND COMPANY FILE:")
print(COMPANY_FILE)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def clean_name(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

def find_column(columns, possible_names):
    clean_map = {clean_name(c): c for c in columns}
    
    for name in possible_names:
        name_clean = clean_name(name)
        if name_clean in clean_map:
            return clean_map[name_clean]
    
    for c in columns:
        c_clean = clean_name(c)
        for name in possible_names:
            name_clean = clean_name(name)
            if name_clean in c_clean:
                return c
    
    return None

def classify_metric(row_text):
    text = str(row_text).lower()
    
    # Job openings
    if (
        "job opening" in text
        or "openings" in text
        or "opening" in text
    ):
        return "job_openings"
    
    # Job closings
    if (
        "job closing" in text
        or "closings" in text
        or "closing" in text
    ):
        return "job_closings"
    
    # Job gains
    if (
        "job gain" in text
        or "gross job gain" in text
        or "gains" in text
    ):
        return "job_gains"
    
    # Job losses
    if (
        "job loss" in text
        or "gross job loss" in text
        or "losses" in text
    ):
        return "job_losses"
    
    # Establishment births
    if (
        "establishment birth" in text
        or "births" in text
        or "birth" in text
    ):
        return "establishment_births"
    
    # Establishment deaths
    if (
        "establishment death" in text
        or "deaths" in text
        or "death" in text
    ):
        return "establishment_deaths"
    
    return None

# ------------------------------------------------------------
# 1. Read only column names first
# ------------------------------------------------------------
columns = list(pd.read_csv(COMPANY_FILE, nrows=0).columns)

column_table = pd.DataFrame({
    "column_number": range(1, len(columns) + 1),
    "column_name": columns
})

print("\nTABLE 0: COMPANY FILE COLUMN NAMES")
display(column_table)

# ------------------------------------------------------------
# 2. Find year column
# ------------------------------------------------------------
year_col = find_column(columns, ["year", "data_year", "time_year"])

if year_col is None:
    raise KeyError("Cannot find year column.")

print("\nYEAR COLUMN USED:")
print(year_col)

# ------------------------------------------------------------
# 3. Try to find wide metric columns first
# ------------------------------------------------------------
wide_metric_candidates = {
    "job_openings": [
        "job_openings", "job openings", "openings", "opening"
    ],
    "job_closings": [
        "job_closings", "job closings", "closings", "closing"
    ],
    "job_gains": [
        "job_gains", "job gains", "gross job gains", "gains"
    ],
    "job_losses": [
        "job_losses", "job losses", "gross job losses", "losses"
    ],
    "establishment_births": [
        "establishment_births", "establishment births", "births", "birth"
    ],
    "establishment_deaths": [
        "establishment_deaths", "establishment deaths", "deaths", "death"
    ],
}

wide_found = {}

for standard_name, possible_names in wide_metric_candidates.items():
    found_col = find_column(columns, possible_names)
    if found_col is not None:
        wide_found[standard_name] = found_col

wide_found_table = pd.DataFrame([
    {
        "standard_metric_needed": k,
        "column_found_in_file": v
    }
    for k, v in wide_found.items()
])

print("\nTABLE 1: WIDE METRIC COLUMN CHECK")
display(wide_found_table)

# ------------------------------------------------------------
# 4. If wide columns exist, use wide mode
# ------------------------------------------------------------
required_metrics = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
]

year_parts = []

if len(wide_found) >= 2:
    print("\nMODE USED: WIDE DATA MODE")
    
    use_cols = [year_col] + list(wide_found.values())
    
    for chunk in pd.read_csv(COMPANY_FILE, usecols=use_cols, chunksize=CHUNKSIZE, low_memory=False):
        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk[(chunk[year_col] >= YEAR_START) & (chunk[year_col] <= YEAR_END)].copy()
        
        if len(chunk) == 0:
            continue
        
        temp = pd.DataFrame()
        temp["year"] = chunk[year_col].astype(int)
        
        for standard_name in required_metrics:
            if standard_name in wide_found:
                temp[standard_name] = pd.to_numeric(chunk[wide_found[standard_name]], errors="coerce").fillna(0)
            else:
                temp[standard_name] = 0
        
        year_parts.append(temp)

# ------------------------------------------------------------
# 5. Otherwise use long mode
# ------------------------------------------------------------
else:
    print("\nMODE USED: LONG DATA MODE")
    print("This means your file probably has one value column and metric names stored in text columns.")
    
    value_col = find_column(columns, [
        "value",
        "data_value",
        "company_value",
        "total_company_value",
        "amount",
        "estimate",
        "count"
    ])
    
    if value_col is None:
        print("\nCould not find a value column.")
        print("Look at TABLE 0 and tell me which column contains the number values.")
        raise KeyError("Cannot find value column.")
    
    print("\nVALUE COLUMN USED:")
    print(value_col)
    
    # text columns are used to find words like openings, closings, gains, losses, births, deaths
    sample = pd.read_csv(COMPANY_FILE, nrows=1000, low_memory=False)
    
    text_cols = []
    for col in sample.columns:
        if col not in [year_col, value_col]:
            if sample[col].dtype == "object":
                text_cols.append(col)
    
    print("\nTEXT COLUMNS USED TO FIND METRIC TYPE:")
    display(pd.DataFrame({"text_column_used": text_cols}))
    
    use_cols = [year_col, value_col] + text_cols
    
    metric_sample_rows = []
    
    for chunk in pd.read_csv(COMPANY_FILE, usecols=use_cols, chunksize=CHUNKSIZE, low_memory=False):
        chunk[year_col] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)
        
        chunk = chunk[(chunk[year_col] >= YEAR_START) & (chunk[year_col] <= YEAR_END)].copy()
        
        if len(chunk) == 0:
            continue
        
        text_block = chunk[text_cols].fillna("").astype(str).agg(" | ".join, axis=1)
        chunk["standard_metric"] = text_block.apply(classify_metric)
        
        found_metric_rows = chunk[chunk["standard_metric"].notna()].copy()
        
        if len(found_metric_rows) == 0:
            continue
        
        if len(metric_sample_rows) < 20:
            metric_sample_rows.append(
                found_metric_rows[[year_col, value_col, "standard_metric"] + text_cols].head(20)
            )
        
        temp = found_metric_rows[[year_col, value_col, "standard_metric"]].copy()
        temp = temp.rename(columns={
            year_col: "year",
            value_col: "value"
        })
        temp["year"] = temp["year"].astype(int)
        
        pivot = (
            temp
            .groupby(["year", "standard_metric"], as_index=False)["value"]
            .sum()
            .pivot(index="year", columns="standard_metric", values="value")
            .reset_index()
        )
        
        for metric in required_metrics:
            if metric not in pivot.columns:
                pivot[metric] = 0
        
        year_parts.append(pivot[["year"] + required_metrics])
    
    if metric_sample_rows:
        print("\nTABLE 2: SAMPLE ROWS WHERE METRIC WAS FOUND")
        display(pd.concat(metric_sample_rows, ignore_index=True).head(30))

# ------------------------------------------------------------
# 6. Build yearly table
# ------------------------------------------------------------
if not year_parts:
    print("\nNo matching metric rows found.")
    print("This means the metric names may use different words than openings, closings, gains, losses, births, deaths.")
    raise ValueError("No yearly metric table could be created.")

all_year_data = pd.concat(year_parts, ignore_index=True)

year_table = (
    all_year_data
    .groupby("year", as_index=False)[required_metrics]
    .sum()
)

# Make sure all years 1984-2024 show, even missing years
all_years = pd.DataFrame({
    "year": list(range(YEAR_START, YEAR_END + 1))
})

year_table = all_years.merge(year_table, on="year", how="left")

for col in required_metrics:
    year_table[col] = year_table[col].fillna(0)

# ------------------------------------------------------------
# 7. Create formulas
# ------------------------------------------------------------
year_table["success_total"] = (
    year_table["job_openings"]
    + year_table["job_gains"]
    + year_table["establishment_births"]
)

year_table["failure_total"] = (
    year_table["job_closings"]
    + year_table["job_losses"]
    + year_table["establishment_deaths"]
)

year_table["total_activity"] = (
    year_table["success_total"]
    + year_table["failure_total"]
)

year_table["net_success_minus_failure"] = (
    year_table["success_total"]
    - year_table["failure_total"]
)

year_table["success_rate"] = np.where(
    year_table["total_activity"] > 0,
    year_table["success_total"] / year_table["total_activity"] * 100,
    np.nan
)

year_table["failure_rate"] = np.where(
    year_table["total_activity"] > 0,
    year_table["failure_total"] / year_table["total_activity"] * 100,
    np.nan
)

year_table["success_rate"] = year_table["success_rate"].round(2)
year_table["failure_rate"] = year_table["failure_rate"].round(2)

# ------------------------------------------------------------
# 8. TABLE 3: Yearly total check
# ------------------------------------------------------------
print("\nTABLE 3: YEARLY TOTAL CHECK")
display(year_table)

# ------------------------------------------------------------
# 9. TABLE 4: Zero check by year
# ------------------------------------------------------------
zero_check = year_table.copy()

check_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
    "success_total",
    "failure_total",
    "total_activity",
]

for col in check_cols:
    zero_check[col + "_is_zero"] = zero_check[col] == 0

zero_check["has_zero_problem"] = zero_check[
    [col + "_is_zero" for col in check_cols]
].any(axis=1)

print("\nTABLE 4: YEAR ZERO CHECK")
display(
    zero_check[
        [
            "year",
            "job_openings_is_zero",
            "job_closings_is_zero",
            "job_gains_is_zero",
            "job_losses_is_zero",
            "establishment_births_is_zero",
            "establishment_deaths_is_zero",
            "success_total_is_zero",
            "failure_total_is_zero",
            "total_activity_is_zero",
            "has_zero_problem",
        ]
    ]
)

# ------------------------------------------------------------
# 10. TABLE 5: Only bad years
# ------------------------------------------------------------
bad_years = zero_check[zero_check["has_zero_problem"] == True].copy()

print("\nTABLE 5: ONLY YEARS WITH 0 PROBLEM")
display(
    bad_years[
        [
            "year",
            "job_openings",
            "job_closings",
            "job_gains",
            "job_losses",
            "establishment_births",
            "establishment_deaths",
            "success_total",
            "failure_total",
            "total_activity",
        ]
    ]
)

# ------------------------------------------------------------
# 11. Final answer
# ------------------------------------------------------------
print("\nFINAL CHECK")
print("Year range checked:", YEAR_START, "to", YEAR_END)
print("Total years expected:", YEAR_END - YEAR_START + 1)
print("Years with 0 problem:", len(bad_years))

if len(bad_years) == 0:
    print("GOOD: No year has 0 problem.")
else:
    print("CHECK TABLE 5: These years have at least one 0 column.")

# Check for 0 - 2

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import gc

# ============================================================
# MEMORY OPTIMIZED
# DEGREE → INDUSTRY → SUCCESS / FAILURE TABLE
# NO CHART
# NO SAVE
# Reads CSV in chunks
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

try:
    from IPython.display import display
except ImportError:
    display = print


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------
DATA_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

YEAR_START = 1984
YEAR_END = 2024

CHUNKSIZE = 100_000
SHOW_TOP_N = 50

# Filter options:
# "total_activity_nonzero"
# "success_and_failure_nonzero"
# "all_six_metrics_nonzero"
MAIN_FILTER = "success_and_failure_nonzero"

COMPACT_EVERY_N_CHUNKS = 10

metric_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
]


# ------------------------------------------------------------
# 2. BASIC HELPERS
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_col_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        base = clean_col_name(col)

        if base not in seen:
            seen[base] = 0
            new_cols.append(base)
        else:
            seen[base] += 1
            new_cols.append(f"{base}_{seen[base]}")

    return new_cols


def detect_standard_metric(text):
    text = str(text).lower()
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if "establishment" in text and "birth" in text:
        return "establishment_births"

    if "establishment" in text and "death" in text:
        return "establishment_deaths"

    if "job opening" in text or "job openings" in text:
        return "job_openings"

    if "job closing" in text or "job closings" in text:
        return "job_closings"

    if "job gain" in text or "job gains" in text or "gross job gain" in text:
        return "job_gains"

    if "job loss" in text or "job losses" in text or "gross job loss" in text:
        return "job_losses"

    if re.search(r"\bopenings\b", text):
        return "job_openings"

    if re.search(r"\bclosings\b", text):
        return "job_closings"

    if re.search(r"\bgains\b", text):
        return "job_gains"

    if re.search(r"\blosses\b", text):
        return "job_losses"

    if re.search(r"\bbirths\b", text):
        return "establishment_births"

    if re.search(r"\bdeaths\b", text):
        return "establishment_deaths"

    return np.nan


# ------------------------------------------------------------
# 3. INDUSTRY SECTOR HELPERS
# ------------------------------------------------------------
def naics_to_sector(code):
    if pd.isna(code):
        return np.nan

    digits = re.sub(r"[^0-9]", "", str(code))

    if len(digits) < 2:
        return np.nan

    two = int(digits[:2])

    if two == 11:
        return "Agriculture, Forestry, Fishing and Hunting"
    if two == 21:
        return "Mining, Quarrying, and Oil and Gas Extraction"
    if two == 22:
        return "Utilities"
    if two == 23:
        return "Construction"
    if 31 <= two <= 33:
        return "Manufacturing"
    if two == 42:
        return "Wholesale Trade"
    if 44 <= two <= 45:
        return "Retail Trade"
    if 48 <= two <= 49:
        return "Transportation and Warehousing"
    if two == 51:
        return "Information"
    if two == 52:
        return "Finance and Insurance"
    if two == 53:
        return "Real Estate and Rental and Leasing"
    if two == 54:
        return "Professional, Scientific, and Technical Services"
    if two == 55:
        return "Management of Companies and Enterprises"
    if two == 56:
        return "Administrative and Support and Waste Management"
    if two == 61:
        return "Educational Services"
    if two == 62:
        return "Health Care and Social Assistance"
    if two == 71:
        return "Arts, Entertainment, and Recreation"
    if two == 72:
        return "Accommodation and Food Services"
    if two == 81:
        return "Other Services"
    if two == 92:
        return "Public Administration"

    return np.nan


def industry_name_to_sector(name):
    if pd.isna(name):
        return np.nan

    text = str(name).lower()

    if "agriculture" in text or "forestry" in text or "fishing" in text:
        return "Agriculture, Forestry, Fishing and Hunting"
    if "mining" in text or "oil" in text or "gas" in text:
        return "Mining, Quarrying, and Oil and Gas Extraction"
    if "utilities" in text:
        return "Utilities"
    if "construction" in text:
        return "Construction"
    if "manufacturing" in text:
        return "Manufacturing"
    if "wholesale" in text:
        return "Wholesale Trade"
    if "retail" in text:
        return "Retail Trade"
    if "transportation" in text or "warehousing" in text:
        return "Transportation and Warehousing"
    if "information" in text:
        return "Information"
    if "finance" in text or "insurance" in text:
        return "Finance and Insurance"
    if "real estate" in text or "rental" in text or "leasing" in text:
        return "Real Estate and Rental and Leasing"
    if "professional" in text or "scientific" in text or "technical" in text:
        return "Professional, Scientific, and Technical Services"
    if "management of companies" in text:
        return "Management of Companies and Enterprises"
    if "administrative" in text or "waste" in text:
        return "Administrative and Support and Waste Management"
    if "educational" in text or "education" in text:
        return "Educational Services"
    if "health care" in text or "healthcare" in text or "social assistance" in text:
        return "Health Care and Social Assistance"
    if "arts" in text or "entertainment" in text or "recreation" in text:
        return "Arts, Entertainment, and Recreation"
    if "accommodation" in text or "food services" in text or "restaurant" in text:
        return "Accommodation and Food Services"
    if "other services" in text:
        return "Other Services"
    if "public administration" in text or "government" in text:
        return "Public Administration"

    return str(name).strip()


def add_connected_industry_sector(df):
    if "connected_industry_sector" in df.columns:
        df["connected_industry_sector"] = (
            df["connected_industry_sector"]
            .fillna("Unknown Sector")
            .astype(str)
            .str.strip()
        )
        return df

    sector = pd.Series(np.nan, index=df.index, dtype="object")

    for code_col in ["naics_code", "industry_code"]:
        if code_col in df.columns:
            sector = sector.fillna(df[code_col].map(naics_to_sector))

    for name_col in ["industry_sector", "sector", "industry_name", "industry"]:
        if name_col in df.columns:
            sector = sector.fillna(df[name_col].map(industry_name_to_sector))

    df["connected_industry_sector"] = (
        sector
        .fillna("Unknown Sector")
        .astype(str)
        .str.strip()
    )

    return df


# ------------------------------------------------------------
# 4. FIELD OF STUDY HELPERS
# ------------------------------------------------------------
def sector_to_fields(sector):
    sector = str(sector).lower()

    if "agriculture" in sector:
        return [
            "Agriculture",
            "Biological and Biomedical Sciences",
            "Natural Resources and Conservation",
        ]

    if "mining" in sector or "oil" in sector or "gas" in sector:
        return [
            "Engineering",
            "Engineering Technologies",
            "Physical Sciences",
        ]

    if "utilities" in sector:
        return [
            "Engineering",
            "Engineering Technologies",
            "Natural Resources and Conservation",
        ]

    if "construction" in sector:
        return [
            "Architecture and Related Services",
            "Engineering",
            "Construction Trades",
        ]

    if "manufacturing" in sector:
        return [
            "Engineering",
            "Engineering Technologies",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "wholesale" in sector or "retail" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Communication, Journalism, and Related Programs",
        ]

    if "transportation" in sector or "warehousing" in sector:
        return [
            "Transportation and Materials Moving",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "information" in sector:
        return [
            "Computer and Information Sciences",
            "Communication, Journalism, and Related Programs",
            "Engineering",
        ]

    if "finance" in sector or "insurance" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Mathematics and Statistics",
            "Social Sciences",
        ]

    if "real estate" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Architecture and Related Services",
        ]

    if "professional" in sector or "scientific" in sector or "technical" in sector:
        return [
            "Computer and Information Sciences",
            "Engineering",
            "Business, Management, Marketing, and Related Support Services",
            "Mathematics and Statistics",
            "Physical Sciences",
        ]

    if "management of companies" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "administrative" in sector or "waste" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Public Administration and Social Service Professions",
            "Natural Resources and Conservation",
        ]

    if "educational" in sector:
        return [
            "Education",
            "Library Science",
        ]

    if "health care" in sector or "social assistance" in sector:
        return [
            "Health Professions and Related Programs",
            "Psychology",
            "Public Administration and Social Service Professions",
            "Biological and Biomedical Sciences",
        ]

    if "arts" in sector or "entertainment" in sector or "recreation" in sector:
        return [
            "Visual and Performing Arts",
            "Communication, Journalism, and Related Programs",
            "Parks, Recreation, Leisure, Fitness, and Kinesiology",
        ]

    if "accommodation" in sector or "food services" in sector:
        return [
            "Culinary, Entertainment, and Personal Services",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "other services" in sector:
        return [
            "Culinary, Entertainment, and Personal Services",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "public administration" in sector:
        return [
            "Public Administration and Social Service Professions",
            "Homeland Security, Law Enforcement, Firefighting and Related Protective Services",
            "Social Sciences",
        ]

    return ["General / Other Field"]


field_candidates = [
    "field_of_study",
    "degree_field",
    "degree_fields",
    "field_group",
    "field_subgroup",
    "major_name",
    "cip_title",
    "cip_name",
]


def add_field_of_study(df, field_col_found):
    if field_col_found is not None and field_col_found in df.columns:
        df["field_of_study"] = (
            df[field_col_found]
            .fillna("Unknown Field")
            .astype(str)
            .str.strip()
        )
        return df

    df["field_of_study"] = df["connected_industry_sector"].map(sector_to_fields)
    df = df.explode("field_of_study", ignore_index=True)
    df["field_of_study"] = df["field_of_study"].astype(str).str.strip()

    return df


# ------------------------------------------------------------
# 5. FORMULA HELPERS
# ------------------------------------------------------------
def make_empty_metric_table(index_cols):
    return pd.DataFrame(columns=index_cols + metric_cols)


def compact_parts(parts, index_cols):
    if len(parts) == 0:
        return []

    temp = pd.concat(parts, ignore_index=True)

    for col in metric_cols:
        if col not in temp.columns:
            temp[col] = 0

    temp = (
        temp
        .groupby(index_cols, as_index=False)[metric_cols]
        .sum()
    )

    return [temp]


def add_formula_columns(df):
    df = df.copy()

    for col in metric_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["success_total"] = (
        df["job_openings"]
        + df["job_gains"]
        + df["establishment_births"]
    )

    df["failure_total"] = (
        df["job_closings"]
        + df["job_losses"]
        + df["establishment_deaths"]
    )

    df["job_net"] = (
        df["job_openings"]
        + df["job_gains"]
        - df["job_closings"]
        - df["job_losses"]
    )

    df["startup_net"] = (
        df["establishment_births"]
        - df["establishment_deaths"]
    )

    df["net_success_minus_failure"] = (
        df["success_total"]
        - df["failure_total"]
    )

    df["total_activity"] = (
        df["success_total"]
        + df["failure_total"]
    )

    df["success_rate"] = np.where(
        df["total_activity"] > 0,
        df["success_total"] / df["total_activity"] * 100,
        np.nan
    )

    df["failure_rate"] = np.where(
        df["total_activity"] > 0,
        df["failure_total"] / df["total_activity"] * 100,
        np.nan
    )

    return df


# ------------------------------------------------------------
# 6. READ HEADER ONLY
# ------------------------------------------------------------
check_file(DATA_FILE)

raw_columns = list(pd.read_csv(DATA_FILE, nrows=0).columns)
clean_columns = make_unique_columns(raw_columns)

column_table = pd.DataFrame({
    "column_number": range(1, len(clean_columns) + 1),
    "column_name": clean_columns
})

print("\nCOLUMNS IN FILE")
display(column_table)


# ------------------------------------------------------------
# 7. DETECT DATA FORMAT FROM COLUMN NAMES
# ------------------------------------------------------------
has_any_wide_metric = any(col in clean_columns for col in metric_cols)

value_candidates = [
    "value",
    "company_value",
    "metric_value",
    "count",
    "amount",
]

label_candidates = [
    "metric_name",
    "metric_type",
    "metric_category",
    "metric_code",
    "source_table",
    "dataset",
    "all_text",
    "notes",
]

value_col = None
for col in value_candidates:
    if col in clean_columns:
        value_col = col
        break

label_cols = [col for col in label_candidates if col in clean_columns]

field_col_found = None
for col in field_candidates:
    if col in clean_columns:
        field_col_found = col
        break

if has_any_wide_metric:
    source_format = "WIDE TABLE"
else:
    source_format = "LONG TABLE"

    if value_col is None:
        raise KeyError("Cannot find value column like value, company_value, metric_value, count, or amount.")

    if len(label_cols) == 0:
        raise KeyError("Cannot find metric label column like metric_name, metric_type, source_table, dataset, or notes.")

print("\nDATA FORMAT FOUND:", source_format)

if source_format == "LONG TABLE":
    print("Value column used:", value_col)
    print("Metric label columns used:", label_cols)

if field_col_found is not None:
    connection_method = f"Used existing degree column: {field_col_found}"
else:
    connection_method = "Created field_of_study from industry-sector mapping because file did not contain a degree field column."

print("\nCONNECTION METHOD:")
print(connection_method)


# ------------------------------------------------------------
# 8. MEMORY OPTIMIZATION: ONLY READ NEEDED COLUMNS
# ------------------------------------------------------------
industry_needed = [
    "connected_industry_sector",
    "naics_code",
    "industry_code",
    "industry_sector",
    "sector",
    "industry_name",
    "industry",
]

needed_clean_cols = set(["year"])
needed_clean_cols.update(industry_needed)
needed_clean_cols.update(field_candidates)

if source_format == "WIDE TABLE":
    needed_clean_cols.update(metric_cols)
else:
    needed_clean_cols.add(value_col)
    needed_clean_cols.update(label_cols)

raw_usecols = [
    raw
    for raw, clean in zip(raw_columns, clean_columns)
    if clean in needed_clean_cols
]

print("\nMEMORY OPTIMIZATION")
print("Total file columns:", len(raw_columns))
print("Columns loaded per chunk:", len(raw_usecols))
print("Chunk size:", CHUNKSIZE)


# ------------------------------------------------------------
# 9. PROCESS CSV IN CHUNKS
# ------------------------------------------------------------
summary_parts = []
year_parts = []
metric_detection_parts = []

total_rows_read = 0
total_rows_used = 0

summary_index_cols = [
    "field_of_study",
    "connected_industry_sector",
]

year_index_cols = [
    "year",
]

reader = pd.read_csv(
    DATA_FILE,
    usecols=raw_usecols,
    chunksize=CHUNKSIZE,
    low_memory=False
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows_read += len(chunk)

    chunk.columns = make_unique_columns(chunk.columns)

    # Year clean and filter
    if "year" in chunk.columns:
        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk = chunk[
            (chunk["year"] >= YEAR_START)
            & (chunk["year"] <= YEAR_END)
        ]

    if chunk.empty:
        del chunk
        gc.collect()
        continue

    if source_format == "WIDE TABLE":
        # Make sure all metric columns exist
        for col in metric_cols:
            if col not in chunk.columns:
                chunk[col] = 0
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").fillna(0)

        total_rows_used += len(chunk)

        # YEAR ZERO CHECK DATA BEFORE degree-field explode
        if "year" in chunk.columns:
            year_chunk = (
                chunk
                .groupby("year", as_index=False)[metric_cols]
                .sum()
            )
            year_parts.append(year_chunk)

        # Add industry + degree field
        chunk = add_connected_industry_sector(chunk)
        chunk = add_field_of_study(chunk, field_col_found)

        grouped = (
            chunk
            .groupby(summary_index_cols, as_index=False)[metric_cols]
            .sum()
        )

        summary_parts.append(grouped)

    else:
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce").fillna(0)

        metric_text = (
            chunk[label_cols]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
        )

        chunk["standard_metric"] = metric_text.map(detect_standard_metric)
        chunk = chunk[chunk["standard_metric"].notna()]

        if chunk.empty:
            del chunk
            gc.collect()
            continue

        total_rows_used += len(chunk)

        # Metric detection summary
        detect_chunk = (
            chunk
            .groupby("standard_metric", as_index=False)
            .agg(
                rows=("standard_metric", "size"),
                total_value=(value_col, "sum")
            )
        )
        metric_detection_parts.append(detect_chunk)

        # YEAR ZERO CHECK DATA BEFORE degree-field explode
        if "year" in chunk.columns:
            year_long = (
                chunk
                .groupby(["year", "standard_metric"], as_index=False)[value_col]
                .sum()
            )

            year_wide = (
                year_long
                .pivot_table(
                    index="year",
                    columns="standard_metric",
                    values=value_col,
                    aggfunc="sum",
                    fill_value=0
                )
                .reset_index()
            )

            year_wide.columns.name = None

            for col in metric_cols:
                if col not in year_wide.columns:
                    year_wide[col] = 0

            year_parts.append(year_wide[["year"] + metric_cols])

        # Add industry + degree field
        chunk = add_connected_industry_sector(chunk)
        chunk = add_field_of_study(chunk, field_col_found)

        grouped_long = (
            chunk
            .groupby(summary_index_cols + ["standard_metric"], as_index=False)[value_col]
            .sum()
        )

        grouped_wide = (
            grouped_long
            .pivot_table(
                index=summary_index_cols,
                columns="standard_metric",
                values=value_col,
                aggfunc="sum",
                fill_value=0
            )
            .reset_index()
        )

        grouped_wide.columns.name = None

        for col in metric_cols:
            if col not in grouped_wide.columns:
                grouped_wide[col] = 0

        summary_parts.append(grouped_wide[summary_index_cols + metric_cols])

    # Compact small grouped results every few chunks
    if chunk_number % COMPACT_EVERY_N_CHUNKS == 0:
        summary_parts = compact_parts(summary_parts, summary_index_cols)
        year_parts = compact_parts(year_parts, year_index_cols)

        if len(metric_detection_parts) > 0:
            metric_detection_parts = [
                pd.concat(metric_detection_parts, ignore_index=True)
                .groupby("standard_metric", as_index=False)
                .agg(
                    rows=("rows", "sum"),
                    total_value=("total_value", "sum")
                )
            ]

        print(f"Processed chunk {chunk_number:,} | rows read: {total_rows_read:,} | rows used: {total_rows_used:,}")

    del chunk
    gc.collect()


print("\nFINISHED READING")
print("Total rows read:", f"{total_rows_read:,}")
print("Total rows used:", f"{total_rows_used:,}")


# ------------------------------------------------------------
# 10. FINAL COMPACT GROUPING
# ------------------------------------------------------------
summary_parts = compact_parts(summary_parts, summary_index_cols)

if len(summary_parts) == 0:
    raise ValueError("No usable rows found after filtering/detecting metrics.")

summary = summary_parts[0]

for col in metric_cols:
    if col not in summary.columns:
        summary[col] = 0

summary = add_formula_columns(summary)


# ------------------------------------------------------------
# 11. LONG TABLE METRIC DETECTION DISPLAY
# ------------------------------------------------------------
if source_format == "LONG TABLE" and len(metric_detection_parts) > 0:
    metric_detection = (
        pd.concat(metric_detection_parts, ignore_index=True)
        .groupby("standard_metric", as_index=False)
        .agg(
            rows=("rows", "sum"),
            total_value=("total_value", "sum")
        )
        .sort_values("standard_metric")
    )

    print("\nMETRICS DETECTED FROM LONG TABLE")
    display(metric_detection)

    missing_detected = [
        col for col in metric_cols
        if col not in metric_detection["standard_metric"].tolist()
    ]

    if missing_detected:
        print("\nWARNING: These metrics were not detected and were filled with 0:")
        for col in missing_detected:
            print("-", col)


# ------------------------------------------------------------
# 12. YEAR ZERO CHECK TABLE
# ------------------------------------------------------------
year_zero_check = None

year_parts = compact_parts(year_parts, year_index_cols)

if len(year_parts) > 0:
    year_zero_check = year_parts[0]

    for col in metric_cols:
        if col not in year_zero_check.columns:
            year_zero_check[col] = 0

    year_zero_check = add_formula_columns(year_zero_check)

    year_zero_check = year_zero_check[
        [
            "year",
            "job_openings",
            "job_closings",
            "job_gains",
            "job_losses",
            "establishment_births",
            "establishment_deaths",
            "success_total",
            "failure_total",
            "total_activity",
        ]
    ].copy()

    year_zero_check["success_is_zero"] = year_zero_check["success_total"] == 0
    year_zero_check["failure_is_zero"] = year_zero_check["failure_total"] == 0
    year_zero_check["total_activity_is_zero"] = year_zero_check["total_activity"] == 0

    year_zero_check = year_zero_check.sort_values("year").reset_index(drop=True)

    print("\nYEAR ZERO CHECK TABLE")
    display(year_zero_check)


# ------------------------------------------------------------
# 13. OVERALL ZERO CHECK TABLE
# ------------------------------------------------------------
check_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
    "success_total",
    "failure_total",
    "total_activity",
]

zero_check_rows = []

for col in check_cols:
    zero_check_rows.append({
        "column": col,
        "zero_rows": int((summary[col] == 0).sum()),
        "nonzero_rows": int((summary[col] != 0).sum()),
        "min_value": summary[col].min(),
        "max_value": summary[col].max(),
        "total_value": summary[col].sum(),
    })

zero_check = pd.DataFrame(zero_check_rows)

print("\nOVERALL ZERO CHECK TABLE")
display(zero_check)


# ------------------------------------------------------------
# 14. FILTER ROWS
# ------------------------------------------------------------
if MAIN_FILTER == "total_activity_nonzero":
    final_table = summary[summary["total_activity"] > 0].copy()

elif MAIN_FILTER == "success_and_failure_nonzero":
    final_table = summary[
        (summary["success_total"] > 0)
        & (summary["failure_total"] > 0)
    ].copy()

elif MAIN_FILTER == "all_six_metrics_nonzero":
    final_table = summary[
        (summary["job_openings"] > 0)
        & (summary["job_closings"] > 0)
        & (summary["job_gains"] > 0)
        & (summary["job_losses"] > 0)
        & (summary["establishment_births"] > 0)
        & (summary["establishment_deaths"] > 0)
    ].copy()

else:
    raise ValueError("MAIN_FILTER is not valid.")

final_table = final_table.sort_values(
    by="net_success_minus_failure",
    ascending=False
).reset_index(drop=True)

print("\nFILTER USED:")
print(MAIN_FILTER)
print("Rows before filter:", len(summary))
print("Rows after filter:", len(final_table))


# ------------------------------------------------------------
# 15. TABLE 1: DEGREE → INDUSTRY → SUCCESS / FAILURE
# ------------------------------------------------------------
degree_industry_success_failure_table = final_table[
    [
        "field_of_study",
        "connected_industry_sector",
        "success_total",
        "failure_total",
    ]
].copy()

print("\nTABLE 1: DEGREE → INDUSTRY → SUCCESS / FAILURE")
display(degree_industry_success_failure_table.head(SHOW_TOP_N))


# ------------------------------------------------------------
# 16. TABLE 2: JOB MARKET ONLY
# ------------------------------------------------------------
job_market_only_table = final_table[
    [
        "field_of_study",
        "connected_industry_sector",
        "job_openings",
        "job_closings",
        "job_gains",
        "job_losses",
        "job_net",
    ]
].copy()

print("\nTABLE 2: JOB MARKET ONLY")
display(job_market_only_table.head(SHOW_TOP_N))


# ------------------------------------------------------------
# 17. TABLE 3: STARTUP ONLY
# ------------------------------------------------------------
startup_only_table = final_table[
    [
        "field_of_study",
        "connected_industry_sector",
        "establishment_births",
        "establishment_deaths",
        "startup_net",
    ]
].copy()

print("\nTABLE 3: STARTUP ONLY")
display(startup_only_table.head(SHOW_TOP_N))


# ------------------------------------------------------------
# 18. TABLE 4: OVERALL GOOD VS BAD + RATE
# ------------------------------------------------------------
overall_good_bad_table = final_table[
    [
        "field_of_study",
        "connected_industry_sector",
        "success_total",
        "failure_total",
        "net_success_minus_failure",
        "success_rate",
        "failure_rate",
    ]
].copy()

overall_good_bad_table["success_rate"] = overall_good_bad_table["success_rate"].round(2)
overall_good_bad_table["failure_rate"] = overall_good_bad_table["failure_rate"].round(2)

print("\nTABLE 4: OVERALL GOOD VS BAD + SUCCESS / FAILURE RATE")
display(overall_good_bad_table.head(SHOW_TOP_N))


# ------------------------------------------------------------
# 19. ROWS WITH ZERO PROBLEM
# ------------------------------------------------------------
zero_problem_rows = summary[
    (summary["success_total"] == 0)
    | (summary["failure_total"] == 0)
    | (summary["total_activity"] == 0)
].copy()

print("\nROWS THAT STILL HAVE ZERO PROBLEM")
display(
    zero_problem_rows[
        [
            "field_of_study",
            "connected_industry_sector",
            "success_total",
            "failure_total",
            "total_activity",
            "success_rate",
            "failure_rate",
        ]
    ].head(SHOW_TOP_N)
)


print("\nDONE.")
print("Created tables:")
print("- year_zero_check")
print("- zero_check")
print("- degree_industry_success_failure_table")
print("- job_market_only_table")
print("- startup_only_table")
print("- overall_good_bad_table")
print("- zero_problem_rows")

# formual code
success_total = job_openings + job_gains + establishment_births

failure_total = job_closings + job_losses + establishment_deaths

net_success_minus_failure = success_total - failure_total

success_rate = success_total / (success_total + failure_total) * 100

# Sackle bar by year success code
This chart shows success and failure activity by year, while keeping each field of study and connected industry sector as separate stacked parts instead of one large yearly total.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
import textwrap

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# STACKED CHART BY YEAR + FIELD OF STUDY + INDUSTRY
# NO SAVE
# MEMORY OPTIMIZED
# This is NOT one big yearly total
# ============================================================

YEAR_START = 1984
YEAR_END = 2023

TOP_N_PAIR = 12          # number of Field → Industry groups shown in chart
SHOW_TABLE_ROWS = 100
CHUNKSIZE = 100_000
COMPACT_EVERY_N_CHUNKS = 10

metric_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
]

# ------------------------------------------------------------
# 1. Check required memory-optimized functions exist
# ------------------------------------------------------------
needed_functions = [
    "make_unique_columns",
    "detect_standard_metric",
    "add_connected_industry_sector",
    "add_field_of_study",
]

missing_functions = [
    name for name in needed_functions
    if name not in globals()
]

if missing_functions:
    print("Missing functions:")
    for name in missing_functions:
        print("-", name)

    raise NameError(
        "Run the memory-optimized code first, then run this chart code."
    )

if "DATA_FILE" not in globals():
    raise NameError(
        "DATA_FILE is missing. Run the memory-optimized code first."
    )


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def compact_year_field_parts(parts):
    if len(parts) == 0:
        return []

    temp = pd.concat(parts, ignore_index=True)

    for col in metric_cols:
        if col not in temp.columns:
            temp[col] = 0

    temp = (
        temp
        .groupby(
            ["year", "field_of_study", "connected_industry_sector"],
            as_index=False
        )[metric_cols]
        .sum()
    )

    return [temp]


def add_formula_columns(df):
    df = df.copy()

    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["success_total"] = (
        df["job_openings"]
        + df["job_gains"]
        + df["establishment_births"]
    )

    df["failure_total"] = (
        df["job_closings"]
        + df["job_losses"]
        + df["establishment_deaths"]
    )

    df["net_success_minus_failure"] = (
        df["success_total"]
        - df["failure_total"]
    )

    df["total_activity"] = (
        df["success_total"]
        + df["failure_total"]
    )

    df["success_rate"] = np.where(
        df["total_activity"] > 0,
        df["success_total"] / df["total_activity"] * 100,
        np.nan
    )

    df["failure_rate"] = np.where(
        df["total_activity"] > 0,
        df["failure_total"] / df["total_activity"] * 100,
        np.nan
    )

    return df


def short_label(text, width=55):
    text = str(text)
    return textwrap.shorten(text, width=width, placeholder="...")


# ------------------------------------------------------------
# 3. Read file header only
# ------------------------------------------------------------
raw_columns = list(pd.read_csv(DATA_FILE, nrows=0).columns)
clean_columns = make_unique_columns(raw_columns)

has_any_wide_metric = any(col in clean_columns for col in metric_cols)

value_candidates = [
    "value",
    "company_value",
    "metric_value",
    "count",
    "amount",
]

label_candidates = [
    "metric_name",
    "metric_type",
    "metric_category",
    "metric_code",
    "source_table",
    "dataset",
    "all_text",
    "notes",
]

field_candidates = [
    "field_of_study",
    "degree_field",
    "degree_fields",
    "field_group",
    "field_subgroup",
    "major_name",
    "cip_title",
    "cip_name",
]

value_col = None
for col in value_candidates:
    if col in clean_columns:
        value_col = col
        break

label_cols = [
    col for col in label_candidates
    if col in clean_columns
]

field_col_found = None
for col in field_candidates:
    if col in clean_columns:
        field_col_found = col
        break

if has_any_wide_metric:
    source_format = "WIDE TABLE"
else:
    source_format = "LONG TABLE"

print("DATA FORMAT:", source_format)

if source_format == "LONG TABLE":
    if value_col is None:
        raise KeyError("Cannot find value column.")

    if len(label_cols) == 0:
        raise KeyError("Cannot find metric label column.")

    print("Value column:", value_col)
    print("Metric label columns:", label_cols)

if field_col_found:
    print("Field of study column:", field_col_found)
else:
    print("Field of study will be created from industry mapping.")


# ------------------------------------------------------------
# 4. Only load needed columns
# ------------------------------------------------------------
industry_needed = [
    "connected_industry_sector",
    "naics_code",
    "industry_code",
    "industry_sector",
    "sector",
    "industry_name",
    "industry",
]

needed_clean_cols = set(["year"])
needed_clean_cols.update(industry_needed)
needed_clean_cols.update(field_candidates)

if source_format == "WIDE TABLE":
    needed_clean_cols.update(metric_cols)
else:
    needed_clean_cols.add(value_col)
    needed_clean_cols.update(label_cols)

raw_usecols = [
    raw
    for raw, clean in zip(raw_columns, clean_columns)
    if clean in needed_clean_cols
]

print("Total file columns:", len(raw_columns))
print("Columns loaded per chunk:", len(raw_usecols))
print("Chunk size:", CHUNKSIZE)


# ------------------------------------------------------------
# 5. Read chunks and group by:
# year + field_of_study + connected_industry_sector
# ------------------------------------------------------------
year_field_parts = []

total_rows_read = 0
total_rows_used = 0

reader = pd.read_csv(
    DATA_FILE,
    usecols=raw_usecols,
    chunksize=CHUNKSIZE,
    low_memory=False
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows_read += len(chunk)

    chunk.columns = make_unique_columns(chunk.columns)

    if "year" not in chunk.columns:
        raise KeyError("No year column found.")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    chunk = chunk[
        (chunk["year"] >= YEAR_START)
        & (chunk["year"] <= YEAR_END)
    ].copy()

    if chunk.empty:
        del chunk
        gc.collect()
        continue

    if source_format == "WIDE TABLE":

        for col in metric_cols:
            if col not in chunk.columns:
                chunk[col] = 0

            chunk[col] = pd.to_numeric(
                chunk[col],
                errors="coerce"
            ).fillna(0)

        total_rows_used += len(chunk)

        chunk = add_connected_industry_sector(chunk)
        chunk = add_field_of_study(chunk, field_col_found)

        grouped = (
            chunk
            .groupby(
                ["year", "field_of_study", "connected_industry_sector"],
                as_index=False
            )[metric_cols]
            .sum()
        )

        year_field_parts.append(grouped)

    else:

        chunk[value_col] = pd.to_numeric(
            chunk[value_col],
            errors="coerce"
        ).fillna(0)

        metric_text = (
            chunk[label_cols]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
        )

        chunk["standard_metric"] = metric_text.map(detect_standard_metric)

        chunk = chunk[chunk["standard_metric"].notna()].copy()

        if chunk.empty:
            del chunk
            gc.collect()
            continue

        total_rows_used += len(chunk)

        chunk = add_connected_industry_sector(chunk)
        chunk = add_field_of_study(chunk, field_col_found)

        grouped_long = (
            chunk
            .groupby(
                [
                    "year",
                    "field_of_study",
                    "connected_industry_sector",
                    "standard_metric",
                ],
                as_index=False
            )[value_col]
            .sum()
        )

        grouped_wide = (
            grouped_long
            .pivot_table(
                index=[
                    "year",
                    "field_of_study",
                    "connected_industry_sector",
                ],
                columns="standard_metric",
                values=value_col,
                aggfunc="sum",
                fill_value=0
            )
            .reset_index()
        )

        grouped_wide.columns.name = None

        for col in metric_cols:
            if col not in grouped_wide.columns:
                grouped_wide[col] = 0

        grouped_wide = grouped_wide[
            [
                "year",
                "field_of_study",
                "connected_industry_sector",
            ]
            + metric_cols
        ]

        year_field_parts.append(grouped_wide)

    if chunk_number % COMPACT_EVERY_N_CHUNKS == 0:
        year_field_parts = compact_year_field_parts(year_field_parts)
        print(
            f"Processed chunk {chunk_number:,} | "
            f"rows read: {total_rows_read:,} | "
            f"rows used: {total_rows_used:,}"
        )

    del chunk
    gc.collect()


print("DONE READING")
print("Rows read:", f"{total_rows_read:,}")
print("Rows used:", f"{total_rows_used:,}")


# ------------------------------------------------------------
# 6. Final grouped table
# ------------------------------------------------------------
year_field_parts = compact_year_field_parts(year_field_parts)

if len(year_field_parts) == 0:
    raise ValueError("No usable data found.")

year_field_industry_table = year_field_parts[0]

year_field_industry_table = add_formula_columns(year_field_industry_table)

year_field_industry_table["year"] = (
    pd.to_numeric(
        year_field_industry_table["year"],
        errors="coerce"
    )
    .astype("Int64")
)

year_field_industry_table = year_field_industry_table[
    (year_field_industry_table["year"].notna())
    & (year_field_industry_table["success_total"] > 0)
    & (year_field_industry_table["failure_total"] > 0)
].copy()

year_field_industry_table["degree_to_industry"] = (
    year_field_industry_table["field_of_study"].astype(str)
    + " → "
    + year_field_industry_table["connected_industry_sector"].astype(str)
)

year_field_industry_table = year_field_industry_table.sort_values(
    ["year", "total_activity"],
    ascending=[True, False]
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Display table
# ------------------------------------------------------------
display_table = year_field_industry_table[
    [
        "year",
        "field_of_study",
        "connected_industry_sector",
        "job_openings",
        "job_closings",
        "job_gains",
        "job_losses",
        "establishment_births",
        "establishment_deaths",
        "success_total",
        "failure_total",
        "net_success_minus_failure",
        "success_rate",
        "failure_rate",
    ]
].copy()

display_table["success_rate"] = display_table["success_rate"].round(2)
display_table["failure_rate"] = display_table["failure_rate"].round(2)

print("TABLE: Year → Field of Study → Industry → Success vs Failure")
display(display_table.head(SHOW_TABLE_ROWS))


# ------------------------------------------------------------
# 8. Choose top Field → Industry pairs for readable stacked chart
# ------------------------------------------------------------
top_pairs = (
    year_field_industry_table
    .groupby("degree_to_industry", as_index=False)["total_activity"]
    .sum()
    .sort_values("total_activity", ascending=False)
    .head(TOP_N_PAIR)["degree_to_industry"]
    .tolist()
)

chart_data = year_field_industry_table[
    year_field_industry_table["degree_to_industry"].isin(top_pairs)
].copy()

chart_data["year"] = chart_data["year"].astype(int)

years = sorted(chart_data["year"].unique())

success_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="degree_to_industry",
        values="success_total",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(years)
)

failure_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="degree_to_industry",
        values="failure_total",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(years)
)

# Keep same column order as top_pairs
success_pivot = success_pivot.reindex(columns=top_pairs, fill_value=0)
failure_pivot = failure_pivot.reindex(columns=top_pairs, fill_value=0)


# ------------------------------------------------------------
# 9. Opposite stacked bar chart
# Success stacks above zero
# Failure stacks below zero
# ------------------------------------------------------------
x = np.arange(len(years))

bottom_success = np.zeros(len(years))
bottom_failure = np.zeros(len(years))

plt.figure(figsize=(22, 11))
ax = plt.gca()

for pair in top_pairs:
    label = short_label(pair, width=65)

    success_values = success_pivot[pair].values
    failure_values = failure_pivot[pair].values

    bars = ax.bar(
        x,
        success_values,
        bottom=bottom_success,
        label=label
    )

    chosen_color = bars.patches[0].get_facecolor()

    ax.bar(
        x,
        -failure_values,
        bottom=bottom_failure,
        color=chosen_color,
        alpha=0.45
    )

    bottom_success = bottom_success + success_values
    bottom_failure = bottom_failure - failure_values

ax.axhline(0, linewidth=1)

ax.set_title(
    "Stacked Success vs Failure by Year, Field of Study, and Industry\n"
    "Success = Job Openings + Job Gains + Establishment Births | "
    "Failure = Job Closings + Job Losses + Establishment Deaths",
    fontsize=14
)

ax.set_xlabel("Year")
ax.set_ylabel("Failure below 0 | Success above 0")

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=90)

ax.legend(
    title="Field of Study → Industry Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)

plt.tight_layout()
plt.show()


print("DONE.")
print("Created:")
print("- year_field_industry_table")
print("- display_table")
print("- chart_data")

# Check for 2024 missing data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import gc

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# CHECK 2024 TABLE
# MEMORY OPTIMIZED
# NO CHART
# NO SAVE
# Checks if 2024 exists before and after zero filter
# ============================================================

DATA_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHECK_YEAR = 2024
CHUNKSIZE = 100_000
COMPACT_EVERY_N_CHUNKS = 10
SHOW_ROWS = 100

metric_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
]


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_col_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        base = clean_col_name(col)

        if base not in seen:
            seen[base] = 0
            new_cols.append(base)
        else:
            seen[base] += 1
            new_cols.append(f"{base}_{seen[base]}")

    return new_cols


def detect_standard_metric(text):
    text = str(text).lower()
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if "establishment" in text and "birth" in text:
        return "establishment_births"

    if "establishment" in text and "death" in text:
        return "establishment_deaths"

    if "job opening" in text or "job openings" in text:
        return "job_openings"

    if "job closing" in text or "job closings" in text:
        return "job_closings"

    if "job gain" in text or "job gains" in text or "gross job gain" in text:
        return "job_gains"

    if "job loss" in text or "job losses" in text or "gross job loss" in text:
        return "job_losses"

    if re.search(r"\bopenings\b", text):
        return "job_openings"

    if re.search(r"\bclosings\b", text):
        return "job_closings"

    if re.search(r"\bgains\b", text):
        return "job_gains"

    if re.search(r"\blosses\b", text):
        return "job_losses"

    if re.search(r"\bbirths\b", text):
        return "establishment_births"

    if re.search(r"\bdeaths\b", text):
        return "establishment_deaths"

    return np.nan


def naics_to_sector(code):
    if pd.isna(code):
        return np.nan

    digits = re.sub(r"[^0-9]", "", str(code))

    if len(digits) < 2:
        return np.nan

    two = int(digits[:2])

    if two == 11:
        return "Agriculture, Forestry, Fishing and Hunting"
    if two == 21:
        return "Mining, Quarrying, and Oil and Gas Extraction"
    if two == 22:
        return "Utilities"
    if two == 23:
        return "Construction"
    if 31 <= two <= 33:
        return "Manufacturing"
    if two == 42:
        return "Wholesale Trade"
    if 44 <= two <= 45:
        return "Retail Trade"
    if 48 <= two <= 49:
        return "Transportation and Warehousing"
    if two == 51:
        return "Information"
    if two == 52:
        return "Finance and Insurance"
    if two == 53:
        return "Real Estate and Rental and Leasing"
    if two == 54:
        return "Professional, Scientific, and Technical Services"
    if two == 55:
        return "Management of Companies and Enterprises"
    if two == 56:
        return "Administrative and Support and Waste Management"
    if two == 61:
        return "Educational Services"
    if two == 62:
        return "Health Care and Social Assistance"
    if two == 71:
        return "Arts, Entertainment, and Recreation"
    if two == 72:
        return "Accommodation and Food Services"
    if two == 81:
        return "Other Services"
    if two == 92:
        return "Public Administration"

    return np.nan


def industry_name_to_sector(name):
    if pd.isna(name):
        return np.nan

    text = str(name).lower()

    if "agriculture" in text or "forestry" in text or "fishing" in text:
        return "Agriculture, Forestry, Fishing and Hunting"
    if "mining" in text or "oil" in text or "gas" in text:
        return "Mining, Quarrying, and Oil and Gas Extraction"
    if "utilities" in text:
        return "Utilities"
    if "construction" in text:
        return "Construction"
    if "manufacturing" in text:
        return "Manufacturing"
    if "wholesale" in text:
        return "Wholesale Trade"
    if "retail" in text:
        return "Retail Trade"
    if "transportation" in text or "warehousing" in text:
        return "Transportation and Warehousing"
    if "information" in text:
        return "Information"
    if "finance" in text or "insurance" in text:
        return "Finance and Insurance"
    if "real estate" in text or "rental" in text or "leasing" in text:
        return "Real Estate and Rental and Leasing"
    if "professional" in text or "scientific" in text or "technical" in text:
        return "Professional, Scientific, and Technical Services"
    if "management of companies" in text:
        return "Management of Companies and Enterprises"
    if "administrative" in text or "waste" in text:
        return "Administrative and Support and Waste Management"
    if "educational" in text or "education" in text:
        return "Educational Services"
    if "health care" in text or "healthcare" in text or "social assistance" in text:
        return "Health Care and Social Assistance"
    if "arts" in text or "entertainment" in text or "recreation" in text:
        return "Arts, Entertainment, and Recreation"
    if "accommodation" in text or "food services" in text or "restaurant" in text:
        return "Accommodation and Food Services"
    if "other services" in text:
        return "Other Services"
    if "public administration" in text or "government" in text:
        return "Public Administration"

    return str(name).strip()


def add_connected_industry_sector(df):
    if "connected_industry_sector" in df.columns:
        df["connected_industry_sector"] = (
            df["connected_industry_sector"]
            .fillna("Unknown Sector")
            .astype(str)
            .str.strip()
        )
        return df

    sector = pd.Series(np.nan, index=df.index, dtype="object")

    for code_col in ["naics_code", "industry_code"]:
        if code_col in df.columns:
            sector = sector.fillna(df[code_col].map(naics_to_sector))

    for name_col in ["industry_sector", "sector", "industry_name", "industry"]:
        if name_col in df.columns:
            sector = sector.fillna(df[name_col].map(industry_name_to_sector))

    df["connected_industry_sector"] = (
        sector
        .fillna("Unknown Sector")
        .astype(str)
        .str.strip()
    )

    return df


def sector_to_fields(sector):
    sector = str(sector).lower()

    if "agriculture" in sector:
        return [
            "Agriculture",
            "Biological and Biomedical Sciences",
            "Natural Resources and Conservation",
        ]

    if "mining" in sector or "oil" in sector or "gas" in sector:
        return [
            "Engineering",
            "Engineering Technologies",
            "Physical Sciences",
        ]

    if "utilities" in sector:
        return [
            "Engineering",
            "Engineering Technologies",
            "Natural Resources and Conservation",
        ]

    if "construction" in sector:
        return [
            "Architecture and Related Services",
            "Engineering",
            "Construction Trades",
        ]

    if "manufacturing" in sector:
        return [
            "Engineering",
            "Engineering Technologies",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "wholesale" in sector or "retail" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Communication, Journalism, and Related Programs",
        ]

    if "transportation" in sector or "warehousing" in sector:
        return [
            "Transportation and Materials Moving",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "information" in sector:
        return [
            "Computer and Information Sciences",
            "Communication, Journalism, and Related Programs",
            "Engineering",
        ]

    if "finance" in sector or "insurance" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Mathematics and Statistics",
            "Social Sciences",
        ]

    if "real estate" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Architecture and Related Services",
        ]

    if "professional" in sector or "scientific" in sector or "technical" in sector:
        return [
            "Computer and Information Sciences",
            "Engineering",
            "Business, Management, Marketing, and Related Support Services",
            "Mathematics and Statistics",
            "Physical Sciences",
        ]

    if "management of companies" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "administrative" in sector or "waste" in sector:
        return [
            "Business, Management, Marketing, and Related Support Services",
            "Public Administration and Social Service Professions",
            "Natural Resources and Conservation",
        ]

    if "educational" in sector:
        return [
            "Education",
            "Library Science",
        ]

    if "health care" in sector or "social assistance" in sector:
        return [
            "Health Professions and Related Programs",
            "Psychology",
            "Public Administration and Social Service Professions",
            "Biological and Biomedical Sciences",
        ]

    if "arts" in sector or "entertainment" in sector or "recreation" in sector:
        return [
            "Visual and Performing Arts",
            "Communication, Journalism, and Related Programs",
            "Parks, Recreation, Leisure, Fitness, and Kinesiology",
        ]

    if "accommodation" in sector or "food services" in sector:
        return [
            "Culinary, Entertainment, and Personal Services",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "other services" in sector:
        return [
            "Culinary, Entertainment, and Personal Services",
            "Business, Management, Marketing, and Related Support Services",
        ]

    if "public administration" in sector:
        return [
            "Public Administration and Social Service Professions",
            "Homeland Security, Law Enforcement, Firefighting and Related Protective Services",
            "Social Sciences",
        ]

    return ["General / Other Field"]


field_candidates = [
    "field_of_study",
    "degree_field",
    "degree_fields",
    "field_group",
    "field_subgroup",
    "major_name",
    "cip_title",
    "cip_name",
]


def add_field_of_study(df, field_col_found):
    if field_col_found is not None and field_col_found in df.columns:
        df["field_of_study"] = (
            df[field_col_found]
            .fillna("Unknown Field")
            .astype(str)
            .str.strip()
        )
        return df

    df["field_of_study"] = df["connected_industry_sector"].map(sector_to_fields)
    df = df.explode("field_of_study", ignore_index=True)
    df["field_of_study"] = df["field_of_study"].astype(str).str.strip()

    return df


def compact_parts(parts):
    if len(parts) == 0:
        return []

    temp = pd.concat(parts, ignore_index=True)

    for col in metric_cols:
        if col not in temp.columns:
            temp[col] = 0

    temp = (
        temp
        .groupby(
            ["year", "field_of_study", "connected_industry_sector"],
            as_index=False
        )[metric_cols]
        .sum()
    )

    return [temp]


def add_formula_columns(df):
    df = df.copy()

    for col in metric_cols:
        if col not in df.columns:
            df[col] = 0

        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["success_total"] = (
        df["job_openings"]
        + df["job_gains"]
        + df["establishment_births"]
    )

    df["failure_total"] = (
        df["job_closings"]
        + df["job_losses"]
        + df["establishment_deaths"]
    )

    df["net_success_minus_failure"] = (
        df["success_total"]
        - df["failure_total"]
    )

    df["total_activity"] = (
        df["success_total"]
        + df["failure_total"]
    )

    df["success_rate"] = np.where(
        df["total_activity"] > 0,
        df["success_total"] / df["total_activity"] * 100,
        np.nan
    )

    df["failure_rate"] = np.where(
        df["total_activity"] > 0,
        df["failure_total"] / df["total_activity"] * 100,
        np.nan
    )

    return df


# ------------------------------------------------------------
# 2. Read header only
# ------------------------------------------------------------
check_file(DATA_FILE)

raw_columns = list(pd.read_csv(DATA_FILE, nrows=0).columns)
clean_columns = make_unique_columns(raw_columns)

column_table = pd.DataFrame({
    "column_number": range(1, len(clean_columns) + 1),
    "column_name": clean_columns
})

print("\nCOLUMNS IN FILE")
display(column_table)


# ------------------------------------------------------------
# 3. Detect wide or long format
# ------------------------------------------------------------
has_any_wide_metric = any(col in clean_columns for col in metric_cols)

value_candidates = [
    "value",
    "company_value",
    "metric_value",
    "count",
    "amount",
]

label_candidates = [
    "metric_name",
    "metric_type",
    "metric_category",
    "metric_code",
    "source_table",
    "dataset",
    "all_text",
    "notes",
]

value_col = None
for col in value_candidates:
    if col in clean_columns:
        value_col = col
        break

label_cols = [
    col for col in label_candidates
    if col in clean_columns
]

field_col_found = None
for col in field_candidates:
    if col in clean_columns:
        field_col_found = col
        break

if has_any_wide_metric:
    source_format = "WIDE TABLE"
else:
    source_format = "LONG TABLE"

    if value_col is None:
        raise KeyError(
            "Cannot find value column like value, company_value, metric_value, count, or amount."
        )

    if len(label_cols) == 0:
        raise KeyError(
            "Cannot find metric label column like metric_name, metric_type, source_table, dataset, or notes."
        )

print("\nDATA FORMAT FOUND:", source_format)

if source_format == "LONG TABLE":
    print("Value column used:", value_col)
    print("Metric label columns used:", label_cols)

if field_col_found is not None:
    print("Field of study column used:", field_col_found)
else:
    print("No field of study column found, so field_of_study will be created from industry mapping.")


# ------------------------------------------------------------
# 4. Memory optimization: only load needed columns
# ------------------------------------------------------------
industry_needed = [
    "connected_industry_sector",
    "naics_code",
    "industry_code",
    "industry_sector",
    "sector",
    "industry_name",
    "industry",
]

needed_clean_cols = set(["year"])
needed_clean_cols.update(industry_needed)
needed_clean_cols.update(field_candidates)

if source_format == "WIDE TABLE":
    needed_clean_cols.update(metric_cols)
else:
    needed_clean_cols.add(value_col)
    needed_clean_cols.update(label_cols)

raw_usecols = [
    raw
    for raw, clean in zip(raw_columns, clean_columns)
    if clean in needed_clean_cols
]

print("\nMEMORY OPTIMIZATION")
print("Total file columns:", len(raw_columns))
print("Columns loaded per chunk:", len(raw_usecols))
print("Chunk size:", CHUNKSIZE)


# ------------------------------------------------------------
# 5. Read only 2024 in chunks
# ------------------------------------------------------------
parts_2024 = []

raw_2024_rows = 0
usable_2024_rows = 0
metric_detection_parts = []

reader = pd.read_csv(
    DATA_FILE,
    usecols=raw_usecols,
    chunksize=CHUNKSIZE,
    low_memory=False
)

for chunk_number, chunk in enumerate(reader, start=1):

    chunk.columns = make_unique_columns(chunk.columns)

    if "year" not in chunk.columns:
        raise KeyError("No year column found.")

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    chunk = chunk[chunk["year"] == CHECK_YEAR].copy()

    raw_2024_rows += len(chunk)

    if chunk.empty:
        del chunk
        gc.collect()
        continue

    if source_format == "WIDE TABLE":

        for col in metric_cols:
            if col not in chunk.columns:
                chunk[col] = 0

            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").fillna(0)

        usable_2024_rows += len(chunk)

        chunk = add_connected_industry_sector(chunk)
        chunk = add_field_of_study(chunk, field_col_found)

        grouped = (
            chunk
            .groupby(
                ["year", "field_of_study", "connected_industry_sector"],
                as_index=False
            )[metric_cols]
            .sum()
        )

        parts_2024.append(grouped)

    else:

        chunk[value_col] = pd.to_numeric(
            chunk[value_col],
            errors="coerce"
        ).fillna(0)

        metric_text = (
            chunk[label_cols]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
        )

        chunk["standard_metric"] = metric_text.map(detect_standard_metric)

        detect_chunk = (
            chunk
            .groupby("standard_metric", dropna=False, as_index=False)
            .agg(
                rows=("standard_metric", "size"),
                total_value=(value_col, "sum")
            )
        )

        metric_detection_parts.append(detect_chunk)

        chunk = chunk[chunk["standard_metric"].notna()].copy()

        usable_2024_rows += len(chunk)

        if chunk.empty:
            del chunk
            gc.collect()
            continue

        chunk = add_connected_industry_sector(chunk)
        chunk = add_field_of_study(chunk, field_col_found)

        grouped_long = (
            chunk
            .groupby(
                [
                    "year",
                    "field_of_study",
                    "connected_industry_sector",
                    "standard_metric",
                ],
                as_index=False
            )[value_col]
            .sum()
        )

        grouped_wide = (
            grouped_long
            .pivot_table(
                index=[
                    "year",
                    "field_of_study",
                    "connected_industry_sector",
                ],
                columns="standard_metric",
                values=value_col,
                aggfunc="sum",
                fill_value=0
            )
            .reset_index()
        )

        grouped_wide.columns.name = None

        for col in metric_cols:
            if col not in grouped_wide.columns:
                grouped_wide[col] = 0

        grouped_wide = grouped_wide[
            [
                "year",
                "field_of_study",
                "connected_industry_sector",
            ]
            + metric_cols
        ]

        parts_2024.append(grouped_wide)

    if chunk_number % COMPACT_EVERY_N_CHUNKS == 0:
        parts_2024 = compact_parts(parts_2024)

        print(
            f"Processed chunk {chunk_number:,} | "
            f"2024 raw rows found: {raw_2024_rows:,} | "
            f"2024 usable metric rows: {usable_2024_rows:,}"
        )

    del chunk
    gc.collect()


print("\nDONE READING 2024")
print("2024 raw rows found:", f"{raw_2024_rows:,}")
print("2024 usable metric rows:", f"{usable_2024_rows:,}")


# ------------------------------------------------------------
# 6. Metric detection table for 2024
# ------------------------------------------------------------
if source_format == "LONG TABLE" and len(metric_detection_parts) > 0:

    metric_detection_2024 = (
        pd.concat(metric_detection_parts, ignore_index=True)
        .groupby("standard_metric", dropna=False, as_index=False)
        .agg(
            rows=("rows", "sum"),
            total_value=("total_value", "sum")
        )
    )

    print("\n2024 METRIC DETECTION TABLE")
    display(metric_detection_2024)


# ------------------------------------------------------------
# 7. Final 2024 grouped table before zero filter
# ------------------------------------------------------------
parts_2024 = compact_parts(parts_2024)

if len(parts_2024) == 0:
    print("\nRESULT:")
    print("No usable 2024 rows were found.")
    print("That means 2024 is not in the data, or 2024 metric names could not be detected.")
else:
    table_2024_before_filter = parts_2024[0]

    table_2024_before_filter = add_formula_columns(table_2024_before_filter)

    table_2024_before_filter["year"] = (
        pd.to_numeric(
            table_2024_before_filter["year"],
            errors="coerce"
        )
        .astype("Int64")
    )

    table_2024_before_filter = table_2024_before_filter.sort_values(
        "total_activity",
        ascending=False
    ).reset_index(drop=True)

    print("\n2024 TABLE BEFORE ZERO FILTER")
    display(
        table_2024_before_filter[
            [
                "year",
                "field_of_study",
                "connected_industry_sector",
                "job_openings",
                "job_closings",
                "job_gains",
                "job_losses",
                "establishment_births",
                "establishment_deaths",
                "success_total",
                "failure_total",
                "total_activity",
                "net_success_minus_failure",
                "success_rate",
                "failure_rate",
            ]
        ].head(SHOW_ROWS)
    )

    # --------------------------------------------------------
    # 8. Check why 2024 may disappear
    # --------------------------------------------------------
    table_2024_filter_check = table_2024_before_filter.copy()

    table_2024_filter_check["success_is_zero"] = (
        table_2024_filter_check["success_total"] == 0
    )

    table_2024_filter_check["failure_is_zero"] = (
        table_2024_filter_check["failure_total"] == 0
    )

    table_2024_filter_check["total_activity_is_zero"] = (
        table_2024_filter_check["total_activity"] == 0
    )

    print("\n2024 ZERO FILTER CHECK")
    display(
        table_2024_filter_check[
            [
                "year",
                "field_of_study",
                "connected_industry_sector",
                "success_total",
                "failure_total",
                "total_activity",
                "success_is_zero",
                "failure_is_zero",
                "total_activity_is_zero",
            ]
        ].head(SHOW_ROWS)
    )

    # --------------------------------------------------------
    # 9. 2024 table after same filter used for chart
    # --------------------------------------------------------
    table_2024_after_filter = table_2024_before_filter[
        (table_2024_before_filter["success_total"] > 0)
        & (table_2024_before_filter["failure_total"] > 0)
    ].copy()

    print("\n2024 TABLE AFTER SUCCESS AND FAILURE NONZERO FILTER")
    print("Rows before filter:", len(table_2024_before_filter))
    print("Rows after filter:", len(table_2024_after_filter))

    display(
        table_2024_after_filter[
            [
                "year",
                "field_of_study",
                "connected_industry_sector",
                "success_total",
                "failure_total",
                "total_activity",
                "net_success_minus_failure",
                "success_rate",
                "failure_rate",
            ]
        ].head(SHOW_ROWS)
    )

    # --------------------------------------------------------
    # 10. Simple answer
    # --------------------------------------------------------
    print("\nFINAL CHECK RESULT")

    if raw_2024_rows == 0:
        print("2024 is NOT in the raw data.")
    elif len(table_2024_before_filter) > 0 and len(table_2024_after_filter) == 0:
        print("2024 is in the data, but it was removed because success_total or failure_total is 0.")
    elif len(table_2024_after_filter) > 0:
        print("2024 is in the data and passes the chart filter.")
    else:
        print("2024 exists but no usable grouped rows were created.")

# 1984 to 2023 use instead

# Degree + company data from 1984 to 2023

layman term:

Computer degree → Information industry

If the right side is bigger, that means more job/startup growth.

If the left side is bigger, that means more job/startup loss.

Short:

This chart compares yearly success vs failure for degree-field and industry-sector connections.

It compares yearly success and failure activity for the top degree-field and industry-sector connections.
Right side shows success:

job openings + job gains + establishment births

Left side shows failure:

job closings + job losses + establishment deaths

In [ ]:
# ------------------------------------------------------------
# 9. Left-right opposite stacked bar chart
# Success goes right
# Failure goes left
# ------------------------------------------------------------
y = np.arange(len(years))

left_success = np.zeros(len(years))
left_failure = np.zeros(len(years))

plt.figure(figsize=(22, 12))
ax = plt.gca()

for pair in top_pairs:
    label = short_label(pair, width=65)

    success_values = success_pivot[pair].values
    failure_values = failure_pivot[pair].values

    bars = ax.barh(
        y,
        success_values,
        left=left_success,
        label=label
    )

    chosen_color = bars.patches[0].get_facecolor()

    ax.barh(
        y,
        -failure_values,
        left=left_failure,
        color=chosen_color,
        alpha=0.45
    )

    left_success = left_success + success_values
    left_failure = left_failure - failure_values

ax.axvline(0, linewidth=1)

ax.set_title(
    "Left-Right Success vs Failure by Year, Field of Study, and Industry\n"
    "Left = Failure (Job Closings + Job Losses + Establishment Deaths) | "
    "Right = Success (Job Openings + Job Gains + Establishment Births)",
    fontsize=14
)

ax.set_xlabel("Failure ← 0 → Success")
ax.set_ylabel("Year")

ax.set_yticks(y)
ax.set_yticklabels(years)

ax.legend(
    title="Field of Study → Industry Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# FORMULAS FOR 2-PATH COMPARISON
# Degree → Industry Job Path vs Degree → Startup Path
# 1984–2023
# ------------------------------------------------------------

1:

industry_success = job_openings + job_gains

industry_failure = job_closings + job_losses

2:

startup_success = establishment_births

startup_failure = establishment_deaths

3:

industry_success_rate = industry_success / (industry_success + industry_failure) * 100

industry_failure_rate = industry_failure / (industry_success + industry_failure) * 100

4:

startup_success_rate = startup_success / (startup_success + startup_failure) * 100

startup_failure_rate = startup_failure / (startup_success + startup_failure) * 100

5:

industry_net = industry_success - industry_failure

startup_net = startup_success - startup_failure

6:

better_path = industry job path if industry_success_rate > startup_success_rate

better_path = startup path if startup_success_rate > industry_success_rate

7:

Industry Job Path = job openings + job gains compared to job closings + job losses

Startup Business Path = establishment births compared to establishment deaths

In [ ]:
# ============================================================
# DEGREE → INDUSTRY JOB PATH VS STARTUP BUSINESS PATH
# 1984–2023
#
# No saving files
# Memory optimized with chunks
#
# Goal:
# Compare whether the connected industry job path looks better
# or the startup/business path looks better after degree.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import re
import time

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

YEAR_START = 1984
YEAR_END = 2023

CHUNKSIZE = 100_000
TOP_N_PAIRS = 12

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


# ------------------------------------------------------------
# 2. BASIC HELPERS
# ------------------------------------------------------------

def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_text(x):
    x = clean_text(x).lower()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def short_label(text, width=55):
    text = str(text)
    return "\n".join(textwrap.wrap(text, width=width))


def find_first_existing_column(columns, possible_names):
    columns_lower = {c.lower(): c for c in columns}
    for name in possible_names:
        if name.lower() in columns_lower:
            return columns_lower[name.lower()]
    return None


def safe_rate(success, failure):
    total = success + failure
    return np.where(total > 0, success / total * 100, np.nan)


# ------------------------------------------------------------
# 3. DEGREE FIELD → CONNECTED INDUSTRY SECTOR
# ------------------------------------------------------------

def map_degree_field_to_industry_sector(field):
    """
    This connects degree fields to the most related industry sector.
    It is a bridge/mapping, not a direct column from the raw company file.
    """

    f = normalize_text(field)

    if f == "":
        return "Unmapped / Other"

    if any(k in f for k in ["computer", "information technology", "data", "informatics"]):
        return "Information"

    if any(k in f for k in ["business", "management", "marketing", "accounting", "finance"]):
        return "Professional and Business Services"

    if any(k in f for k in ["engineering", "mechanic", "precision production", "manufacturing"]):
        return "Manufacturing"

    if any(k in f for k in ["health", "nursing", "medicine", "dental", "pharmacy", "therapy"]):
        return "Health Care and Social Assistance"

    if any(k in f for k in ["education", "teacher", "teaching"]):
        return "Educational Services"

    if any(k in f for k in ["agriculture", "natural resources", "animal", "plant"]):
        return "Agriculture, Forestry, Fishing and Hunting"

    if any(k in f for k in ["communication", "journalism", "media"]):
        return "Information"

    if any(k in f for k in ["visual", "performing", "arts", "music", "design", "drama"]):
        return "Arts, Entertainment, and Recreation"

    if any(k in f for k in ["construction", "architecture"]):
        return "Construction"

    if any(k in f for k in ["law", "legal", "public administration", "homeland", "security", "fire", "protective"]):
        return "Public Administration"

    if any(k in f for k in ["transportation", "logistics"]):
        return "Transportation and Warehousing"

    if any(k in f for k in ["culinary", "hospitality", "restaurant", "tourism"]):
        return "Accommodation and Food Services"

    if any(k in f for k in ["psychology", "social work", "family", "human services"]):
        return "Health Care and Social Assistance"

    if any(k in f for k in ["mathematics", "statistics", "physical sciences", "biology", "biological", "science"]):
        return "Professional and Business Services"

    if any(k in f for k in ["english", "language", "history", "liberal arts", "philosophy"]):
        return "Educational Services"

    return "Unmapped / Other"


# ------------------------------------------------------------
# 4. COMPANY INDUSTRY NAME / NAICS → INDUSTRY SECTOR
# ------------------------------------------------------------

NAICS_SECTOR_MAP = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31": "Manufacturing",
    "32": "Manufacturing",
    "33": "Manufacturing",
    "42": "Wholesale Trade",
    "44": "Retail Trade",
    "45": "Retail Trade",
    "48": "Transportation and Warehousing",
    "49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional and Business Services",
    "55": "Professional and Business Services",
    "56": "Professional and Business Services",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services",
    "92": "Public Administration",
}


def sector_from_naics(x):
    if pd.isna(x):
        return ""

    s = str(x).strip()

    # examples: 51, 510000, 51-Information, 31-33
    match = re.search(r"\d{2}", s)
    if not match:
        return ""

    prefix = match.group(0)
    return NAICS_SECTOR_MAP.get(prefix, "")


def map_company_industry_to_sector(industry_name, naics_code=None, industry_code=None):
    sector = sector_from_naics(naics_code)
    if sector:
        return sector

    sector = sector_from_naics(industry_code)
    if sector:
        return sector

    name = normalize_text(industry_name)

    if name == "":
        return "Unmapped / Other"

    if "agriculture" in name or "forestry" in name or "fishing" in name:
        return "Agriculture, Forestry, Fishing and Hunting"

    if "mining" in name or "oil" in name or "gas" in name:
        return "Mining, Quarrying, and Oil and Gas Extraction"

    if "utilities" in name:
        return "Utilities"

    if "construction" in name:
        return "Construction"

    if "manufacturing" in name:
        return "Manufacturing"

    if "wholesale" in name:
        return "Wholesale Trade"

    if "retail" in name:
        return "Retail Trade"

    if "transportation" in name or "warehousing" in name:
        return "Transportation and Warehousing"

    if "information" in name or "software" in name or "data" in name or "telecommunications" in name:
        return "Information"

    if "finance" in name or "insurance" in name:
        return "Finance and Insurance"

    if "real estate" in name or "rental" in name or "leasing" in name:
        return "Real Estate and Rental and Leasing"

    if "professional" in name or "scientific" in name or "technical" in name:
        return "Professional and Business Services"

    if "management" in name or "administrative" in name or "support" in name:
        return "Professional and Business Services"

    if "education" in name:
        return "Educational Services"

    if "health" in name or "social assistance" in name:
        return "Health Care and Social Assistance"

    if "arts" in name or "entertainment" in name or "recreation" in name:
        return "Arts, Entertainment, and Recreation"

    if "accommodation" in name or "food" in name:
        return "Accommodation and Food Services"

    if "public administration" in name:
        return "Public Administration"

    if "other services" in name:
        return "Other Services"

    return "Unmapped / Other"


# ------------------------------------------------------------
# 5. METRIC NAME CLEANING
# ------------------------------------------------------------

def standardize_metric_name(metric_name, metric_code=None):
    """
    Converts metric names into the 6 metrics needed for this project.
    """

    m = normalize_text(metric_name)
    c = normalize_text(metric_code)

    combined = m + " " + c

    if "job opening" in combined or "openings" in combined:
        return "job_openings"

    if "job closing" in combined or "closings" in combined:
        return "job_closings"

    if "job gain" in combined or "gains" in combined:
        return "job_gains"

    if "job loss" in combined or "losses" in combined:
        return "job_losses"

    if "establishment birth" in combined or "births" in combined:
        return "establishment_births"

    if "establishment death" in combined or "deaths" in combined:
        return "establishment_deaths"

    return None


# ------------------------------------------------------------
# 6. READ DEGREE DATA
# ------------------------------------------------------------

def read_degree_data(path):
    print("\nREADING DEGREE DATA...")
    start = time.time()

    columns = list(pd.read_csv(path, nrows=0).columns)

    year_col = find_first_existing_column(columns, ["year"])
    degree_count_col = find_first_existing_column(columns, ["degree_count", "count", "value"])

    field_col = find_first_existing_column(
        columns,
        [
            "field_group",
            "field_of_study",
            "degree_field",
            "cip_field",
            "field_subgroup",
            "major_name",
            "degree_group"
        ]
    )

    if year_col is None:
        raise KeyError("Cannot find year column in degree file.")

    if degree_count_col is None:
        raise KeyError("Cannot find degree count column in degree file.")

    if field_col is None:
        raise KeyError("Cannot find field of study column in degree file.")

    usecols = [year_col, degree_count_col, field_col]

    print("Degree year column:", year_col)
    print("Degree count column:", degree_count_col)
    print("Field of study column:", field_col)

    pieces = []

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False):
        chunk = chunk.rename(
            columns={
                year_col: "year",
                degree_count_col: "degree_count",
                field_col: "field_of_study"
            }
        )

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["degree_count"] = pd.to_numeric(chunk["degree_count"], errors="coerce").fillna(0)

        chunk = chunk[
            (chunk["year"] >= YEAR_START) &
            (chunk["year"] <= YEAR_END) &
            (chunk["degree_count"] > 0)
        ].copy()

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)
        chunk["field_of_study"] = chunk["field_of_study"].fillna("Unknown").astype(str).str.strip()
        chunk["connected_industry_sector"] = chunk["field_of_study"].apply(map_degree_field_to_industry_sector)

        grouped = (
            chunk
            .groupby(["year", "field_of_study", "connected_industry_sector"], as_index=False)["degree_count"]
            .sum()
        )

        pieces.append(grouped)

    if not pieces:
        raise ValueError("No usable degree data found for selected years.")

    degree = pd.concat(pieces, ignore_index=True)

    degree = (
        degree
        .groupby(["year", "field_of_study", "connected_industry_sector"], as_index=False)["degree_count"]
        .sum()
    )

    print("Degree rows after grouping:", len(degree))
    print("Degree year range:", degree["year"].min(), "-", degree["year"].max())
    print("Finished degree data in", round(time.time() - start, 2), "seconds")

    return degree


# ------------------------------------------------------------
# 7. READ COMPANY DATA
# ------------------------------------------------------------

def read_company_data(path):
    print("\nREADING COMPANY DATA...")
    start = time.time()

    columns = list(pd.read_csv(path, nrows=0).columns)

    year_col = find_first_existing_column(columns, ["year"])
    value_col = find_first_existing_column(columns, ["value"])
    metric_name_col = find_first_existing_column(columns, ["metric_name", "metric"])
    metric_code_col = find_first_existing_column(columns, ["metric_code"])
    industry_name_col = find_first_existing_column(columns, ["industry_name", "industry"])
    naics_col = find_first_existing_column(columns, ["naics_code"])
    industry_code_col = find_first_existing_column(columns, ["industry_code"])

    if year_col is None:
        raise KeyError("Cannot find year column in company file.")

    if value_col is None:
        raise KeyError("Cannot find value column in company file.")

    if metric_name_col is None and metric_code_col is None:
        raise KeyError("Cannot find metric_name or metric_code column in company file.")

    if industry_name_col is None and naics_col is None and industry_code_col is None:
        raise KeyError("Cannot find industry_name, naics_code, or industry_code column in company file.")

    usecols = [year_col, value_col]

    for c in [metric_name_col, metric_code_col, industry_name_col, naics_col, industry_code_col]:
        if c is not None and c not in usecols:
            usecols.append(c)

    print("Company year column:", year_col)
    print("Company value column:", value_col)
    print("Company metric name column:", metric_name_col)
    print("Company metric code column:", metric_code_col)
    print("Company industry name column:", industry_name_col)
    print("Company NAICS column:", naics_col)
    print("Company industry code column:", industry_code_col)

    pieces = []

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False):
        rename_map = {
            year_col: "year",
            value_col: "value"
        }

        if metric_name_col is not None:
            rename_map[metric_name_col] = "metric_name"
        else:
            chunk["metric_name"] = ""

        if metric_code_col is not None:
            rename_map[metric_code_col] = "metric_code"
        else:
            chunk["metric_code"] = ""

        if industry_name_col is not None:
            rename_map[industry_name_col] = "industry_name"
        else:
            chunk["industry_name"] = ""

        if naics_col is not None:
            rename_map[naics_col] = "naics_code"
        else:
            chunk["naics_code"] = ""

        if industry_code_col is not None:
            rename_map[industry_code_col] = "industry_code"
        else:
            chunk["industry_code"] = ""

        chunk = chunk.rename(columns=rename_map)

        for needed in ["metric_name", "metric_code", "industry_name", "naics_code", "industry_code"]:
            if needed not in chunk.columns:
                chunk[needed] = ""

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce").fillna(0)

        chunk = chunk[
            (chunk["year"] >= YEAR_START) &
            (chunk["year"] <= YEAR_END) &
            (chunk["value"] > 0)
        ].copy()

        if chunk.empty:
            continue

        chunk["year"] = chunk["year"].astype(int)

        chunk["standard_metric"] = chunk.apply(
            lambda row: standardize_metric_name(row["metric_name"], row["metric_code"]),
            axis=1
        )

        chunk = chunk[chunk["standard_metric"].notna()].copy()

        if chunk.empty:
            continue

        chunk["connected_industry_sector"] = chunk.apply(
            lambda row: map_company_industry_to_sector(
                row["industry_name"],
                row["naics_code"],
                row["industry_code"]
            ),
            axis=1
        )

        grouped = (
            chunk
            .groupby(["year", "connected_industry_sector", "standard_metric"], as_index=False)["value"]
            .sum()
        )

        pieces.append(grouped)

    if not pieces:
        raise ValueError("No usable company data found for selected years and selected metrics.")

    company_long = pd.concat(pieces, ignore_index=True)

    company_long = (
        company_long
        .groupby(["year", "connected_industry_sector", "standard_metric"], as_index=False)["value"]
        .sum()
    )

    company = (
        company_long
        .pivot_table(
            index=["year", "connected_industry_sector"],
            columns="standard_metric",
            values="value",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    needed_metrics = [
        "job_openings",
        "job_closings",
        "job_gains",
        "job_losses",
        "establishment_births",
        "establishment_deaths"
    ]

    for col in needed_metrics:
        if col not in company.columns:
            company[col] = 0

    company = company[["year", "connected_industry_sector"] + needed_metrics]

    print("Company rows after grouping:", len(company))
    print("Company year range:", company["year"].min(), "-", company["year"].max())
    print("Finished company data in", round(time.time() - start, 2), "seconds")

    return company


# ------------------------------------------------------------
# 8. BUILD CONNECTED TABLE
# ------------------------------------------------------------

check_file(DEGREE_FILE)
check_file(COMPANY_FILE)

degree_table = read_degree_data(DEGREE_FILE)
company_table = read_company_data(COMPANY_FILE)

print("\nBUILDING DEGREE → INDUSTRY CONNECTION TABLE...")

# This prevents double-counting.
# If 3 degree fields connect to the same industry sector,
# company values are split by degree share.
sector_degree_total = (
    degree_table
    .groupby(["year", "connected_industry_sector"], as_index=False)["degree_count"]
    .sum()
    .rename(columns={"degree_count": "sector_degree_count"})
)

degree_table = degree_table.merge(
    sector_degree_total,
    on=["year", "connected_industry_sector"],
    how="left"
)

degree_table["degree_share_in_sector"] = (
    degree_table["degree_count"] / degree_table["sector_degree_count"]
)

connected = degree_table.merge(
    company_table,
    on=["year", "connected_industry_sector"],
    how="inner"
)

if connected.empty:
    raise ValueError("No matching rows between degree sectors and company sectors.")

metric_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths"
]

for col in metric_cols:
    connected[col + "_weighted"] = connected[col] * connected["degree_share_in_sector"]


# ------------------------------------------------------------
# 9. FORMULAS
# ------------------------------------------------------------

connected["industry_success"] = (
    connected["job_openings_weighted"] +
    connected["job_gains_weighted"]
)

connected["industry_failure"] = (
    connected["job_closings_weighted"] +
    connected["job_losses_weighted"]
)

connected["startup_success"] = connected["establishment_births_weighted"]

connected["startup_failure"] = connected["establishment_deaths_weighted"]

connected["industry_success_rate"] = safe_rate(
    connected["industry_success"],
    connected["industry_failure"]
)

connected["industry_failure_rate"] = safe_rate(
    connected["industry_failure"],
    connected["industry_success"]
)

connected["startup_success_rate"] = safe_rate(
    connected["startup_success"],
    connected["startup_failure"]
)

connected["startup_failure_rate"] = safe_rate(
    connected["startup_failure"],
    connected["startup_success"]
)

connected["industry_net"] = connected["industry_success"] - connected["industry_failure"]
connected["startup_net"] = connected["startup_success"] - connected["startup_failure"]

connected["better_path_by_rate"] = np.where(
    connected["industry_success_rate"] > connected["startup_success_rate"],
    "Industry Job Path",
    np.where(
        connected["startup_success_rate"] > connected["industry_success_rate"],
        "Startup Business Path",
        "Tie"
    )
)

connected["field_to_sector"] = (
    connected["field_of_study"].astype(str) +
    " → " +
    connected["connected_industry_sector"].astype(str)
)


# ------------------------------------------------------------
# 10. MAIN COMPARISON TABLE
# ------------------------------------------------------------

comparison_table = connected[
    [
        "year",
        "field_of_study",
        "connected_industry_sector",
        "degree_count",
        "industry_success",
        "industry_failure",
        "industry_success_rate",
        "industry_net",
        "startup_success",
        "startup_failure",
        "startup_success_rate",
        "startup_net",
        "better_path_by_rate"
    ]
].copy()

comparison_table = comparison_table.sort_values(
    ["year", "degree_count"],
    ascending=[True, False]
)

print("\nMAIN COMPARISON TABLE")
print("This separates the two paths:")
print("Column group 1 = Industry Job Path")
print("Column group 2 = Startup Business Path")
display(comparison_table.head(50))


# ------------------------------------------------------------
# 11. SUMMARY BY YEAR
# ------------------------------------------------------------

year_summary = (
    connected
    .groupby("year", as_index=False)
    .agg(
        degree_count=("degree_count", "sum"),
        industry_success=("industry_success", "sum"),
        industry_failure=("industry_failure", "sum"),
        startup_success=("startup_success", "sum"),
        startup_failure=("startup_failure", "sum")
    )
)

year_summary["industry_success_rate"] = safe_rate(
    year_summary["industry_success"],
    year_summary["industry_failure"]
)

year_summary["startup_success_rate"] = safe_rate(
    year_summary["startup_success"],
    year_summary["startup_failure"]
)

year_summary["industry_net"] = year_summary["industry_success"] - year_summary["industry_failure"]
year_summary["startup_net"] = year_summary["startup_success"] - year_summary["startup_failure"]

year_summary["better_path_by_rate"] = np.where(
    year_summary["industry_success_rate"] > year_summary["startup_success_rate"],
    "Industry Job Path",
    np.where(
        year_summary["startup_success_rate"] > year_summary["industry_success_rate"],
        "Startup Business Path",
        "Tie"
    )
)

print("\nSUMMARY BY YEAR")
display(year_summary)


# ------------------------------------------------------------
# 12. TOP FIELD → INDUSTRY PAIRS
# ------------------------------------------------------------

top_pairs = (
    connected
    .groupby("field_to_sector", as_index=False)
    .agg(
        total_degree_count=("degree_count", "sum"),
        industry_success=("industry_success", "sum"),
        industry_failure=("industry_failure", "sum"),
        startup_success=("startup_success", "sum"),
        startup_failure=("startup_failure", "sum")
    )
)

top_pairs["industry_success_rate"] = safe_rate(
    top_pairs["industry_success"],
    top_pairs["industry_failure"]
)

top_pairs["startup_success_rate"] = safe_rate(
    top_pairs["startup_success"],
    top_pairs["startup_failure"]
)

top_pairs["industry_net"] = top_pairs["industry_success"] - top_pairs["industry_failure"]
top_pairs["startup_net"] = top_pairs["startup_success"] - top_pairs["startup_failure"]

top_pairs["better_path_by_rate"] = np.where(
    top_pairs["industry_success_rate"] > top_pairs["startup_success_rate"],
    "Industry Job Path",
    np.where(
        top_pairs["startup_success_rate"] > top_pairs["industry_success_rate"],
        "Startup Business Path",
        "Tie"
    )
)

top_pairs = top_pairs.sort_values("total_degree_count", ascending=False)

print("\nTOP FIELD OF STUDY → INDUSTRY SECTOR PAIRS")
display(top_pairs.head(TOP_N_PAIRS))

selected_pairs = top_pairs.head(TOP_N_PAIRS)["field_to_sector"].tolist()


# ------------------------------------------------------------
# 13. CHART 1: TWO-COLUMN SUCCESS RATE BY YEAR
# ------------------------------------------------------------

plt.figure(figsize=(18, 8))

x = np.arange(len(year_summary))
bar_width = 0.42

plt.bar(
    x - bar_width / 2,
    year_summary["industry_success_rate"],
    width=bar_width,
    label="Industry Job Path Success Rate"
)

plt.bar(
    x + bar_width / 2,
    year_summary["startup_success_rate"],
    width=bar_width,
    label="Startup Business Path Success Rate"
)

plt.title(
    "Two-Column Comparison by Year: Industry Job Path vs Startup Business Path\n"
    "Success Rate = Success / (Success + Failure) × 100",
    fontsize=14
)

plt.xlabel("Year")
plt.ylabel("Success Rate (%)")

plt.xticks(x, year_summary["year"], rotation=90)
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 14. CHART 2: INDUSTRY JOB PATH LEFT-RIGHT CHART
# Failure left, success right
# ------------------------------------------------------------

chart_data = connected[connected["field_to_sector"].isin(selected_pairs)].copy()

industry_success_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="field_to_sector",
        values="industry_success",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

industry_failure_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="field_to_sector",
        values="industry_failure",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

years = industry_success_pivot.index.tolist()
y = np.arange(len(years))

left_success = np.zeros(len(years))
left_failure = np.zeros(len(years))

plt.figure(figsize=(22, 12))
ax = plt.gca()

for pair in selected_pairs:
    if pair not in industry_success_pivot.columns:
        continue

    label = short_label(pair, width=65)

    success_values = industry_success_pivot[pair].values
    failure_values = industry_failure_pivot[pair].values

    bars = ax.barh(
        y,
        success_values,
        left=left_success,
        label=label
    )

    chosen_color = bars.patches[0].get_facecolor()

    ax.barh(
        y,
        -failure_values,
        left=left_failure,
        color=chosen_color,
        alpha=0.45
    )

    left_success = left_success + success_values
    left_failure = left_failure - failure_values

ax.axvline(0, linewidth=1)

ax.set_title(
    "Industry Job Path: Success vs Failure by Year, Field of Study, and Industry\n"
    "Left = Failure (Job Closings + Job Losses) | "
    "Right = Success (Job Openings + Job Gains)",
    fontsize=14
)

ax.set_xlabel("Failure ← 0 → Success")
ax.set_ylabel("Year")

ax.set_yticks(y)
ax.set_yticklabels(years)

ax.legend(
    title="Field of Study → Industry Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 15. CHART 3: STARTUP BUSINESS PATH LEFT-RIGHT CHART
# Failure left, success right
# ------------------------------------------------------------

startup_success_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="field_to_sector",
        values="startup_success",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

startup_failure_pivot = (
    chart_data
    .pivot_table(
        index="year",
        columns="field_to_sector",
        values="startup_failure",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

years = startup_success_pivot.index.tolist()
y = np.arange(len(years))

left_success = np.zeros(len(years))
left_failure = np.zeros(len(years))

plt.figure(figsize=(22, 12))
ax = plt.gca()

for pair in selected_pairs:
    if pair not in startup_success_pivot.columns:
        continue

    label = short_label(pair, width=65)

    success_values = startup_success_pivot[pair].values
    failure_values = startup_failure_pivot[pair].values

    bars = ax.barh(
        y,
        success_values,
        left=left_success,
        label=label
    )

    chosen_color = bars.patches[0].get_facecolor()

    ax.barh(
        y,
        -failure_values,
        left=left_failure,
        color=chosen_color,
        alpha=0.45
    )

    left_success = left_success + success_values
    left_failure = left_failure - failure_values

ax.axvline(0, linewidth=1)

ax.set_title(
    "Startup Business Path: Success vs Failure by Year, Field of Study, and Industry\n"
    "Left = Failure (Establishment Deaths) | "
    "Right = Success (Establishment Births)",
    fontsize=14
)

ax.set_xlabel("Failure ← 0 → Success")
ax.set_ylabel("Year")

ax.set_yticks(y)
ax.set_yticklabels(years)

ax.legend(
    title="Field of Study → Industry Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 16. FINAL SIMPLE ANSWER TABLE
# ------------------------------------------------------------

final_answer = year_summary[
    [
        "year",
        "industry_success_rate",
        "startup_success_rate",
        "industry_net",
        "startup_net",
        "better_path_by_rate"
    ]
].copy()

print("\nFINAL SIMPLE ANSWER TABLE")
print("This table answers: which path looks better each year?")
display(final_answer)


print("\nDONE")
print("Formula used:")
print("Industry success = job_openings + job_gains")
print("Industry failure = job_closings + job_losses")
print("Startup success = establishment_births")
print("Startup failure = establishment_deaths")
print("Success rate = success / (success + failure) * 100")

# Fix before 2008 missing - fail?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# SEPARATE THE SUCCESSFUL STACKED JOIN INTO 2 PATHS
#
# Path 1:
# Degree → Industry Job Path
# success = job_openings + job_gains
# failure = job_closings + job_losses
#
# Path 2:
# Degree → Startup Path
# success = establishment_births
# failure = establishment_deaths
#
# Uses the successful table:
# year_field_industry_table
#
# Does not save files.
# Shows table + charts only.
# ============================================================

TOP_N_PAIR = 12
SHOW_TABLE_ROWS = 150

needed_cols = [
    "year",
    "field_of_study",
    "connected_industry_sector",
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
]

# ------------------------------------------------------------
# 1. Check table exists
# ------------------------------------------------------------
if "year_field_industry_table" not in globals():
    raise NameError(
        "year_field_industry_table does not exist. "
        "Run the successful stacked chart code first, then run this."
    )

df = year_field_industry_table.copy()

missing_cols = [c for c in needed_cols if c not in df.columns]

if missing_cols:
    print("Missing columns:")
    for c in missing_cols:
        print("-", c)
    raise KeyError("The successful joined table is missing required columns.")

# ------------------------------------------------------------
# 2. Clean columns
# ------------------------------------------------------------
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

metric_cols = [
    "job_openings",
    "job_closings",
    "job_gains",
    "job_losses",
    "establishment_births",
    "establishment_deaths",
]

for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["field_of_study"] = df["field_of_study"].astype(str).str.strip()
df["connected_industry_sector"] = df["connected_industry_sector"].astype(str).str.strip()

df = df[df["year"].notna()].copy()
df["year"] = df["year"].astype(int)

df["degree_to_industry"] = (
    df["field_of_study"]
    + " → "
    + df["connected_industry_sector"]
)

# ------------------------------------------------------------
# 3. Separate formulas
# ------------------------------------------------------------

# Degree → Industry Job Path
df["job_path_success"] = df["job_openings"] + df["job_gains"]
df["job_path_failure"] = df["job_closings"] + df["job_losses"]
df["job_path_net"] = df["job_path_success"] - df["job_path_failure"]
df["job_path_total"] = df["job_path_success"] + df["job_path_failure"]

df["job_path_success_rate"] = np.where(
    df["job_path_total"] > 0,
    df["job_path_success"] / df["job_path_total"] * 100,
    np.nan
)

df["job_path_failure_rate"] = np.where(
    df["job_path_total"] > 0,
    df["job_path_failure"] / df["job_path_total"] * 100,
    np.nan
)

df["job_path_failed"] = df["job_path_net"] < 0


# Degree → Startup Path
df["startup_path_success"] = df["establishment_births"]
df["startup_path_failure"] = df["establishment_deaths"]
df["startup_path_net"] = df["startup_path_success"] - df["startup_path_failure"]
df["startup_path_total"] = df["startup_path_success"] + df["startup_path_failure"]

df["startup_path_success_rate"] = np.where(
    df["startup_path_total"] > 0,
    df["startup_path_success"] / df["startup_path_total"] * 100,
    np.nan
)

df["startup_path_failure_rate"] = np.where(
    df["startup_path_total"] > 0,
    df["startup_path_failure"] / df["startup_path_total"] * 100,
    np.nan
)

df["startup_path_failed"] = df["startup_path_net"] < 0


# Compare path
df["better_path_by_rate"] = np.select(
    [
        df["job_path_success_rate"] > df["startup_path_success_rate"],
        df["startup_path_success_rate"] > df["job_path_success_rate"],
        df["job_path_success_rate"].eq(df["startup_path_success_rate"]),
    ],
    [
        "Degree → Industry Job Path",
        "Degree → Startup Path",
        "Tie",
    ],
    default="No rate data"
)

df["rate_gap_job_minus_startup"] = (
    df["job_path_success_rate"] - df["startup_path_success_rate"]
)

# ------------------------------------------------------------
# 4. Keep rows that have at least one real path
# ------------------------------------------------------------
separated_path_table = df[
    (df["job_path_total"] > 0)
    | (df["startup_path_total"] > 0)
].copy()

separated_path_table = separated_path_table.sort_values(
    ["year", "degree_to_industry"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Display separated table
# ------------------------------------------------------------
display_cols = [
    "year",
    "field_of_study",
    "connected_industry_sector",

    "job_openings",
    "job_gains",
    "job_closings",
    "job_losses",
    "job_path_success",
    "job_path_failure",
    "job_path_net",
    "job_path_success_rate",
    "job_path_failure_rate",
    "job_path_failed",

    "establishment_births",
    "establishment_deaths",
    "startup_path_success",
    "startup_path_failure",
    "startup_path_net",
    "startup_path_success_rate",
    "startup_path_failure_rate",
    "startup_path_failed",

    "better_path_by_rate",
    "rate_gap_job_minus_startup",
]

display_table_separated = separated_path_table[display_cols].copy()

rate_cols = [
    "job_path_success_rate",
    "job_path_failure_rate",
    "startup_path_success_rate",
    "startup_path_failure_rate",
    "rate_gap_job_minus_startup",
]

for col in rate_cols:
    display_table_separated[col] = display_table_separated[col].round(2)

print("=" * 100)
print("SEPARATED TABLE: Degree → Job Path vs Degree → Startup Path")
print("=" * 100)
display(display_table_separated.head(SHOW_TABLE_ROWS))


# ------------------------------------------------------------
# 6. Annual summary table
# ------------------------------------------------------------
annual_separated = (
    separated_path_table
    .groupby("year", as_index=False)
    .agg(
        job_openings=("job_openings", "sum"),
        job_gains=("job_gains", "sum"),
        job_closings=("job_closings", "sum"),
        job_losses=("job_losses", "sum"),
        establishment_births=("establishment_births", "sum"),
        establishment_deaths=("establishment_deaths", "sum"),
    )
)

annual_separated["job_path_success"] = (
    annual_separated["job_openings"]
    + annual_separated["job_gains"]
)

annual_separated["job_path_failure"] = (
    annual_separated["job_closings"]
    + annual_separated["job_losses"]
)

annual_separated["job_path_net"] = (
    annual_separated["job_path_success"]
    - annual_separated["job_path_failure"]
)

annual_separated["job_path_total"] = (
    annual_separated["job_path_success"]
    + annual_separated["job_path_failure"]
)

annual_separated["job_path_success_rate"] = np.where(
    annual_separated["job_path_total"] > 0,
    annual_separated["job_path_success"] / annual_separated["job_path_total"] * 100,
    np.nan
)

annual_separated["startup_path_success"] = annual_separated["establishment_births"]
annual_separated["startup_path_failure"] = annual_separated["establishment_deaths"]

annual_separated["startup_path_net"] = (
    annual_separated["startup_path_success"]
    - annual_separated["startup_path_failure"]
)

annual_separated["startup_path_total"] = (
    annual_separated["startup_path_success"]
    + annual_separated["startup_path_failure"]
)

annual_separated["startup_path_success_rate"] = np.where(
    annual_separated["startup_path_total"] > 0,
    annual_separated["startup_path_success"] / annual_separated["startup_path_total"] * 100,
    np.nan
)

annual_separated["job_path_failed"] = annual_separated["job_path_net"] < 0
annual_separated["startup_path_failed"] = annual_separated["startup_path_net"] < 0

annual_separated["better_path_by_rate"] = np.select(
    [
        annual_separated["job_path_success_rate"] > annual_separated["startup_path_success_rate"],
        annual_separated["startup_path_success_rate"] > annual_separated["job_path_success_rate"],
        annual_separated["job_path_success_rate"].eq(annual_separated["startup_path_success_rate"]),
    ],
    [
        "Degree → Industry Job Path",
        "Degree → Startup Path",
        "Tie",
    ],
    default="No rate data"
)

annual_separated["rate_gap_job_minus_startup"] = (
    annual_separated["job_path_success_rate"]
    - annual_separated["startup_path_success_rate"]
)

annual_display_cols = [
    "year",
    "job_path_success",
    "job_path_failure",
    "job_path_net",
    "job_path_success_rate",
    "job_path_failed",
    "startup_path_success",
    "startup_path_failure",
    "startup_path_net",
    "startup_path_success_rate",
    "startup_path_failed",
    "better_path_by_rate",
    "rate_gap_job_minus_startup",
]

annual_display = annual_separated[annual_display_cols].copy()

for col in [
    "job_path_success_rate",
    "startup_path_success_rate",
    "rate_gap_job_minus_startup",
]:
    annual_display[col] = annual_display[col].round(2)

print("=" * 100)
print("ANNUAL SUMMARY: Job Path vs Startup Path")
print("=" * 100)
display(annual_display)


# ------------------------------------------------------------
# 7. Failure tables
# ------------------------------------------------------------
annual_failure_table = annual_display[
    (annual_separated["job_path_failed"])
    | (annual_separated["startup_path_failed"])
].copy()

sector_failure_table = display_table_separated[
    (separated_path_table["job_path_failed"])
    | (separated_path_table["startup_path_failed"])
].copy()

print("=" * 100)
print("FAILURE TABLES")
print("=" * 100)

if annual_failure_table.empty:
    print("No annual failure found.")
else:
    print("Annual failure table:")
    display(annual_failure_table)

if sector_failure_table.empty:
    print("No field/industry failure found.")
else:
    print("Field/industry failure table:")
    display(sector_failure_table.head(SHOW_TABLE_ROWS))


# ------------------------------------------------------------
# 8. Chart helper
# ------------------------------------------------------------
def short_label(text, width=65):
    return textwrap.shorten(str(text), width=width, placeholder="...")


def choose_top_pairs(table, value_col, top_n=12):
    return (
        table
        .groupby("degree_to_industry", as_index=False)[value_col]
        .sum()
        .sort_values(value_col, ascending=False)
        .head(top_n)["degree_to_industry"]
        .tolist()
    )


# ------------------------------------------------------------
# 9. Chart 1: Job Path only
# ------------------------------------------------------------
job_chart_data = separated_path_table[
    separated_path_table["job_path_total"] > 0
].copy()

top_job_pairs = choose_top_pairs(
    job_chart_data,
    "job_path_total",
    TOP_N_PAIR
)

job_chart_data = job_chart_data[
    job_chart_data["degree_to_industry"].isin(top_job_pairs)
].copy()

years = sorted(job_chart_data["year"].unique())
x = np.arange(len(years))

job_success_pivot = (
    job_chart_data
    .pivot_table(
        index="year",
        columns="degree_to_industry",
        values="job_path_success",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(years)
    .reindex(columns=top_job_pairs, fill_value=0)
)

job_failure_pivot = (
    job_chart_data
    .pivot_table(
        index="year",
        columns="degree_to_industry",
        values="job_path_failure",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(years)
    .reindex(columns=top_job_pairs, fill_value=0)
)

bottom_success = np.zeros(len(years))
bottom_failure = np.zeros(len(years))

plt.figure(figsize=(22, 11))
ax = plt.gca()

for pair in top_job_pairs:
    label = short_label(pair)

    success_values = job_success_pivot[pair].values
    failure_values = job_failure_pivot[pair].values

    bars = ax.bar(
        x,
        success_values,
        bottom=bottom_success,
        label=label
    )

    chosen_color = bars.patches[0].get_facecolor()

    ax.bar(
        x,
        -failure_values,
        bottom=bottom_failure,
        color=chosen_color,
        alpha=0.45
    )

    bottom_success += success_values
    bottom_failure -= failure_values

ax.axhline(0, linewidth=1)
ax.set_title(
    "Degree → Industry Job Path Only\n"
    "Success = Job Openings + Job Gains | Failure = Job Closings + Job Losses",
    fontsize=14
)
ax.set_xlabel("Year")
ax.set_ylabel("Failure below 0 | Success above 0")
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=90)
ax.legend(
    title="Field of Study → Industry Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. Chart 2: Startup Path only
# ------------------------------------------------------------
startup_chart_data = separated_path_table[
    separated_path_table["startup_path_total"] > 0
].copy()

top_startup_pairs = choose_top_pairs(
    startup_chart_data,
    "startup_path_total",
    TOP_N_PAIR
)

startup_chart_data = startup_chart_data[
    startup_chart_data["degree_to_industry"].isin(top_startup_pairs)
].copy()

years = sorted(startup_chart_data["year"].unique())
x = np.arange(len(years))

startup_success_pivot = (
    startup_chart_data
    .pivot_table(
        index="year",
        columns="degree_to_industry",
        values="startup_path_success",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(years)
    .reindex(columns=top_startup_pairs, fill_value=0)
)

startup_failure_pivot = (
    startup_chart_data
    .pivot_table(
        index="year",
        columns="degree_to_industry",
        values="startup_path_failure",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(years)
    .reindex(columns=top_startup_pairs, fill_value=0)
)

bottom_success = np.zeros(len(years))
bottom_failure = np.zeros(len(years))

plt.figure(figsize=(22, 11))
ax = plt.gca()

for pair in top_startup_pairs:
    label = short_label(pair)

    success_values = startup_success_pivot[pair].values
    failure_values = startup_failure_pivot[pair].values

    bars = ax.bar(
        x,
        success_values,
        bottom=bottom_success,
        label=label
    )

    chosen_color = bars.patches[0].get_facecolor()

    ax.bar(
        x,
        -failure_values,
        bottom=bottom_failure,
        color=chosen_color,
        alpha=0.45
    )

    bottom_success += success_values
    bottom_failure -= failure_values

ax.axhline(0, linewidth=1)
ax.set_title(
    "Degree → Startup Path Only\n"
    "Success = Establishment Births | Failure = Establishment Deaths",
    fontsize=14
)
ax.set_xlabel("Year")
ax.set_ylabel("Failure below 0 | Success above 0")
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=90)
ax.legend(
    title="Field of Study → Industry Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11. Chart 3: Job Path Net vs Startup Path Net by year
# ------------------------------------------------------------
annual_plot = annual_separated.sort_values("year").copy()

x = np.arange(len(annual_plot))
width = 0.42

plt.figure(figsize=(22, 8))
ax = plt.gca()

ax.bar(
    x - width / 2,
    annual_plot["job_path_net"],
    width,
    label="Degree → Industry Job Path net"
)

ax.bar(
    x + width / 2,
    annual_plot["startup_path_net"],
    width,
    label="Degree → Startup Path net"
)

ax.axhline(0, linewidth=1)

ax.set_title(
    "Separated Path Comparison by Year\n"
    "Net = Success - Failure",
    fontsize=14
)

ax.set_xlabel("Year")
ax.set_ylabel("Net Success Minus Failure")
ax.set_xticks(x)
ax.set_xticklabels(annual_plot["year"], rotation=90)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


print("DONE.")
print("Created tables:")
print("- separated_path_table")
print("- display_table_separated")
print("- annual_separated")
print("- annual_failure_table")
print("- sector_failure_table")
print()
print("Formula used:")
print("Job Path Success = job_openings + job_gains")
print("Job Path Failure = job_closings + job_losses")
print("Startup Path Success = establishment_births")
print("Startup Path Failure = establishment_deaths")
print("Failure means net < 0")

# read row by year column

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# LARGE CSV RANDOM RAW ROW CHECK
# Does NOT edit original files
# Does NOT save files
# Reads in chunks so 500MB+ files are okay
# Shows random raw rows for every year found
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 50_000
ROWS_PER_YEAR = 3
RANDOM_SEED = 42

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 120)


def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"FOUND: {path}")
    print(f"SIZE: {size_mb:,.2f} MB")


def get_columns(path):
    return list(pd.read_csv(path, nrows=0).columns)


def find_year_column(columns):
    possible = [
        "year",
        "Year",
        "YEAR",
        "survey_year",
        "academic_year",
        "report_year",
        "data_year"
    ]

    for col in possible:
        if col in columns:
            return col

    raise KeyError(
        "No year column found. Available columns are:\n"
        + "\n".join(columns)
    )


def show_column_list(path, dataset_name):
    print("\n" + "=" * 100)
    print(f"{dataset_name} COLUMNS")
    print("=" * 100)

    columns = get_columns(path)

    col_table = pd.DataFrame({
        "column_number": range(1, len(columns) + 1),
        "column_name": columns
    })

    print(f"Total columns: {len(columns)}")
    display(col_table)

    return columns


def random_raw_rows_by_year(path, dataset_name, rows_per_year=3, chunksize=50_000):
    print("\n" + "=" * 100)
    print(f"RANDOM RAW ROWS BY YEAR: {dataset_name}")
    print("=" * 100)

    check_file(path)

    columns = get_columns(path)
    year_col = find_year_column(columns)

    print(f"Using year column: {year_col}")

    rng = np.random.default_rng(RANDOM_SEED)

    samples_by_year = {}
    year_counts = {}
    total_rows_read = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            path,
            chunksize=chunksize,
            dtype=str
        ),
        start=1
    ):
        chunk_rows = len(chunk)

        # Keep real CSV line number for checking
        chunk.insert(0, "__csv_row_number__", range(total_rows_read + 2, total_rows_read + 2 + chunk_rows))

        # Clean year only for grouping
        chunk["__year_clean__"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk.dropna(subset=["__year_clean__"])
        chunk["__year_clean__"] = chunk["__year_clean__"].astype(int)

        # Random key for true random sample
        chunk["__random_key__"] = rng.random(len(chunk))

        # Count rows by year
        counts = chunk["__year_clean__"].value_counts()
        for year, count in counts.items():
            year_counts[year] = year_counts.get(year, 0) + int(count)

        # Keep only random rows per year
        for year, group in chunk.groupby("__year_clean__"):
            group_sample = group.nsmallest(rows_per_year, "__random_key__")

            if year not in samples_by_year:
                samples_by_year[year] = group_sample.copy()
            else:
                combined = pd.concat([samples_by_year[year], group_sample], ignore_index=True)
                samples_by_year[year] = combined.nsmallest(rows_per_year, "__random_key__").copy()

        total_rows_read += chunk_rows

        if chunk_number % 10 == 0:
            print(f"Read {total_rows_read:,} rows so far...")

    print(f"\nDONE reading {total_rows_read:,} rows.")

    # Year count table
    count_table = (
        pd.DataFrame({
            "year": list(year_counts.keys()),
            "row_count": list(year_counts.values())
        })
        .sort_values("year")
        .reset_index(drop=True)
    )

    print("\n" + "-" * 100)
    print("ROW COUNT BY YEAR")
    print("-" * 100)
    display(count_table)

    # Random sample table
    if not samples_by_year:
        print("No rows found.")
        return count_table, pd.DataFrame()

    sample_table = (
        pd.concat(samples_by_year.values(), ignore_index=True)
        .sort_values(["__year_clean__", "__random_key__"])
        .reset_index(drop=True)
    )

    # Move helper columns to front
    front_cols = ["__year_clean__", "__csv_row_number__", year_col]
    other_cols = [c for c in sample_table.columns if c not in front_cols + ["__random_key__"]]

    sample_table = sample_table[front_cols + other_cols]

    sample_table = sample_table.rename(columns={
        "__year_clean__": "clean_year",
        "__csv_row_number__": "csv_row_number"
    })

    print("\n" + "-" * 100)
    print(f"RANDOM RAW ROWS: {rows_per_year} ROWS PER YEAR")
    print("-" * 100)
    display(sample_table)

    return count_table, sample_table


# ============================================================
# RUN DEGREE FILE
# ============================================================

degree_columns = show_column_list(DEGREE_FILE, "DEGREE FILE")

degree_year_counts, degree_random_rows = random_raw_rows_by_year(
    DEGREE_FILE,
    "DEGREE FILE",
    rows_per_year=ROWS_PER_YEAR,
    chunksize=CHUNKSIZE
)


# ============================================================
# RUN COMPANY FILE
# ============================================================

company_columns = show_column_list(COMPANY_FILE, "COMPANY FILE")

company_year_counts, company_random_rows = random_raw_rows_by_year(
    COMPANY_FILE,
    "COMPANY FILE",
    rows_per_year=ROWS_PER_YEAR,
    chunksize=CHUNKSIZE
)


# info maybe

# Code?

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

warnings.filterwarnings("ignore")

# ============================================================
# DEGREE FIELD → INDUSTRY JOB PATH
# VS
# DEGREE FIELD → STARTUP PATH
# 1984–2023 ONLY
#
# Memory optimized with pandas chunks.
# Does NOT save files.
# Displays tables and charts only.
# ============================================================

# =========================
# FILE PATHS
# =========================
degree_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
company_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

FIRST_YEAR = 1984
LAST_YEAR = 2023
YEARS = list(range(FIRST_YEAR, LAST_YEAR + 1))

DEGREE_CHUNKSIZE = 250_000
COMPANY_CHUNKSIZE = 300_000

TOP_FIELDS_CHART = 10
TOP_FIELDS_HEATMAP = 15
TOP_FINAL_PAIRS = 10

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 5000)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 120)


# ============================================================
# PRINT / DISPLAY HELPERS
# ============================================================
def banner(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def subbanner(title):
    print("\n" + "-" * 100)
    print(title)
    print("-" * 100)


def show_table(title, df, max_rows=None):
    subbanner(title)
    if df is None or len(df) == 0:
        print("EMPTY TABLE")
        return
    if max_rows is None:
        display(df)
    else:
        display(df.head(max_rows))
        if len(df) > max_rows:
            print(f"Showing first {max_rows:,} rows out of {len(df):,} total rows.")


def compact_num(x, pos=None):
    x = float(x)
    ax = abs(x)
    if ax >= 1_000_000_000:
        return f"{x/1_000_000_000:.1f}B"
    if ax >= 1_000_000:
        return f"{x/1_000_000:.1f}M"
    if ax >= 1_000:
        return f"{x/1_000:.1f}K"
    return f"{x:.0f}"


def check_file(path, label):
    banner(f"CHECKING {label}")
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print("FOUND:", path)
    print("SIZE:", f"{path.stat().st_size / (1024**2):,.2f} MB")


# ============================================================
# COLUMN HELPERS
# ============================================================
def unique_preserve(items):
    seen = set()
    out = []
    for item in items:
        if item is None:
            continue
        key = str(item)
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def read_columns(path):
    return pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()


def find_exact_case_insensitive(columns, wanted_name):
    wanted = wanted_name.casefold()
    return unique_preserve([c for c in columns if str(c).casefold() == wanted])


def first_existing_case_insensitive(columns, possible_names):
    for name in possible_names:
        found = find_exact_case_insensitive(columns, name)
        if found:
            return found[0]
    return None


def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def first_positive_across(df, cols):
    """
    Important fix:
    If both crace15 and CRACE15 exist, do NOT add both.
    They are case variants of the same old IPEDS value.
    This returns the first positive value across case variants.
    """
    out = pd.Series(0.0, index=df.index)

    for col in unique_preserve(cols):
        if col not in df.columns:
            continue
        s = safe_numeric(df[col])
        s = s.where(s > 0)

        mask = (out <= 0) & s.notna()
        out.loc[mask] = s.loc[mask]

    return out.fillna(0.0)


# ============================================================
# DEGREE FIELD HELPERS
# ============================================================
CIP2_TO_FIELD = {
    "01": "AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED FIELDS",
    "03": "NATURAL RESOURCES AND CONSERVATION",
    "04": "ARCHITECTURE AND RELATED SERVICES",
    "05": "AREA, ETHNIC, CULTURAL, GENDER, AND GROUP STUDIES",
    "09": "COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS",
    "10": "COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES",
    "11": "COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES",
    "12": "CULINARY, ENTERTAINMENT, AND PERSONAL SERVICES",
    "13": "EDUCATION",
    "14": "ENGINEERING",
    "15": "ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS",
    "16": "FOREIGN LANGUAGES, LITERATURES, AND LINGUISTICS",
    "19": "FAMILY AND CONSUMER SCIENCES/HUMAN SCIENCES",
    "22": "LEGAL PROFESSIONS AND STUDIES",
    "23": "ENGLISH LANGUAGE AND LITERATURE/LETTERS",
    "24": "LIBERAL ARTS AND SCIENCES, GENERAL STUDIES AND HUMANITIES",
    "25": "LIBRARY SCIENCE",
    "26": "BIOLOGICAL AND BIOMEDICAL SCIENCES",
    "27": "MATHEMATICS AND STATISTICS",
    "30": "MULTI/INTERDISCIPLINARY STUDIES",
    "31": "PARKS, RECREATION, LEISURE, FITNESS, AND KINESIOLOGY",
    "38": "PHILOSOPHY AND RELIGIOUS STUDIES",
    "39": "THEOLOGY AND RELIGIOUS VOCATIONS",
    "40": "PHYSICAL SCIENCES",
    "41": "SCIENCE TECHNOLOGIES/TECHNICIANS",
    "42": "PSYCHOLOGY",
    "43": "HOMELAND SECURITY, LAW ENFORCEMENT, FIREFIGHTING AND RELATED PROTECTIVE SERVICES",
    "44": "PUBLIC ADMINISTRATION AND SOCIAL SERVICE PROFESSIONS",
    "45": "SOCIAL SCIENCES",
    "46": "CONSTRUCTION TRADES",
    "47": "MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS",
    "48": "PRECISION PRODUCTION",
    "49": "TRANSPORTATION AND MATERIALS MOVING",
    "50": "VISUAL AND PERFORMING ARTS",
    "51": "HEALTH PROFESSIONS AND RELATED PROGRAMS",
    "52": "BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES",
    "54": "HISTORY",
    "95": "OTHER / UNKNOWN CIP FIELD",
    "99": "OTHER / UNKNOWN CIP FIELD",
}


def cip_prefix(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    text = text.replace(".0", "")

    if text == "" or text.lower() in {"nan", "none", "not listed"}:
        return None

    if "." in text:
        left = text.split(".")[0]
        left = re.sub(r"\D", "", left)
        if left:
            return left.zfill(2)[:2]

    digits = re.sub(r"\D", "", text)
    if len(digits) >= 2:
        return digits[:2].zfill(2)

    return None


def field_from_cip(value):
    prefix = cip_prefix(value)
    if prefix is None:
        return "OTHER / UNKNOWN CIP FIELD"
    return CIP2_TO_FIELD.get(prefix, "OTHER / UNKNOWN CIP FIELD")


def canonical_field_name(value):
    if pd.isna(value):
        return "OTHER / UNKNOWN CIP FIELD"

    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "unknown", "not listed"}:
        return "OTHER / UNKNOWN CIP FIELD"

    u = text.upper()
    u = re.sub(r"\s+", " ", u)

    # Fix common spelling / older label differences
    if "BUSINESS" in u or "MARKETING" in u or "MANAGEMENT" in u:
        return "BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES"
    if "HEALTH PROFESSIONS" in u or "HEALTH AND MEDICAL" in u or "NURSING" in u:
        return "HEALTH PROFESSIONS AND RELATED PROGRAMS"
    if "COMPUTER" in u or "INFORMATION SCIENCES" in u:
        return "COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES"
    if "ENGINEERING/ENGINEERING" in u or "ENGINEERING-RELATED" in u or "ENGINEERING TECHNOLOG" in u:
        return "ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS"
    if u == "ENGINEERING" or u.startswith("ENGINEERING "):
        return "ENGINEERING"
    if "EDUCATION" in u or "TEACHER" in u:
        return "EDUCATION"
    if "LIBERAL ARTS" in u or "GENERAL STUDIES" in u or "HUMANITIES" in u:
        return "LIBERAL ARTS AND SCIENCES, GENERAL STUDIES AND HUMANITIES"
    if "VISUAL" in u or "PERFORMING ARTS" in u or "FINE ARTS" in u:
        return "VISUAL AND PERFORMING ARTS"
    if "BIOLOGICAL" in u or "BIOMEDICAL" in u or "BIOLOGY" in u:
        return "BIOLOGICAL AND BIOMEDICAL SCIENCES"
    if "COMMUNICATION, JOURNALISM" in u:
        return "COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS"
    if "COMMUNICATIONS TECHNOLOG" in u:
        return "COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES"
    if "SOCIAL SCIENCES" in u:
        return "SOCIAL SCIENCES"
    if "PSYCHOLOGY" in u:
        return "PSYCHOLOGY"
    if "PHYSICAL SCIENCES" in u or "CHEMISTRY" in u or "PHYSICS" in u:
        return "PHYSICAL SCIENCES"
    if "MATHEMATICS" in u or "STATISTICS" in u:
        return "MATHEMATICS AND STATISTICS"
    if "CONSTRUCTION" in u:
        return "CONSTRUCTION TRADES"
    if "MECHANIC" in u or "REPAIR" in u:
        return "MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS"
    if "TRANSPORTATION" in u:
        return "TRANSPORTATION AND MATERIALS MOVING"
    if "NATURAL RESOURCES" in u or "CONSERVATION" in u:
        return "NATURAL RESOURCES AND CONSERVATION"
    if "AGRICULT" in u or "ANIMAL" in u or "VETERINARY" in u:
        return "AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED FIELDS"
    if "LAW ENFORCEMENT" in u or "HOMELAND" in u or "FIREFIGHTING" in u:
        return "HOMELAND SECURITY, LAW ENFORCEMENT, FIREFIGHTING AND RELATED PROTECTIVE SERVICES"
    if "PUBLIC ADMINISTRATION" in u or "SOCIAL SERVICE" in u:
        return "PUBLIC ADMINISTRATION AND SOCIAL SERVICE PROFESSIONS"
    if "CULINARY" in u or "PERSONAL SERVICES" in u:
        return "CULINARY, ENTERTAINMENT, AND PERSONAL SERVICES"
    if "FAMILY AND CONSUMER" in u:
        return "FAMILY AND CONSUMER SCIENCES/HUMAN SCIENCES"
    if "FOREIGN LANGUAGES" in u or "LINGUISTICS" in u:
        return "FOREIGN LANGUAGES, LITERATURES, AND LINGUISTICS"
    if "ENGLISH" in u or "LETTERS" in u:
        return "ENGLISH LANGUAGE AND LITERATURE/LETTERS"
    if "HISTORY" in u:
        return "HISTORY"
    if "LEGAL" in u:
        return "LEGAL PROFESSIONS AND STUDIES"
    if "LIBRARY" in u:
        return "LIBRARY SCIENCE"
    if "PARKS" in u or "RECREATION" in u or "KINESIOLOGY" in u:
        return "PARKS, RECREATION, LEISURE, FITNESS, AND KINESIOLOGY"
    if "PHILOSOPHY" in u:
        return "PHILOSOPHY AND RELIGIOUS STUDIES"
    if "THEOLOGY" in u or "RELIGIOUS" in u:
        return "THEOLOGY AND RELIGIOUS VOCATIONS"
    if "PRECISION PRODUCTION" in u:
        return "PRECISION PRODUCTION"
    if "SCIENCE TECHNOLOG" in u:
        return "SCIENCE TECHNOLOGIES/TECHNICIANS"
    if "MULTI" in u or "INTERDISCIPLINARY" in u:
        return "MULTI/INTERDISCIPLINARY STUDIES"
    if "ARCHITECTURE" in u:
        return "ARCHITECTURE AND RELATED SERVICES"
    if "AREA, ETHNIC" in u or "CULTURAL" in u or "GENDER" in u:
        return "AREA, ETHNIC, CULTURAL, GENDER, AND GROUP STUDIES"
    if "MILITARY" in u:
        return "MILITARY SCIENCE, LEADERSHIP AND OPERATIONAL ART"
    if "OTHER" in u or "UNKNOWN" in u or "UNMAPPED" in u or "NONSTANDARD" in u or "NON-CIP" in u:
        return "OTHER / UNKNOWN CIP FIELD"

    return u


def make_field_group(chunk, field_col, cip_col):
    if field_col and field_col in chunk.columns:
        raw = chunk[field_col].astype(str).str.strip()
    else:
        raw = pd.Series("", index=chunk.index)

    if cip_col and cip_col in chunk.columns:
        derived = chunk[cip_col].apply(field_from_cip)
    else:
        derived = pd.Series("OTHER / UNKNOWN CIP FIELD", index=chunk.index)

    bad_raw = (
        raw.eq("")
        | raw.str.lower().isin(["nan", "none", "unknown", "not listed"])
        | raw.str.contains("unmapped|nonstandard|non-cip", case=False, na=False)
    )

    mixed = raw.where(~bad_raw, derived)
    return mixed.apply(canonical_field_name)


# ============================================================
# DEGREE FIELD → INDUSTRY SECTOR BUILT-IN MAPPING
# ============================================================
FIELD_TO_SECTORS = {
    "AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED FIELDS": ["Natural Resources and Mining"],
    "NATURAL RESOURCES AND CONSERVATION": ["Natural Resources and Mining", "Professional and Business Services"],
    "ARCHITECTURE AND RELATED SERVICES": ["Construction", "Professional and Business Services"],
    "AREA, ETHNIC, CULTURAL, GENDER, AND GROUP STUDIES": ["Education and Health Services", "Professional and Business Services"],
    "BIOLOGICAL AND BIOMEDICAL SCIENCES": ["Education and Health Services", "Professional and Business Services"],
    "BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES": [
        "Financial Activities",
        "Professional and Business Services",
        "Trade, Transportation, and Utilities",
    ],
    "COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS": ["Information", "Professional and Business Services"],
    "COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES": ["Information"],
    "COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES": ["Information", "Professional and Business Services"],
    "CONSTRUCTION TRADES": ["Construction"],
    "CULINARY, ENTERTAINMENT, AND PERSONAL SERVICES": ["Leisure and Hospitality", "Other Services"],
    "EDUCATION": ["Education and Health Services"],
    "ENGINEERING": ["Construction", "Manufacturing", "Professional and Business Services", "Trade, Transportation, and Utilities"],
    "ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS": [
        "Construction",
        "Manufacturing",
        "Trade, Transportation, and Utilities",
    ],
    "ENGLISH LANGUAGE AND LITERATURE/LETTERS": ["Education and Health Services", "Information", "Professional and Business Services"],
    "FAMILY AND CONSUMER SCIENCES/HUMAN SCIENCES": ["Education and Health Services", "Leisure and Hospitality", "Other Services"],
    "FOREIGN LANGUAGES, LITERATURES, AND LINGUISTICS": ["Education and Health Services", "Professional and Business Services"],
    "HEALTH PROFESSIONS AND RELATED PROGRAMS": ["Education and Health Services"],
    "HISTORY": ["Education and Health Services", "Information"],
    "HOMELAND SECURITY, LAW ENFORCEMENT, FIREFIGHTING AND RELATED PROTECTIVE SERVICES": [
        "Other Services",
        "Professional and Business Services",
        "Public Administration",
    ],
    "LEGAL PROFESSIONS AND STUDIES": ["Professional and Business Services"],
    "LIBERAL ARTS AND SCIENCES, GENERAL STUDIES AND HUMANITIES": [
        "Education and Health Services",
        "Professional and Business Services",
    ],
    "LIBRARY SCIENCE": ["Education and Health Services", "Information"],
    "MATHEMATICS AND STATISTICS": ["Financial Activities", "Information", "Professional and Business Services"],
    "MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS": [
        "Manufacturing",
        "Other Services",
        "Trade, Transportation, and Utilities",
    ],
    "MILITARY SCIENCE, LEADERSHIP AND OPERATIONAL ART": ["Professional and Business Services", "Public Administration"],
    "MILITARY TECHNOLOGIES AND APPLIED SCIENCES": ["Professional and Business Services", "Public Administration"],
    "MULTI/INTERDISCIPLINARY STUDIES": ["Education and Health Services", "Professional and Business Services"],
    "OTHER / UNKNOWN CIP FIELD": ["Professional and Business Services"],
    "PARKS, RECREATION, LEISURE, FITNESS, AND KINESIOLOGY": ["Education and Health Services", "Leisure and Hospitality"],
    "PHILOSOPHY AND RELIGIOUS STUDIES": ["Education and Health Services", "Other Services"],
    "PHYSICAL SCIENCES": ["Education and Health Services", "Manufacturing", "Professional and Business Services"],
    "PRECISION PRODUCTION": ["Manufacturing"],
    "PSYCHOLOGY": ["Education and Health Services", "Professional and Business Services"],
    "PUBLIC ADMINISTRATION AND SOCIAL SERVICE PROFESSIONS": ["Education and Health Services", "Public Administration"],
    "SCIENCE TECHNOLOGIES/TECHNICIANS": ["Manufacturing", "Professional and Business Services"],
    "SOCIAL SCIENCES": ["Education and Health Services", "Professional and Business Services", "Public Administration"],
    "THEOLOGY AND RELIGIOUS VOCATIONS": ["Education and Health Services", "Other Services"],
    "TRANSPORTATION AND MATERIALS MOVING": ["Trade, Transportation, and Utilities"],
    "VISUAL AND PERFORMING ARTS": ["Information", "Leisure and Hospitality"],
}


def build_field_sector_mapping(field_values):
    rows = []

    for field in sorted(pd.Series(field_values).dropna().unique()):
        canonical = canonical_field_name(field)
        sectors = FIELD_TO_SECTORS.get(canonical, ["Professional and Business Services"])

        for sector in sectors:
            rows.append({
                "field_group": canonical,
                "connected_industry_sector": sector
            })

    return pd.DataFrame(rows).drop_duplicates().sort_values(
        ["field_group", "connected_industry_sector"]
    ).reset_index(drop=True)


# ============================================================
# READ DEGREE FILE
# ============================================================
def read_degree_file(path):
    banner("READING DEGREE FILE WITH CHUNKS")

    cols = read_columns(path)

    year_col = first_existing_case_insensitive(cols, ["year"])
    cip_col = first_existing_case_insensitive(cols, ["cipcode", "cip_code", "cip"])
    degree_count_col = first_existing_case_insensitive(cols, ["degree_count", "ctotalt", "total_degree_count"])

    # Important fix:
    # Find ALL actual case variants, but never duplicate the same exact column.
    old_male_cols = unique_preserve(
        find_exact_case_insensitive(cols, "crace15")
        + find_exact_case_insensitive(cols, "CRACE15")
    )
    old_female_cols = unique_preserve(
        find_exact_case_insensitive(cols, "crace16")
        + find_exact_case_insensitive(cols, "CRACE16")
    )

    field_col = first_existing_case_insensitive(cols, ["field_group", "degree_field", "field_of_study", "cip_field"])

    if year_col is None:
        raise KeyError("Could not find year column in degree file.")
    if cip_col is None:
        print("WARNING: Could not find CIP column. Field group will use existing field column or Unknown.")
    if degree_count_col is None and not old_male_cols and not old_female_cols:
        raise KeyError("Could not find degree_count or old IPEDS crace15/crace16 columns.")

    print("Using year column:", year_col)
    print("Using CIP column:", cip_col)
    print("Using degree_count column:", degree_count_col)
    print("Using old IPEDS male column(s), no duplicates:", old_male_cols)
    print("Using old IPEDS female column(s), no duplicates:", old_female_cols)
    print("Existing field column:", field_col)

    usecols = unique_preserve(
        [year_col, cip_col, degree_count_col, field_col] + old_male_cols + old_female_cols
    )
    usecols = [c for c in usecols if c is not None]

    year_parts = []
    source_parts = []
    field_parts = []

    total_read = 0

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=DEGREE_CHUNKSIZE, low_memory=False):
        total_read += len(chunk)

        chunk["year_clean"] = safe_numeric(chunk[year_col])
        chunk = chunk[chunk["year_clean"].between(FIRST_YEAR, LAST_YEAR)].copy()

        if chunk.empty:
            continue

        chunk["year"] = chunk["year_clean"].astype(int)

        if degree_count_col and degree_count_col in chunk.columns:
            degree_count = safe_numeric(chunk[degree_count_col]).fillna(0.0)
        else:
            degree_count = pd.Series(0.0, index=chunk.index)

        old_male = first_positive_across(chunk, old_male_cols)
        old_female = first_positive_across(chunk, old_female_cols)
        old_value = old_male + old_female

        # Main usable degree rule:
        # 1. Use degree_count when positive.
        # 2. Otherwise use old IPEDS crace15 + crace16 case-insensitive variants.
        usable = degree_count.where(degree_count > 0, old_value.where(old_value > 0, 0.0))
        chunk["usable_degree_value"] = usable.fillna(0.0)

        chunk["degree_source_used"] = np.select(
            [
                degree_count > 0,
                (degree_count <= 0) & (old_value > 0)
            ],
            [
                "IPEDS degree_count",
                "IPEDS old columns"
            ],
            default="missing"
        )

        chunk["field_group"] = make_field_group(chunk, field_col, cip_col)

        chunk["has_real_degree_value"] = chunk["usable_degree_value"] > 0

        year_part = chunk.groupby("year", as_index=False).agg(
            total_rows_in_file=("usable_degree_value", "size"),
            rows_with_real_value=("has_real_degree_value", "sum"),
            total_degree_value=("usable_degree_value", "sum")
        )
        year_part["rows_without_real_value"] = (
            year_part["total_rows_in_file"] - year_part["rows_with_real_value"]
        )
        year_parts.append(year_part)

        source_part = chunk.groupby(["year", "degree_source_used"], as_index=False).agg(
            source_rows=("usable_degree_value", "size"),
            source_degree_value=("usable_degree_value", "sum")
        )
        source_parts.append(source_part)

        valid = chunk[chunk["has_real_degree_value"]].copy()
        if not valid.empty:
            field_part = valid.groupby(["year", "field_group"], as_index=False).agg(
                degree_value=("usable_degree_value", "sum"),
                degree_rows=("usable_degree_value", "size")
            )
            field_parts.append(field_part)

        if total_read % 500_000 < DEGREE_CHUNKSIZE:
            print(f"Read {total_read:,} degree rows so far...")

    print(f"DONE reading degree file rows: {total_read:,}")

    if not year_parts:
        raise ValueError("No usable degree rows found for 1984–2023.")

    degree_year = pd.concat(year_parts, ignore_index=True).groupby("year", as_index=False).agg(
        total_rows_in_file=("total_rows_in_file", "sum"),
        rows_with_real_value=("rows_with_real_value", "sum"),
        total_degree_value=("total_degree_value", "sum")
    )
    degree_year["rows_without_real_value"] = (
        degree_year["total_rows_in_file"] - degree_year["rows_with_real_value"]
    )

    degree_year = pd.DataFrame({"year": YEARS}).merge(degree_year, on="year", how="left")
    for col in ["total_rows_in_file", "rows_with_real_value", "rows_without_real_value", "total_degree_value"]:
        degree_year[col] = degree_year[col].fillna(0)

    source_df = pd.concat(source_parts, ignore_index=True)
    source_df = source_df.groupby(["year", "degree_source_used"], as_index=False).agg(
        source_rows=("source_rows", "sum"),
        source_degree_value=("source_degree_value", "sum")
    )

    def choose_degree_source(g):
        good = g[g["source_degree_value"] > 0].sort_values("source_degree_value", ascending=False)
        if good.empty:
            return "missing"
        return " + ".join(good["degree_source_used"].drop_duplicates().tolist())

    source_by_year = source_df.groupby("year").apply(choose_degree_source).reset_index(name="degree_source_used")

    degree_year = degree_year.merge(source_by_year, on="year", how="left")
    degree_year["degree_source_used"] = degree_year["degree_source_used"].fillna("missing")

    degree_year = degree_year[
        [
            "year",
            "total_rows_in_file",
            "rows_with_real_value",
            "rows_without_real_value",
            "total_degree_value",
            "degree_source_used",
        ]
    ]

    if field_parts:
        degree_field_year = pd.concat(field_parts, ignore_index=True).groupby(
            ["year", "field_group"], as_index=False
        ).agg(
            degree_value=("degree_value", "sum"),
            degree_rows=("degree_rows", "sum")
        )
    else:
        degree_field_year = pd.DataFrame(columns=["year", "field_group", "degree_value", "degree_rows"])

    degree_field_year = degree_field_year.sort_values(
        ["year", "degree_value"], ascending=[True, False]
    ).reset_index(drop=True)

    return degree_year, degree_field_year


# ============================================================
# COMPANY HELPERS
# ============================================================
def source_family_vector(df, source_col, dataset_col):
    source = df[source_col].astype(str) if source_col in df.columns else pd.Series("", index=df.index)
    dataset = df[dataset_col].astype(str) if dataset_col in df.columns else pd.Series("", index=df.index)
    text = (source + " " + dataset).str.lower()

    out = pd.Series("Other", index=df.index)
    out.loc[text.str.contains("bls|bed|bdm|business employment dynamics", na=False)] = "BLS actual job-flow"
    out.loc[text.str.contains("bds|business dynamics statistics", na=False)] = "Census BDS fallback"
    out.loc[text.str.contains("bfs|business formation statistics", na=False)] = "Census BFS startup proxy"
    return out


def map_industry_sector_one(code, name):
    text = "" if pd.isna(name) else str(name).strip().lower()
    code_text = "" if pd.isna(code) else str(code).strip().upper()

    # Prefer explicit industry name when available.
    if "natural resources" in text or "mining" in text or text in {"agriculture", "agriculture forestry fishing and hunting"}:
        return "Natural Resources and Mining"
    if "construction" in text:
        return "Construction"
    if "manufacturing" in text:
        return "Manufacturing"
    if "trade, transportation" in text or "transportation and utilities" in text or "utilities" in text or "retail trade" in text or "wholesale trade" in text:
        return "Trade, Transportation, and Utilities"
    if "information" in text:
        return "Information"
    if "financial activities" in text or "finance" in text or "insurance" in text or "real estate" in text:
        return "Financial Activities"
    if "professional and business" in text or "professional" in text or "management" in text or "administrative" in text:
        return "Professional and Business Services"
    if "education and health" in text or "educational services" in text or "health care" in text or "social assistance" in text:
        return "Education and Health Services"
    if "leisure and hospitality" in text or "arts, entertainment" in text or "accommodation" in text or "food services" in text:
        return "Leisure and Hospitality"
    if "other services" in text:
        return "Other Services"
    if "public administration" in text:
        return "Public Administration"

    # Avoid broad totals because they double-count sectors.
    if text in {"goods-producing", "service-providing", "total private", "private", "not listed", ""}:
        pass

    c = code_text.replace("NAICS", "")
    c = c.replace(".0", "")
    c = c.strip()

    # NAICS two-digit or range mapping
    if c.startswith("11") or c.startswith("21"):
        return "Natural Resources and Mining"
    if c.startswith("23"):
        return "Construction"
    if c.startswith("31") or c.startswith("32") or c.startswith("33") or c == "31-33":
        return "Manufacturing"
    if c.startswith("22") or c.startswith("42") or c.startswith("44") or c.startswith("45") or c.startswith("48") or c.startswith("49") or c in {"44-45", "48-49"}:
        return "Trade, Transportation, and Utilities"
    if c.startswith("51"):
        return "Information"
    if c.startswith("52") or c.startswith("53"):
        return "Financial Activities"
    if c.startswith("54") or c.startswith("55") or c.startswith("56"):
        return "Professional and Business Services"
    if c.startswith("61") or c.startswith("62"):
        return "Education and Health Services"
    if c.startswith("71") or c.startswith("72"):
        return "Leisure and Hospitality"
    if c.startswith("81"):
        return "Other Services"
    if c.startswith("92"):
        return "Public Administration"

    return None


def map_industry_sector_vector(df, industry_code_col, industry_name_col):
    codes = df[industry_code_col] if industry_code_col in df.columns else pd.Series("", index=df.index)
    names = df[industry_name_col] if industry_name_col in df.columns else pd.Series("", index=df.index)
    return pd.Series(
        [map_industry_sector_one(c, n) for c, n in zip(codes, names)],
        index=df.index
    )


def classify_metric_vector(df, metric_code_col, metric_name_col, metric_category_col, source_col, dataset_col):
    code = df[metric_code_col].astype(str) if metric_code_col in df.columns else pd.Series("", index=df.index)
    name = df[metric_name_col].astype(str) if metric_name_col in df.columns else pd.Series("", index=df.index)
    cat = df[metric_category_col].astype(str) if metric_category_col in df.columns else pd.Series("", index=df.index)
    source = df[source_col].astype(str) if source_col in df.columns else pd.Series("", index=df.index)
    dataset = df[dataset_col].astype(str) if dataset_col in df.columns else pd.Series("", index=df.index)

    text = (code + " " + name + " " + cat).str.lower()
    source_text = (source + " " + dataset).str.lower()

    metric_class = pd.Series(pd.NA, index=df.index, dtype="object")

    def put(mask, label):
        metric_class.loc[mask & metric_class.isna()] = label

    # Startup / establishment birth/death first
    put(text.str.contains(r"establishment\s+birth|establishments?\s+birth|firmbirth|firm_birth|firm\s+birth|birth_firms|births_firms", na=False), "establishment_births")
    put(text.str.contains(r"establishment\s+death|establishments?\s+death|firmdeath|firm_death|firm\s+death|death_firms|deaths_firms", na=False), "establishment_deaths")

    # BDS startup proxy: job_creation_births means jobs created by establishment births.
    # Use as startup_success proxy if no true establishment birth column exists.
    put(text.str.contains(r"job_creation_births|job creation births|jobcreationbirths", na=False), "establishment_births")

    # BLS job-flow exact labels
    put(text.str.contains(r"\bopenings?\b", na=False), "job_openings")
    put(text.str.contains(r"\bclosings?\b", na=False), "job_closings")
    put(text.str.contains(r"gross job gains|job gains|gross_job_gains", na=False), "job_gains")
    put(text.str.contains(r"gross job losses|job losses|gross_job_losses", na=False), "job_losses")

    # Census BDS fallback job creation/destruction counts
    put(text.str.contains(r"job_creation|job creation", na=False), "job_gains")
    put(text.str.contains(r"job_destruction|job destruction", na=False), "job_losses")

    # Census BFS is startup formation proxy.
    bfs_proxy = (
        source_text.str.contains("bfs|business formation", na=False)
        | text.str.contains(r"\bba_|bf_|business application|formation", na=False)
    )
    put(bfs_proxy, "establishment_births")

    # Do not mix rates / denominators into level counts.
    rate_or_denom = text.str.contains(r"rate|denom|reallocation", na=False)
    metric_class.loc[rate_or_denom] = pd.NA

    # Do not use generic employment or firms as success/failure, except firmbirth/firmdeath already caught.
    generic_not_metric = text.str.fullmatch(r".*\b(emp|employment|firms)\b.*", na=False)
    keep_firm_birth_death = text.str.contains(r"firmbirth|firmdeath|firm birth|firm death", na=False)
    metric_class.loc[generic_not_metric & ~keep_firm_birth_death] = pd.NA

    return metric_class


def metric_scan_mask(df, metric_code_col, metric_name_col, metric_category_col):
    code = df[metric_code_col].astype(str) if metric_code_col in df.columns else pd.Series("", index=df.index)
    name = df[metric_name_col].astype(str) if metric_name_col in df.columns else pd.Series("", index=df.index)
    cat = df[metric_category_col].astype(str) if metric_category_col in df.columns else pd.Series("", index=df.index)
    text = (code + " " + name + " " + cat).str.lower()

    return text.str.contains(
        "openings|closings|gross job gains|gross job losses|job creation|job destruction|"
        "establishment births|establishment deaths|firm births|firm deaths|"
        "firmbirth|firmdeath|births|deaths|job gains|job losses",
        na=False
    )


# ============================================================
# READ COMPANY FILE
# ============================================================
def read_company_file(path):
    banner("READING COMPANY FILE WITH CHUNKS")

    cols = read_columns(path)

    year_col = first_existing_case_insensitive(cols, ["year"])
    value_col = first_existing_case_insensitive(cols, ["value"])
    source_col = first_existing_case_insensitive(cols, ["source"])
    dataset_col = first_existing_case_insensitive(cols, ["dataset"])
    industry_code_col = first_existing_case_insensitive(cols, ["industry_code", "naics_code"])
    industry_name_col = first_existing_case_insensitive(cols, ["industry_name"])
    metric_code_col = first_existing_case_insensitive(cols, ["metric_code"])
    metric_name_col = first_existing_case_insensitive(cols, ["metric_name"])
    metric_category_col = first_existing_case_insensitive(cols, ["metric_category"])

    required = {
        "year": year_col,
        "value": value_col,
        "metric_code": metric_code_col,
        "metric_name": metric_name_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise KeyError(f"Missing required company columns: {missing}")

    print("Using year column:", year_col)
    print("Using value column:", value_col)
    print("Using source column:", source_col)
    print("Using dataset column:", dataset_col)
    print("Using industry code column:", industry_code_col)
    print("Using industry name column:", industry_name_col)
    print("Using metric code column:", metric_code_col)
    print("Using metric name column:", metric_name_col)
    print("Using metric category column:", metric_category_col)

    usecols = unique_preserve([
        year_col,
        value_col,
        source_col,
        dataset_col,
        industry_code_col,
        industry_name_col,
        metric_code_col,
        metric_name_col,
        metric_category_col,
    ])
    usecols = [c for c in usecols if c is not None]

    year_parts = []
    metric_scan_parts = []
    metric_long_parts = []

    total_read = 0

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=COMPANY_CHUNKSIZE, low_memory=False):
        total_read += len(chunk)

        chunk["year_clean"] = safe_numeric(chunk[year_col])
        chunk = chunk[chunk["year_clean"].between(FIRST_YEAR, LAST_YEAR)].copy()

        if chunk.empty:
            continue

        chunk["year"] = chunk["year_clean"].astype(int)
        chunk["value_clean"] = safe_numeric(chunk[value_col]).fillna(0.0)
        chunk["has_real_value"] = chunk["value_clean"] > 0

        year_part = chunk.groupby("year", as_index=False).agg(
            total_rows_in_file=("value_clean", "size"),
            rows_with_real_value=("has_real_value", "sum")
        )
        year_part["rows_without_real_value"] = (
            year_part["total_rows_in_file"] - year_part["rows_with_real_value"]
        )
        year_parts.append(year_part)

        scan = chunk[metric_scan_mask(chunk, metric_code_col, metric_name_col, metric_category_col)].copy()
        if not scan.empty:
            scan_cols = unique_preserve([
                source_col,
                dataset_col,
                metric_code_col,
                metric_name_col,
                metric_category_col,
            ])
            scan_cols = [c for c in scan_cols if c is not None and c in scan.columns]
            metric_scan_parts.append(scan[scan_cols].drop_duplicates())

        chunk["connected_industry_sector"] = map_industry_sector_vector(
            chunk,
            industry_code_col,
            industry_name_col
        )

        chunk["source_family"] = source_family_vector(chunk, source_col, dataset_col)

        chunk["metric_class"] = classify_metric_vector(
            chunk,
            metric_code_col,
            metric_name_col,
            metric_category_col,
            source_col,
            dataset_col
        )

        keep = (
            chunk["connected_industry_sector"].notna()
            & chunk["metric_class"].notna()
            & chunk["has_real_value"]
        )

        usable = chunk.loc[keep, [
            "year",
            "connected_industry_sector",
            "source_family",
            "metric_class",
            "value_clean"
        ]].copy()

        if not usable.empty:
            part = usable.groupby(
                ["year", "connected_industry_sector", "source_family", "metric_class"],
                as_index=False
            ).agg(
                value=("value_clean", "sum"),
                rows=("value_clean", "size")
            )
            metric_long_parts.append(part)

        if total_read % 600_000 < COMPANY_CHUNKSIZE:
            print(f"Read {total_read:,} company rows so far...")

    print(f"DONE reading company file rows: {total_read:,}")

    if not year_parts:
        raise ValueError("No company rows found for 1984–2023.")

    company_year = pd.concat(year_parts, ignore_index=True).groupby("year", as_index=False).agg(
        total_rows_in_file=("total_rows_in_file", "sum"),
        rows_with_real_value=("rows_with_real_value", "sum")
    )
    company_year["rows_without_real_value"] = (
        company_year["total_rows_in_file"] - company_year["rows_with_real_value"]
    )

    company_year = pd.DataFrame({"year": YEARS}).merge(company_year, on="year", how="left")
    for col in ["total_rows_in_file", "rows_with_real_value", "rows_without_real_value"]:
        company_year[col] = company_year[col].fillna(0)

    if metric_scan_parts:
        metric_scan_df = pd.concat(metric_scan_parts, ignore_index=True).drop_duplicates()
    else:
        metric_scan_df = pd.DataFrame()

    if metric_long_parts:
        metric_long = pd.concat(metric_long_parts, ignore_index=True).groupby(
            ["year", "connected_industry_sector", "source_family", "metric_class"],
            as_index=False
        ).agg(
            value=("value", "sum"),
            rows=("rows", "sum")
        )
    else:
        metric_long = pd.DataFrame(columns=[
            "year", "connected_industry_sector", "source_family", "metric_class", "value", "rows"
        ])

    return company_year, metric_scan_df, metric_long


# ============================================================
# CHOOSE BLS FIRST, THEN BDS, THEN BFS
# ============================================================
def build_company_metric_table(metric_long):
    banner("BUILDING COMPANY METRIC TABLE BY YEAR AND INDUSTRY SECTOR")

    source_priority = [
        "BLS actual job-flow",
        "Census BDS fallback",
        "Census BFS startup proxy",
        "Other"
    ]

    if metric_long.empty:
        raise ValueError("No company metrics classified. Check metric names printed above.")

    pivot_source = metric_long.pivot_table(
        index=["year", "connected_industry_sector", "metric_class"],
        columns="source_family",
        values="value",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    chosen_rows = []

    for _, row in pivot_source.iterrows():
        chosen_value = 0.0
        chosen_source = "missing"

        for src in source_priority:
            if src in pivot_source.columns and row[src] > 0:
                chosen_value = float(row[src])
                chosen_source = src
                break

        chosen_rows.append({
            "year": int(row["year"]),
            "connected_industry_sector": row["connected_industry_sector"],
            "metric_class": row["metric_class"],
            "value": chosen_value,
            "chosen_source": chosen_source
        })

    chosen = pd.DataFrame(chosen_rows)

    company_metric = chosen.pivot_table(
        index=["year", "connected_industry_sector"],
        columns="metric_class",
        values="value",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    needed_metric_cols = [
        "job_openings",
        "job_closings",
        "job_gains",
        "job_losses",
        "establishment_births",
        "establishment_deaths"
    ]

    for col in needed_metric_cols:
        if col not in company_metric.columns:
            company_metric[col] = 0.0

    source_used = chosen[chosen["value"] > 0].groupby(
        ["year", "connected_industry_sector"]
    )["chosen_source"].apply(
        lambda s: " + ".join(sorted(set(s)))
    ).reset_index(name="source_used")

    company_metric = company_metric.merge(
        source_used,
        on=["year", "connected_industry_sector"],
        how="left"
    )
    company_metric["source_used"] = company_metric["source_used"].fillna("missing")

    company_metric = company_metric[
        [
            "year",
            "connected_industry_sector",
            "job_openings",
            "job_closings",
            "job_gains",
            "job_losses",
            "establishment_births",
            "establishment_deaths",
            "source_used",
        ]
    ]

    company_metric["industry_job_success"] = company_metric["job_openings"] + company_metric["job_gains"]
    company_metric["industry_job_failure"] = company_metric["job_closings"] + company_metric["job_losses"]
    company_metric["startup_success"] = company_metric["establishment_births"]
    company_metric["startup_failure"] = company_metric["establishment_deaths"]

    company_metric = company_metric.sort_values(
        ["year", "connected_industry_sector"]
    ).reset_index(drop=True)

    return company_metric


# ============================================================
# FINAL FORMULA TABLE
# ============================================================
def build_final_formula_table(degree_year, degree_field_year, field_sector_map, company_metric):
    banner("BUILDING FINAL FORMULA TABLE")

    degree_connected = degree_field_year.merge(
        field_sector_map,
        on="field_group",
        how="left"
    )

    degree_connected["connected_industry_sector"] = degree_connected["connected_industry_sector"].fillna(
        "Professional and Business Services"
    )

    degree_source_year = degree_year[["year", "degree_source_used", "total_degree_value"]].copy()

    final = degree_connected.merge(
        company_metric,
        on=["year", "connected_industry_sector"],
        how="left"
    )

    final = final.merge(degree_source_year, on="year", how="left")

    numeric_cols = [
        "degree_value",
        "degree_rows",
        "job_openings",
        "job_closings",
        "job_gains",
        "job_losses",
        "establishment_births",
        "establishment_deaths",
        "industry_job_success",
        "industry_job_failure",
        "startup_success",
        "startup_failure",
    ]

    missing_company = final[
        final["source_used"].isna()
    ][["year", "field_group", "connected_industry_sector", "degree_value"]].copy()

    if not missing_company.empty:
        show_table(
            "WARNING: These degree field → sector rows did not find company metrics before fillna",
            missing_company.sort_values(["year", "field_group"]).head(300),
            max_rows=300
        )

    for col in numeric_cols:
        if col not in final.columns:
            final[col] = 0.0
        final[col] = final[col].fillna(0.0)

    final["source_used"] = final["degree_source_used"].fillna("missing degree") + " + " + final["source_used"].fillna("missing company")

    final["total_success"] = final["industry_job_success"] + final["startup_success"]
    final["total_failure"] = final["industry_job_failure"] + final["startup_failure"]
    final["net_result"] = final["total_success"] - final["total_failure"]

    denom = final["total_success"] + final["total_failure"]
    final["success_rate"] = np.where(denom > 0, final["total_success"] / denom * 100, 0.0)
    final["failure_rate"] = np.where(denom > 0, final["total_failure"] / denom * 100, 0.0)

    final["success_rate"] = final["success_rate"].round(2)
    final["failure_rate"] = final["failure_rate"].round(2)

    final_cols = [
        "year",
        "field_group",
        "connected_industry_sector",
        "degree_value",
        "industry_job_success",
        "industry_job_failure",
        "startup_success",
        "startup_failure",
        "total_success",
        "total_failure",
        "net_result",
        "success_rate",
        "failure_rate",
        "source_used",
        "job_openings",
        "job_closings",
        "job_gains",
        "job_losses",
        "establishment_births",
        "establishment_deaths",
    ]

    final = final[final_cols].sort_values(
        ["year", "field_group", "connected_industry_sector"]
    ).reset_index(drop=True)

    return final


# ============================================================
# WARNING CHECK BEFORE CHARTING
# ============================================================
def warning_check(degree_year, company_year, company_metric, final_formula):
    banner("CHECKING FOR MISSING OR ZERO YEARS BEFORE CHARTING")

    company_year_formula = company_metric.groupby("year", as_index=False).agg(
        industry_job_success=("industry_job_success", "sum"),
        industry_job_failure=("industry_job_failure", "sum"),
        startup_success=("startup_success", "sum"),
        startup_failure=("startup_failure", "sum")
    )

    check = pd.DataFrame({"year": YEARS})
    check = check.merge(
        degree_year[["year", "rows_with_real_value"]],
        on="year",
        how="left"
    ).rename(columns={"rows_with_real_value": "degree_rows_with_real_value"})

    check = check.merge(
        company_year[["year", "rows_with_real_value"]],
        on="year",
        how="left"
    ).rename(columns={"rows_with_real_value": "company_rows_with_real_value"})

    check = check.merge(company_year_formula, on="year", how="left")

    for col in [
        "degree_rows_with_real_value",
        "company_rows_with_real_value",
        "industry_job_success",
        "industry_job_failure",
        "startup_success",
        "startup_failure",
    ]:
        check[col] = check[col].fillna(0)

    problems = []
    for _, row in check.iterrows():
        msg = ""
        if row["degree_rows_with_real_value"] <= 0:
            msg += "missing degree real values; "
        if row["company_rows_with_real_value"] <= 0:
            msg += "missing company real values; "
        if row["industry_job_success"] <= 0:
            msg += "zero industry job success; "
        if row["industry_job_failure"] <= 0:
            msg += "zero industry job failure; "
        if row["startup_success"] <= 0:
            msg += "zero startup success; "
        if row["startup_failure"] <= 0:
            msg += "zero startup failure; "
        problems.append(msg)

    check["problem"] = problems
    warning_rows = check[check["problem"].str.len() > 0].copy()

    if warning_rows.empty:
        print("No missing or zero-value year problems found before charting.")
    else:
        show_table(
            "WARNING TABLE BEFORE CHARTING",
            warning_rows,
            max_rows=None
        )

    return check


# ============================================================
# CHARTS
# ============================================================
def chart_1_degree_totals_by_year(degree_field_year):
    banner("CHART 1: DEGREE FIELD TOTALS BY YEAR")

    top_fields = degree_field_year.groupby("field_group")["degree_value"].sum().sort_values(ascending=False).head(TOP_FIELDS_CHART).index

    plot_df = degree_field_year[degree_field_year["field_group"].isin(top_fields)].copy()

    pivot = plot_df.pivot_table(
        index="year",
        columns="field_group",
        values="degree_value",
        aggfunc="sum",
        fill_value=0
    ).reindex(YEARS, fill_value=0)

    plt.figure(figsize=(18, 8))
    for col in pivot.columns:
        plt.plot(pivot.index, pivot[col], marker="o", linewidth=2, label=col)

    plt.title("Chart 1: Degree Field Totals by Year, Top 10 Fields, 1984–2023")
    plt.xlabel("Year")
    plt.ylabel("Degree value")
    plt.gca().yaxis.set_major_formatter(FuncFormatter(compact_num))
    plt.xticks(YEARS[::2], rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9)
    plt.tight_layout()
    plt.show()


def chart_2_heatmap(degree_connected):
    banner("CHART 2: DEGREE FIELD → INDUSTRY SECTOR CONNECTION HEATMAP")

    top_fields = degree_connected.groupby("field_group")["degree_value"].sum().sort_values(ascending=False).head(TOP_FIELDS_HEATMAP).index

    heat_df = degree_connected[degree_connected["field_group"].isin(top_fields)].pivot_table(
        index="field_group",
        columns="connected_industry_sector",
        values="degree_value",
        aggfunc="sum",
        fill_value=0
    )

    heat_df = heat_df.loc[heat_df.sum(axis=1).sort_values(ascending=True).index]

    plt.figure(figsize=(18, 10))
    plt.imshow(heat_df.values, aspect="auto")
    plt.colorbar(label="Degree value")
    plt.title("Chart 2: Degree Field → Industry Sector Connection, Top 15 Fields")
    plt.xlabel("Connected industry sector")
    plt.ylabel("Degree field")
    plt.xticks(range(len(heat_df.columns)), heat_df.columns, rotation=45, ha="right")
    plt.yticks(range(len(heat_df.index)), heat_df.index)
    plt.tight_layout()
    plt.show()


def chart_3_industry_vs_startup(company_metric):
    banner("CHART 3: INDUSTRY JOB PATH VS STARTUP PATH BY YEAR")

    year_df = company_metric.groupby("year", as_index=False).agg(
        industry_job_success=("industry_job_success", "sum"),
        industry_job_failure=("industry_job_failure", "sum"),
        startup_success=("startup_success", "sum"),
        startup_failure=("startup_failure", "sum")
    ).sort_values("year")

    plt.figure(figsize=(16, 7))
    plt.plot(year_df["year"], year_df["industry_job_success"], marker="o", linewidth=2, label="Industry job success = job_openings + job_gains")
    plt.plot(year_df["year"], year_df["industry_job_failure"], marker="o", linewidth=2, label="Industry job failure = job_closings + job_losses")
    plt.title("Chart 3A: Industry Job Path Success vs Failure, 1984–2023")
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.gca().yaxis.set_major_formatter(FuncFormatter(compact_num))
    plt.xticks(YEARS[::2], rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(16, 7))
    plt.plot(year_df["year"], year_df["startup_success"], marker="o", linewidth=2, label="Startup success = establishment_births")
    plt.plot(year_df["year"], year_df["startup_failure"], marker="o", linewidth=2, label="Startup failure = establishment_deaths")
    plt.title("Chart 3B: Startup Path Success vs Failure, 1984–2023")
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.gca().yaxis.set_major_formatter(FuncFormatter(compact_num))
    plt.xticks(YEARS[::2], rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return year_df


def chart_4_success_rate(company_metric):
    banner("CHART 4: SUCCESS RATE BY YEAR")

    year_df = company_metric.groupby("year", as_index=False).agg(
        industry_job_success=("industry_job_success", "sum"),
        industry_job_failure=("industry_job_failure", "sum"),
        startup_success=("startup_success", "sum"),
        startup_failure=("startup_failure", "sum")
    )

    year_df["total_success"] = year_df["industry_job_success"] + year_df["startup_success"]
    year_df["total_failure"] = year_df["industry_job_failure"] + year_df["startup_failure"]
    denom = year_df["total_success"] + year_df["total_failure"]

    year_df["success_rate"] = np.where(denom > 0, year_df["total_success"] / denom * 100, 0.0)

    plt.figure(figsize=(16, 7))
    plt.plot(year_df["year"], year_df["success_rate"], marker="o", linewidth=2)
    plt.title("Chart 4: Success Rate by Year, 1984–2023")
    plt.xlabel("Year")
    plt.ylabel("Success rate (%)")
    plt.xticks(YEARS[::2], rotation=45)
    plt.grid(True, alpha=0.3)
    plt.figtext(
        0.01,
        -0.03,
        "Formula: success_rate = total_success / (total_success + total_failure) * 100",
        ha="left",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

    return year_df


def chart_5_final_opposite_bar(final_formula):
    banner("CHART 5: FINAL COMPARISON OPPOSITE HORIZONTAL BAR CHART")

    pair_summary = final_formula.groupby(
        ["field_group", "connected_industry_sector"],
        as_index=False
    ).agg(
        degree_value=("degree_value", "sum"),
        total_success=("total_success", "sum"),
        total_failure=("total_failure", "sum"),
        net_result=("net_result", "sum"),
        success_rate_mean=("success_rate", "mean")
    )

    pair_summary = pair_summary.sort_values("net_result", ascending=False).head(TOP_FINAL_PAIRS)
    pair_summary = pair_summary.sort_values("net_result", ascending=True)

    labels = (
        pair_summary["field_group"].str.slice(0, 45)
        + " → "
        + pair_summary["connected_industry_sector"]
    )

    y = np.arange(len(pair_summary))

    plt.figure(figsize=(18, 9))
    plt.barh(y, pair_summary["total_success"], label="Right side = total_success")
    plt.barh(y, -pair_summary["total_failure"], label="Left side = total_failure")

    plt.axvline(0, linewidth=1)
    plt.yticks(y, labels)
    plt.xlabel("Failure ← 0 → Success")
    plt.title("Degree → Industry Job Path vs Degree → Startup Path\nSuccess vs Failure, Top 10 Degree Field + Industry Pairs, 1984–2023")
    plt.gca().xaxis.set_major_formatter(FuncFormatter(compact_num))
    plt.grid(True, axis="x", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return pair_summary


# ============================================================
# FORMULAS PRINT
# ============================================================
def print_formulas():
    banner("FORMULAS USED")
    print("industry_job_success = job_openings + job_gains")
    print("industry_job_failure = job_closings + job_losses")
    print("startup_success = establishment_births")
    print("startup_failure = establishment_deaths")
    print("total_success = industry_job_success + startup_success")
    print("total_failure = industry_job_failure + startup_failure")
    print("net_result = total_success - total_failure")
    print("success_rate = total_success / (total_success + total_failure) * 100")
    print("failure_rate = total_failure / (total_success + total_failure) * 100")


# ============================================================
# RUN EVERYTHING
# ============================================================
check_file(degree_file, "DEGREE FILE")
check_file(company_file, "COMPANY FILE")

degree_year_check, degree_field_table = read_degree_file(degree_file)

show_table(
    "1. DEGREE YEAR CHECK TABLE",
    degree_year_check,
    max_rows=None
)

show_table(
    "3. DEGREE FIELD TABLE: year, field_group, degree_value, degree_rows",
    degree_field_table,
    max_rows=None
)

field_sector_map = build_field_sector_mapping(degree_field_table["field_group"])

show_table(
    "4. DEGREE FIELD → INDUSTRY SECTOR MAPPING TABLE",
    field_sector_map,
    max_rows=None
)

degree_connected_for_heatmap = degree_field_table.merge(
    field_sector_map,
    on="field_group",
    how="left"
)

company_year_check, company_metric_scan, company_metric_long = read_company_file(company_file)

show_table(
    "2. COMPANY YEAR CHECK TABLE",
    company_year_check,
    max_rows=None
)

show_table(
    "COMPANY UNIQUE METRICS FOUND BEFORE FORMULAS",
    company_metric_scan.sort_values(company_metric_scan.columns.tolist()) if not company_metric_scan.empty else company_metric_scan,
    max_rows=None
)

show_table(
    "COMPANY METRICS FOUND AFTER CLASSIFICATION",
    company_metric_long.sort_values(["year", "source_family", "metric_class", "connected_industry_sector"]),
    max_rows=500
)

company_metric_table = build_company_metric_table(company_metric_long)

show_table(
    "5. COMPANY METRIC TABLE BY YEAR AND INDUSTRY SECTOR",
    company_metric_table,
    max_rows=None
)

final_formula_table = build_final_formula_table(
    degree_year_check,
    degree_field_table,
    field_sector_map,
    company_metric_table
)

warning_quality_table = warning_check(
    degree_year_check,
    company_year_check,
    company_metric_table,
    final_formula_table
)

print_formulas()

# ============================================================
# CHARTS
# ============================================================
chart_1_degree_totals_by_year(degree_field_table)

chart_2_heatmap(degree_connected_for_heatmap)

company_year_chart_table = chart_3_industry_vs_startup(company_metric_table)

success_rate_year_table = chart_4_success_rate(company_metric_table)

top_pair_chart_table = chart_5_final_opposite_bar(final_formula_table)

# ============================================================
# TABLES AFTER CHARTS
# ============================================================
show_table(
    "TABLE AFTER CHARTS: FINAL FORMULA TABLE BY YEAR, DEGREE FIELD, AND INDUSTRY SECTOR",
    final_formula_table[
        [
            "year",
            "field_group",
            "connected_industry_sector",
            "degree_value",
            "industry_job_success",
            "industry_job_failure",
            "startup_success",
            "startup_failure",
            "total_success",
            "total_failure",
            "net_result",
            "success_rate",
            "failure_rate",
            "source_used",
        ]
    ],
    max_rows=None
)

summary_table = final_formula_table.groupby(
    ["field_group", "connected_industry_sector"],
    as_index=False
).agg(
    total_success=("total_success", "sum"),
    total_failure=("total_failure", "sum"),
    net_result=("net_result", "sum")
)

denom = summary_table["total_success"] + summary_table["total_failure"]
summary_table["success_rate"] = np.where(
    denom > 0,
    summary_table["total_success"] / denom * 100,
    0.0
).round(2)
summary_table["failure_rate"] = np.where(
    denom > 0,
    summary_table["total_failure"] / denom * 100,
    0.0
).round(2)

summary_table = summary_table.sort_values(
    ["net_result", "success_rate"],
    ascending=[False, False]
).reset_index(drop=True)

show_table(
    "SMALLER SUMMARY TABLE RANKED BY BEST RESULT",
    summary_table,
    max_rows=None
)

banner("DONE")
print("This result answers:")
print("Which degree fields connect to which industries, and does the industry job path or startup path look better by success vs failure?")